# Dhātu: executable Grammar and Logic

All 48 kārikās, extracted from the Organon studies. The Markdown
studies remain authoritative; this notebook is an editable snapshot.
Run with a Jupyter Python 3 kernel containing **NLTK**,
from inside the Yoga repository. No NLTK corpus downloads are needed.
Choose **Run All**, or run setup and then any individual study.

Each study has editable source blocks followed by a checking cell.
`studies[48]` (for example) exposes `tree`, `sem`, `theories`,
`valuation`, `model`, `dependencies`, and `results`.
These are finite-model checks, not general theorem proofs.
The dependency grammar is loaded, but only its vocabulary is checked;
this is not a dependency parse.

Edit a valuation to explore countermodels. The checking cells enforce
the source `# EXPECT` values; for intentional counterexamples use
`satisfies(changed_valuation, formula)` as demonstrated at the end.
Re-running setup clears `studies`; re-run the desired study cells.
Regeneration replaces notebook edits, so save experiments separately.


In [1]:
from pathlib import Path
import re
import sys

from nltk.grammar import DependencyGrammar, FeatureGrammar
from nltk.parse import FeatureChartParser
from nltk.sem import Assignment, Expression, Model, Valuation

# Use the same loader and strict checks as the source studies.
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / "Vasubandhu" / "Organon")
    organon = next(
        (p for p in candidates if (p / "tests" / "audit_nltk_blocks.py").is_file()),
        None,
    )
    if organon is not None:
        break
else:
    raise FileNotFoundError("Run this notebook from inside the Yoga repository.")
sys.path.insert(0, str(organon / "tests"))
from audit_nltk_blocks import check_grammar, check_logic, load_valuation


def satisfies(valuation, formula):
    """Evaluate one formula in a valuation, including an edited copy."""
    if isinstance(formula, str):
        formula = Expression.fromstring(formula)
    model = Model(valuation.domain, valuation)
    return model.satisfy(formula, Assignment(model.domain))


def run_study(fcfg_text, dependency_text, sem_text, fol_text, valuation_text):
    """Check the edited blocks and expose the parse, theories, and model."""
    token_line = re.search(r"^# tokens: (.*)$", fcfg_text, re.M)
    if token_line is None:
        raise ValueError("The feature grammar needs a '# tokens:' line.")
    text = "## 3. Lexical Analysis\n" + token_line.group(0) + "\n"
    text += "## 4. Grammar\n"
    text += "```nltk-fcfg\n" + fcfg_text + "```\n"
    text += "```nltk-dep\n" + dependency_text + "```\n"
    text += "## 8. Logical Determination\n"
    text += "```nltk-sem\n" + sem_text + "```\n"
    text += "```nltk-fol\n" + fol_text + "```\n"
    text += "```nltk-valuation\n" + valuation_text + "```\n"
    grammar_summary, grammar_errors = check_grammar(text, strict=True)
    logic_summary, logic_errors = check_logic(text, strict=True)
    if grammar_errors or logic_errors:
        raise ValueError("\n".join(grammar_errors + logic_errors))

    tokens = token_line.group(1).split()
    grammar = FeatureGrammar.fromstring(fcfg_text)
    tree = next(FeatureChartParser(grammar).parse(tokens))
    theories = {}
    current = "default"
    for line in fol_text.splitlines():
        if line.startswith("# THEORY"):
            current = line.split()[2]
            theories.setdefault(current, [])
        elif line.strip() and not line.startswith("#"):
            theories.setdefault(current, []).append(Expression.fromstring(line))
    valuation = load_valuation(valuation_text)
    results = dict(item.split("=") for item in logic_summary.split())
    print(grammar_summary)
    print(logic_summary)
    return {
        "tokens": tokens,
        "grammar": grammar,
        "dependencies": DependencyGrammar.fromstring(dependency_text),
        "tree": tree,
        "sem": tree.label()["SEM"].simplify(),
        "theories": theories,
        "valuation": valuation,
        "model": Model(valuation.domain, valuation),
        "results": results,
    }


studies = {}


In [2]:
from nltk.sem.logic import (
    BinaryExpression, NegatedExpression,
    AllExpression, ExistsExpression, LambdaExpression,
)

def pretty_sem(expr, level=0):
    pad = "  " * level

    if isinstance(expr, BinaryExpression):
        return "\n".join([
            pad + "(",
            pretty_sem(expr.first, level + 1),
            pad + "  " + expr.getOp(),
            pretty_sem(expr.second, level + 1),
            pad + ")",
        ])

    for kind, symbol in (
        (AllExpression, "forall"),
        (ExistsExpression, "exists"),
        (LambdaExpression, "lambda"),
    ):
        if isinstance(expr, kind):
            return (
                f"{pad}{symbol} {expr.variable}.\n"
                + pretty_sem(expr.term, level + 1)
            )

    if isinstance(expr, NegatedExpression):
        return pad + "not\n" + pretty_sem(expr.term, level + 1)

    return pad + str(expr)

# VAK_1.01 — Liberation through Instruction According to Reality

Source: `VAK_1.01.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
yaḥ sarvathāsarvahatāndhakāraḥ saṃsārapaṅkājjagadujjahāra /
tasmai namaskṛtya yathārthaśāstre śāstraṃ pravakṣyāmy abhidharmakośam // 1.01 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| yaḥ | nominative masculine singular | he who |
| sarvathā | indeclinable | in every way |
| sarva-hata-andhakāraḥ | nominative masculine singular | whose darkness is destroyed concerning everything |
| saṃsāra-paṅkāt | ablative singular | from the mire of saṃsāra |
| jagat | accusative neuter singular | the world |
| ujjahāra | third-person singular perfect | lifted up, raised out |
| tasmai | dative masculine singular | to him |
| namaskṛtya | absolutive | having bowed |
| yathārtha-śāstre | dative singular of *yathārtha-śāstṛ* | to the teacher according to reality |
| śāstram | accusative neuter singular | treatise, teaching |
| pravakṣyāmi | first-person singular future | I shall proclaim |
| abhidharmakośam | accusative neuter singular | Treasury of Abhidharma |

**Grammar tokens.** Normalized IAST in verse order; sandhi is opened,
not the Sanskrit witness changed. Hyphenated compounds remain single
tokens corresponding to the table rows. No supplied token is needed.

```text
# tokens: yaḥ sarvathā sarva-hata-andhakāraḥ saṃsāra-paṅkāt jagat ujjahāra tasmai namaskṛtya yathārtha-śāstre śāstram pravakṣyāmi abhidharmakośam
```

## 5. Translation

### Literal Translation

> Having bowed to him whose darkness has been destroyed in every way and
> concerning everything, who raised the world from the mire of saṃsāra — to
> that teacher according to reality — I shall proclaim the treatise, the
> *Abhidharmakośa*.

### Bhāṣya-informed study translation

> Having bowed to the teacher who teaches according to reality, whose
> darkness has been destroyed in every way and with respect to all, and who
> raised the world from the mire of saṃsāra, I shall expound the
> *Abhidharmakośa* treatise.

## 7. Technical Vocabulary

| Sanskrit | Project rendering | Note |
|---|---|---|
| andhakāra | darkness | ignorance / obscuration |
| saṃsāra-paṅka | mire of saṃsāra | entangled cyclic appearance |
| jagat | world | beings/world-process caught in saṃsāra |
| yathārtha-śāstṛ | teacher according to reality | epithet of the Buddha in the dative singular `yathārthaśāstre` |
| śāstra | treatise / teaching | principled doctrinal exposition |
| abhidharmakośa | Treasury of Abhidharma | treasury of Dharma-analysis |

In [3]:
fcfg_text = r"""% start S
# tokens: yaḥ sarvathā sarva-hata-andhakāraḥ saṃsāra-paṅkāt jagat ujjahāra tasmai namaskṛtya yathārtha-śāstre śāstram pravakṣyāmi abhidharmakośam
S[SEM=<?r(?h)(?a)>] -> RELCL[NUM=?n,GEN=?g,SEM=?r] HOMAGE[NUM=?n,GEN=?g,SEM=?h] ANNOUNCE[SEM=?a]
RELCL[NUM=?n,GEN=?g,SEM=<?rel(?quality)(?rescue)>] -> REL[CASE=nom,NUM=?n,GEN=?g,SEM=?rel] QUALITY[CASE=nom,NUM=?n,GEN=?g,SEM=?quality] RESCUE[NUM=?n,SEM=?rescue]
QUALITY[CASE=?c,NUM=?n,GEN=?g,SEM=<?mode(?quality)>] -> ADV[SEM=?mode] ADJ[CASE=?c,NUM=?n,GEN=?g,SEM=?quality]
RESCUE[NUM=?n,SEM=<?verb(?world)(?source)>] -> SOURCE[CASE=abl,SEM=?source] N[CASE=acc,NUM=sg,GEN=n,SEM=?world] V[NUM=?n,PERS=third,TENSE=perfect,SEM=?verb]
HOMAGE[NUM=?n,GEN=?g,SEM=<?bow(?teacher)>] -> CORR[CASE=dat,NUM=?n,GEN=?g] ABS[PERS=first,SEM=?bow] TEACHER[CASE=dat,NUM=?n,GEN=?g,SEM=?teacher]
ANNOUNCE[SEM=<?verb(?work)>] -> WORK[CASE=acc,NUM=?n,GEN=?g] V[NUM=sg,PERS=first,TENSE=future,SEM=?verb] TITLE[CASE=acc,NUM=?n,GEN=?g,SEM=?work]
REL[CASE=nom,NUM=sg,GEN=m,SEM=<\P.\Q.\H.\A.exists x.(P(x) & Q(x) & H(x) & A)>] -> 'yaḥ'
ADV[SEM=<\P.\x.(P(x) & DarknessDestroyedEveryWay(x))>] -> 'sarvathā'
ADJ[CASE=nom,NUM=sg,GEN=m,SEM=<\x.DarknessDestroyedRegardingAll(x)>] -> 'sarva-hata-andhakāraḥ'
SOURCE[CASE=abl,SEM=<samsara_mire>] -> 'saṃsāra-paṅkāt'
N[CASE=acc,NUM=sg,GEN=n,SEM=<world>] -> 'jagat'
V[NUM=sg,PERS=third,TENSE=perfect,SEM=<\w.\s.\x.RaisedFrom(x,w,s)>] -> 'ujjahāra'
CORR[CASE=dat,NUM=sg,GEN=m] -> 'tasmai'
ABS[PERS=first,SEM=<\P.\x.(P(x) & BowedTo(speaker,x))>] -> 'namaskṛtya'
TEACHER[CASE=dat,NUM=sg,GEN=m,SEM=<\x.TeachesAccordingToReality(x)>] -> 'yathārtha-śāstre'
WORK[CASE=acc,NUM=sg,GEN=n] -> 'śāstram'
V[NUM=sg,PERS=first,TENSE=future,SEM=<\w.(Treatise(w) & AnnouncesFutureExposition(speaker,w))>] -> 'pravakṣyāmi'
TITLE[CASE=acc,NUM=sg,GEN=n,SEM=<kosa>] -> 'abhidharmakośam'
"""

dependency_text = r"""'pravakṣyāmi' -> 'ujjahāra' | 'namaskṛtya' | 'śāstram'
'ujjahāra' -> 'yaḥ' | 'sarva-hata-andhakāraḥ' | 'saṃsāra-paṅkāt' | 'jagat'
'sarva-hata-andhakāraḥ' -> 'sarvathā'
'namaskṛtya' -> 'tasmai'
'tasmai' -> 'yathārtha-śāstre'
'śāstram' -> 'abhidharmakośam'
"""

sem_text = r"""# derived by the section 4 parse
exists x.(DarknessDestroyedRegardingAll(x) & DarknessDestroyedEveryWay(x) & RaisedFrom(x,world,samsara_mire) & TeachesAccordingToReality(x) & BowedTo(speaker,x) & Treatise(kosa) & AnnouncesFutureExposition(speaker,kosa))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
exists x.(DarknessDestroyedRegardingAll(x) & DarknessDestroyedEveryWay(x) & RaisedFrom(x,world,samsara_mire) & TeachesAccordingToReality(x) & BowedTo(speaker,x) & Treatise(kosa) & AnnouncesFutureExposition(speaker,kosa))
"""

valuation_text = r"""speaker => author
kosa => work
world => beings
samsara_mire => mire
DarknessDestroyedRegardingAll => {teacher}
DarknessDestroyedEveryWay => {teacher}
RaisedFrom => {(teacher,beings,mire)}
TeachesAccordingToReality => {teacher}
BowedTo => {(author,teacher)}
Treatise => {work}
AnnouncesFutureExposition => {(author,work)}
"""


In [4]:
studies[1] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[1]["tree"].pformat())
print("Derived SEM:", pretty_sem(studies[1]["sem"]))


parses 1 SEM=ok
verse=T
(S[SEM=<exists x.(DarknessDestroyedRegardingAll(x) & DarknessDestroyedEveryWay(x) & RaisedFrom(x,world,samsara_mire) & TeachesAccordingToReality(x) & BowedTo(speaker,x) & Treatise(kosa) & AnnouncesFutureExposition(speaker,kosa))>]
  (RELCL[GEN='m', NUM='sg', SEM=<\H A.exists x.(DarknessDestroyedRegardingAll(x) & DarknessDestroyedEveryWay(x) & RaisedFrom(x,world,samsara_mire) & H(x) & A)>]
    (REL[CASE='nom', GEN='m', NUM='sg', SEM=<\P Q H A.exists x.(P(x) & Q(x) & H(x) & A)>]
      yaḥ)
    (QUALITY[CASE='nom', GEN='m', NUM='sg', SEM=<\x.(DarknessDestroyedRegardingAll(x) & DarknessDestroyedEveryWay(x))>]
      (ADV[SEM=<\P x.(P(x) & DarknessDestroyedEveryWay(x))>]
        sarvathā)
      (ADJ[CASE='nom', GEN='m', NUM='sg', SEM=<\x.DarknessDestroyedRegardingAll(x)>]
        sarva-hata-andhakāraḥ))
    (RESCUE[NUM='sg', SEM=<\x.RaisedFrom(x,world,samsara_mire)>]
      (SOURCE[CASE='abl', SEM=<samsara_mire>] saṃsāra-paṅkāt)
      (N[CASE='acc', GEN='n', NUM='sg', 

# VAK_1.02 — Abhidharma as Attained Knowing, Path, and Treasury

Source: `VAK_1.02.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
prajñāmalā sānucarābhidharmaḥ tatprāptaye yāpi ca yacca śāstram /
tasyārthato 'smin samanupraveśāt sa cāśrayo 'syety abhidharmakośam // 1.02 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| prajñā | noun; feminine; nominative singular | discriminative comprehension; subject of the first definition |
| amalā | adjective; feminine; nominative singular (lemma: amala) | stainless in Literal and Bhāṣya-informed |
| sānucarā | adjective; feminine; nominative singular (lemma: sa-anucara) | qualifies `prajñā`: with attendants |
| abhidharmaḥ | noun; masculine; nominative singular (lemma: abhidharma) | predicate nominative |
| tatprāptaye | compound noun; feminine; dative singular (lemma: tad-prāpti) | purpose: for attainment of that |
| yā | relative pronoun; feminine; nominative singular (lemma: yad) | supplies a second `prajñā` by ellipsis |
| api | indeclinable | inclusion: also |
| ca | indeclinable | coordination |
| yat | relative pronoun; neuter; nominative singular (lemma: yad) | qualifies `śāstram` |
| ca | indeclinable | coordination |
| śāstram | noun; neuter; nominative singular (lemma: śāstra) | treatise included in the definition |
| tasya | pronoun; masculine/neuter; genitive singular (lemma: tad) | of that Abhidharma |
| arthataḥ | indeclinable ablatival adverb (lemma: artha) | by meaning; in respect of meaning |
| asmin | pronoun; masculine/neuter; locative singular (lemma: idam) | in this treatise |
| samanupraveśāt | action noun; masculine; ablative singular (lemma: sam-anu-praveśa) | causal: because of comprehensive entry |
| saḥ | pronoun; masculine; nominative singular (lemma: tad) | that Abhidharma |
| ca | indeclinable | introduces the further derivation |
| āśrayaḥ | noun; masculine; nominative singular (lemma: āśraya) | predicate: supporting basis |
| asya | pronoun; masculine/neuter; genitive singular (lemma: idam) | of this treatise |
| iti | indeclinable | closes the title derivation |
| abhidharmakośam | compound noun; neuter; accusative singular (lemma: abhidharma-kośa) | the title predicated of the treatise |

Sandhi to retain: *sānucarā* = *sa-anucarā*; *tasyārthato 'smin* =
*tasya arthataḥ asmin*; *sa cāśrayo 'syety* =
*saḥ ca āśrayaḥ asya iti*.

**Grammar tokens.** Normalized IAST in verse order. `∅-prajñā` supplies
the noun qualified by *yā*; `∅-tatprāptaye` repeats the purpose for the
treatise; `∅-abhidharmaḥ` repeats the predicate for both conventional
members. These are Bhāṣya-resolved ellipses, not witness words.

```text
# tokens: prajñā amalā sānucarā abhidharmaḥ tatprāptaye yā ∅-prajñā api ca yat ca śāstram ∅-tatprāptaye ∅-abhidharmaḥ tasya arthataḥ asmin samanupraveśāt saḥ ca āśrayaḥ asya iti abhidharmakośam
```

## 5. Translation

### Literal Translation

> Stainless prajñā, with its attendants, is Abhidharma; so also is whatever
> [prajñā] is for the attainment of that, and whatever treatise [is for its
> attainment]. Because that enters into this by meaning—and because that is
> the supporting basis of this—[this treatise is called] the
> *Abhidharmakośa*.

### Bhāṣya-informed study translation

> Stainless discriminative comprehension with its attendants is Abhidharma.
> Also included are what serves to attain it and the treatise. The meaning
> of that Abhidharma enters this treatise, and Abhidharma is its source;
> hence this is called the *Abhidharmakośa*.

## 7. Technical Vocabulary

### Kārikā terms

| Sanskrit | Project rendering | Determination |
|---|---|---|
| prajñā | discriminative comprehension | Bhāṣya: `dharmapravicaya`, discrimination of dharmas. A science |
| amala | stainless | Literal and Bhāṣya-informed. Organon: Pure Prajñā. Not outflow-free |
| sānucara | with attendants | Prajñā together with its retinue; the attendants are not enumerated here |
| tatprāpti | attainment of that | Attainment of stainless prajñā with attendants |
| śāstra | treatise | Conventionally Abhidharma because it is equipment for that attainment |
| arthataḥ | by meaning | The mode in which Abhidharma is included in this work |
| samanupraveśa | comprehensive entry | Inclusion of the principal meaning of śāstric Abhidharma |
| āśraya | supporting basis | Abhidharma as the source from which the treatise is drawn |
| kośa | treasury | Either this work as container or Abhidharma as source-treasury |

### Bhāṣya determinations

| Sanskrit | Project rendering | Function here |
|---|---|---|
| pāramārthika | in the ultimate sense | Classifies stainless prajñā with attendants |
| sāṃketika | conventional / designated | Classifies path-prajñā and the treatise |
| anāsrava | Pure | Organon name of the Bhāṣya gloss of `amala`. Always First. Not the verse English |
| sāsrava | with outflows | Conventional. Organon: Impure. Presupposes a before |
| pañcaskandhaka | consisting of five Bases | Concrete complex of ultimate Abhidharma |
| saṃbhāra | equipment | Why the treatise is conventionally Abhidharma |
| dharmapravicaya | discrimination of dharmas | Operational definition of prajñā |
| dharma | dharma | `svalakṣaṇa-dhāraṇāt`: because it bears its own mark |
| dhāraṇa | bearing | The holding. Root `dhṛ`. Not dropped from the definition |
| svalakṣaṇa | own mark | What a dharma bears |
| pratyabhimukha | facing toward | The prefix `abhi-`. Two facings, both kept |
| paramārtha-dharma | ultimate dharma | Identified here with nirvāṇa. First facing |
| dharma-lakṣaṇa | mark of a dharma | Second facing. The Sāṃkhya hinge. Not dropped |

In [5]:
fcfg_text = r"""% start S
# tokens: prajñā amalā sānucarā abhidharmaḥ tatprāptaye yā ∅-prajñā api ca yat ca śāstram ∅-tatprāptaye ∅-abhidharmaḥ tasya arthataḥ asmin samanupraveśāt saḥ ca āśrayaḥ asya iti abhidharmakośam
S[SEM=<(?u & ?c & ?t)>] -> ULTIMATE[SEM=?u] CONVENTIONAL[SEM=?c] TITLECLAUSE[SEM=?t]
ULTIMATE[SEM=<all x.(?retinue(?pure(?prajna))(x) -> ?pred(x))>] -> N[K=prajna,CASE=nom,NUM=?n,GEN=?g,SEM=?prajna] ADJ[K=stainless,CASE=nom,NUM=?n,GEN=?g,SEM=?pure] ADJ[K=retinue,CASE=nom,NUM=?n,GEN=?g,SEM=?retinue] PRED[CASE=nom,NUM=?n,SEM=?pred]
CONVENTIONAL[SEM=<(?q(?p)(?purpose)(?pred) & ?qt(?t)(?purpose2)(?pred))>] -> PURPOSE[CASE=dat,SEM=?purpose] REL[CASE=nom,NUM=?n,GEN=?g,SEM=?q] COVERT[K=prajna,CASE=nom,NUM=?n,GEN=?g,SUPPLIED=prose,SEM=?p] PTCL[K=also] PTCL[K=and] REL[CASE=nom,NUM=?nt,GEN=?gt,SEM=?qt] PTCL[K=and] N[K=treatise,CASE=nom,NUM=?nt,GEN=?gt,SEM=?t] COVERT[K=purpose,CASE=dat,SUPPLIED=prose,SEM=?purpose2] COVERT[K=predicate,CASE=nom,SUPPLIED=prose,SEM=?pred]
TITLECLAUSE[SEM=<(?entry & ?support & ?name)>] -> ENTRY[SEM=?entry] SUPPORT[SEM=?support] PTCL[K=iti] NAME[CASE=acc,SEM=?name]
ENTRY[SEM=<?entry(?mode)(?source)(?work)>] -> PRON[K=that,CASE=gen,SEM=?source] ADV[SEM=?mode] PRON[K=this,CASE=loc,SEM=?work] ENTRYHEAD[CASE=abl,SEM=?entry]
SUPPORT[SEM=<?support(?source)(?work)>] -> PRON[K=that,CASE=nom,NUM=?n,GEN=?g,SEM=?source] PTCL[K=and] SUPPORTHEAD[CASE=nom,NUM=?n,GEN=?g,SEM=?support] PRON[K=this,CASE=gen,SEM=?work]
N[K=prajna,CASE=nom,NUM=sg,GEN=f,SEM=<\x.Prajna(x)>] -> 'prajñā'
ADJ[K=stainless,CASE=nom,NUM=sg,GEN=f,SEM=<\P.\x.(P(x) & Stainless(x))>] -> 'amalā'
ADJ[K=retinue,CASE=nom,NUM=sg,GEN=f,SEM=<\P.\x.(P(x) & WithRetinue(x))>] -> 'sānucarā'
PRED[CASE=nom,NUM=sg,GEN=m,SEM=<\x.Abhidharma(x)>] -> 'abhidharmaḥ'
PURPOSE[CASE=dat,SEM=<\x.ForAttainmentOf(x,stainless_prajna)>] -> 'tatprāptaye'
REL[CASE=nom,NUM=sg,GEN=f,SEM=<\P.\Q.\R.all x.((P(x) & Q(x)) -> R(x))>] -> 'yā'
COVERT[K=prajna,CASE=nom,NUM=sg,GEN=f,SUPPLIED=prose,SEM=<\x.Prajna(x)>] -> '∅-prajñā'
PTCL[K=also] -> 'api'
PTCL[K=and] -> 'ca'
REL[CASE=nom,NUM=sg,GEN=n,SEM=<\P.\Q.\R.all x.((P(x) & Q(x)) -> R(x))>] -> 'yat'
N[K=treatise,CASE=nom,NUM=sg,GEN=n,SEM=<\x.Treatise(x)>] -> 'śāstram'
COVERT[K=purpose,CASE=dat,SUPPLIED=prose,SEM=<\x.ForAttainmentOf(x,stainless_prajna)>] -> '∅-tatprāptaye'
COVERT[K=predicate,CASE=nom,SUPPLIED=prose,SEM=<\x.Abhidharma(x)>] -> '∅-abhidharmaḥ'
PRON[K=that,CASE=gen,SEM=<abhidharma_source>] -> 'tasya'
ADV[SEM=<principal_meaning>] -> 'arthataḥ'
PRON[K=this,CASE=loc,SEM=<kosa>] -> 'asmin'
ENTRYHEAD[CASE=abl,SEM=<\m.\s.\w.ContainsByMeaning(w,s,m)>] -> 'samanupraveśāt'
PRON[K=that,CASE=nom,NUM=sg,GEN=m,SEM=<abhidharma_source>] -> 'saḥ'
SUPPORTHEAD[CASE=nom,NUM=sg,GEN=m,SEM=<\s.\w.SourceFor(s,w)>] -> 'āśrayaḥ'
PRON[K=this,CASE=gen,SEM=<kosa>] -> 'asya'
PTCL[K=iti] -> 'iti'
NAME[CASE=acc,SEM=<Named(kosa,kosa_title)>] -> 'abhidharmakośam'
"""

dependency_text = r"""'abhidharmaḥ' -> 'prajñā' | '∅-abhidharmaḥ' | 'abhidharmakośam'
'prajñā' -> 'amalā' | 'sānucarā'
'∅-abhidharmaḥ' -> '∅-prajñā' | 'śāstram'
'∅-prajñā' -> 'tatprāptaye' | 'yā' | 'api' | 'ca'
'śāstram' -> 'yat' | 'ca' | '∅-tatprāptaye'
'abhidharmakośam' -> 'samanupraveśāt' | 'āśrayaḥ' | 'iti'
'samanupraveśāt' -> 'tasya' | 'arthataḥ' | 'asmin'
'āśrayaḥ' -> 'saḥ' | 'ca' | 'asya'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(((Prajna(x) & Stainless(x)) & WithRetinue(x)) -> Abhidharma(x))) & ((all x.((Prajna(x) & ForAttainmentOf(x,stainless_prajna)) -> Abhidharma(x))) & (all x.((Treatise(x) & ForAttainmentOf(x,stainless_prajna)) -> Abhidharma(x)))) & (ContainsByMeaning(kosa,abhidharma_source,principal_meaning) & SourceFor(abhidharma_source,kosa) & Named(kosa,kosa_title)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(((Prajna(x) & Stainless(x)) & WithRetinue(x)) -> Abhidharma(x))) & ((all x.((Prajna(x) & ForAttainmentOf(x,stainless_prajna)) -> Abhidharma(x))) & (all x.((Treatise(x) & ForAttainmentOf(x,stainless_prajna)) -> Abhidharma(x)))) & (ContainsByMeaning(kosa,abhidharma_source,principal_meaning) & SourceFor(abhidharma_source,kosa) & Named(kosa,kosa_title)))
"""

valuation_text = r"""stainless_prajna => pure
abhidharma_source => source
principal_meaning => meaning
kosa => work
kosa_title => title
Prajna => {pure,path}
Stainless => {pure}
WithRetinue => {pure,path}
Treatise => {work}
ForAttainmentOf => {(path,pure),(work,pure)}
Abhidharma => {pure,path,work}
ContainsByMeaning => {(work,source,meaning)}
SourceFor => {(source,work)}
Named => {(work,title)}
"""


In [6]:
studies[2] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[2]["tree"].pformat())
print("Derived SEM:", studies[2]["sem"])


parses 1 SEM=ok
verse=T
(S[SEM=<(all x.((Prajna(x) & Stainless(x) & WithRetinue(x)) -> Abhidharma(x)) & all x.((Prajna(x) & ForAttainmentOf(x,stainless_prajna)) -> Abhidharma(x)) & all x.((Treatise(x) & ForAttainmentOf(x,stainless_prajna)) -> Abhidharma(x)) & ContainsByMeaning(kosa,abhidharma_source,principal_meaning) & SourceFor(abhidharma_source,kosa) & Named(kosa,kosa_title))>]
  (ULTIMATE[SEM=<all x.((Prajna(x) & Stainless(x) & WithRetinue(x)) -> Abhidharma(x))>]
    (N[CASE='nom', GEN='f', K='prajna', NUM='sg', SEM=<\x.Prajna(x)>]
      prajñā)
    (ADJ[CASE='nom', GEN='f', K='stainless', NUM='sg', SEM=<\P x.(P(x) & Stainless(x))>]
      amalā)
    (ADJ[CASE='nom', GEN='f', K='retinue', NUM='sg', SEM=<\P x.(P(x) & WithRetinue(x))>]
      sānucarā)
    (PRED[CASE='nom', GEN='m', NUM='sg', SEM=<\x.Abhidharma(x)>]
      abhidharmaḥ))
  (CONVENTIONAL[SEM=<(all x.((Prajna(x) & ForAttainmentOf(x,stainless_prajna)) -> Abhidharma(x)) & all x.((Treatise(x) & ForAttainmentOf(x,stainless_pra

# VAK_1.03 — Why Abhidharma Instruction Is Necessary

Source: `VAK_1.03.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
dharmāṇāṃ pravicayam antareṇa nāsti kleśānāṃ yata upaśāntaye 'bhyupāyaḥ /
kleśaiś ca bhramati bhavārṇave 'tra lokas taddhetor ata uditaḥ kilaiṣa śāstraḥ // 1.03 //
```

The running Sanskrit witness reads `śāstrā` at the end of this verse.
`Śāstraḥ` above is an editorial conjecture, not a reading established by that
witness. It fits `uditaḥ` and `eṣaḥ` formally, but `śāstra` is ordinarily
neuter; the exact ending remains unsettled. The Devanāgarī line reflects the
same conjecture. The translation conveys the contextual sense without
claiming that the transmitted ending has been resolved.

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| dharmāṇām | genitive plural | dharmas to be discriminated |
| pravicayam | accusative singular | governed by `antareṇa` |
| antareṇa | indeclinable postposition | without |
| kleśānām | genitive plural | afflictions to be pacified |
| upaśāntaye | dative singular | purpose: for pacification |
| abhyupāyaḥ | nominative singular | adequate means |
| kleśaiḥ | instrumental plural | cause of wandering |
| bhramati | 3rd singular present | wanders |
| bhavārṇave | locative singular | in the ocean of becoming |
| lokaḥ | nominative singular | subject of `bhramati` |
| tad-dhetoḥ / ataḥ | ablatival expressions | for that reason / therefore |
| uditaḥ | past participle, nominative singular | taught, declared |
| kila | particle | reportedly; according to the received account |
| eṣaḥ śāstraḥ | nominative singular in the conjectural reading | this treatise; transmitted `śāstrā` remains unsettled |

**Grammar tokens.** Normalized IAST in verse order, including *na asti*,
*yataḥ*, and *atra* present in the Sanskrit. The compressed lexical
row *eṣaḥ śāstraḥ* becomes two tokens. `śāstraḥ` retains the displayed
conjecture, explicitly marked `READING=conjecture`; the grammar does not
establish it as the transmitted reading. No covert token is needed.

```text
# tokens: dharmāṇām pravicayam antareṇa na asti kleśānām yataḥ upaśāntaye abhyupāyaḥ kleśaiḥ ca bhramati bhavārṇave atra lokaḥ tad-dhetoḥ ataḥ uditaḥ kila eṣaḥ śāstraḥ
```

## 5. Translation

### Literal Translation

> Since without discrimination of dharmas there is no means for the
> pacification of the afflictions, and since through the afflictions the
> world wanders here in the ocean of becoming, for that reason this treatise
> is said to have been taught.

### Bhāṣya-informed study translation

> Without discrimination of dharmas, there is no means to pacify the
> afflictions; through the afflictions, the world wanders in this ocean of
> becoming. For this reason, this treatise is said to have been taught.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Determination |
|---|---|---|
| dharmapravicaya | discrimination of dharmas | indispensable cognitive operation |
| kleśopaśānti | pacification of affliction | practical end of discrimination |
| abhyupāya | adequate means | here denied apart from pravicaya |
| bhavārṇava | ocean of becoming | glossed as the great ocean of saṃsāra |
| abhidharmopadeśa | Abhidharma instruction | Bhāṣya: pedagogical mediation |
| kila | reportedly | preserves distance from received doctrine |
| prakīrṇa | dispersed | Bhāṣya: mode of Buddha's teaching in the Vaibhāṣika account |

In [7]:
fcfg_text = r"""% start S
# tokens: dharmāṇām pravicayam antareṇa na asti kleśānām yataḥ upaśāntaye abhyupāyaḥ kleśaiḥ ca bhramati bhavārṇave atra lokaḥ tad-dhetoḥ ataḥ uditaḥ kila eṣaḥ śāstraḥ
S[SEM=<?reason(?need)(?wandering)(?report)>] -> NECESSITY[SEM=?need] WANDERING[SEM=?wandering] REASON[SEM=?reason] REPORT[SEM=?report]
NECESSITY[SEM=<?without(?discriminate)(?neg(?means(?afflictions)(?purpose)))>] -> DISCRIMINATION[CASE=acc,SEM=?discriminate] WITHOUT[GOVERNS=acc,SEM=?without] NEG[SEM=?neg] COPULA[NUM=?n] AFFLICTIONS[CASE=gen,SEM=?afflictions] PTCL[K=because] PURPOSE[CASE=dat,SEM=?purpose] MEANS[CASE=nom,NUM=?n,GEN=m,SEM=?means]
DISCRIMINATION[CASE=?c,SEM=<?verb(?dharmas)>] -> DHARMAS[CASE=gen,SEM=?dharmas] DISCRIM[CASE=?c,SEM=?verb]
WANDERING[SEM=<?verb(?cause)(?place)(?world)>] -> CAUSE[CASE=ins,SEM=?cause] PTCL[K=and] WANDER[NUM=?n,PERS=third,TENSE=present,SEM=?verb] PLACE[CASE=loc,SEM=?place] PTCL[K=here] WORLD[CASE=nom,NUM=?n,SEM=?world]
REASON[SEM=?reason] -> REASONHEAD[SEM=?reason] PTCL[K=therefore]
REPORT[SEM=<?report(?taught(?work))>] -> TAUGHT[CASE=nom,NUM=?n,GEN=?g,SEM=?taught] REPORTER[SEM=?report] DEM[CASE=nom,NUM=?n,GEN=?g] WORK[CASE=nom,NUM=?n,GEN=?g,READING=conjecture,SEM=?work]
DHARMAS[CASE=gen,SEM=<dharmas>] -> 'dharmāṇām'
DISCRIM[CASE=acc,SEM=<\d.\x.Discriminates(x,d)>] -> 'pravicayam'
WITHOUT[GOVERNS=acc,SEM=<\D.\M.all x.(-D(x) -> M(x))>] -> 'antareṇa'
NEG[SEM=<\P.\x.-P(x)>] -> 'na'
COPULA[NUM=sg,TENSE=present] -> 'asti'
AFFLICTIONS[CASE=gen,SEM=<afflictions>] -> 'kleśānām'
PTCL[K=because] -> 'yataḥ'
PURPOSE[CASE=dat,SEM=<pacification>] -> 'upaśāntaye'
MEANS[CASE=nom,NUM=sg,GEN=m,SEM=<\a.\p.\x.HasMeansFor(x,p,a)>] -> 'abhyupāyaḥ'
CAUSE[CASE=ins,SEM=<afflictions>] -> 'kleśaiḥ'
PTCL[K=and] -> 'ca'
WANDER[NUM=sg,PERS=third,TENSE=present,SEM=<\c.\p.\w.WandersThrough(w,c,p)>] -> 'bhramati'
PLACE[CASE=loc,SEM=<becoming_ocean>] -> 'bhavārṇave'
PTCL[K=here] -> 'atra'
WORLD[CASE=nom,NUM=sg,SEM=<world>] -> 'lokaḥ'
REASONHEAD[SEM=<\P.\Q.\R.(P & Q & R)>] -> 'tad-dhetoḥ'
PTCL[K=therefore] -> 'ataḥ'
TAUGHT[CASE=nom,NUM=sg,GEN=m,VOICE=passive,SEM=<\w.\r.ReportsTaught(r,w)>] -> 'uditaḥ'
REPORTER[SEM=<\P.(ReceivedReport(teaching_report) & P(teaching_report))>] -> 'kila'
DEM[CASE=nom,NUM=sg,GEN=m] -> 'eṣaḥ'
WORK[CASE=nom,NUM=sg,GEN=m,READING=conjecture,SEM=<treatise>] -> 'śāstraḥ'
"""

dependency_text = r"""'uditaḥ' -> 'asti' | 'bhramati' | 'tad-dhetoḥ' | 'ataḥ' | 'kila' | 'śāstraḥ'
'asti' -> 'pravicayam' | 'na' | 'abhyupāyaḥ' | 'yataḥ'
'pravicayam' -> 'dharmāṇām' | 'antareṇa'
'abhyupāyaḥ' -> 'upaśāntaye'
'upaśāntaye' -> 'kleśānām'
'bhramati' -> 'kleśaiḥ' | 'ca' | 'bhavārṇave' | 'atra' | 'lokaḥ'
'śāstraḥ' -> 'eṣaḥ'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(-Discriminates(x,dharmas) -> -HasMeansFor(x,pacification,afflictions))) & WandersThrough(world,afflictions,becoming_ocean) & (ReceivedReport(teaching_report) & ReportsTaught(teaching_report,treatise)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(-Discriminates(x,dharmas) -> -HasMeansFor(x,pacification,afflictions))) & WandersThrough(world,afflictions,becoming_ocean) & (ReceivedReport(teaching_report) & ReportsTaught(teaching_report,treatise)))
"""

valuation_text = r"""dharmas => dharmas
pacification => pacification
afflictions => afflictions
world => beings
becoming_ocean => ocean
teaching_report => report
treatise => work
Discriminates => {(disciple,dharmas),(learner,dharmas)}
HasMeansFor => {(disciple,pacification,afflictions)}
WandersThrough => {(beings,afflictions,ocean)}
ReceivedReport => {report}
ReportsTaught => {(report,work)}
"""


In [8]:
studies[3] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[3]["tree"].pformat())
print("Derived SEM:", studies[3]["sem"])


parses 1 SEM=ok
verse=T
(S[SEM=<(all x.(-Discriminates(x,dharmas) -> -HasMeansFor(x,pacification,afflictions)) & WandersThrough(world,afflictions,becoming_ocean) & ReceivedReport(teaching_report) & ReportsTaught(teaching_report,treatise))>]
  (NECESSITY[SEM=<all x.(-Discriminates(x,dharmas) -> -HasMeansFor(x,pacification,afflictions))>]
    (DISCRIMINATION[CASE='acc', SEM=<\x.Discriminates(x,dharmas)>]
      (DHARMAS[CASE='gen', SEM=<dharmas>] dharmāṇām)
      (DISCRIM[CASE='acc', SEM=<\d x.Discriminates(x,d)>] pravicayam))
    (WITHOUT[GOVERNS='acc', SEM=<\D M.all x.(-D(x) -> M(x))>]
      antareṇa)
    (NEG[SEM=<\P x.-P(x)>] na)
    (COPULA[NUM='sg', TENSE='present'] asti)
    (AFFLICTIONS[CASE='gen', SEM=<afflictions>] kleśānām)
    (PTCL[K='because'] yataḥ)
    (PURPOSE[CASE='dat', SEM=<pacification>] upaśāntaye)
    (MEANS[CASE='nom', GEN='m', NUM='sg', SEM=<\a p x.HasMeansFor(x,p,a)>]
      abhyupāyaḥ))
  (WANDERING[SEM=<WandersThrough(world,afflictions,becoming_ocean)>]
    (CAU

# VAK_1.04 — Impure and Pure

Source: `VAK_1.04.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
sāsravānāsravā dharmāḥ saṃskṛtā mārgavarjitāḥ /
sāsravāḥ āsravās teṣu yasmāt samanuśerate // 1.04 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| sāsravānāsravāḥ | sāsravāḥ + anāsravāḥ; nominative plural | impure and pure; the first division of the dharmas |
| dharmāḥ | nominative masculine plural | the dharmas so divided |
| saṃskṛtāḥ | participial adjective; nominative plural | conditioned |
| mārgavarjitāḥ | mārga-varjitāḥ; nominative plural | excluding the Path |
| sāsravāḥ | nominative plural | impure; the predicate resumed |
| āsravāḥ | nominative masculine plural | the *āsravas*; subject of the persistence clause |
| teṣu | locative masculine plural pronoun | in those dharmas; refers to the conditioned other than the Path |
| yasmāt | causal indeclinable | because |
| samanuśerate | 3rd plural middle, present | persist |

**Grammar tokens.** Verse-order IAST with sandhi opened. The coordinated
first table row becomes two leaves; the repeated *sāsravāḥ* is retained.
No covert word is inserted.

```text
# tokens: sāsravāḥ anāsravāḥ dharmāḥ saṃskṛtāḥ mārgavarjitāḥ sāsravāḥ āsravāḥ teṣu yasmāt samanuśerate
```

## 5. Translation

### Literal Translation

> Dharmas are impure and pure. Conditioned, excluding the Path, are impure,
> because the *āsravas* persist in them.

### Bhāṣya-informed study translation

> Dharmas are impure and pure. Conditioned dharmas other than the Truth of
> the Path are impure, because the *āsravas* persist in them.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| dharma | dharma | not a second word for the division |
| sāsrava | impure | conditioned, excluding the Path |
| anāsrava | pure | the Path, and the unconditioned of the next verse |
| saṃskṛta | conditioned | produced through conditions |
| mārga | Path | the commentary: the Truth of the Path |
| āsrava | *āsrava* | subject of the persistence clause; distinct from *sāsrava*, the classification |
| samanuśerate | persist | technical persistence of the *āsravas*, not mere temporal duration |
| teṣu | in those dharmas | locative relation, not a separate enduring bearer |
| ālambana | *ālambana* | what an arising takes; that relation does not entail persistence |

In [9]:
fcfg_text = r"""% start S
# tokens: sāsravāḥ anāsravāḥ dharmāḥ saṃskṛtāḥ mārgavarjitāḥ sāsravāḥ āsravāḥ teṣu yasmāt samanuśerate
S[SEM=<(?division & ?classification & ?reason)>] -> CL[K=divide,SEM=?division] CL[K=classify,SEM=?classification] CL[K=reason,SEM=?reason]
CL[K=divide,SEM=<all x.(?d(x) -> ((?i(x) | ?p(x)) & -(?i(x) & ?p(x))))>] -> ADJ[CASE=nom,NUM=?n,GEN=?g,STATUS=impure,SEM=?i] ADJ[CASE=nom,NUM=?n,GEN=?g,STATUS=pure,SEM=?p] N[CASE=nom,NUM=?n,GEN=?g,K=dharma,SEM=?d]
CL[K=classify,SEM=<all x.(Dharma(x) -> (?i(x) <-> (?c(x) & ?e(x))))>] -> ADJ[CASE=nom,NUM=?n,GEN=?g,K=conditioned,SEM=?c] ADJ[CASE=nom,NUM=?n,GEN=?g,K=exclusion,SEM=?e] ADJ[CASE=nom,NUM=?n,GEN=?g,STATUS=impure,SEM=?i]
CL[K=reason,SEM=<?l(\x.?q(?v(x)))>] -> N[CASE=nom,NUM=?n,GEN=m,K=asrava,SEM=?q] PRON[CASE=loc,NUM=pl,GEN=m,REF=classified,SEM=?l] CAUS[LINK=reason] V[NUM=?n,PERSON=third,VOICE=middle,LOC=required,SEM=?v]
ADJ[CASE=nom,NUM=pl,GEN=m,STATUS=impure,SEM=<\x.Impure(x)>] -> 'sāsravāḥ'
ADJ[CASE=nom,NUM=pl,GEN=m,STATUS=pure,SEM=<\x.Pure(x)>] -> 'anāsravāḥ'
N[CASE=nom,NUM=pl,GEN=m,K=dharma,SEM=<\x.Dharma(x)>] -> 'dharmāḥ'
ADJ[CASE=nom,NUM=pl,GEN=m,K=conditioned,SEM=<\x.Conditioned(x)>] -> 'saṃskṛtāḥ'
ADJ[CASE=nom,NUM=pl,GEN=m,K=exclusion,SEM=<\x.-PathTruth(x)>] -> 'mārgavarjitāḥ'
N[CASE=nom,NUM=pl,GEN=m,K=asrava,SEM=<\P.exists a.(Asrava(a) & P(a))>] -> 'āsravāḥ'
PRON[CASE=loc,NUM=pl,GEN=m,REF=classified,SEM=<\P.all x.((Dharma(x) & Conditioned(x) & -PathTruth(x)) -> P(x))>] -> 'teṣu'
CAUS[LINK=reason] -> 'yasmāt'
V[NUM=pl,PERSON=third,VOICE=middle,LOC=required,SEM=<\x.\a.PersistsIn(a,x)>] -> 'samanuśerate'
"""

dependency_text = r"""'dharmāḥ' -> 'sāsravāḥ' | 'anāsravāḥ'
'sāsravāḥ' -> 'saṃskṛtāḥ' | 'mārgavarjitāḥ'
'samanuśerate' -> 'āsravāḥ' | 'teṣu' | 'yasmāt'
"""

sem_text = r"""impure : \x.Impure(x)
conditioned : \x.Conditioned(x)
excluded : \x.-PathTruth(x)
asravas : \P.exists a.(Asrava(a) & P(a))
locus : \P.all x.((Dharma(x) & Conditioned(x) & -PathTruth(x)) -> P(x))
persist : \x.\a.PersistsIn(a,x)
reason : (\P.all x.((Dharma(x) & Conditioned(x) & -PathTruth(x)) -> P(x)))(\x.(\P.exists a.(Asrava(a) & P(a)))((\x.\a.PersistsIn(a,x))(x)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
# [verse] division, class specification, and reason-clause content
(all x.(Dharma(x) -> ((Impure(x) | Pure(x)) & -(Impure(x) & Pure(x)))) & all x.(Dharma(x) -> (Impure(x) <-> (Conditioned(x) & -PathTruth(x)))) & all x.((Dharma(x) & Conditioned(x) & -PathTruth(x)) -> exists a.(Asrava(a) & PersistsIn(a,x))))

# THEORY prose
# EXPECT prose T
# [prose] the Path is conditioned and pure; no asrava persists in either Truth
all x.(PathTruth(x) -> (Dharma(x) & Conditioned(x) & Pure(x)))
all x.(CessationTruth(x) -> Dharma(x))
all a x.((Asrava(a) & (PathTruth(x) | CessationTruth(x))) -> -PersistsIn(a,x))
# [prose] taking either Truth as alambana is nevertheless possible
all x.((PathTruth(x) | CessationTruth(x)) -> exists a.(Asrava(a) & CanTakeAsAlambana(a,x)))

# THEORY counterexamples
# EXPECT counterexamples T
# [prose] possible taking does not require persistence in what is taken
exists a x.(Asrava(a) & CanTakeAsAlambana(a,x) & -PersistsIn(a,x))
# [prose] conditioned and pure is an admitted case
exists x.(Dharma(x) & Conditioned(x) & Pure(x))

# THEORY rejected_taking_entails_persistence
# EXPECT rejected_taking_entails_persistence F
all a x.((Asrava(a) & CanTakeAsAlambana(a,x)) -> PersistsIn(a,x))

# THEORY rejected_conditioned_entails_impure
# EXPECT rejected_conditioned_entails_impure F
all x.((Dharma(x) & Conditioned(x)) -> Impure(x))

# THEORY rejected_pure_entails_unconditioned
# EXPECT rejected_pure_entails_unconditioned F
all x.((Dharma(x) & Pure(x)) -> -Conditioned(x))
"""

valuation_text = r"""ordinary => ordinary
path => path
cessation => cessation
influx => influx
Dharma => {ordinary, path, cessation}
Impure => {ordinary}
Pure => {path, cessation}
Conditioned => {ordinary, path}
PathTruth => {path}
CessationTruth => {cessation}
Asrava => {influx}
PersistsIn => {(influx,ordinary)}
CanTakeAsAlambana => {(influx,path), (influx,cessation)}
"""


In [10]:
studies[4] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[4]["tree"].pformat())
print("Derived SEM:", studies[4]["sem"])


parses 1 SEM=ok
verse=T prose=T counterexamples=T rejected_taking_entails_persistence=F rejected_conditioned_entails_impure=F rejected_pure_entails_unconditioned=F
(S[SEM=<(all x.(Dharma(x) -> ((Impure(x) | Pure(x)) & -(Impure(x) & Pure(x)))) & all x.(Dharma(x) -> (Impure(x) <-> (Conditioned(x) & -PathTruth(x)))) & all x.((Dharma(x) & Conditioned(x) & -PathTruth(x)) -> exists a.(Asrava(a) & PersistsIn(a,x))))>]
  (CL[K='divide', SEM=<all x.(Dharma(x) -> ((Impure(x) | Pure(x)) & -(Impure(x) & Pure(x))))>]
    (ADJ[CASE='nom', GEN='m', NUM='pl', SEM=<\x.Impure(x)>, STATUS='impure']
      sāsravāḥ)
    (ADJ[CASE='nom', GEN='m', NUM='pl', SEM=<\x.Pure(x)>, STATUS='pure']
      anāsravāḥ)
    (N[CASE='nom', GEN='m', K='dharma', NUM='pl', SEM=<\x.Dharma(x)>]
      dharmāḥ))
  (CL[K='classify', SEM=<all x.(Dharma(x) -> (Impure(x) <-> (Conditioned(x) & -PathTruth(x))))>]
    (ADJ[CASE='nom', GEN='m', K='conditioned', NUM='pl', SEM=<\x.Conditioned(x)>]
      saṃskṛtāḥ)
    (ADJ[CASE='nom', GEN=

# VAK_1.05 — The Pure, and Space

Source: `VAK_1.05.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
anāsravā mārgasatyaṃ trividhaṃ cāpy asaṃskṛtam /
ākāśaṃ dvau nirodhau ca tatrākāśam anāvṛtiḥ // 1.05 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| anāsravāḥ | adjective; nominative plural | pure; the subject supplied from 1.04 |
| mārgasatyam | mārga-satyam; nominative neuter singular | the Truth of the Path |
| trividham | nominative neuter singular | threefold |
| ca api | particles | and also |
| asaṃskṛtam | nominative neuter singular | the unconditioned |
| ākāśam | nominative neuter singular | space |
| dvau nirodhau | masculine dual nominative | the two cessations |
| tatra | indeclinable | among these |
| anāvṛtiḥ | an-āvṛtiḥ; nominative feminine singular | non-obstruction; the predicate of space |

**Grammar tokens.** Verse-order IAST with sandhi opened; *ca api* and
*dvau nirodhau* become separate leaves. Space occurs twice. The Pure
Dharmas are recovered from 1.04 semantically, without a covert noun leaf.
The Bhāṣya names the two cessations; the verse supplies their dual count.

```text
# tokens: anāsravāḥ mārgasatyam trividham ca api asaṃskṛtam ākāśam dvau nirodhau ca tatra ākāśam anāvṛtiḥ
```

## 5. Translation

### Literal Translation

> The pure are the Truth of the Path and also the threefold unconditioned:
> space and the two cessations. Among these, space is non-obstruction.

### Bhāṣya-informed study translation

> The Path Truth is Pure, and the unconditioned is also threefold: space
> and the two cessations, cessation through discrimination and cessation
> not through discrimination. Among these, space is non-obstruction, where
> Form can move.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| anāsrava | pure | the *āsravas* do not persist |
| mārga-satya | Truth of the Path | conditioned and pure |
| asaṃskṛta | unconditioned | space and the two cessations |
| ākāśa | space | non-obstruction |
| anāvṛti | non-obstruction | where Form can move |
| pratisaṃkhyā-nirodha | cessation through discrimination | named here; defined in 1.06 |
| apratisaṃkhyā-nirodha | cessation not through discrimination | named here; defined in 1.06 |
| anuśerate | persist | the verb that fails in the pure |

In [11]:
fcfg_text = r"""% start S
# tokens: anāsravāḥ mārgasatyam trividham ca api asaṃskṛtam ākāśam dvau nirodhau ca tatra ākāśam anāvṛtiḥ
S[SEM=<(?p(?path)(?u) & ?roster & ?space)>] -> PURE[SEM=?p] PATH[SEM=?path] UNCONDITIONED[SEM=?u] ROSTER[SEM=?roster] SPACEDEF[SEM=?space]
UNCONDITIONED[SEM=<?three(?u)>] -> THREE[CASE=nom,NUM=?n,GEN=?g,SEM=?three] AND ALSO UNCOND[CASE=nom,NUM=?n,GEN=?g,SEM=?u]
ROSTER[SEM=<all x.(Unconditioned(x) <-> (x=?space | ?cessations(x)))>] -> SPACE[CASE=nom,SEM=?space] TWO[CASE=nom,NUM=?n,GEN=?g] CESSATIONS[CASE=nom,NUM=?n,GEN=?g,SEM=?cessations] AND
SPACEDEF[SEM=<?pred(?space)>] -> AMONG SPACE[CASE=nom,NUM=?n,SEM=?space] NONOBSTRUCTION[CASE=nom,NUM=?n,SEM=?pred]
PURE[CASE=nom,NUM=pl,SEM=<\P.\Q.all x.(Dharma(x) -> (Pure(x) <-> (P(x) | Q(x))))>] -> 'anāsravāḥ'
PATH[CASE=nom,NUM=sg,GEN=n,SEM=<\x.PathTruth(x)>] -> 'mārgasatyam'
THREE[CASE=nom,NUM=sg,GEN=n,SEM=<\P.\x.P(x)>] -> 'trividham'
AND -> 'ca'
ALSO -> 'api'
UNCOND[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Unconditioned(x)>] -> 'asaṃskṛtam'
SPACE[CASE=nom,NUM=sg,GEN=n,SEM=<space>] -> 'ākāśam'
TWO[CASE=nom,NUM=dual,GEN=m] -> 'dvau'
CESSATIONS[CASE=nom,NUM=dual,GEN=m,SEM=<\x.(x=discriminative_cessation | x=nondiscriminative_cessation)>] -> 'nirodhau'
AMONG -> 'tatra'
NONOBSTRUCTION[CASE=nom,NUM=sg,GEN=f,SEM=<\x.NonObstruction(x)>] -> 'anāvṛtiḥ'
"""

dependency_text = r"""'anāsravāḥ' -> 'mārgasatyam' | 'asaṃskṛtam'
'asaṃskṛtam' -> 'trividham' | 'ca' | 'api' | 'ākāśam' | 'nirodhau'
'nirodhau' -> 'dvau' | 'ca'
'anāvṛtiḥ' -> 'tatra' | 'ākāśam'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(Dharma(x) -> (Pure(x) <-> (PathTruth(x) | Unconditioned(x))))) & (all x.(Unconditioned(x) <-> (x=space | (x=discriminative_cessation | x=nondiscriminative_cessation)))) & NonObstruction(space))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(Dharma(x) -> (Pure(x) <-> (PathTruth(x) | Unconditioned(x))))) & (all x.(Unconditioned(x) <-> (x=space | (x=discriminative_cessation | x=nondiscriminative_cessation)))) & NonObstruction(space))
# THEORY prose
# EXPECT prose T
all x.(PathTruth(x) -> (Dharma(x) & Conditioned(x)))
all x.(Unconditioned(x) -> (Dharma(x) & -Conditioned(x)))
# THEORY rejected_pure_means_unconditioned
# EXPECT rejected_pure_means_unconditioned F
all x.(Pure(x) -> Unconditioned(x))
"""

valuation_text = r"""space => space_case
discriminative_cessation => discriminative
nondiscriminative_cessation => nondiscriminative
Dharma => {path,ordinary,space_case,discriminative,nondiscriminative}
Pure => {path,space_case,discriminative,nondiscriminative}
PathTruth => {path}
Unconditioned => {space_case,discriminative,nondiscriminative}
Conditioned => {path,ordinary}
NonObstruction => {space_case}
"""


In [12]:
studies[5] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[5]["tree"].pformat())
print("Derived SEM:", studies[5]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_pure_means_unconditioned=F
(S[SEM=<(all x.(Dharma(x) -> (Pure(x) <-> (PathTruth(x) | Unconditioned(x)))) & all x.(Unconditioned(x) <-> ((x = space) | (x = discriminative_cessation) | (x = nondiscriminative_cessation))) & NonObstruction(space))>]
  (PURE[CASE='nom', NUM='pl', SEM=<\P Q.all x.(Dharma(x) -> (Pure(x) <-> (P(x) | Q(x))))>]
    anāsravāḥ)
  (PATH[CASE='nom', GEN='n', NUM='sg', SEM=<\x.PathTruth(x)>]
    mārgasatyam)
  (UNCONDITIONED[SEM=<\x.Unconditioned(x)>]
    (THREE[CASE='nom', GEN='n', NUM='sg', SEM=<\P x.P(x)>] trividham)
    (AND[] ca)
    (ALSO[] api)
    (UNCOND[CASE='nom', GEN='n', NUM='sg', SEM=<\x.Unconditioned(x)>]
      asaṃskṛtam))
  (ROSTER[SEM=<all x.(Unconditioned(x) <-> ((x = space) | (x = discriminative_cessation) | (x = nondiscriminative_cessation)))>]
    (SPACE[CASE='nom', GEN='n', NUM='sg', SEM=<space>] ākāśam)
    (TWO[CASE='nom', GEN='m', NUM='dual'] dvau)
    (CESSATIONS[CASE='nom', GEN='m', NUM='dual', SEM=

# VAK_1.06 — The Two Cessations

Source: `VAK_1.06.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
pratisaṃkhyānirodho yo visaṃyogaḥ pṛthak pṛthak /
utpādātyantavighno 'nyo nirodho 'pratisaṃkhyayā // 1.06 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| pratisaṃkhyā-nirodhaḥ | tatpuruṣa; nominative singular | cessation through discrimination; the first named in 1.05 |
| yaḥ | relative pronoun | which |
| visaṃyogaḥ | nominative singular | disjunction; the definition |
| pṛthak pṛthak | iterated indeclinable | separately, and separately |
| utpāda-atyanta-vighnaḥ | compound; nominative singular | absolute obstruction of arising |
| anyaḥ | nominative singular | the other |
| nirodhaḥ | nominative singular | cessation |
| apratisaṃkhyayā | instrumental | not through discrimination |

**Grammar tokens.** Verse-order IAST with sandhi opened. Both occurrences
of *pṛthak* are retained. The two defining compounds remain whole leaves.
No covert word is needed; Impure dharmas and future arising are restrictions
resolved from the Bhāṣya in the semantic vocabulary.

```text
# tokens: pratisaṃkhyā-nirodhaḥ yaḥ visaṃyogaḥ pṛthak pṛthak utpāda-atyanta-vighnaḥ anyaḥ nirodhaḥ apratisaṃkhyayā
```

## 5. Translation

### Literal Translation

> Cessation through discrimination is disjunction, separately and
> separately. The other cessation, not through discrimination, is the
> absolute obstruction of arising.

### Bhāṣya-informed study translation

> Cessation through discriminative comprehension is disjunction from
> Impure dharmas, separately in each case. The other cessation, not through
> discriminative comprehension, is an absolute obstruction to the arising
> of a future dharma.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| pratisaṃkhyā | discrimination | a particular prajñā of the Noble Truths |
| nirodha | cessation | unconditioned here; the two are not one |
| visaṃyoga | disjunction | from impure dharmas |
| pṛthak pṛthak | separately | as many disjunctions as conjunctions |
| sāsrava | impure | the field of the first cessation |
| anāsrava | pure | no disjunction required where it can arise |
| apratisaṃkhyā | not through discrimination | not attained by that prajñā |
| utpāda-atyanta-vighna | absolute obstruction of arising | the second definition |
| pratyaya-vaikalya | deficiency of conditions | how the second is got |

In [13]:
fcfg_text = r"""% start S
# tokens: pratisaṃkhyā-nirodhaḥ yaḥ visaṃyogaḥ pṛthak pṛthak utpāda-atyanta-vighnaḥ anyaḥ nirodhaḥ apratisaṃkhyayā
S[SEM=<(?first & ?second)>] -> FIRST[SEM=?first] SECOND[SEM=?second]
FIRST[SEM=<?each(?definition(?kind)(?disjunction))>] -> KIND[CASE=nom,NUM=?n,GEN=?g,SEM=?kind] REL[CASE=nom,NUM=?n,GEN=?g,SEM=?definition] DISJUNCTION[CASE=nom,NUM=?n,GEN=?g,SEM=?disjunction] EACH[SEM=?each] EACH
SECOND[SEM=<?without(?other(?block)(?cessation))>] -> BLOCK[CASE=nom,NUM=?n,GEN=?g,SEM=?block] OTHER[CASE=nom,NUM=?n,GEN=?g,SEM=?other] CESSATION[CASE=nom,NUM=?n,GEN=?g,SEM=?cessation] WITHOUT[CASE=ins,SEM=?without]
KIND[CASE=nom,NUM=sg,GEN=m,SEM=<\x.DiscriminativeCessation(x)>] -> 'pratisaṃkhyā-nirodhaḥ'
REL[CASE=nom,NUM=sg,GEN=m,SEM=<\P.\Q.all x.(P(x) <-> Q(x))>] -> 'yaḥ'
DISJUNCTION[CASE=nom,NUM=sg,GEN=m,SEM=<\x.DisjunctionFromImpure(x)>] -> 'visaṃyogaḥ'
EACH[SEM=<\P.(P & all x.(DiscriminativeCessation(x) -> Casewise(x)))>] -> 'pṛthak'
BLOCK[CASE=nom,NUM=sg,GEN=m,SEM=<\x.AbsoluteObstructionOfFutureArising(x)>] -> 'utpāda-atyanta-vighnaḥ'
OTHER[CASE=nom,NUM=sg,GEN=m,SEM=<\P.\Q.all x.(OtherCessation(x) -> (P(x) & Q(x)))>] -> 'anyaḥ'
CESSATION[CASE=nom,NUM=sg,GEN=m,SEM=<\x.Cessation(x)>] -> 'nirodhaḥ'
WITHOUT[CASE=ins,SEM=<\P.(P & all x.(OtherCessation(x) -> -AttainedByDiscrimination(x)))>] -> 'apratisaṃkhyayā'
"""

dependency_text = r"""'pratisaṃkhyā-nirodhaḥ' -> 'yaḥ' | 'visaṃyogaḥ'
'visaṃyogaḥ' -> 'pṛthak'
'nirodhaḥ' -> 'utpāda-atyanta-vighnaḥ' | 'anyaḥ' | 'apratisaṃkhyayā'
"""

sem_text = r"""# derived by the section 4 parse
(((all x.(DiscriminativeCessation(x) <-> DisjunctionFromImpure(x))) & (all x.(DiscriminativeCessation(x) -> Casewise(x)))) & ((all x.(OtherCessation(x) -> (AbsoluteObstructionOfFutureArising(x) & Cessation(x)))) & (all x.(OtherCessation(x) -> -AttainedByDiscrimination(x)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(((all x.(DiscriminativeCessation(x) <-> DisjunctionFromImpure(x))) & (all x.(DiscriminativeCessation(x) -> Casewise(x)))) & ((all x.(OtherCessation(x) -> (AbsoluteObstructionOfFutureArising(x) & Cessation(x)))) & (all x.(OtherCessation(x) -> -AttainedByDiscrimination(x)))))
# THEORY prose
# EXPECT prose T
all x.(DiscriminativeCessation(x) -> (Cessation(x) & AttainedByDiscrimination(x)))
all x.(OtherCessation(x) -> ThroughDeficiencyOfConditions(x))
all x.(Cessation(x) -> Unconditioned(x))
# THEORY rejected_obstruction_is_disjunction
# EXPECT rejected_obstruction_is_disjunction F
all x.(OtherCessation(x) -> DisjunctionFromImpure(x))
"""

valuation_text = r"""DiscriminativeCessation => {release_one,release_two}
DisjunctionFromImpure => {release_one,release_two}
Casewise => {release_one,release_two}
OtherCessation => {blocked}
AbsoluteObstructionOfFutureArising => {blocked}
Cessation => {release_one,release_two,blocked}
AttainedByDiscrimination => {release_one,release_two}
ThroughDeficiencyOfConditions => {blocked}
Unconditioned => {release_one,release_two,blocked}
"""


In [14]:
studies[6] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[6]["tree"].pformat())
print("Derived SEM:", studies[6]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_obstruction_is_disjunction=F
(S[SEM=<(all x.(DiscriminativeCessation(x) <-> DisjunctionFromImpure(x)) & all x.(DiscriminativeCessation(x) -> Casewise(x)) & all x.(OtherCessation(x) -> (AbsoluteObstructionOfFutureArising(x) & Cessation(x))) & all x.(OtherCessation(x) -> -AttainedByDiscrimination(x)))>]
  (FIRST[SEM=<(all x.(DiscriminativeCessation(x) <-> DisjunctionFromImpure(x)) & all x.(DiscriminativeCessation(x) -> Casewise(x)))>]
    (KIND[CASE='nom', GEN='m', NUM='sg', SEM=<\x.DiscriminativeCessation(x)>]
      pratisaṃkhyā-nirodhaḥ)
    (REL[CASE='nom', GEN='m', NUM='sg', SEM=<\P Q.all x.(P(x) <-> Q(x))>]
      yaḥ)
    (DISJUNCTION[CASE='nom', GEN='m', NUM='sg', SEM=<\x.DisjunctionFromImpure(x)>]
      visaṃyogaḥ)
    (EACH[SEM=<\P.(P & all x.(DiscriminativeCessation(x) -> Casewise(x)))>]
      pṛthak)
    (EACH[SEM=<\P.(P & all x.(DiscriminativeCessation(x) -> Casewise(x)))>]
      pṛthak))
  (SECOND[SEM=<(all x.(OtherCessation(x) -> (Abs

# VAK_1.07 — The Wheel in a Human Language

Source: `VAK_1.07.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
te punaḥ saṃskṛtā dharmā rūpādiskandhapañcakam /
sa evādhvā kathāvastu saniḥsārāḥ savastukāḥ // 1.07 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| te | nominative masculine plural | those, resumed from 1.04 |
| punaḥ | indeclinable | resumes. Not “again” |
| saṃskṛtāḥ | nominative masculine plural | Conditioned |
| dharmāḥ | nominative masculine plural | dharmas |
| rūpādiskandhapañcakam | rūpa-ādi-skandha-pañcakam; collective singular predicate | the five Bases beginning with Form, gathered as one |
| saḥ eva | restrictive | that same field. The second line is not four extra heaps |
| adhvā | nominative masculine singular | the course |
| kathāvastu | kathā-vastu; compound | the reflection-term. Not “basis of discourse” |
| saniḥsārāḥ | sa-niḥsāra; bahuvrīhi, nominative masculine plural | the sāra term. Not “possessed of an exit” |
| savastukāḥ | sa-vastuka; bahuvrīhi, nominative masculine plural | possessed of vastu |

**Grammar tokens.** Normalized IAST in verse order, with sandhi opened.
The collective compound remains one leaf; *saḥ eva* becomes two.
No supplied token is needed. The singular resumption takes up the
collective field, while the final plural predicates resume its Dharmas.

```text
# tokens: te punaḥ saṃskṛtāḥ dharmāḥ rūpādi-skandha-pañcakam saḥ eva adhvā kathāvastu saniḥsārāḥ savastukāḥ
```

## 5. Translation

### Literal Translation

> Those Conditioned dharmas are the five Bases beginning with Form. That
> same field is the course, the *kathāvastu*, the *saniḥsāra*, and the
> *savastuka*.

### Bhāṣya-informed study translation

> The conditioned dharmas are the five Bases beginning with the Form Base.
> These same dharmas are past, present, and future; subjects of discourse;
> have an exit in nirvāṇa; and possess a real basis.

## 7. Technical Vocabulary

| Sanskrit | Do not use as the reading | Determination |
|---|---|---|
| saṃskṛta | created; constructed | Conditioned |
| skandha | a list; heap as dismissive | Base |
| rūpa | purely material Form | Form; the first member is the Form Base |
| adhvā | a clock | Path of Dharma |
| kathāvastu | basis of discourse; topic | Essential Reflection |
| saniḥsāra | possessed of an exit | Essential Relation. A Dharma is an End |
| vastu | cause; topic; substrate | Appearance |
| savastuka | possessed of causes | possessed of that Appearance |
| punaḥ | again | resumes |

The Vaibhāṣika gloss *vastu* = *hetu* is the armor-moment. Not the reading.

In [15]:
fcfg_text = r"""% start S
# tokens: te punaḥ saṃskṛtāḥ dharmāḥ rūpādi-skandha-pañcakam saḥ eva adhvā kathāvastu saniḥsārāḥ savastukāḥ
S[SEM=<(?classification & ?determinations)>] -> CLASSIFICATION[SEM=?classification] RESUMPTION[SEM=?determinations]
CLASSIFICATION[SEM=<all x.(?dharma(x) -> (?conditioned(x) <-> ?bases(x)))>] -> DEM[CASE=nom,NUM=?n,GEN=?g] RESUME CONDITIONED[CASE=nom,NUM=?n,GEN=?g,SEM=?conditioned] DHARMA[CASE=nom,NUM=?n,GEN=?g,SEM=?dharma] COLLECTIVE[CASE=nom,NUM=sg,GEN=n,SEM=?bases]
RESUMPTION[SEM=<?same(?course)(?discourse)(?end)(?vastu)>] -> SAME[CASE=nom,NUM=sg,GEN=m,SEM=?same] ONLY COURSE[CASE=nom,NUM=sg,GEN=m,SEM=?course] DISCOURSE[CASE=nom,NUM=sg,GEN=n,SEM=?discourse] END[CASE=nom,NUM=?n,GEN=?g,SEM=?end] VASTU[CASE=nom,NUM=?n,GEN=?g,SEM=?vastu]
DEM[CASE=nom,NUM=pl,GEN=m] -> 'te'
RESUME -> 'punaḥ'
CONDITIONED[CASE=nom,NUM=pl,GEN=m,SEM=<\x.Conditioned(x)>] -> 'saṃskṛtāḥ'
DHARMA[CASE=nom,NUM=pl,GEN=m,SEM=<\x.Dharma(x)>] -> 'dharmāḥ'
COLLECTIVE[CASE=nom,NUM=sg,GEN=n,SEM=<\x.FiveBaseMember(x)>] -> 'rūpādi-skandha-pañcakam'
SAME[CASE=nom,NUM=sg,GEN=m,SEM=<\P.\Q.\R.\T.all x.((Dharma(x) & Conditioned(x)) -> (P(x) & Q(x) & R(x) & T(x)))>] -> 'saḥ'
ONLY -> 'eva'
COURSE[CASE=nom,NUM=sg,GEN=m,SEM=<\x.CourseDetermination(x)>] -> 'adhvā'
DISCOURSE[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Kathavastu(x)>] -> 'kathāvastu'
END[CASE=nom,NUM=pl,GEN=m,SEM=<\x.HasNihsara(x)>] -> 'saniḥsārāḥ'
VASTU[CASE=nom,NUM=pl,GEN=m,SEM=<\x.HasVastu(x)>] -> 'savastukāḥ'
"""

dependency_text = r"""'rūpādi-skandha-pañcakam' -> 'dharmāḥ'
'dharmāḥ' -> 'te' | 'punaḥ' | 'saṃskṛtāḥ'
'adhvā' -> 'saḥ' | 'kathāvastu' | 'saniḥsārāḥ' | 'savastukāḥ'
'saḥ' -> 'eva'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(Dharma(x) -> (Conditioned(x) <-> FiveBaseMember(x)))) & (all x.((Dharma(x) & Conditioned(x)) -> (CourseDetermination(x) & Kathavastu(x) & HasNihsara(x) & HasVastu(x)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(Dharma(x) -> (Conditioned(x) <-> FiveBaseMember(x)))) & (all x.((Dharma(x) & Conditioned(x)) -> (CourseDetermination(x) & Kathavastu(x) & HasNihsara(x) & HasVastu(x)))))
# THEORY preceding_classification
# EXPECT preceding_classification T
all x.(Unconditioned(x) -> (Dharma(x) & -FiveBaseMember(x)))
exists x.(Dharma(x) & Conditioned(x) & Pure(x) & -Impure(x))
# THEORY rejected_conditioned_means_impure
# EXPECT rejected_conditioned_means_impure F
all x.((Dharma(x) & Conditioned(x)) -> Impure(x))
"""

valuation_text = r"""Dharma => {form,feeling,reflection,formation,cognition,path,space_case}
Conditioned => {form,feeling,reflection,formation,cognition,path}
FiveBaseMember => {form,feeling,reflection,formation,cognition,path}
CourseDetermination => {form,feeling,reflection,formation,cognition,path}
Kathavastu => {form,feeling,reflection,formation,cognition,path}
HasNihsara => {form,feeling,reflection,formation,cognition,path}
HasVastu => {form,feeling,reflection,formation,cognition,path}
Unconditioned => {space_case}
Pure => {path,space_case}
Impure => {form,feeling,reflection,formation,cognition}
"""


In [16]:
studies[7] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[7]["tree"].pformat())
print("Derived SEM:", studies[7]["sem"])


parses 1 SEM=ok
verse=T preceding_classification=T rejected_conditioned_means_impure=F
(S[SEM=<(all x.(Dharma(x) -> (Conditioned(x) <-> FiveBaseMember(x))) & all x.((Dharma(x) & Conditioned(x)) -> (CourseDetermination(x) & Kathavastu(x) & HasNihsara(x) & HasVastu(x))))>]
  (CLASSIFICATION[SEM=<all x.(Dharma(x) -> (Conditioned(x) <-> FiveBaseMember(x)))>]
    (DEM[CASE='nom', GEN='m', NUM='pl'] te)
    (RESUME[] punaḥ)
    (CONDITIONED[CASE='nom', GEN='m', NUM='pl', SEM=<\x.Conditioned(x)>]
      saṃskṛtāḥ)
    (DHARMA[CASE='nom', GEN='m', NUM='pl', SEM=<\x.Dharma(x)>]
      dharmāḥ)
    (COLLECTIVE[CASE='nom', GEN='n', NUM='sg', SEM=<\x.FiveBaseMember(x)>]
      rūpādi-skandha-pañcakam))
  (RESUMPTION[SEM=<all x.((Dharma(x) & Conditioned(x)) -> (CourseDetermination(x) & Kathavastu(x) & HasNihsara(x) & HasVastu(x)))>]
    (SAME[CASE='nom', GEN='m', NUM='sg', SEM=<\P Q R T.all x.((Dharma(x) & Conditioned(x)) -> (P(x) & Q(x) & R(x) & T(x)))>]
      saḥ)
    (ONLY[] eva)
    (COURSE[CASE='

# VAK_1.08 — The Impure Division of the Bases

Source: `VAK_1.08.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
ye sāsravā upādānaskandhās te saraṇā api /
duḥkhaṃ samudayo loko dṛṣṭisthānaṃ bhavaś ca te // 1.08 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| ye | nominative masculine plural | those which. Restricts the five Bases of 1.07 |
| sāsravāḥ | sa-āsrava; nominative masculine plural | Impure. The summary division |
| upādānaskandhāḥ | upādāna-skandha; nominative masculine plural | Bases of appropriation. Not every Base |
| te | nominative masculine plural | they. Same field |
| saraṇāḥ | sa-raṇa; nominative masculine plural | with conflict. Coextensive with the Impure field |
| api | particle | also. Not a second inventory |
| duḥkham | nominative neuter singular | suffering |
| samudayaḥ | nominative masculine singular | origin |
| lokaḥ | nominative masculine singular | world. Here, what dissolves |
| dṛṣṭisthānam | dṛṣṭi-sthāna; nominative neuter singular | station of views. Not the station of prajñā |
| bhavaḥ | nominative masculine singular | becoming |
| ca | particle | and. The five names are one field |
| te | nominative masculine plural | they, resumed |

**Grammar tokens.** Verse-order IAST with sandhi opened. Both occurrences
of *te* remain; each resumes the Impure restriction, not every Base.
The compound *upādāna-skandhāḥ* remains one leaf. No covert token is needed.

```text
# tokens: ye sāsravāḥ upādāna-skandhāḥ te saraṇāḥ api duḥkham samudayaḥ lokaḥ dṛṣṭisthānam bhavaḥ ca te
```

## 5. Translation

### Literal Translation

> Those which are Impure are the Bases of appropriation. They are also with
> conflict. They are suffering, origin, world, the station of views, and
> becoming.

### Bhāṣya-informed study translation

> Those conditioned dharmas that are with outflows are Bases of
> appropriation; they too are with conflict. These same dharmas are
> suffering, origin, the world, the station of views, and becoming.

## 7. Technical Vocabulary

| Sanskrit | Do not use as the reading | Determination |
|---|---|---|
| sāsrava | with outflows; lodging | Impure. Summary division |
| anāsrava | outflow-free | Pure. The Path is Conditioned and Pure |
| upādānaskandha | Base of clinging as a stain-word | Base of appropriation |
| saraṇa | with defilement, unspecified | with conflict |
| loka | a place | here, what dissolves |
| dṛṣṭisthāna | a viewpoint | station of views |
| bhava | existence as a realm-name | becoming |

In [17]:
fcfg_text = r"""% start S
# tokens: ye sāsravāḥ upādāna-skandhāḥ te saraṇāḥ api duḥkham samudayaḥ lokaḥ dṛṣṭisthānam bhavaḥ ca te
S[SEM=<(?a & ?c & ?names)>] -> APPROPRIATION[SEM=?a] CONFLICT[SEM=?c] NAMES[SEM=?names]
APPROPRIATION[SEM=<?rel(?impure)(?base)>] -> REL[CASE=nom,NUM=?n,GEN=?g,SEM=?rel] IMPURE[CASE=nom,NUM=?n,GEN=?g,SEM=?impure] BASE[CASE=nom,NUM=?n,GEN=?g,SEM=?base]
CONFLICT[SEM=<?scope(?conflict)>] -> CORR[CASE=nom,NUM=?n,GEN=?g,SEM=?scope] CONFLICTHEAD[CASE=nom,NUM=?n,GEN=?g,SEM=?conflict] ALSO
NAMES[SEM=<?scope(\x.(?suffering(x) & ?origin(x) & ?world(x) & ?views(x) & ?becoming(x)))>] -> SUFFERING[CASE=nom,SEM=?suffering] ORIGIN[CASE=nom,SEM=?origin] WORLD[CASE=nom,SEM=?world] VIEWS[CASE=nom,SEM=?views] BECOMING[CASE=nom,SEM=?becoming] AND CORR[CASE=nom,NUM=pl,GEN=m,SEM=?scope]
REL[CASE=nom,NUM=pl,GEN=m,SEM=<\P.\Q.all x.((Dharma(x) & Conditioned(x)) -> (P(x) <-> Q(x)))>] -> 'ye'
IMPURE[CASE=nom,NUM=pl,GEN=m,SEM=<\x.Impure(x)>] -> 'sāsravāḥ'
BASE[CASE=nom,NUM=pl,GEN=m,SEM=<\x.AppropriationBaseMember(x)>] -> 'upādāna-skandhāḥ'
CORR[CASE=nom,NUM=pl,GEN=m,SEM=<\P.all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> P(x))>] -> 'te'
CONFLICTHEAD[CASE=nom,NUM=pl,GEN=m,SEM=<\x.WithConflict(x)>] -> 'saraṇāḥ'
ALSO -> 'api'
SUFFERING[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Suffering(x)>] -> 'duḥkham'
ORIGIN[CASE=nom,NUM=sg,GEN=m,SEM=<\x.Origin(x)>] -> 'samudayaḥ'
WORLD[CASE=nom,NUM=sg,GEN=m,SEM=<\x.DissolvingWorld(x)>] -> 'lokaḥ'
VIEWS[CASE=nom,NUM=sg,GEN=n,SEM=<\x.StationOfViews(x)>] -> 'dṛṣṭisthānam'
BECOMING[CASE=nom,NUM=sg,GEN=m,SEM=<\x.Becoming(x)>] -> 'bhavaḥ'
AND -> 'ca'
"""

dependency_text = r"""'upādāna-skandhāḥ' -> 'ye' | 'sāsravāḥ'
'saraṇāḥ' -> 'te' | 'api'
'duḥkham' -> 'samudayaḥ' | 'lokaḥ' | 'dṛṣṭisthānam' | 'bhavaḥ' | 'te'
'bhavaḥ' -> 'ca'
"""

sem_text = r"""# derived by the section 4 parse
((all x.((Dharma(x) & Conditioned(x)) -> (Impure(x) <-> AppropriationBaseMember(x)))) & (all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> WithConflict(x))) & (all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> (Suffering(x) & Origin(x) & DissolvingWorld(x) & StationOfViews(x) & Becoming(x)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.((Dharma(x) & Conditioned(x)) -> (Impure(x) <-> AppropriationBaseMember(x)))) & (all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> WithConflict(x))) & (all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> (Suffering(x) & Origin(x) & DissolvingWorld(x) & StationOfViews(x) & Becoming(x)))))
# THEORY prose
# EXPECT prose T
all x.(AppropriationBaseMember(x) -> FiveBaseMember(x))
exists x.(Dharma(x) & Conditioned(x) & Pure(x) & FiveBaseMember(x) & -AppropriationBaseMember(x))
# THEORY rejected_every_base_is_appropriated
# EXPECT rejected_every_base_is_appropriated F
all x.(FiveBaseMember(x) -> AppropriationBaseMember(x))
"""

valuation_text = r"""Dharma => {ordinary,path,space_case}
Conditioned => {ordinary,path}
Impure => {ordinary}
Pure => {path,space_case}
FiveBaseMember => {ordinary,path}
AppropriationBaseMember => {ordinary}
WithConflict => {ordinary}
Suffering => {ordinary}
Origin => {ordinary}
DissolvingWorld => {ordinary}
StationOfViews => {ordinary}
Becoming => {ordinary}
"""


In [18]:
studies[8] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[8]["tree"].pformat())
print("Derived SEM:", studies[8]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_every_base_is_appropriated=F
(S[SEM=<(all x.((Dharma(x) & Conditioned(x)) -> (Impure(x) <-> AppropriationBaseMember(x))) & all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> WithConflict(x)) & all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> (Suffering(x) & Origin(x) & DissolvingWorld(x) & StationOfViews(x) & Becoming(x))))>]
  (APPROPRIATION[SEM=<all x.((Dharma(x) & Conditioned(x)) -> (Impure(x) <-> AppropriationBaseMember(x)))>]
    (REL[CASE='nom', GEN='m', NUM='pl', SEM=<\P Q.all x.((Dharma(x) & Conditioned(x)) -> (P(x) <-> Q(x)))>]
      ye)
    (IMPURE[CASE='nom', GEN='m', NUM='pl', SEM=<\x.Impure(x)>]
      sāsravāḥ)
    (BASE[CASE='nom', GEN='m', NUM='pl', SEM=<\x.AppropriationBaseMember(x)>]
      upādāna-skandhāḥ))
  (CONFLICT[SEM=<all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> WithConflict(x))>]
    (CORR[CASE='nom', GEN='m', NUM='pl', SEM=<\P.all x.((Dharma(x) & Conditioned(x) & Impure(x)) -> P(x))>]
      te)
    (CONFLICTHEAD

# VAK_1.09 — The Form Base

Source: `VAK_1.09.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
rūpaṃ pañcendriyāṇy arthāḥ pañcāvijñaptir eva ca /
tadvijñānāśrayā rūpaprasādāś cakṣurādayaḥ // 1.09 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| rūpam | nominative neuter singular | Form; the Form Base |
| pañcendriyāṇi | pañca + indriyāṇi; nominative neuter plural | five Faculties |
| arthāḥ | nominative masculine plural | meanings |
| pañca | numeral | counts the meanings, not the faculties again |
| avijñaptiḥ | nominative feminine singular | one member; named, not defined |
| eva ca | particles | and just that |
| tadvijñānāśrayāḥ | tad-vijñāna-āśrayāḥ; nominative plural | supports of the corresponding Cognition |
| rūpaprasādāḥ | rūpa-prasādāḥ; nominative plural | clarities of Form |
| cakṣurādayaḥ | cakṣus-ādayaḥ; nominative plural | beginning with the eye |

**Grammar tokens.** Verse-order IAST with external sandhi opened.
*Pañcendriyāṇi* becomes *pañca indriyāṇi*; both occurrences of *pañca*
remain, counting different groups. The two support compounds remain
whole leaves. No covert token is needed.

```text
# tokens: rūpam pañca indriyāṇi arthāḥ pañca avijñaptiḥ eva ca tadvijñānāśrayāḥ rūpaprasādāḥ cakṣurādayaḥ
```

## 5. Translation

### Literal Translation

> Form is the five Faculties, the five meanings, and *avijñapti*. The
> clarities of Form, beginning with the eye, are supports of the
> corresponding Cognition.

### Bhāṣya-informed study translation

> The Form Base comprises five Faculties—eye, ear, nose, tongue, and
> body—their five corresponding meanings—visible Form, sound, odor, taste,
> and the tangible—and *avijñapti*. The five Faculties are clarities of
> Form, each serving as the support of the Cognition corresponding to its
> meaning.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| rūpa | Form | the Form Base; eleven |
| skandha | Base | the first collection; not the whole |
| indriya | Faculty | one of the five; not a synonym for an anatomical organ |
| artha | meaning | the five |
| viṣaya | condition | the commentary’s gloss of meaning; not a second list |
| avijñapti | *avijñapti* | named, not defined |
| rūpa-prasāda | clarity of Form | what the five Faculties are |
| vijñāna | Cognition | not Principle |
| āśraya | support | of the corresponding Cognition; not a twelfth member |
| āyatana | Essence | cited for the eye; not enumerated |
| upādāya | depending on | the four Great Elements; not the appropriation of 1.08 |
| dhātu | Principle | not a member of this count |

In [19]:
fcfg_text = r"""% start S
# tokens: rūpam pañca indriyāṇi arthāḥ pañca avijñaptiḥ eva ca tadvijñānāśrayāḥ rūpaprasādāḥ cakṣurādayaḥ
S[SEM=?definition] -> CL[K=enumerate] CL[K=define,SEM=?definition]
CL[K=enumerate] -> N[CASE=nom,NUM=sg,GEN=n,K=form] NUMR[COUNT=five] N[CASE=nom,NUM=pl,GEN=n,K=faculty] N[CASE=nom,NUM=pl,GEN=m,K=meaning] NUMR[COUNT=five] N[CASE=nom,NUM=sg,GEN=f,K=avijnapti] PTCL[FOCUS=yes] CONJ[COORD=and]
CL[K=define,SEM=<?q(\x.(?s(x) & ?p(x)))>] -> N[CASE=nom,NUM=?n,GEN=?g,K=support,SEM=?s] N[CASE=nom,NUM=?n,GEN=?g,K=clarity,SEM=?p] N[CASE=nom,NUM=?n,GEN=?g,K=faculties,SEM=?q]
N[CASE=nom,NUM=sg,GEN=n,K=form] -> 'rūpam'
NUMR[COUNT=five] -> 'pañca'
N[CASE=nom,NUM=pl,GEN=n,K=faculty] -> 'indriyāṇi'
N[CASE=nom,NUM=pl,GEN=m,K=meaning] -> 'arthāḥ'
N[CASE=nom,NUM=sg,GEN=f,K=avijnapti] -> 'avijñaptiḥ'
PTCL[FOCUS=yes] -> 'eva'
CONJ[COORD=and] -> 'ca'
N[CASE=nom,NUM=pl,GEN=m,K=support,SEM=<\x.exists c.(Cognition(c) & Corresponds(c,x) & Supports(x,c))>] -> 'tadvijñānāśrayāḥ'
N[CASE=nom,NUM=pl,GEN=m,K=clarity,SEM=<\x.ClarityOfForm(x)>] -> 'rūpaprasādāḥ'
N[CASE=nom,NUM=pl,GEN=m,K=faculties,SEM=<\P.all x.(Faculty(x) -> P(x))>] -> 'cakṣurādayaḥ'
"""

dependency_text = r"""'rūpam' -> 'indriyāṇi' | 'arthāḥ' | 'avijñaptiḥ'
'indriyāṇi' -> 'pañca'
'arthāḥ' -> 'pañca'
'avijñaptiḥ' -> 'eva' | 'ca'
'cakṣurādayaḥ' -> 'tadvijñānāśrayāḥ' | 'rūpaprasādāḥ'
"""

sem_text = r"""faculties : \P.all x.(Faculty(x) -> P(x))
clarities : \x.ClarityOfForm(x)
supports : \x.exists c.(Cognition(c) & Corresponds(c,x) & Supports(x,c))
definition : (\P.all x.(Faculty(x) -> P(x)))(\x.((\x.exists c.(Cognition(c) & Corresponds(c,x) & Supports(x,c)))(x) & ClarityOfForm(x)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
# [verse; prose resolves the five Faculties and corresponding Cognitions]
all x.(Faculty(x) -> (exists c.(Cognition(c) & Corresponds(c,x) & Supports(x,c)) & ClarityOfForm(x)))
# [verse; prose: etavan rupaskandhah] exhaustive membership by the three groups
all x.(FormMember(x) <-> (Faculty(x) | Meaning(x) | Avijnapti(x)))

# THEORY prose
# EXPECT prose T
# [prose] the two fivefold lists, with avijnapti named but not defined
all x.(Faculty(x) <-> (x = eye | x = ear | x = nose | x = tongue | x = body))
all x.(Meaning(x) <-> (x = visible | x = sound | x = odor | x = taste | x = tangible))
all x.(Avijnapti(x) <-> x = avijnapti)
# [prose] corresponding support, not arbitrary support of any Cognition
Supports(eye,eyecognition) & Corresponds(eyecognition,eye)
Supports(ear,earcognition) & Corresponds(earcognition,ear)
Supports(nose,nosecognition) & Corresponds(nosecognition,nose)
Supports(tongue,tonguecognition) & Corresponds(tonguecognition,tongue)
Supports(body,bodycognition) & Corresponds(bodycognition,body)
# [verse enumeration; prose closure] supported Cognition is not an extra member
all c.(Cognition(c) -> -FormMember(c))

# THEORY counterexamples
# EXPECT counterexamples T
exists f c.(Faculty(f) & Cognition(c) & Supports(f,c) & -(f = c))
exists x.(FormMember(x) & -VisibleMeaning(x))

# THEORY rejected_support_is_identity
# EXPECT rejected_support_is_identity F
all f c.(Supports(f,c) -> f = c)

# THEORY rejected_form_is_only_visible
# EXPECT rejected_form_is_only_visible F
all x.(FormMember(x) -> VisibleMeaning(x))
"""

valuation_text = r"""eye => eye
ear => ear
nose => nose
tongue => tongue
body => body
visible => visible
sound => sound
odor => odor
taste => taste
tangible => tangible
avijnapti => avijnapti
eyecognition => eyecognition
earcognition => earcognition
nosecognition => nosecognition
tonguecognition => tonguecognition
bodycognition => bodycognition
FormMember => {eye, ear, nose, tongue, body, visible, sound, odor, taste, tangible, avijnapti}
Faculty => {eye, ear, nose, tongue, body}
Meaning => {visible, sound, odor, taste, tangible}
Avijnapti => {avijnapti}
ClarityOfForm => {eye, ear, nose, tongue, body}
Cognition => {eyecognition, earcognition, nosecognition, tonguecognition, bodycognition}
Corresponds => {(eyecognition,eye), (earcognition,ear), (nosecognition,nose), (tonguecognition,tongue), (bodycognition,body)}
Supports => {(eye,eyecognition), (ear,earcognition), (nose,nosecognition), (tongue,tonguecognition), (body,bodycognition)}
VisibleMeaning => {visible}
"""


In [20]:
studies[9] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[9]["tree"].pformat())
print("Derived SEM:", studies[9]["sem"])


parses 1 SEM=ok
verse=T prose=T counterexamples=T rejected_support_is_identity=F rejected_form_is_only_visible=F
(S[SEM=<all x.(Faculty(x) -> (exists c.(Cognition(c) & Corresponds(c,x) & Supports(x,c)) & ClarityOfForm(x)))>]
  (CL[K='enumerate']
    (N[CASE='nom', GEN='n', K='form', NUM='sg'] rūpam)
    (NUMR[COUNT='five'] pañca)
    (N[CASE='nom', GEN='n', K='faculty', NUM='pl'] indriyāṇi)
    (N[CASE='nom', GEN='m', K='meaning', NUM='pl'] arthāḥ)
    (NUMR[COUNT='five'] pañca)
    (N[CASE='nom', GEN='f', K='avijnapti', NUM='sg'] avijñaptiḥ)
    (PTCL[FOCUS='yes'] eva)
    (CONJ[COORD='and'] ca))
  (CL[K='define', SEM=<all x.(Faculty(x) -> (exists c.(Cognition(c) & Corresponds(c,x) & Supports(x,c)) & ClarityOfForm(x)))>]
    (N[CASE='nom', GEN='m', K='support', NUM='pl', SEM=<\x.exists c.(Cognition(c) & Corresponds(c,x) & Supports(x,c))>]
      tadvijñānāśrayāḥ)
    (N[CASE='nom', GEN='m', K='clarity', NUM='pl', SEM=<\x.ClarityOfForm(x)>]
      rūpaprasādāḥ)
    (N[CASE='nom', GEN='m'

# VAK_1.10 — Counts Within the Meanings

Source: `VAK_1.10.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
rūpaṃ dvidhā viṃśatidhā śabdas tv aṣṭavidhaḥ rasaḥ /
ṣoḍhā caturvidho gandhaḥ spṛśyam ekādaśātmakam // 1.10 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| rūpam | nominative neuter singular | visible Form; one meaning, not the Form Base |
| dvidhā | adverb | twofold |
| viṃśatidhā | adverb | twentyfold; the same, counted |
| śabdaḥ | nominative masculine | sound |
| tu | particle | turns to the next |
| aṣṭavidhaḥ | nominative masculine | eightfold |
| rasaḥ | nominative masculine | taste |
| ṣoḍhā | adverb | sixfold; construes with taste |
| caturvidhaḥ | nominative masculine | fourfold; construes with odor |
| gandhaḥ | nominative masculine | odor |
| spṛśyam | nominative neuter | the tangible |
| ekādaśātmakam | bahuvrīhi; nominative neuter | having eleven as its nature |

*Rūpa* here is one of the five meanings. Not the Form Base of 1.09.

**Grammar tokens.** Verse-order IAST with sandhi opened. *Caturvidhaḥ*
precedes *gandhaḥ*, as in the lexical table. Compounds remain whole leaves; no covert token
is required.

```text
# tokens: rūpam dvidhā viṃśatidhā śabdaḥ tu aṣṭavidhaḥ rasaḥ ṣoḍhā caturvidhaḥ gandhaḥ spṛśyam ekādaśātmakam
```

## 5. Translation

### Literal Translation

> Visible Form is twofold and twentyfold. Sound, however, is eightfold.
> Taste is sixfold. Odor is fourfold. The tangible has eleven as its nature.

### Bhāṣya-informed study translation

> Visible Form is twofold, color and configuration, and is further counted
> in twenty kinds. Sound has eight kinds, taste six, and odor four. The
> tangible has eleven kinds: the four Great Elements and seven derived
> tangibles.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| rūpa | visible Form | one meaning; not the Form Base |
| varṇa | color | blue, yellow, red, white |
| saṃsthāna | configuration | long through uneven; eight |
| viṃśatidhā | twentyfold | the same meaning, counted; not twenty-one |
| śabda | sound | eightfold |
| rasa | taste | sixfold |
| gandha | odor | fourfold; the treatise says three |
| spṛśya | tangible | eleven as its nature |
| mahābhūta | Great Element | four of the eleven; defined later |
| vid | knowing | not existence; the prose |
| āyatana | Essence | the field of the count; not a new list |
| vijñāna | Cognition | what takes one or many; not a member |
| rūpadhātu | Form Realm | where hunger and thirst are absent |

In [21]:
fcfg_text = r"""% start S
# tokens: rūpam dvidhā viṃśatidhā śabdaḥ tu aṣṭavidhaḥ rasaḥ ṣoḍhā caturvidhaḥ gandhaḥ spṛśyam ekādaśātmakam
S[SEM=<(?v & ?s & ?t & ?o & ?b)>] -> VISIBLE[SEM=?v] SOUND[SEM=?s] TASTE[SEM=?t] ODOR[SEM=?o] TANGIBLE[SEM=?b]
VISIBLE[SEM=<(?two(?meaning) & ?twenty(?meaning))>] -> N[K=visible,CASE=nom,SEM=?meaning] ADV[K=binary,SEM=?two] ADV[K=enumerated,SEM=?twenty]
SOUND[SEM=<?count(?meaning)>] -> N[K=sound,CASE=nom,NUM=?n,GEN=?g,SEM=?meaning] CONTRAST COUNT[K=eight,CASE=nom,NUM=?n,GEN=?g,SEM=?count]
TASTE[SEM=<?count(?meaning)>] -> N[K=taste,CASE=nom,SEM=?meaning] ADV[K=six,SEM=?count]
ODOR[SEM=<?count(?meaning)>] -> COUNT[K=four,CASE=nom,NUM=?n,GEN=?g,SEM=?count] N[K=odor,CASE=nom,NUM=?n,GEN=?g,SEM=?meaning]
TANGIBLE[SEM=<?count(?meaning)>] -> N[K=tangible,CASE=nom,NUM=?n,GEN=?g,SEM=?meaning] COUNT[K=eleven,CASE=nom,NUM=?n,GEN=?g,SEM=?count]
N[K=visible,CASE=nom,NUM=sg,GEN=n,SEM=<visible_meaning>] -> 'rūpam'
ADV[K=binary,SEM=<\x.TwofoldAnalysis(x)>] -> 'dvidhā'
ADV[K=enumerated,SEM=<\x.EnumeratedKinds(x,twenty)>] -> 'viṃśatidhā'
N[K=sound,CASE=nom,NUM=sg,GEN=m,SEM=<sound_meaning>] -> 'śabdaḥ'
CONTRAST -> 'tu'
COUNT[K=eight,CASE=nom,NUM=sg,GEN=m,SEM=<\x.EnumeratedKinds(x,eight)>] -> 'aṣṭavidhaḥ'
N[K=taste,CASE=nom,NUM=sg,GEN=m,SEM=<taste_meaning>] -> 'rasaḥ'
ADV[K=six,SEM=<\x.EnumeratedKinds(x,six)>] -> 'ṣoḍhā'
COUNT[K=four,CASE=nom,NUM=sg,GEN=m,SEM=<\x.EnumeratedKinds(x,four)>] -> 'caturvidhaḥ'
N[K=odor,CASE=nom,NUM=sg,GEN=m,SEM=<odor_meaning>] -> 'gandhaḥ'
N[K=tangible,CASE=nom,NUM=sg,GEN=n,SEM=<tangible_meaning>] -> 'spṛśyam'
COUNT[K=eleven,CASE=nom,NUM=sg,GEN=n,SEM=<\x.ElevenfoldConstitution(x)>] -> 'ekādaśātmakam'
"""

dependency_text = r"""'rūpam' -> 'dvidhā' | 'viṃśatidhā'
'śabdaḥ' -> 'tu' | 'aṣṭavidhaḥ'
'rasaḥ' -> 'ṣoḍhā'
'gandhaḥ' -> 'caturvidhaḥ'
'spṛśyam' -> 'ekādaśātmakam'
"""

sem_text = r"""# derived by the section 4 parse
((TwofoldAnalysis(visible_meaning) & EnumeratedKinds(visible_meaning,twenty)) & EnumeratedKinds(sound_meaning,eight) & EnumeratedKinds(taste_meaning,six) & EnumeratedKinds(odor_meaning,four) & ElevenfoldConstitution(tangible_meaning))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((TwofoldAnalysis(visible_meaning) & EnumeratedKinds(visible_meaning,twenty)) & EnumeratedKinds(sound_meaning,eight) & EnumeratedKinds(taste_meaning,six) & EnumeratedKinds(odor_meaning,four) & ElevenfoldConstitution(tangible_meaning))
# THEORY prose
# EXPECT prose T
BinaryAspect(visible_meaning,color) & BinaryAspect(visible_meaning,configuration)
all x.(TangibleConstituent(x) <-> (GreatElement(x) | DerivedTangible(x)))
# THEORY rejected_visible_means_form_base
# EXPECT rejected_visible_means_form_base F
visible_meaning = form_base
"""

valuation_text = r"""visible_meaning => visible_class
sound_meaning => sound_class
taste_meaning => taste_class
odor_meaning => odor_class
tangible_meaning => tangible_class
form_base => form_collection
twenty => count_twenty
eight => count_eight
six => count_six
four => count_four
color => color_aspect
configuration => configuration_aspect
TwofoldAnalysis => {visible_class}
EnumeratedKinds => {(visible_class,count_twenty),(sound_class,count_eight),(taste_class,count_six),(odor_class,count_four)}
ElevenfoldConstitution => {tangible_class}
BinaryAspect => {(visible_class,color_aspect),(visible_class,configuration_aspect)}
VisibleKind => {blue,yellow,red,white,long,short,round,circular,raised,lowered,even,uneven,cloud,smoke,dust,mist,shadow,sunlight,light,darkness}
TasteKind => {sweet,sour,salty,pungent,bitter,astringent}
GreatElement => {earth,water,fire,wind}
DerivedTangible => {smoothness,roughness,heaviness,lightness,cold,hunger,thirst}
TangibleConstituent => {earth,water,fire,wind,smoothness,roughness,heaviness,lightness,cold,hunger,thirst}
"""


In [22]:
studies[10] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[10]["tree"].pformat())
print("Derived SEM:", studies[10]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_visible_means_form_base=F
(S[SEM=<(TwofoldAnalysis(visible_meaning) & EnumeratedKinds(visible_meaning,twenty) & EnumeratedKinds(sound_meaning,eight) & EnumeratedKinds(taste_meaning,six) & EnumeratedKinds(odor_meaning,four) & ElevenfoldConstitution(tangible_meaning))>]
  (VISIBLE[SEM=<(TwofoldAnalysis(visible_meaning) & EnumeratedKinds(visible_meaning,twenty))>]
    (N[CASE='nom', GEN='n', K='visible', NUM='sg', SEM=<visible_meaning>]
      rūpam)
    (ADV[K='binary', SEM=<\x.TwofoldAnalysis(x)>] dvidhā)
    (ADV[K='enumerated', SEM=<\x.EnumeratedKinds(x,twenty)>]
      viṃśatidhā))
  (SOUND[SEM=<EnumeratedKinds(sound_meaning,eight)>]
    (N[CASE='nom', GEN='m', K='sound', NUM='sg', SEM=<sound_meaning>]
      śabdaḥ)
    (CONTRAST[] tu)
    (COUNT[CASE='nom', GEN='m', K='eight', NUM='sg', SEM=<\x.EnumeratedKinds(x,eight)>]
      aṣṭavidhaḥ))
  (TASTE[SEM=<EnumeratedKinds(taste_meaning,six)>]
    (N[CASE='nom', GEN='m', K='taste', NUM='sg', SEM=<t

# VAK_1.11 — The Continuity Called Avijñapti

Source: `VAK_1.11.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
vikṣiptācittakasyāpi yo 'nubandhaḥ śubhāśubhaḥ /
mahābhūtāny upādāya sa hy avijñaptir ucyate // 1.11 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| vikṣipta-acittakasya | genitive singular | of one otherwise directed, or without consciousness |
| api | particle | even; the scope is wider |
| yaḥ | nominative masculine singular relative pronoun | which continuity |
| anubandhaḥ | nominative masculine | the continuity |
| śubha-aśubhaḥ | nominative masculine | wholesome or unwholesome |
| mahābhūtāni | accusative plural | the Great Elements |
| upādāya | absolutive | depending on |
| saḥ | nominative masculine singular correlative pronoun | that same continuity |
| hi | particle | indeed; the naming |
| avijñaptiḥ | nominative feminine | the name |
| ucyate | passive | is called |

The compound is compressed. The commentary separates *vikṣiptacittakasya*
and *acittakasya*. *Api* also brings in one whose consciousness is present
and not otherwise directed.

**Grammar tokens.** Verse-order IAST with external sandhi opened. The
compressed compound remains one leaf; its two cases are interpreted with
the Bhāṣya. No consciousness noun is inserted as an overt token.

```text
# tokens: vikṣipta-acittakasya api yaḥ anubandhaḥ śubha-aśubhaḥ mahābhūtāni upādāya saḥ hi avijñaptiḥ ucyate
```

## 5. Translation

### Literal Translation

> The wholesome or unwholesome continuity which, depending on the
> Great Elements, belongs even to one whose consciousness is otherwise
> directed or absent: that indeed is called *avijñapti*.

### Bhāṣya-informed study translation

> The wholesome or unwholesome continuity that persists even when
> consciousness is directed elsewhere or absent depends on the Great
> Elements and is called *avijñapti*.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| avijñapti | *avijñapti* | the name stays |
| anubandha | continuity | glossed as stream |
| pravāha | stream | the commentary's word |
| śubha / aśubha | wholesome / unwholesome | *kuśala* / *akuśala* |
| citta | consciousness | otherwise directed, absent, or present |
| vikṣipta | otherwise directed | a different consciousness |
| acittaka | without consciousness | two attainments |
| api | even | includes the undistracted |
| mahābhūta | Great Element | the differentiator; defined next |
| upādāya | depending on | causal reading is Vaibhāṣika |
| prāpti-vāha | stream of acquisition | the similar, distinguished |
| vijñapti | *vijñapti* | informs another; the contrast |
| samādhi | absorption | one origin, in the summary |

In [23]:
fcfg_text = r"""% start S
# tokens: vikṣipta-acittakasya api yaḥ anubandhaḥ śubha-aśubhaḥ mahābhūtāni upādāya saḥ hi avijñaptiḥ ucyate
S[SEM=<?relative(?naming)>] -> RELATIVE[NUM=?n,GEN=?g,SEM=?relative] NAMING[NUM=?n,GEN=?g,SEM=?naming]
RELATIVE[NUM=?n,GEN=?g,SEM=<\Q.all x.((?stream(x) & ?ethical(x) & ?depend(x)) -> (?scope(x) & Q(x)))>] -> HOLDER[CASE=gen,NUM=sg,SEM=?scope] EVEN REL[CASE=nom,NUM=?n,GEN=?g] STREAM[CASE=nom,NUM=?n,GEN=?g,SEM=?stream] ETHICAL[CASE=nom,NUM=?n,GEN=?g,SEM=?ethical] DEPENDENCE[SEM=?depend]
DEPENDENCE[SEM=<?verb(?basis)>] -> ELEMENTS[CASE=?case,NUM=pl,SEM=?basis] DEPEND[FORM=absolutive,OBJCASE=?case,SEM=?verb]
NAMING[NUM=?n,GEN=?g,SEM=<?verb(?name)>] -> CORREL[CASE=nom,NUM=?n,GEN=?g] NAMINGMARK NAME[CASE=nom,NUM=?n,SEM=?name] CALLED[NUM=?n,PERSON=third,VOICE=passive,SEM=?verb]
HOLDER[CASE=gen,NUM=sg,SEM=<\x.(ScopeIncludes(x,otherwise_directed) & ScopeIncludes(x,absent))>] -> 'vikṣipta-acittakasya'
EVEN -> 'api'
REL[CASE=nom,NUM=sg,GEN=m] -> 'yaḥ'
STREAM[CASE=nom,NUM=sg,GEN=m,SEM=<\x.Stream(x)>] -> 'anubandhaḥ'
ETHICAL[CASE=nom,NUM=sg,GEN=m,SEM=<\x.(Wholesome(x) | Unwholesome(x))>] -> 'śubha-aśubhaḥ'
ELEMENTS[CASE=acc,NUM=pl,SEM=<great_elements>] -> 'mahābhūtāni'
DEPEND[FORM=absolutive,OBJCASE=acc,SEM=<\b.\x.DependsOn(x,b)>] -> 'upādāya'
CORREL[CASE=nom,NUM=sg,GEN=m] -> 'saḥ'
NAMINGMARK -> 'hi'
NAME[CASE=nom,NUM=sg,GEN=f,SEM=<\x.CalledAvijnapti(x)>] -> 'avijñaptiḥ'
CALLED[NUM=sg,PERSON=third,VOICE=passive,SEM=<\P.\x.P(x)>] -> 'ucyate'
"""

dependency_text = r"""'ucyate' -> 'saḥ' | 'hi' | 'avijñaptiḥ'
'saḥ' -> 'anubandhaḥ'
'anubandhaḥ' -> 'vikṣipta-acittakasya' | 'yaḥ' | 'śubha-aśubhaḥ' | 'upādāya'
'vikṣipta-acittakasya' -> 'api'
'upādāya' -> 'mahābhūtāni'
"""

sem_text = r"""# derived by the section 4 parse
all x.((Stream(x) & (Wholesome(x) | Unwholesome(x)) & DependsOn(x,great_elements)) -> ((ScopeIncludes(x,otherwise_directed) & ScopeIncludes(x,absent)) & CalledAvijnapti(x)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
all x.((Stream(x) & (Wholesome(x) | Unwholesome(x)) & DependsOn(x,great_elements)) -> ((ScopeIncludes(x,otherwise_directed) & ScopeIncludes(x,absent)) & CalledAvijnapti(x)))
# THEORY prose
# EXPECT prose T
all x.(CalledAvijnapti(x) -> (ScopeIncludes(x,undistracted) & FormAndAction(x) & -InformsAnother(x)))
all x.(CalledAvijnapti(x) -> (Origin(x,vijnapti) | Origin(x,absorption)))
exists x.(Stream(x) & Wholesome(x) & AcquisitionStream(x) & -DependsOn(x,great_elements) & -CalledAvijnapti(x))
# THEORY rejected_ethical_stream_alone_suffices
# EXPECT rejected_ethical_stream_alone_suffices F
all x.((Stream(x) & (Wholesome(x) | Unwholesome(x))) -> CalledAvijnapti(x))
# THEORY rejected_elemental_dependence_alone_suffices
# EXPECT rejected_elemental_dependence_alone_suffices F
all x.(DependsOn(x,great_elements) -> CalledAvijnapti(x))
# THEORY rejected_avijnapti_informs_another
# EXPECT rejected_avijnapti_informs_another F
all x.(CalledAvijnapti(x) -> InformsAnother(x))
# THEORY rejected_only_absent_consciousness
# EXPECT rejected_only_absent_consciousness F
all x.(CalledAvijnapti(x) -> -ScopeIncludes(x,undistracted))
"""

valuation_text = r"""great_elements => elemental_basis
otherwise_directed => other_consciousness_case
absent => no_consciousness_case
undistracted => undistracted_consciousness_case
vijnapti => manifestation_origin
absorption => absorption_origin
Stream => {wholesome_continuity,unwholesome_continuity,acquisition_case}
Wholesome => {wholesome_continuity,acquisition_case}
Unwholesome => {unwholesome_continuity}
DependsOn => {(wholesome_continuity,elemental_basis),(unwholesome_continuity,elemental_basis),(neutral_form_case,elemental_basis)}
CalledAvijnapti => {wholesome_continuity,unwholesome_continuity}
ScopeIncludes => {(wholesome_continuity,other_consciousness_case),(wholesome_continuity,no_consciousness_case),(wholesome_continuity,undistracted_consciousness_case),(unwholesome_continuity,other_consciousness_case),(unwholesome_continuity,no_consciousness_case),(unwholesome_continuity,undistracted_consciousness_case)}
AcquisitionStream => {acquisition_case}
FormAndAction => {wholesome_continuity,unwholesome_continuity}
InformsAnother => {}
Origin => {(wholesome_continuity,absorption_origin),(unwholesome_continuity,manifestation_origin)}
"""


In [24]:
studies[11] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[11]["tree"].pformat())
print("Derived SEM:", studies[11]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_ethical_stream_alone_suffices=F rejected_elemental_dependence_alone_suffices=F rejected_avijnapti_informs_another=F rejected_only_absent_consciousness=F
(S[SEM=<all x.((Stream(x) & (Wholesome(x) | Unwholesome(x)) & DependsOn(x,great_elements)) -> (ScopeIncludes(x,otherwise_directed) & ScopeIncludes(x,absent) & CalledAvijnapti(x)))>]
  (RELATIVE[GEN='m', NUM='sg', SEM=<\Q.all x.((Stream(x) & (Wholesome(x) | Unwholesome(x)) & DependsOn(x,great_elements)) -> (ScopeIncludes(x,otherwise_directed) & ScopeIncludes(x,absent) & Q(x)))>]
    (HOLDER[CASE='gen', NUM='sg', SEM=<\x.(ScopeIncludes(x,otherwise_directed) & ScopeIncludes(x,absent))>]
      vikṣipta-acittakasya)
    (EVEN[] api)
    (REL[CASE='nom', GEN='m', NUM='sg'] yaḥ)
    (STREAM[CASE='nom', GEN='m', NUM='sg', SEM=<\x.Stream(x)>]
      anubandhaḥ)
    (ETHICAL[CASE='nom', GEN='m', NUM='sg', SEM=<\x.(Wholesome(x) | Unwholesome(x))>]
      śubha-aśubhaḥ)
    (DEPENDENCE[SEM=<\x.DependsOn(x,gre

# VAK_1.12 — The Four Great Elements

Source: `VAK_1.12.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
bhūtāni pṛthivīdhātur aptejovāyudhātavaḥ /
dhṛtyādikarmasaṃsiddhāḥ kharasnehoṣṇateraṇāḥ // 1.12 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| bhūtāni | nominative neuter plural | the four Elements |
| pṛthivī-dhātuḥ | nominative masculine singular | the Earth Principle |
| ap-tejo-vāyu-dhātavaḥ | nominative masculine plural copulative compound | Water, Fire, Wind Principles |
| dhṛty-ādi-karma-saṃsiddhāḥ | nominative masculine plural compound | established in functions beginning with support |
| khara-sneha-uṣṇatā-īraṇāḥ | nominative masculine plural copulative predicate compound | hardness, cohesion, heat, impulsion |

*Dhātu* is the chapter word for Principle; *bhūta* names an Element. The four
Elements are identified as Earth, Water, Fire, and Wind Principles. In the
commentary’s mass-sentence, *skandha* means a mass, not a technical Base.

**Grammar tokens.** Verse-order IAST with external sandhi opened and
compound boundaries marked. Compounds remain whole leaves: the Bhāṣya
explains their members and order. No covert token is needed.

```text
# tokens: bhūtāni pṛthivī-dhātuḥ ap-tejo-vāyu-dhātavaḥ dhṛty-ādi-karma-saṃsiddhāḥ khara-sneha-uṣṇatā-īraṇāḥ
```

## 5. Translation

### Literal Translation

> The Elements are the Earth Principle and the Water, Fire, and Wind Principles.
> They are established in functions beginning with support: hardness,
> cohesion, heat, and impulsion.

### Bhāṣya-informed study translation

> The four Great Elements are the Earth, Water, Fire, and Wind Principles,
> each sustaining its own-character and derived Form. In that order, their
> functions are support, gathering, ripening, and spreading (growth and
> extension), while their own-characters are hardness, cohesion, heat, and
> impulsion.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| bhūta | Element | the four |
| mahābhūta | Great Element | magnitude, or gathering in masses |
| dhātu | Principle | here it names the principled determination |
| dhṛti | support | earth |
| saṃgraha | gathering | water |
| pakti | ripening | fire |
| vyūhana | spreading | wind; increase and extension |
| khara | hardness | own-character of earth |
| sneha | cohesion | own-character of water |
| uṣṇatā | heat | own-character of fire |
| īraṇā | impulsion | own-character of wind |
| svabhāva | own-character | the second list |
| laghutva | lightness | derived Form; not the Wind Principle |
| skandha | mass | in the commentary’s heap-sentence; not a technical Base |

In [25]:
fcfg_text = r"""% start S
# tokens: bhūtāni pṛthivī-dhātuḥ ap-tejo-vāyu-dhātavaḥ dhṛty-ādi-karma-saṃsiddhāḥ khara-sneha-uṣṇatā-īraṇāḥ
S[SEM=<?roster(?description)>] -> ROSTER[NUM=?n,GEN=?g,SEM=?roster] DESCRIPTION[NUM=?n,GEN=?g,SEM=?description]
ROSTER[NUM=pl,GEN=m,SEM=<?rest(?earth)(?class)>] -> CATEGORY[CASE=nom,NUM=pl,GEN=n,SEM=?class] EARTH[CASE=nom,NUM=sg,GEN=m,SEM=?earth] OTHERS[CASE=nom,NUM=pl,GEN=m,SEM=?rest]
DESCRIPTION[NUM=?n,GEN=?g,SEM=<?functions(?characters)>] -> FUNCTIONS[CASE=nom,NUM=?n,GEN=?g,SEM=?functions] CHARACTERS[CASE=nom,NUM=?n,GEN=?g,SEM=?characters]
CATEGORY[CASE=nom,NUM=pl,GEN=n,SEM=<\x.GreatElement(x)>] -> 'bhūtāni'
EARTH[CASE=nom,NUM=sg,GEN=m,SEM=<earth>] -> 'pṛthivī-dhātuḥ'
OTHERS[CASE=nom,NUM=pl,GEN=m,SEM=<\e.\C.\P.((all x.(C(x) <-> (x = e | x = water | x = fire | x = wind))) & P(e)(water)(fire)(wind))>] -> 'ap-tejo-vāyu-dhātavaḥ'
FUNCTIONS[CASE=nom,NUM=pl,GEN=m,SEM=<\P.\e.\w.\f.\v.(EstablishedInSupportEtc(e) & EstablishedInSupportEtc(w) & EstablishedInSupportEtc(f) & EstablishedInSupportEtc(v) & P(e)(w)(f)(v))>] -> 'dhṛty-ādi-karma-saṃsiddhāḥ'
CHARACTERS[CASE=nom,NUM=pl,GEN=m,SEM=<\e.\w.\f.\v.(OwnCharacter(e,hardness) & OwnCharacter(w,cohesion) & OwnCharacter(f,heat) & OwnCharacter(v,impulsion))>] -> 'khara-sneha-uṣṇatā-īraṇāḥ'
"""

dependency_text = r"""'bhūtāni' -> 'pṛthivī-dhātuḥ' | 'ap-tejo-vāyu-dhātavaḥ'
'ap-tejo-vāyu-dhātavaḥ' -> 'dhṛty-ādi-karma-saṃsiddhāḥ' | 'khara-sneha-uṣṇatā-īraṇāḥ'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(GreatElement(x) <-> (x = earth | x = water | x = fire | x = wind))) & (EstablishedInSupportEtc(earth) & EstablishedInSupportEtc(water) & EstablishedInSupportEtc(fire) & EstablishedInSupportEtc(wind) & OwnCharacter(earth,hardness) & OwnCharacter(water,cohesion) & OwnCharacter(fire,heat) & OwnCharacter(wind,impulsion)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(GreatElement(x) <-> (x = earth | x = water | x = fire | x = wind))) & (EstablishedInSupportEtc(earth) & EstablishedInSupportEtc(water) & EstablishedInSupportEtc(fire) & EstablishedInSupportEtc(wind) & OwnCharacter(earth,hardness) & OwnCharacter(water,cohesion) & OwnCharacter(fire,heat) & OwnCharacter(wind,impulsion)))
# THEORY prose
# EXPECT prose T
Function(earth,support) & Function(water,gathering) & Function(fire,ripening) & Function(wind,spreading)
all x.(GreatElement(x) -> (Principle(x) & HoldsOwnCharacter(x) & HoldsDerivedForm(x)))
Elaboration(spreading,growth) & Elaboration(spreading,extension)
DerivedForm(lightness) & -OwnCharacter(wind,lightness)
# THEORY rejected_function_equals_character
# EXPECT rejected_function_equals_character F
support = hardness
# THEORY rejected_wind_is_lightness
# EXPECT rejected_wind_is_lightness F
all x.((GreatElement(x) & x = wind) -> OwnCharacter(x,lightness))
# THEORY rejected_lightness_is_fifth_great_element
# EXPECT rejected_lightness_is_fifth_great_element F
exists x.(GreatElement(x) & x = lightness)
"""

valuation_text = r"""earth => earth_principle
water => water_principle
fire => fire_principle
wind => wind_principle
support => support_function
gathering => gathering_function
ripening => ripening_function
spreading => spreading_function
hardness => hardness_character
cohesion => cohesion_character
heat => heat_character
impulsion => impulsion_character
lightness => lightness_case
growth => growth_determination
extension => extension_determination
GreatElement => {earth_principle,water_principle,fire_principle,wind_principle}
Principle => {earth_principle,water_principle,fire_principle,wind_principle}
EstablishedInSupportEtc => {earth_principle,water_principle,fire_principle,wind_principle}
OwnCharacter => {(earth_principle,hardness_character),(water_principle,cohesion_character),(fire_principle,heat_character),(wind_principle,impulsion_character)}
Function => {(earth_principle,support_function),(water_principle,gathering_function),(fire_principle,ripening_function),(wind_principle,spreading_function)}
HoldsOwnCharacter => {earth_principle,water_principle,fire_principle,wind_principle}
HoldsDerivedForm => {earth_principle,water_principle,fire_principle,wind_principle}
DerivedForm => {lightness_case}
Elaboration => {(spreading_function,growth_determination),(spreading_function,extension_determination)}
"""


In [26]:
studies[12] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[12]["tree"].pformat())
print("Derived SEM:", studies[12]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_function_equals_character=F rejected_wind_is_lightness=F rejected_lightness_is_fifth_great_element=F
(S[SEM=<(all x.(GreatElement(x) <-> ((x = earth) | (x = water) | (x = fire) | (x = wind))) & EstablishedInSupportEtc(earth) & EstablishedInSupportEtc(water) & EstablishedInSupportEtc(fire) & EstablishedInSupportEtc(wind) & OwnCharacter(earth,hardness) & OwnCharacter(water,cohesion) & OwnCharacter(fire,heat) & OwnCharacter(wind,impulsion))>]
  (ROSTER[GEN='m', NUM='pl', SEM=<\P.(all x.(GreatElement(x) <-> ((x = earth) | (x = water) | (x = fire) | (x = wind))) & P(earth,water,fire,wind))>]
    (CATEGORY[CASE='nom', GEN='n', NUM='pl', SEM=<\x.GreatElement(x)>]
      bhūtāni)
    (EARTH[CASE='nom', GEN='m', NUM='sg', SEM=<earth>]
      pṛthivī-dhātuḥ)
    (OTHERS[CASE='nom', GEN='m', NUM='pl', SEM=<\e C P.(all x.(C(x) <-> ((x = e) | (x = water) | (x = fire) | (x = wind))) & P(e,water,fire,wind))>]
      ap-tejo-vāyu-dhātavaḥ))
  (DESCRIPTION[GEN='m',

# VAK_1.13 — Worldly Designation and the Principle

Source: `VAK_1.13.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
pṛthivī varṇasaṃsthānam ucyate lokasaṃjñayā /
āpas tejaś ca vāyus tu dhātur eva tathāpi ca // 1.13 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| pṛthivī | nominative feminine singular | the worldly name |
| varṇa-saṃsthānam | nominative neuter singular collective compound | color and configuration |
| ucyate | third-person singular present passive | is called |
| loka-saṃjñayā | instrumental feminine singular | by worldly designation |
| āpaḥ | nominative feminine plural | water |
| tejaḥ | nominative neuter singular | fire |
| ca | conjunction | and; extends the first predication |
| vāyuḥ | nominative masculine singular | wind |
| tu | particle | the contrast |
| dhātuḥ | nominative masculine singular | the Principle |
| eva | restrictive particle | itself |
| tathā | adverb | in that way; resumes visible designation |
| api | additive particle | also |
| ca | conjunction | and |

*Loka-saṃjñā* is designation in ordinary use. It is not the reflection
defined in the next verse. *Varṇa* and *saṃsthāna* are the pair already
counted under visible Form.

**Grammar tokens.** Verse-order IAST with sandhi opened and compound
boundaries marked. The repeated *ca* remains in both positions. Elliptical
predication is resolved in the productions, not by invented verse words.

```text
# tokens: pṛthivī varṇa-saṃsthānam ucyate loka-saṃjñayā āpaḥ tejaḥ ca vāyuḥ tu dhātuḥ eva tathā api ca
```

## 5. Translation

### Literal Translation

> By worldly designation, color and configuration are called earth. Water
> and fire too. Wind, however, is the Principle itself — and yet also in
> that way.

### Bhāṣya-informed study translation

> By worldly designation, earth, water, and fire are color and
> configuration; wind, however, is also the Principle itself, though in
> that way too it is called from color and configuration.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| loka-saṃjñā | worldly designation | not the reflection of 1.14 |
| varṇa | color | what is shown |
| saṃsthāna | configuration | what is shown |
| pṛthivī, āpas, tejas | earth, water, fire | ordinary names |
| vāyu | wind | the exception |
| dhātu | Principle | the chapter word |
| vāyu-dhātu | Wind Principle | impulsion, from 1.12 |
| tathāpi ca | and yet also | the visible storm |
| rūpa-skandha | Form Base | the commentary's question |
| raupaṇa | being affected | the reason offered |
| avijñapti | *avijñapti* | the hard member; the name stays |
| vijñapti | *vijñapti* | informs another; the contrast |

In [27]:
fcfg_text = r"""% start S
# tokens: pṛthivī varṇa-saṃsthānam ucyate loka-saṃjñayā āpaḥ tejaḥ ca vāyuḥ tu dhātuḥ eva tathā api ca
S[SEM=<(?first & ?others & ?wind)>] -> FIRST[SEM=?first] EXTENSION[SEM=?others] WINDCLAUSE[SEM=?wind]
FIRST[SEM=<?mode(?verb(?label)(?target))>] -> EARTH[CASE=nom,NUM=sg,SEM=?label] VISIBLE[CASE=nom,NUM=?n,SEM=?target] CALLED[NUM=?n,PERSON=third,VOICE=passive,SEM=?verb] WORLDLY[CASE=ins,SEM=?mode]
EXTENSION[SEM=<(WorldlyDesignates(?water,visible_complex) & WorldlyDesignates(?fire,visible_complex))>] -> WATER[CASE=nom,NUM=pl,SEM=?water] FIRE[CASE=nom,NUM=sg,SEM=?fire] AND
WINDCLAUSE[SEM=<(?principle(?label) & ?also(?label))>] -> WIND[CASE=nom,NUM=?n,GEN=?g,SEM=?label] CONTRAST PRINCIPLE[CASE=nom,NUM=?n,GEN=?g,SEM=?principle] ONLY ALSO[SEM=?also]
ALSO[SEM=<\l.WorldlyDesignates(l,visible_complex)>] -> THUS ADDITIVE AND
EARTH[CASE=nom,NUM=sg,GEN=f,SEM=<earth_name>] -> 'pṛthivī'
VISIBLE[CASE=nom,NUM=sg,GEN=n,SEM=<visible_complex>] -> 'varṇa-saṃsthānam'
CALLED[NUM=sg,PERSON=third,VOICE=passive,SEM=<\l.\v.WorldlyDesignates(l,v)>] -> 'ucyate'
WORLDLY[CASE=ins,NUM=sg,SEM=<\P.P>] -> 'loka-saṃjñayā'
WATER[CASE=nom,NUM=pl,GEN=f,SEM=<water_name>] -> 'āpaḥ'
FIRE[CASE=nom,NUM=sg,GEN=n,SEM=<fire_name>] -> 'tejaḥ'
AND -> 'ca'
WIND[CASE=nom,NUM=sg,GEN=m,SEM=<wind_name>] -> 'vāyuḥ'
CONTRAST -> 'tu'
PRINCIPLE[CASE=nom,NUM=sg,GEN=m,SEM=<\l.WorldlyDesignates(l,wind_principle)>] -> 'dhātuḥ'
ONLY -> 'eva'
THUS -> 'tathā'
ADDITIVE -> 'api'
"""

dependency_text = r"""'ucyate' -> 'pṛthivī' | 'varṇa-saṃsthānam' | 'loka-saṃjñayā' | 'āpaḥ' | 'tejaḥ'
'tejaḥ' -> 'ca'
'dhātuḥ' -> 'vāyuḥ' | 'tu' | 'eva' | 'tathā'
'tathā' -> 'api' | 'ca'
"""

sem_text = r"""# derived by the section 4 parse
(WorldlyDesignates(earth_name,visible_complex) & (WorldlyDesignates(water_name,visible_complex) & WorldlyDesignates(fire_name,visible_complex)) & (WorldlyDesignates(wind_name,wind_principle) & WorldlyDesignates(wind_name,visible_complex)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(WorldlyDesignates(earth_name,visible_complex) & (WorldlyDesignates(water_name,visible_complex) & WorldlyDesignates(fire_name,visible_complex)) & (WorldlyDesignates(wind_name,wind_principle) & WorldlyDesignates(wind_name,visible_complex)))
# THEORY prose
# EXPECT prose T
-(visible_complex = wind_principle)
FormBaseMember(continuity) & -FormBaseMember(eye_cognition)
OriginatingManifestation(continuity,manifestation) & OriginatingElementSupport(continuity,elemental_support)
Ceased(manifestation,later_case) & Ceased(elemental_support,later_case) & Continues(continuity,later_case)
# THEORY rejected_wind_only_names_principle
# EXPECT rejected_wind_only_names_principle F
all x.(WorldlyDesignates(wind_name,x) -> x = wind_principle)
# THEORY rejected_all_names_designate_principles_here
# EXPECT rejected_all_names_designate_principles_here F
WorldlyDesignates(earth_name,earth_principle) & WorldlyDesignates(water_name,water_principle) & WorldlyDesignates(fire_name,fire_principle)
# THEORY rejected_shadow_like_cessation
# EXPECT rejected_shadow_like_cessation F
all x.all y.all t.((OriginatingManifestation(x,y) & Ceased(y,t)) -> -Continues(x,t))
# THEORY rejected_originating_support_must_persist
# EXPECT rejected_originating_support_must_persist F
all x.all y.all t.((OriginatingElementSupport(x,y) & Continues(x,t)) -> -Ceased(y,t))
"""

valuation_text = r"""earth_name => earth_label
water_name => water_label
fire_name => fire_label
wind_name => wind_label
visible_complex => color_configuration_kind
earth_principle => earth_determination
water_principle => water_determination
fire_principle => fire_determination
wind_principle => wind_determination
continuity => avijnapti_case
manifestation => originating_manifestation_case
elemental_support => originating_element_case
eye_cognition => eye_cognition_case
later_case => later_stage
WorldlyDesignates => {(earth_label,color_configuration_kind),(water_label,color_configuration_kind),(fire_label,color_configuration_kind),(wind_label,wind_determination),(wind_label,color_configuration_kind)}
FormBaseMember => {avijnapti_case}
OriginatingManifestation => {(avijnapti_case,originating_manifestation_case)}
OriginatingElementSupport => {(avijnapti_case,originating_element_case)}
Ceased => {(originating_manifestation_case,later_stage),(originating_element_case,later_stage)}
Continues => {(avijnapti_case,later_stage)}
"""


In [28]:
studies[13] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[13]["tree"].pformat())
print("Derived SEM:", studies[13]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_wind_only_names_principle=F rejected_all_names_designate_principles_here=F rejected_shadow_like_cessation=F rejected_originating_support_must_persist=F
(S[SEM=<(WorldlyDesignates(earth_name,visible_complex) & WorldlyDesignates(water_name,visible_complex) & WorldlyDesignates(fire_name,visible_complex) & WorldlyDesignates(wind_name,wind_principle) & WorldlyDesignates(wind_name,visible_complex))>]
  (FIRST[SEM=<WorldlyDesignates(earth_name,visible_complex)>]
    (EARTH[CASE='nom', GEN='f', NUM='sg', SEM=<earth_name>] pṛthivī)
    (VISIBLE[CASE='nom', GEN='n', NUM='sg', SEM=<visible_complex>]
      varṇa-saṃsthānam)
    (CALLED[NUM='sg', PERSON='third', SEM=<\l v.WorldlyDesignates(l,v)>, VOICE='passive']
      ucyate)
    (WORLDLY[CASE='ins', NUM='sg', SEM=<\P.P>] loka-saṃjñayā))
  (EXTENSION[SEM=<(WorldlyDesignates(water_name,visible_complex) & WorldlyDesignates(fire_name,visible_complex))>]
    (WATER[CASE='nom', GEN='f', NUM='pl', SEM=<water_name

# VAK_1.14 — The Same Ten, Feeling, and Reflection

Source: `VAK_1.14.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
indriyārthās ta eveṣṭā daśāyatanadhātavaḥ /
vedanānubhavaḥ saṃjñā nimittodgrahaṇātmikā // 1.14 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| indriya-arthāḥ | nominative masculine plural copulative compound | Faculties and meanings; the ten already stated |
| te | nominative masculine plural pronoun | these same ones |
| eva | restrictive particle | these very ones, not another inventory |
| iṣṭāḥ | nominative masculine plural participle | accepted |
| daśa-āyatana-dhātavaḥ | nominative masculine plural compound | ten Essences and ten Principles |
| vedanā | nominative feminine singular | Feeling |
| anubhavaḥ | nominative masculine singular predicate noun | undergoing |
| saṃjñā | nominative feminine singular | Reflection |
| nimitta-udgrahaṇa-ātmikā | nominative feminine singular adjective | having taking up a mark as its nature |

*Indriya* is Faculty throughout this study. Here the five Faculties are counted
with their meanings. *Ārtha* is meaning; *āyatana* is Essence; *dhātu* is
Principle; *saṃjñā* is Reflection. The reflective mark is the commentary’s
example, not the headword.

**Grammar tokens.** Verse-order IAST with sandhi opened and compound
boundaries marked. The tenfold Essence–Principle compound stays one leaf;
the Bhāṣya resolves its distributed count. No covert copula is inserted.

```text
# tokens: indriya-arthāḥ te eva iṣṭāḥ daśa-āyatana-dhātavaḥ vedanā anubhavaḥ saṃjñā nimitta-udgrahaṇa-ātmikā
```

## 5. Translation

### Literal Translation

> These very Faculties and meanings are accepted as the ten Essences and the
> ten Principles. Feeling is undergoing. Reflection has the taking up of the
> mark as its nature.

### Bhāṣya-informed study translation

> The five Faculties and five meanings are accepted as ten Essences and the
> same ten Principles. Feeling is experience; Reflection has taking up a
> mark as its nature.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| indriya | Faculty | one of the five |
| artha | meaning | the five |
| skandha | Base | the Form Base, closed here |
| āyatana | Essence | ten |
| dhātu | Principle | the same ten |
| te eva | these very ones | identity across the arrangements |
| vedanā | feeling | undergoing |
| anubhava | undergoing | three tones |
| saṃsparśa | contact | the six groups |
| saṃjñā | Reflection | not worldly designation |
| nimitta | mark | what is taken up |
| udgrahaṇa | taking up | the operation |
| manas | Mind | Mind-contact; the sixth |

In [29]:
fcfg_text = r"""% start S
# tokens: indriya-arthāḥ te eva iṣṭāḥ daśa-āyatana-dhātavaḥ vedanā anubhavaḥ saṃjñā nimitta-udgrahaṇa-ātmikā
S[SEM=<(?classification & ?feeling & ?reflection)>] -> CLASSIFICATION[SEM=?classification] FEELINGDEF[SEM=?feeling] REFLECTIONDEF[SEM=?reflection]
CLASSIFICATION[SEM=<?accepted(?members)(?slots)>] -> MEMBERS[CASE=nom,NUM=?n,GEN=?g,SEM=?members] SAME[CASE=nom,NUM=?n,GEN=?g] ONLY ACCEPTED[CASE=nom,NUM=?n,GEN=?g,SEM=?accepted] SLOTS[CASE=nom,NUM=?n,GEN=?g,SEM=?slots]
FEELINGDEF[SEM=<all x.(?feeling(x) <-> ?experience(x))>] -> FEELING[CASE=nom,NUM=?n,SEM=?feeling] EXPERIENCE[CASE=nom,NUM=?n,SEM=?experience]
REFLECTIONDEF[SEM=<all x.(?reflection(x) <-> ?taking(x))>] -> REFLECTION[CASE=nom,NUM=?n,GEN=?g,SEM=?reflection] TAKING[CASE=nom,NUM=?n,GEN=?g,SEM=?taking]
MEMBERS[CASE=nom,NUM=pl,GEN=m,SEM=<\x.FacultyOrMeaning(x)>] -> 'indriya-arthāḥ'
SAME[CASE=nom,NUM=pl,GEN=m] -> 'te'
ONLY -> 'eva'
ACCEPTED[CASE=nom,NUM=pl,GEN=m,SEM=<\P.\Q.all x.(P(x) <-> Q(x))>] -> 'iṣṭāḥ'
SLOTS[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(TenFormEssenceMember(x) & TenFormPrincipleMember(x))>] -> 'daśa-āyatana-dhātavaḥ'
FEELING[CASE=nom,NUM=sg,GEN=f,SEM=<\x.Feeling(x)>] -> 'vedanā'
EXPERIENCE[CASE=nom,NUM=sg,GEN=m,SEM=<\x.Experience(x)>] -> 'anubhavaḥ'
REFLECTION[CASE=nom,NUM=sg,GEN=f,SEM=<\x.Reflection(x)>] -> 'saṃjñā'
TAKING[CASE=nom,NUM=sg,GEN=f,SEM=<\x.MarkTaking(x)>] -> 'nimitta-udgrahaṇa-ātmikā'
"""

dependency_text = r"""'iṣṭāḥ' -> 'indriya-arthāḥ' | 'te' | 'daśa-āyatana-dhātavaḥ'
'te' -> 'eva'
'vedanā' -> 'anubhavaḥ'
'saṃjñā' -> 'nimitta-udgrahaṇa-ātmikā'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(FacultyOrMeaning(x) <-> (TenFormEssenceMember(x) & TenFormPrincipleMember(x)))) & (all x.(Feeling(x) <-> Experience(x))) & (all x.(Reflection(x) <-> MarkTaking(x))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(FacultyOrMeaning(x) <-> (TenFormEssenceMember(x) & TenFormPrincipleMember(x)))) & (all x.(Feeling(x) <-> Experience(x))) & (all x.(Reflection(x) <-> MarkTaking(x))))
# THEORY prose
# EXPECT prose T
all x.(FacultyOrMeaning(x) <-> (SensoryFaculty(x) | SensoryMeaning(x)))
all x.(TenFormEssenceMember(x) <-> FacultyOrMeaning(x))
all x.(TenFormPrincipleMember(x) <-> FacultyOrMeaning(x))
FormBaseMember(avijnapti) & -FacultyOrMeaning(avijnapti)
all x.(Feeling(x) -> exists t.(FeelingTone(t) & HasTone(x,t)))
all x.(FeelingContactGroup(x) <-> ReflectionContactGroup(x))
Reflection(reflection_case) & TakesMark(reflection_case,blue_mark)
# THEORY rejected_avijnapti_in_these_ten
# EXPECT rejected_avijnapti_in_these_ten F
exists x.(x = avijnapti & TenFormEssenceMember(x))
# THEORY rejected_reflection_is_feeling
# EXPECT rejected_reflection_is_feeling F
all x.(Reflection(x) <-> Feeling(x))
# THEORY rejected_six_groups_are_three_tones
# EXPECT rejected_six_groups_are_three_tones F
all x.(FeelingContactGroup(x) <-> FeelingTone(x))
"""

valuation_text = r"""avijnapti => avijnapti_case
reflection_case => blue_reflection
blue_mark => blue
SensoryFaculty => {eye,ear,nose,tongue,body}
SensoryMeaning => {visible,sound,odor,taste,tangible}
FacultyOrMeaning => {eye,ear,nose,tongue,body,visible,sound,odor,taste,tangible}
TenFormEssenceMember => {eye,ear,nose,tongue,body,visible,sound,odor,taste,tangible}
TenFormPrincipleMember => {eye,ear,nose,tongue,body,visible,sound,odor,taste,tangible}
FormBaseMember => {eye,ear,nose,tongue,body,visible,sound,odor,taste,tangible,avijnapti_case}
Feeling => {pleasant_case,painful_case,neutral_case}
Experience => {pleasant_case,painful_case,neutral_case}
Reflection => {blue_reflection}
MarkTaking => {blue_reflection}
FeelingTone => {pleasant,painful,neither}
HasTone => {(pleasant_case,pleasant),(painful_case,painful),(neutral_case,neither)}
FeelingContactGroup => {eye_contact,ear_contact,nose_contact,tongue_contact,body_contact,mind_contact}
ReflectionContactGroup => {eye_contact,ear_contact,nose_contact,tongue_contact,body_contact,mind_contact}
TakesMark => {(blue_reflection,blue)}
"""


In [30]:
studies[14] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[14]["tree"].pformat())
print("Derived SEM:", studies[14]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_avijnapti_in_these_ten=F rejected_reflection_is_feeling=F rejected_six_groups_are_three_tones=F
(S[SEM=<(all x.(FacultyOrMeaning(x) <-> (TenFormEssenceMember(x) & TenFormPrincipleMember(x))) & all x.(Feeling(x) <-> Experience(x)) & all x.(Reflection(x) <-> MarkTaking(x)))>]
  (CLASSIFICATION[SEM=<all x.(FacultyOrMeaning(x) <-> (TenFormEssenceMember(x) & TenFormPrincipleMember(x)))>]
    (MEMBERS[CASE='nom', GEN='m', NUM='pl', SEM=<\x.FacultyOrMeaning(x)>]
      indriya-arthāḥ)
    (SAME[CASE='nom', GEN='m', NUM='pl'] te)
    (ONLY[] eva)
    (ACCEPTED[CASE='nom', GEN='m', NUM='pl', SEM=<\P Q.all x.(P(x) <-> Q(x))>]
      iṣṭāḥ)
    (SLOTS[CASE='nom', GEN='m', NUM='pl', SEM=<\x.(TenFormEssenceMember(x) & TenFormPrincipleMember(x))>]
      daśa-āyatana-dhātavaḥ))
  (FEELINGDEF[SEM=<all x.(Feeling(x) <-> Experience(x))>]
    (FEELING[CASE='nom', GEN='f', NUM='sg', SEM=<\x.Feeling(x)>]
      vedanā)
    (EXPERIENCE[CASE='nom', GEN='m', NUM='sg', SEM

# VAK_1.15 — Formations, and the Dharma Essence and Principle

Source: `VAK_1.15.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
caturbhyo 'nye tu saṃskāraskandhaḥ ete punas trayaḥ /
dharmāyatanadhātvākhyāḥ sahāvijñaptyasaṃskṛtaiḥ // 1.15 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| caturbhyaḥ | ablative plural numeral | apart from the four Bases |
| anye | nominative masculine plural | the others |
| tu | particle | the remainder |
| saṃskāra-skandhaḥ | nominative masculine singular | one Formations Base |
| ete | nominative masculine plural pronoun | these |
| punaḥ | particle | resumes them under the further name |
| trayaḥ | nominative masculine numeral | three |
| dharma-āyatana-dhātu-ākhyāḥ | nominative masculine plural compound | bearing the names Dharma Essence and Dharma Principle |
| saha | governs instrumental | together with |
| avijñaptyasaṃskṛtaiḥ | instrumental plural | the two accompaniments |

*Saṃskāra* is formation. *Asaṃskṛta* is unconditioned, the negation of
conditioned. *Āyatana* is Essence. *Dhātu* is Principle. *Ākhyā* is the name.

**Grammar tokens.** Verse-order IAST with external sandhi opened and
compound boundaries marked. *Saha* is separated from the instrumental
compound it governs. The four and three are resolved with the Bhāṣya;
their names are not inserted into the verse.

```text
# tokens: caturbhyaḥ anye tu saṃskāra-skandhaḥ ete punaḥ trayaḥ dharma-āyatana-dhātu-ākhyāḥ saha avijñapti-asaṃskṛtaiḥ
```

## 5. Translation

### Literal Translation

> The others, apart from the four, are the Formations Base. These three,
> again, together with *avijñapti* and the unconditioned, are called the
> Dharma Essence and the Dharma Principle.

### Bhāṣya-informed study translation

> Apart from the four, the others are the Formations Base. These three—
> Feeling, Reflection, and Formations—together with *avijñapti* and the
> unconditioned are also named the Dharma Essence and the Dharma Principle.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| caturbhyaḥ | apart from the four | Form, feeling, Reflection, Cognition Base |
| saṃskāra | formation | the remainder |
| skandha | Base | one Formations Base |
| cetanā | volition | predominant, not exhaustive |
| abhisaṃskaroti | actively forms | the conditioned |
| caitasika | associated factor | must be included |
| viprayukta | dissociated | must be included |
| ete trayaḥ | these three | feeling, Reflection, formations |
| dharma-āyatana | Dharma Essence | the name |
| dharma-dhātu | Dharma Principle | the name |
| avijñapti | *avijñapti* | remains Form |
| asaṃskṛta | unconditioned | no Base |
| sapta dravyāṇi | seven | the commentary's count |

In [31]:
fcfg_text = r"""% start S
# tokens: caturbhyaḥ anye tu saṃskāra-skandhaḥ ete punaḥ trayaḥ dharma-āyatana-dhātu-ākhyāḥ saha avijñapti-asaṃskṛtaiḥ
S[SEM=<(?remainder & ?classification)>] -> REMAINDER[SEM=?remainder] CLASSIFICATION[SEM=?classification]
REMAINDER[SEM=<?other(?four)(?base)>] -> FOUR[CASE=abl,NUM=pl,SEM=?four] OTHER[CASE=nom,NUM=pl,GEN=m,SEM=?other] CONTRAST BASE[CASE=nom,NUM=sg,GEN=m,SEM=?base]
CLASSIFICATION[SEM=<?names(?three)(?accomp)>] -> THESE[CASE=nom,NUM=?n,GEN=?g] AGAIN THREE[CASE=nom,NUM=?n,GEN=?g,SEM=?three] NAMES[CASE=nom,NUM=?n,GEN=?g,SEM=?names] ACCOMPANIMENT[SEM=?accomp]
ACCOMPANIMENT[SEM=?members] -> WITH[OBJCASE=?case] ACCOMP[CASE=?case,NUM=pl,SEM=?members]
FOUR[CASE=abl,NUM=pl,SEM=<\x.OtherFourBaseMember(x)>] -> 'caturbhyaḥ'
OTHER[CASE=nom,NUM=pl,GEN=m,SEM=<\P.\Q.all x.(Conditioned(x) -> (Q(x) <-> -P(x)))>] -> 'anye'
CONTRAST -> 'tu'
BASE[CASE=nom,NUM=sg,GEN=m,SEM=<\x.FormationsBaseMember(x)>] -> 'saṃskāra-skandhaḥ'
THESE[CASE=nom,NUM=pl,GEN=m] -> 'ete'
AGAIN -> 'punaḥ'
THREE[CASE=nom,NUM=pl,GEN=m,SEM=<\x.ThreeBaseMember(x)>] -> 'trayaḥ'
NAMES[CASE=nom,NUM=pl,GEN=m,SEM=<\P.\Q.all x.((DharmaEssenceMember(x) <-> (P(x) | Q(x))) & (DharmaPrincipleMember(x) <-> (P(x) | Q(x))))>] -> 'dharma-āyatana-dhātu-ākhyāḥ'
WITH[OBJCASE=ins] -> 'saha'
ACCOMP[CASE=ins,NUM=pl,SEM=<\x.(Avijnapti(x) | Unconditioned(x))>] -> 'avijñapti-asaṃskṛtaiḥ'
"""

dependency_text = r"""'saṃskāra-skandhaḥ' -> 'anye'
'anye' -> 'caturbhyaḥ' | 'tu'
'dharma-āyatana-dhātu-ākhyāḥ' -> 'ete' | 'saha'
'ete' -> 'punaḥ' | 'trayaḥ'
'saha' -> 'avijñapti-asaṃskṛtaiḥ'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(Conditioned(x) -> (FormationsBaseMember(x) <-> -OtherFourBaseMember(x)))) & (all x.((DharmaEssenceMember(x) <-> (ThreeBaseMember(x) | (Avijnapti(x) | Unconditioned(x)))) & (DharmaPrincipleMember(x) <-> (ThreeBaseMember(x) | (Avijnapti(x) | Unconditioned(x)))))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(Conditioned(x) -> (FormationsBaseMember(x) <-> -OtherFourBaseMember(x)))) & (all x.((DharmaEssenceMember(x) <-> (ThreeBaseMember(x) | (Avijnapti(x) | Unconditioned(x)))) & (DharmaPrincipleMember(x) <-> (ThreeBaseMember(x) | (Avijnapti(x) | Unconditioned(x)))))))
# THEORY prose
# EXPECT prose T
all x.(ThreeBaseMember(x) <-> (FeelingBaseMember(x) | ReflectionBaseMember(x) | FormationsBaseMember(x)))
all x.(OtherFourBaseMember(x) <-> (FormBaseMember(x) | FeelingBaseMember(x) | ReflectionBaseMember(x) | CognitionBaseMember(x)))
all x.(FormationsBaseMember(x) -> (Conditioned(x) & BaseMember(x)))
all x.(Unconditioned(x) -> (-BaseMember(x) & -Conditioned(x)))
all x.(Avijnapti(x) -> (FormBaseMember(x) & BaseMember(x)))
all x.(Volition(x) -> FormationsBaseMember(x))
exists x.(AssociatedOther(x) & FormationsBaseMember(x) & -Volition(x))
exists x.(DissociatedFormation(x) & FormationsBaseMember(x) & -Volition(x))
exists x.(FormationsBaseMember(x) & Pure(x))
# THEORY rejected_formations_are_only_volition
# EXPECT rejected_formations_are_only_volition F
all x.(FormationsBaseMember(x) -> Volition(x))
# THEORY rejected_unrestricted_remainder
# EXPECT rejected_unrestricted_remainder F
all x.(FormationsBaseMember(x) <-> -OtherFourBaseMember(x))
# THEORY rejected_dharma_members_all_have_a_base
# EXPECT rejected_dharma_members_all_have_a_base F
all x.(DharmaEssenceMember(x) -> BaseMember(x))
# THEORY rejected_cognition_is_one_of_these_three
# EXPECT rejected_cognition_is_one_of_these_three F
all x.(CognitionBaseMember(x) -> ThreeBaseMember(x))
"""

valuation_text = r"""Conditioned => {ordinary_form,avijnapti_case,feeling_case,reflection_case,cognition_case,volition_case,associated_case,dissociated_case,pure_path_case}
FormBaseMember => {ordinary_form,avijnapti_case}
FeelingBaseMember => {feeling_case}
ReflectionBaseMember => {reflection_case}
CognitionBaseMember => {cognition_case}
OtherFourBaseMember => {ordinary_form,avijnapti_case,feeling_case,reflection_case,cognition_case}
FormationsBaseMember => {volition_case,associated_case,dissociated_case,pure_path_case}
ThreeBaseMember => {feeling_case,reflection_case,volition_case,associated_case,dissociated_case,pure_path_case}
Avijnapti => {avijnapti_case}
Unconditioned => {space_case,discrimination_cessation,nonarising_cessation}
BaseMember => {ordinary_form,avijnapti_case,feeling_case,reflection_case,cognition_case,volition_case,associated_case,dissociated_case,pure_path_case}
DharmaEssenceMember => {feeling_case,reflection_case,volition_case,associated_case,dissociated_case,pure_path_case,avijnapti_case,space_case,discrimination_cessation,nonarising_cessation}
DharmaPrincipleMember => {feeling_case,reflection_case,volition_case,associated_case,dissociated_case,pure_path_case,avijnapti_case,space_case,discrimination_cessation,nonarising_cessation}
Volition => {volition_case}
AssociatedOther => {associated_case}
DissociatedFormation => {dissociated_case}
Pure => {pure_path_case}
"""


In [32]:
studies[15] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[15]["tree"].pformat())
print("Derived SEM:", studies[15]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_formations_are_only_volition=F rejected_unrestricted_remainder=F rejected_dharma_members_all_have_a_base=F rejected_cognition_is_one_of_these_three=F
(S[SEM=<(all x.(Conditioned(x) -> (FormationsBaseMember(x) <-> -OtherFourBaseMember(x))) & all x.((DharmaEssenceMember(x) <-> (ThreeBaseMember(x) | Avijnapti(x) | Unconditioned(x))) & (DharmaPrincipleMember(x) <-> (ThreeBaseMember(x) | Avijnapti(x) | Unconditioned(x)))))>]
  (REMAINDER[SEM=<all x.(Conditioned(x) -> (FormationsBaseMember(x) <-> -OtherFourBaseMember(x)))>]
    (FOUR[CASE='abl', NUM='pl', SEM=<\x.OtherFourBaseMember(x)>]
      caturbhyaḥ)
    (OTHER[CASE='nom', GEN='m', NUM='pl', SEM=<\P Q.all x.(Conditioned(x) -> (Q(x) <-> -P(x)))>]
      anye)
    (CONTRAST[] tu)
    (BASE[CASE='nom', GEN='m', NUM='sg', SEM=<\x.FormationsBaseMember(x)>]
      saṃskāra-skandhaḥ))
  (CLASSIFICATION[SEM=<all x.((DharmaEssenceMember(x) <-> (ThreeBaseMember(x) | Avijnapti(x) | Unconditioned(x))) & (Dharm

# VAK_1.16 — The Cognition Base

Source: `VAK_1.16.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
vijñānaṃ prativijñaptiḥ mana āyatanaṃ ca tat /
dhātavaḥ sapta ca matāḥ ṣaḍvijñānāny atho manaḥ // 1.16 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| vijñānam | nominative neuter | Cognition |
| prativijñaptiḥ | nominative feminine | respective apprehension |
| mana āyatanam | nominative | Mind Essence |
| tat | nominative | that |
| dhātavaḥ | nominative plural | Principles |
| sapta matāḥ | numeral and participle | seven are accepted |
| ṣaṭ vijñānāni | numeral and plural | six cognitions |
| atho manaḥ | connective and nominative | and Mind |

**Grammar tokens.** Verse-order IAST with sandhi opened; combined table
rows are separated below. *Mana āyatanam* is kept as the Mind-Essence
compound, normalized as `mana-āyatanam`. Final *manaḥ* is read as Mind
Principle by the Bhāṣya. No supplied leaf is required.

```text
# tokens: vijñānam prativijñaptiḥ mana-āyatanam ca tat dhātavaḥ sapta ca matāḥ ṣaṭ-vijñānāni atho manaḥ
```

## 5. Translation

### Literal Translation

> Cognition is respective apprehension. And that is the Mind Essence. Seven
> principles are accepted: the six Cognitions, and Mind.

### Bhāṣya-informed study translation

> The Cognition Base is respective apprehension of each Condition. It is
> also the Mind Essence. In the Principle arrangement, it is counted as
> seven Principles: the six Cognition Principles and the Mind Principle.

## 7. Technical Vocabulary

| Sanskrit | Rendering |
|---|---|
| vijñāna-skandha | Cognition Base |
| prativijñapti | respective apprehension |
| mana-āyatana | Mind Essence |
| manovijñānadhātu | Mind-Cognition Principle |
| manodhātu | Mind Principle |
| upalabdhi | apprehension |

In [33]:
fcfg_text = r"""% start S
# tokens: vijñānam prativijñaptiḥ mana-āyatanam ca tat dhātavaḥ sapta ca matāḥ ṣaṭ-vijñānāni atho manaḥ
S[SEM=<(?d & ?e & ?p)>] -> DEFINITION[SEM=?d] ESSENCE[SEM=?e] PRINCIPLES[SEM=?p]
DEFINITION[SEM=<?pred(?base)>] -> BASE[CASE=nom,NUM=?n,SEM=?base] APPREHENSION[CASE=nom,NUM=?n,SEM=?pred]
ESSENCE[SEM=<?essence(?base)>] -> ESSENCEHEAD[CASE=nom,NUM=?n,GEN=?g,SEM=?essence] AND BASEPRON[CASE=nom,NUM=?n,GEN=?g,SEM=?base]
PRINCIPLES[SEM=<?count(?six(?mind))>] -> PRINCIPLEHEAD[CASE=nom,NUM=?n,GEN=?g] COUNT[SEM=?count] AND ACCEPTED[CASE=nom,NUM=?n,GEN=?g] SIX[SEM=?six] ALSO MIND[CASE=nom,SEM=?mind]
BASE[CASE=nom,NUM=sg,GEN=n,SEM=<cognition_base>] -> 'vijñānam'
APPREHENSION[CASE=nom,NUM=sg,GEN=f,SEM=<\b.RespectiveApprehension(b)>] -> 'prativijñaptiḥ'
ESSENCEHEAD[CASE=nom,NUM=sg,GEN=n,SEM=<\b.EssenceProjection(b,mind_essence)>] -> 'mana-āyatanam'
AND -> 'ca'
BASEPRON[CASE=nom,NUM=sg,GEN=n,SEM=<cognition_base>] -> 'tat'
PRINCIPLEHEAD[CASE=nom,NUM=pl,GEN=m] -> 'dhātavaḥ'
COUNT[SEM=<\P.(P & CountedPrinciples(cognition_base,seven))>] -> 'sapta'
ACCEPTED[CASE=nom,NUM=pl,GEN=m] -> 'matāḥ'
SIX[CASE=nom,NUM=pl,GEN=n,SEM=<\m.all x.(PrincipleMember(x,cognition_base) <-> (SixCognitionPrinciple(x) | x=m))>] -> 'ṣaṭ-vijñānāni'
ALSO -> 'atho'
MIND[CASE=nom,NUM=sg,GEN=n,SEM=<mind_principle>] -> 'manaḥ'
"""

dependency_text = r"""'prativijñaptiḥ' -> 'vijñānam'
'mana-āyatanam' -> 'ca' | 'tat'
'matāḥ' -> 'dhātavaḥ' | 'ṣaṭ-vijñānāni'
'dhātavaḥ' -> 'sapta' | 'ca'
'ṣaṭ-vijñānāni' -> 'atho' | 'manaḥ'
"""

sem_text = r"""# derived by the section 4 parse
(RespectiveApprehension(cognition_base) & EssenceProjection(cognition_base,mind_essence) & ((all x.(PrincipleMember(x,cognition_base) <-> (SixCognitionPrinciple(x) | x=mind_principle))) & CountedPrinciples(cognition_base,seven)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(RespectiveApprehension(cognition_base) & EssenceProjection(cognition_base,mind_essence) & ((all x.(PrincipleMember(x,cognition_base) <-> (SixCognitionPrinciple(x) | x=mind_principle))) & CountedPrinciples(cognition_base,seven)))
"""

valuation_text = r"""cognition_base => cognition_class
mind_essence => mind_capacity
mind_principle => mind_role
seven => seven_count
RespectiveApprehension => {cognition_class}
EssenceProjection => {(cognition_class,mind_capacity)}
SixCognitionPrinciple => {eye_cog,ear_cog,nose_cog,tongue_cog,body_cog,mental_cog}
PrincipleMember => {(eye_cog,cognition_class),(ear_cog,cognition_class),(nose_cog,cognition_class),(tongue_cog,cognition_class),(body_cog,cognition_class),(mental_cog,cognition_class),(mind_role,cognition_class)}
CountedPrinciples => {(cognition_class,seven_count)}
"""


In [34]:
studies[16] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[16]["tree"].pformat())
print("Derived SEM:", studies[16]["sem"])


parses 1 SEM=ok
verse=T
(S[SEM=<(RespectiveApprehension(cognition_base) & EssenceProjection(cognition_base,mind_essence) & all x.(PrincipleMember(x,cognition_base) <-> (SixCognitionPrinciple(x) | (x = mind_principle))) & CountedPrinciples(cognition_base,seven))>]
  (DEFINITION[SEM=<RespectiveApprehension(cognition_base)>]
    (BASE[CASE='nom', GEN='n', NUM='sg', SEM=<cognition_base>]
      vijñānam)
    (APPREHENSION[CASE='nom', GEN='f', NUM='sg', SEM=<\b.RespectiveApprehension(b)>]
      prativijñaptiḥ))
  (ESSENCE[SEM=<EssenceProjection(cognition_base,mind_essence)>]
    (ESSENCEHEAD[CASE='nom', GEN='n', NUM='sg', SEM=<\b.EssenceProjection(b,mind_essence)>]
      mana-āyatanam)
    (AND[] ca)
    (BASEPRON[CASE='nom', GEN='n', NUM='sg', SEM=<cognition_base>]
      tat))
  (PRINCIPLES[SEM=<(all x.(PrincipleMember(x,cognition_base) <-> (SixCognitionPrinciple(x) | (x = mind_principle))) & CountedPrinciples(cognition_base,seven))>]
    (PRINCIPLEHEAD[CASE='nom', GEN='m', NUM='pl'] dhātav

# VAK_1.17 — Mind Is the Sixth Support

Source: `VAK_1.17.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
ṣaṇṇām anantarātītaṃ vijñānaṃ yad dhi tan manaḥ /
ṣaṣṭhāśrayaprasiddhyarthaṃ dhātavo 'ṣṭādaśa smṛtāḥ // 1.17 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| ṣaṇṇām | genitive plural | of the six |
| anantara-atītam | nominative neuter | immediately past; just ceased |
| vijñānam | nominative neuter | a cognition, one of the six |
| yat / tat | nominative neuter | whichever, that |
| hi | particle | the reason |
| manaḥ | nominative neuter | mind |
| ṣaṣṭha-āśraya-prasiddhi-artham | accusative of purpose | so that the sixth support be established |
| dhātavaḥ | nominative plural | principles |
| aṣṭādaśa smṛtāḥ | numeral and participle | eighteen are taught |

The commentary tightens *anantarātīta* to *samanantara-niruddha*: ceased immediately before, nothing between.

**Grammar tokens.** Verse-order IAST, with sandhi opened and combined
lexical rows separated. The purpose compound remains one token. No covert
successor is supplied: its actual arising is not required for Mind status.

```text
# tokens: ṣaṇṇām anantara-atītam vijñānam yat hi tat manaḥ ṣaṣṭha-āśraya-prasiddhi-artham dhātavaḥ aṣṭādaśa smṛtāḥ
```

## 5. Translation

### Literal Translation

> Whichever cognition among the six is immediately past, that indeed is
> mind. The eighteen principles are taught in order to establish the sixth
> support.

### Bhāṣya-informed study translation

> Of the six Cognitions, whichever has just ceased is Mind. The eighteen
> Principles are taught so that Mind may be established as the sixth support.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| vijñāna | cognition | one of the six; not consciousness; not principle |
| manas | mind | the same cognition, just ceased |
| manodhātu | mind-principle | the office, not a seventh substance |
| manovijñāna-dhātu | mental cognition-principle | the sixth of the six; the supported |
| anantarātīta | immediately past | ceased immediately before |
| āśraya | support | the office mind fills |
| āśrita | the supported | the six cognitions |
| ālambana | the borne | not the support |
| citta | consciousness | the arhat's final consciousness; not mind |
| dravyataḥ | by substance | the objection's count |
| prasiddhi | establishment | so that the sixth support be known |

In [35]:
fcfg_text = r"""% start S
# tokens: ṣaṇṇām anantara-atītam vijñānam yat hi tat manaḥ ṣaṣṭha-āśraya-prasiddhi-artham dhātavaḥ aṣṭādaśa smṛtāḥ
S[SEM=<(?m & ?t)>] -> MINDCLAUSE[SEM=?m] TEACHING[SEM=?t]
MINDCLAUSE[SEM=<?rel(?six(?past(?cog)))(?mind)>] -> SIX[CASE=gen,SEM=?six] PAST[CASE=nom,NUM=?n,GEN=?g,SEM=?past] COG[CASE=nom,NUM=?n,GEN=?g,SEM=?cog] REL[CASE=nom,NUM=?n,GEN=?g,SEM=?rel] HI CORR[CASE=nom,NUM=?n,GEN=?g] MIND[CASE=nom,NUM=?n,GEN=?g,SEM=?mind]
TEACHING[SEM=<?purpose(?taught(?count))>] -> PURPOSE[SEM=?purpose] PRINCIPLES[CASE=nom,NUM=?n,GEN=?g] COUNT[SEM=?count] TAUGHT[CASE=nom,NUM=?n,GEN=?g,SEM=?taught]
SIX[CASE=gen,NUM=pl,SEM=<\P.\x.(P(x) & OfSixCognitions(x))>] -> 'ṣaṇṇām'
PAST[CASE=nom,NUM=sg,GEN=n,SEM=<\P.\x.(P(x) & ImmediatelyPast(x))>] -> 'anantara-atītam'
COG[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Cognition(x)>] -> 'vijñānam'
REL[CASE=nom,NUM=sg,GEN=n,SEM=<\P.\Q.all x.(P(x) -> Q(x))>] -> 'yat'
HI -> 'hi'
CORR[CASE=nom,NUM=sg,GEN=n] -> 'tat'
MIND[CASE=nom,NUM=sg,GEN=n,SEM=<\x.MindRole(x)>] -> 'manaḥ'
PURPOSE[SEM=<\P.(P & ForEstablishing(principle_teaching,sixth_support))>] -> 'ṣaṣṭha-āśraya-prasiddhi-artham'
PRINCIPLES[CASE=nom,NUM=pl,GEN=m] -> 'dhātavaḥ'
COUNT[SEM=<eighteen>] -> 'aṣṭādaśa'
TAUGHT[CASE=nom,NUM=pl,GEN=m,SEM=<\n.TaughtCount(principle_teaching,n)>] -> 'smṛtāḥ'
"""

dependency_text = r"""'manaḥ' -> 'vijñānam' | 'tat'
'vijñānam' -> 'ṣaṇṇām' | 'anantara-atītam' | 'yat' | 'hi'
'smṛtāḥ' -> 'ṣaṣṭha-āśraya-prasiddhi-artham' | 'dhātavaḥ'
'dhātavaḥ' -> 'aṣṭādaśa'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(((Cognition(x) & ImmediatelyPast(x)) & OfSixCognitions(x)) -> MindRole(x))) & (TaughtCount(principle_teaching,eighteen) & ForEstablishing(principle_teaching,sixth_support)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(((Cognition(x) & ImmediatelyPast(x)) & OfSixCognitions(x)) -> MindRole(x))) & (TaughtCount(principle_teaching,eighteen) & ForEstablishing(principle_teaching,sixth_support)))
"""

valuation_text = r"""principle_teaching => teaching
eighteen => eighteen_count
sixth_support => mind_support
Cognition => {ceased,current,final}
OfSixCognitions => {ceased,current,final}
ImmediatelyPast => {ceased,final}
MindRole => {ceased,final}
TaughtCount => {(teaching,eighteen_count)}
ForEstablishing => {(teaching,mind_support)}
HasSuccessor => {ceased}
"""


In [36]:
studies[17] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[17]["tree"].pformat())
print("Derived SEM:", studies[17]["sem"])


parses 1 SEM=ok
verse=T
(S[SEM=<(all x.((Cognition(x) & ImmediatelyPast(x) & OfSixCognitions(x)) -> MindRole(x)) & TaughtCount(principle_teaching,eighteen) & ForEstablishing(principle_teaching,sixth_support))>]
  (MINDCLAUSE[SEM=<all x.((Cognition(x) & ImmediatelyPast(x) & OfSixCognitions(x)) -> MindRole(x))>]
    (SIX[CASE='gen', NUM='pl', SEM=<\P x.(P(x) & OfSixCognitions(x))>]
      ṣaṇṇām)
    (PAST[CASE='nom', GEN='n', NUM='sg', SEM=<\P x.(P(x) & ImmediatelyPast(x))>]
      anantara-atītam)
    (COG[CASE='nom', GEN='n', NUM='sg', SEM=<\x.Cognition(x)>]
      vijñānam)
    (REL[CASE='nom', GEN='n', NUM='sg', SEM=<\P Q.all x.(P(x) -> Q(x))>]
      yat)
    (HI[] hi)
    (CORR[CASE='nom', GEN='n', NUM='sg'] tat)
    (MIND[CASE='nom', GEN='n', NUM='sg', SEM=<\x.MindRole(x)>] manaḥ))
  (TEACHING[SEM=<(TaughtCount(principle_teaching,eighteen) & ForEstablishing(principle_teaching,sixth_support))>]
    (PURPOSE[SEM=<\P.(P & ForEstablishing(principle_teaching,sixth_support))>]
      ṣaṣṭha

# VAK_1.18 — Inclusion by Own-Nature

Source: `VAK_1.18.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
sarvasaṃgraha ekena skandhenāyatanena ca /
dhātunā ca svabhāvena parabhāvaviyogataḥ // 1.18 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| sarva-saṃgrahaḥ | nominative | the inclusion of all |
| ekena | instrumental | by one; carries across the three |
| skandhena | instrumental | by a base |
| āyatanena | instrumental | by an essence |
| dhātunā | instrumental | by a principle |
| svabhāvena | instrumental | by own-nature |
| parabhāva-viyogataḥ | ablatival adverb | separated from another's nature |

*Saṃgraha* is inclusion. The commentary also uses it for the gathering of an
assembly. The two uses are not the same ground. The verse's use is the first.

**Grammar tokens.** Verse-order IAST with sandhi opened. The two *ca*
tokens are retained. The Bhāṣya names Form Base, Mind Essence, and Dharma
Principle; these referents are lexical semantic resolutions, not additional
overt words or new categories invented by the grammar.

```text
# tokens: sarva-saṃgrahaḥ ekena skandhena āyatanena ca dhātunā ca svabhāvena parabhāva-viyogataḥ
```

## 5. Translation

### Literal Translation

> The inclusion of all is by one base, and by one essence, and by one
> principle, by own-nature, because of separation from another's nature.

### Bhāṣya-informed study translation

> All dharmas are included together by one Base, the Form Base; one Essence,
> the Mind Essence; and one Principle, the Dharma Principle. This inclusion is
> by own-nature, through separation from another's nature.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| saṃgraha | inclusion | also, in the example, a gathering |
| sarva-saṃgraha | inclusion of all | joint, not each alone |
| skandha | base | conditioned only |
| upādāna-skandha | appropriated Base | the Impure |
| āyatana | essence | all dharmas |
| dhātu | principle | all dharmas |
| rūpa-skandha | form-base | the Form System |
| mana-āyatana | mind-essence | the cognition-base, under essence |
| dharma-dhātu | dharma-principle | feeling, reflection, formations, *avijñapti*, the unconditioned |
| svabhāva | own-nature | the ground of inclusion |
| parabhāva | another's nature | what will not include it |
| sāsrava | Impure | |
| sāṃketika | conventional | the assembly |
| kādācitka | occasional | why the assembly is conventional |

In [37]:
fcfg_text = r"""% start S
# tokens: sarva-saṃgrahaḥ ekena skandhena āyatanena ca dhātunā ca svabhāvena parabhāva-viyogataḥ
S[SEM=<?all(?one(?base)(?essence)(?principle))(?own)(?separate)>] -> ALL[SEM=?all] ONE[CASE=ins,NUM=?n,SEM=?one] BASE[CASE=ins,NUM=?n,SEM=?base] ESSENCE[CASE=ins,NUM=?n,SEM=?essence] AND PRINCIPLE[CASE=ins,NUM=?n,SEM=?principle] AND OWN[CASE=ins,SEM=?own] SEPARATION[SEM=?separate]
ALL[CASE=nom,NUM=sg,SEM=<\P.\O.\R.all x.(Dharma(x) -> (P(x) & O(x) & R(x)))>] -> 'sarva-saṃgrahaḥ'
ONE[CASE=ins,NUM=sg,SEM=<\b.\a.\d.\x.(IncludedByOwnNature(x,b) | IncludedByOwnNature(x,a) | IncludedByOwnNature(x,d))>] -> 'ekena'
BASE[CASE=ins,NUM=sg,GEN=m,SEM=<form_base>] -> 'skandhena'
ESSENCE[CASE=ins,NUM=sg,GEN=n,SEM=<mind_essence>] -> 'āyatanena'
AND -> 'ca'
PRINCIPLE[CASE=ins,NUM=sg,GEN=m,SEM=<dharma_principle>] -> 'dhātunā'
OWN[CASE=ins,SEM=<\x.InclusionByOwnNature(x)>] -> 'svabhāvena'
SEPARATION[SEM=<\x.SeparateFromOtherNature(x)>] -> 'parabhāva-viyogataḥ'
"""

dependency_text = r"""'sarva-saṃgrahaḥ' -> 'skandhena' | 'āyatanena' | 'dhātunā' | 'svabhāvena' | 'parabhāva-viyogataḥ'
'skandhena' -> 'ekena'
'āyatanena' -> 'ca'
'dhātunā' -> 'ca'
"""

sem_text = r"""# derived by the section 4 parse
all x.(Dharma(x) -> (((IncludedByOwnNature(x,form_base) | IncludedByOwnNature(x,mind_essence)) | IncludedByOwnNature(x,dharma_principle)) & InclusionByOwnNature(x) & SeparateFromOtherNature(x)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
all x.(Dharma(x) -> (((IncludedByOwnNature(x,form_base) | IncludedByOwnNature(x,mind_essence)) | IncludedByOwnNature(x,dharma_principle)) & InclusionByOwnNature(x) & SeparateFromOtherNature(x)))
"""

valuation_text = r"""form_base => form_class
mind_essence => mind_class
dharma_principle => dharma_class
Dharma => {eye,cognition,feeling,avijnapti,unconditioned}
IncludedByOwnNature => {(eye,form_class),(cognition,mind_class),(feeling,dharma_class),(avijnapti,form_class),(avijnapti,dharma_class),(unconditioned,dharma_class)}
InclusionByOwnNature => {eye,cognition,feeling,avijnapti,unconditioned}
SeparateFromOtherNature => {eye,cognition,feeling,avijnapti,unconditioned}
"""


In [38]:
studies[18] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[18]["tree"].pformat())
print("Derived SEM:", studies[18]["sem"])


parses 1 SEM=ok
verse=T
(S[SEM=<all x.(Dharma(x) -> ((IncludedByOwnNature(x,form_base) | IncludedByOwnNature(x,mind_essence) | IncludedByOwnNature(x,dharma_principle)) & InclusionByOwnNature(x) & SeparateFromOtherNature(x)))>]
  (ALL[CASE='nom', NUM='sg', SEM=<\P O R.all x.(Dharma(x) -> (P(x) & O(x) & R(x)))>]
    sarva-saṃgrahaḥ)
  (ONE[CASE='ins', NUM='sg', SEM=<\b a d x.(IncludedByOwnNature(x,b) | IncludedByOwnNature(x,a) | IncludedByOwnNature(x,d))>]
    ekena)
  (BASE[CASE='ins', GEN='m', NUM='sg', SEM=<form_base>] skandhena)
  (ESSENCE[CASE='ins', GEN='n', NUM='sg', SEM=<mind_essence>]
    āyatanena)
  (AND[] ca)
  (PRINCIPLE[CASE='ins', GEN='m', NUM='sg', SEM=<dharma_principle>]
    dhātunā)
  (AND[] ca)
  (OWN[CASE='ins', SEM=<\x.InclusionByOwnNature(x)>] svabhāvena)
  (SEPARATION[SEM=<\x.SeparateFromOtherNature(x)>]
    parabhāva-viyogataḥ))
Derived SEM: all x.(Dharma(x) -> ((IncludedByOwnNature(x,form_base) | IncludedByOwnNature(x,mind_essence) | IncludedByOwnNature(x,dharma_

# VAK_1.19 — One Principle, Two Sites

Source: `VAK_1.19.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
jātigocaravijñānasāmānyād ekadhātutā /
dvitve 'pi cakṣurādīnāṃ śobhārthaṃ tu dvayodbhavaḥ // 1.19 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| jāti | compound member | kind; the nature of the eye |
| gocara | compound member | range |
| vijñāna | compound member | the cognition |
| sāmānyāt | ablative | because of commonality |
| eka-dhātutā | nominative | the status of one principle |
| dvitve api | locative and particle | even though two |
| cakṣus-ādīnām | genitive plural | of the eye and the rest |
| śobhā-artham | accusative of purpose | for beauty |
| tu | particle | a second question |
| dvaya-udbhavaḥ | nominative | arising as a pair |

The commentary restricts the eye and the rest to eye, ear, and nose. Tongue and body do not raise the count.

**Grammar tokens.** Verse-order IAST with sandhi opened. The first
compound is one leaf, combining the first four lexical rows; *dvitve api*
is separated. *Cakṣus-ādi* is restricted by the Bhāṣya to eye, ear, and
nose. No covert token is needed.

```text
# tokens: jāti-gocara-vijñāna-sāmānyāt eka-dhātutā dvitve api cakṣus-ādīnām śobhā-artham tu dvaya-udbhavaḥ
```

## 5. Translation

### Literal Translation

> From commonality of kind, range, and Cognition, there is the status of one
> principle, even in the duality of the eye and the rest. But the arising as
> a pair is for the sake of beauty.

### Bhāṣya-informed study translation

> Although the eyes, ears, and nostrils occur in pairs, each pair is one
> Principle through commonality of kind, range, and Cognition. Yet they arise
> in pairs for the beauty of the bodily support.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| jāti | kind | the nature of the eye; not birth |
| gocara | range | both take visible form |
| vijñāna | Cognition | both sites support one Eye-Cognition |
| sāmānya | commonality | the three together |
| eka-dhātutā | status of one principle | the count |
| dvitva | duality | two sites, conceded |
| āśraya | support | the bodily support |
| śobhā | beauty | why the pair; not why the one |
| vairūpya | disfigurement | the commentary's contrast |
| rūpa-viṣaya | visible form as condition | the range |

In [39]:
fcfg_text = r"""% start S
# tokens: jāti-gocara-vijñāna-sāmānyāt eka-dhātutā dvitve api cakṣus-ādīnām śobhā-artham tu dvaya-udbhavaḥ
S[SEM=<(?identity & ?arising)>] -> IDENTITY[SEM=?identity] ARISING[SEM=?arising]
IDENTITY[SEM=<?scope(?shared)(?single)>] -> COMMON[CASE=abl,SEM=?shared] SINGLE[CASE=nom,NUM=?n,SEM=?single] DUAL[CASE=loc] EVEN SCOPE[CASE=gen,NUM=pl,SEM=?scope]
ARISING[SEM=<?purpose(?arising)>] -> PURPOSE[SEM=?purpose] CONTRAST PAIR[CASE=nom,NUM=sg,SEM=?arising]
COMMON[CASE=abl,SEM=<\x.(CommonKind(x) & CommonRange(x) & CommonCognition(x))>] -> 'jāti-gocara-vijñāna-sāmānyāt'
SINGLE[CASE=nom,NUM=sg,GEN=f,SEM=<\x.OnePrinciple(x)>] -> 'eka-dhātutā'
DUAL[CASE=loc,NUM=sg] -> 'dvitve'
EVEN -> 'api'
SCOPE[CASE=gen,NUM=pl,SEM=<\P.\Q.all x.((PairedEyeEarNose(x) & P(x)) -> Q(x))>] -> 'cakṣus-ādīnām'
PURPOSE[SEM=<\P.all x.(PairedEyeEarNose(x) -> (P(x) & ForBeautyOfSupport(x)))>] -> 'śobhā-artham'
CONTRAST -> 'tu'
PAIR[CASE=nom,NUM=sg,GEN=m,SEM=<\x.ArisesAsPair(x)>] -> 'dvaya-udbhavaḥ'
"""

dependency_text = r"""'eka-dhātutā' -> 'jāti-gocara-vijñāna-sāmānyāt' | 'dvitve' | 'cakṣus-ādīnām'
'dvitve' -> 'api'
'dvaya-udbhavaḥ' -> 'śobhā-artham' | 'tu'
"""

sem_text = r"""# derived by the section 4 parse
((all x.((PairedEyeEarNose(x) & (CommonKind(x) & CommonRange(x) & CommonCognition(x))) -> OnePrinciple(x))) & (all x.(PairedEyeEarNose(x) -> (ArisesAsPair(x) & ForBeautyOfSupport(x)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.((PairedEyeEarNose(x) & (CommonKind(x) & CommonRange(x) & CommonCognition(x))) -> OnePrinciple(x))) & (all x.(PairedEyeEarNose(x) -> (ArisesAsPair(x) & ForBeautyOfSupport(x)))))
"""

valuation_text = r"""PairedEyeEarNose => {eyes,ears,nostrils}
CommonKind => {eyes,ears,nostrils}
CommonRange => {eyes,ears,nostrils}
CommonCognition => {eyes,ears,nostrils}
OnePrinciple => {eyes,ears,nostrils}
ArisesAsPair => {eyes,ears,nostrils}
ForBeautyOfSupport => {eyes,ears,nostrils}
"""


In [40]:
studies[19] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[19]["tree"].pformat())
print("Derived SEM:", studies[19]["sem"])


parses 1 SEM=ok
verse=T
(S[SEM=<(all x.((PairedEyeEarNose(x) & CommonKind(x) & CommonRange(x) & CommonCognition(x)) -> OnePrinciple(x)) & all x.(PairedEyeEarNose(x) -> (ArisesAsPair(x) & ForBeautyOfSupport(x))))>]
  (IDENTITY[SEM=<all x.((PairedEyeEarNose(x) & CommonKind(x) & CommonRange(x) & CommonCognition(x)) -> OnePrinciple(x))>]
    (COMMON[CASE='abl', SEM=<\x.(CommonKind(x) & CommonRange(x) & CommonCognition(x))>]
      jāti-gocara-vijñāna-sāmānyāt)
    (SINGLE[CASE='nom', GEN='f', NUM='sg', SEM=<\x.OnePrinciple(x)>]
      eka-dhātutā)
    (DUAL[CASE='loc', NUM='sg'] dvitve)
    (EVEN[] api)
    (SCOPE[CASE='gen', NUM='pl', SEM=<\P Q.all x.((PairedEyeEarNose(x) & P(x)) -> Q(x))>]
      cakṣus-ādīnām))
  (ARISING[SEM=<all x.(PairedEyeEarNose(x) -> (ArisesAsPair(x) & ForBeautyOfSupport(x)))>]
    (PURPOSE[SEM=<\P.all x.(PairedEyeEarNose(x) -> (P(x) & ForBeautyOfSupport(x)))>]
      śobhā-artham)
    (CONTRAST[] tu)
    (PAIR[CASE='nom', GEN='m', NUM='sg', SEM=<\x.ArisesAsPair(x)>]


# VAK_1.20 — Three Meanings, Three Teachings

Source: `VAK_1.20.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
rāśyāyadvāragotrārthāḥ skandhāyatanadhātavaḥ /
mohendriyarucitraidhāt tisraḥ skandhādideśanāḥ // 1.20 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| rāśi-āya-dvāra-gotra-arthāḥ | nominative masculine plural compound | having collection, door of arrival, and source-kind as their respective meanings |
| skandha-āyatana-dhātavaḥ | nominative masculine plural copulative compound | Bases, Essences, and Principles, in that order |
| moha-indriya-ruci-traidhāt | ablative singular compound | because of threefoldness in confusion, Faculty, and inclination |
| tisraḥ | nominative feminine numeral | three |
| skandha-ādi-deśanāḥ | nominative feminine plural compound | teachings beginning with the Bases |

The first compound distributes: Base means collection, Essence means door
of arrival, and Principle means source-kind.

**Grammar tokens.** Verse-order IAST with sandhi opened and compound
boundaries marked. Compounds remain whole leaves; *āya-dvāra* is one
meaning, not two separate terms in a four-way distribution.

```text
# tokens: rāśi-āya-dvāra-gotra-arthāḥ skandha-āyatana-dhātavaḥ moha-indriya-ruci-traidhāt tisraḥ skandha-ādi-deśanāḥ
```

## 5. Translation

### Literal Translation

> Bases, essences, and principles have the meanings of collection, door of
> arrival, and source-kind. Because confusion, faculty, and inclination are
> threefold, there are three teachings, beginning with the bases.

### Bhāṣya-informed study translation

> Bases mean collections; Essences, doors of arrival; Principles,
> source-kinds. Because trainees differ threefold in confusion, Faculty, and
> inclination, there are three teachings, beginning with the Bases.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| skandha | base | the collection |
| rāśi | collection | the sūtra's meaning |
| prajñapti | designation | how a base exists |
| āyatana | essence | the door |
| āya | arrival | of consciousness and the mental factors |
| dvāra | door | they extend the arrival |
| dhātu | principle | source-kind |
| gotra | source-kind | glossed as mine |
| sabhāga-hetu | homogeneous cause | of its own kind; the conditioned |
| svabhāva | own-nature | with kind, for the unconditioned |
| citta | consciousness | what arrives through the door |
| moha | confusion | lump-self; form; form and consciousness |
| indriya | faculty | here the learner's |
| ruci | inclination | brief, intermediate, extensive |
| deśanā | teaching | Base, Essence, Principle, in order |

In [41]:
fcfg_text = r"""% start S
# tokens: rāśi-āya-dvāra-gotra-arthāḥ skandha-āyatana-dhātavaḥ moha-indriya-ruci-traidhāt tisraḥ skandha-ādi-deśanāḥ
S[SEM=<(?meanings & ?teaching)>] -> MEANINGS[SEM=?meanings] EXPLANATION[SEM=?teaching]
MEANINGS[SEM=<?roster(?senses)>] -> SENSES[CASE=nom,NUM=?n,GEN=?g,SEM=?senses] ROSTER[CASE=nom,NUM=?n,GEN=?g,SEM=?roster]
EXPLANATION[SEM=<?reason(?number(?teaching))>] -> REASON[CASE=abl,SEM=?reason] THREE[CASE=nom,NUM=?n,GEN=?g,SEM=?number] TEACHING[CASE=nom,NUM=?n,GEN=?g,SEM=?teaching]
SENSES[CASE=nom,NUM=pl,GEN=m,SEM=<\b.\a.\d.(Meaning(b,collection) & Meaning(a,arrival_door) & Meaning(d,source_kind))>] -> 'rāśi-āya-dvāra-gotra-arthāḥ'
ROSTER[CASE=nom,NUM=pl,GEN=m,SEM=<\P.(P(base)(essence)(principle))>] -> 'skandha-āyatana-dhātavaḥ'
REASON[CASE=abl,SEM=<\P.(Threefold(confusion) & Threefold(faculty) & Threefold(inclination) & ExplainsThreeTeachings(confusion) & ExplainsThreeTeachings(faculty) & ExplainsThreeTeachings(inclination) & P)>] -> 'moha-indriya-ruci-traidhāt'
THREE[CASE=nom,NUM=pl,GEN=f,SEM=<\s.ThreeTeachings(s)>] -> 'tisraḥ'
TEACHING[CASE=nom,NUM=pl,GEN=f,SEM=<base_etc_teaching>] -> 'skandha-ādi-deśanāḥ'
"""

dependency_text = r"""'skandha-āyatana-dhātavaḥ' -> 'rāśi-āya-dvāra-gotra-arthāḥ'
'skandha-ādi-deśanāḥ' -> 'moha-indriya-ruci-traidhāt' | 'tisraḥ'
"""

sem_text = r"""# derived by the section 4 parse
((Meaning(base,collection) & Meaning(essence,arrival_door) & Meaning(principle,source_kind)) & (Threefold(confusion) & Threefold(faculty) & Threefold(inclination) & ExplainsThreeTeachings(confusion) & ExplainsThreeTeachings(faculty) & ExplainsThreeTeachings(inclination) & ThreeTeachings(base_etc_teaching)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((Meaning(base,collection) & Meaning(essence,arrival_door) & Meaning(principle,source_kind)) & (Threefold(confusion) & Threefold(faculty) & Threefold(inclination) & ExplainsThreeTeachings(confusion) & ExplainsThreeTeachings(faculty) & ExplainsThreeTeachings(inclination) & ThreeTeachings(base_etc_teaching)))
# THEORY prose
# EXPECT prose T
Teaching(base) & Teaching(essence) & Teaching(principle)
SuitedToConfusion(base,mental_factors) & SuitedToConfusion(essence,form) & SuitedToConfusion(principle,form_and_consciousness)
SuitedToFaculty(base,sharp) & SuitedToFaculty(essence,middling) & SuitedToFaculty(principle,weak)
SuitedToInclination(base,brief) & SuitedToInclination(essence,intermediate) & SuitedToInclination(principle,extensive)
exists x.(Unconditioned(x) & PrincipleMember(x) & -HomogeneousCause(x))
# THEORY rejected_door_is_collection
# EXPECT rejected_door_is_collection F
arrival_door = collection
# THEORY rejected_extensive_means_sharp_faculty
# EXPECT rejected_extensive_means_sharp_faculty F
all x.(SuitedToInclination(x,extensive) -> SuitedToFaculty(x,sharp))
# THEORY rejected_all_principles_are_homogeneous_causes
# EXPECT rejected_all_principles_are_homogeneous_causes F
all x.(PrincipleMember(x) -> HomogeneousCause(x))
"""

valuation_text = r"""base => base_teaching
essence => essence_teaching
principle => principle_teaching
collection => collection_meaning
arrival_door => arrival_door_meaning
source_kind => source_kind_meaning
confusion => confusion_axis
faculty => faculty_axis
inclination => inclination_axis
base_etc_teaching => threefold_exposition
mental_factors => mental_factor_confusion
form => form_confusion
form_and_consciousness => form_consciousness_confusion
sharp => sharp_faculty
middling => middling_faculty
weak => weak_faculty
brief => brief_inclination
intermediate => intermediate_inclination
extensive => extensive_inclination
Meaning => {(base_teaching,collection_meaning),(essence_teaching,arrival_door_meaning),(principle_teaching,source_kind_meaning)}
Threefold => {confusion_axis,faculty_axis,inclination_axis}
ExplainsThreeTeachings => {confusion_axis,faculty_axis,inclination_axis}
ThreeTeachings => {threefold_exposition}
Teaching => {base_teaching,essence_teaching,principle_teaching}
SuitedToConfusion => {(base_teaching,mental_factor_confusion),(essence_teaching,form_confusion),(principle_teaching,form_consciousness_confusion)}
SuitedToFaculty => {(base_teaching,sharp_faculty),(essence_teaching,middling_faculty),(principle_teaching,weak_faculty)}
SuitedToInclination => {(base_teaching,brief_inclination),(essence_teaching,intermediate_inclination),(principle_teaching,extensive_inclination)}
PrincipleMember => {eye_case,space_case}
HomogeneousCause => {eye_case}
Unconditioned => {space_case}
"""


In [42]:
studies[20] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[20]["tree"].pformat())
print("Derived SEM:", studies[20]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_door_is_collection=F rejected_extensive_means_sharp_faculty=F rejected_all_principles_are_homogeneous_causes=F
(S[SEM=<(Meaning(base,collection) & Meaning(essence,arrival_door) & Meaning(principle,source_kind) & Threefold(confusion) & Threefold(faculty) & Threefold(inclination) & ExplainsThreeTeachings(confusion) & ExplainsThreeTeachings(faculty) & ExplainsThreeTeachings(inclination) & ThreeTeachings(base_etc_teaching))>]
  (MEANINGS[SEM=<(Meaning(base,collection) & Meaning(essence,arrival_door) & Meaning(principle,source_kind))>]
    (SENSES[CASE='nom', GEN='m', NUM='pl', SEM=<\b a d.(Meaning(b,collection) & Meaning(a,arrival_door) & Meaning(d,source_kind))>]
      rāśi-āya-dvāra-gotra-arthāḥ)
    (ROSTER[CASE='nom', GEN='m', NUM='pl', SEM=<\P.P(base,essence,principle)>]
      skandha-āyatana-dhātavaḥ))
  (EXPLANATION[SEM=<(Threefold(confusion) & Threefold(faculty) & Threefold(inclination) & ExplainsThreeTeachings(confusion) & ExplainsThreeTeac

# VAK_1.21 — Feeling and Reflection as Distinct Bases

Source: `VAK_1.21.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
vivādamūlasaṃsārahetutvāt kramakāraṇāt /
caittebhyo vedanāsaṃjñe pṛthakskandhau niveśitau // 1.21 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| vivāda-mūla-saṃsāra-hetutvāt | ablative singular abstract compound | because of being causes of the roots of dispute and of saṃsāra |
| krama-kāraṇāt | ablative neuter singular | because of the reason of order |
| caittebhyaḥ | ablative plural | from the other mental factors |
| vedanā-saṃjñe | feminine dual | feeling and reflection |
| pṛthak-skandhau | masculine dual | two distinct bases |
| niveśitau | past participle dual | established |

The commentary supplies *pradhāna-hetu*: principal cause. Feeling and reflection are not the only causes. The local Devanāgarī sometimes reads *cittebhyaḥ*. The prose and the IAST support *caittebhyaḥ*.

**Grammar tokens.** Verse-order IAST with external sandhi opened and
compound boundaries marked. The compound containing both causal fields
remains one leaf. *Pṛthak* is separated from *skandhau*, as in the prose.
No covert token is supplied.

```text
# tokens: vivāda-mūla-saṃsāra-hetutvāt krama-kāraṇāt caittebhyaḥ vedanā-saṃjñe pṛthak skandhau niveśitau
```

## 5. Translation

### Literal Translation

> Because they are causes of the roots of dispute and of saṃsāra, and
> because of the reason of order, feeling and reflection are established as
> two distinct bases apart from the other mental factors.

### Bhāṣya-informed study translation

> Because they are causes of the roots of dispute and of saṃsāra, and because
> of the reason of order, feeling and reflection were established as distinct
> Bases apart from the other mental factors.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| caitta | mental factor | the class they leave, as bases |
| vedanā | feeling | undergoing; not the fixation |
| saṃjñā | reflection | takes up the mark |
| saṃskāra-skandha | formations-base | where the other mental factors remain |
| vivāda-mūla | root of dispute | the two fixations |
| kāmādhyavasāna | fixation upon sensual desire | principal cause: feeling |
| dṛṣṭyadhyavasāna | fixation upon views | principal cause: reflection |
| vedanā-svāda | savor of feeling | the mediation |
| viparīta-saṃjñā | inverted reflection | not every reflection |
| pradhāna-hetu | principal cause | not sole |
| krama | order | deferred |

In [43]:
fcfg_text = r"""% start S
# tokens: vivāda-mūla-saṃsāra-hetutvāt krama-kāraṇāt caittebhyaḥ vedanā-saṃjñe pṛthak skandhau niveśitau
S[SEM=<?reasons(?placement)>] -> REASONS[SEM=?reasons] PLACEMENT[SEM=?placement]
REASONS[SEM=<\P.(?causal & ?order & P)>] -> CAUSAL[CASE=abl,SEM=?causal] ORDER[CASE=abl,SEM=?order]
PLACEMENT[SEM=<?pair(?separate(?other)(?placed(?bases)))>] -> OTHER[CASE=?case,NUM=pl,SEM=?other] PAIR[CASE=nom,NUM=?n,GEN=f,SEM=?pair] SEPARATE[OBJCASE=?case,SEM=?separate] BASES[CASE=nom,NUM=?n,GEN=?g,SEM=?bases] PLACED[CASE=nom,NUM=?n,GEN=?g,SEM=?placed]
CAUSAL[CASE=abl,SEM=<(CauseOfDisputeRoots(feeling) & CauseOfDisputeRoots(reflection) & CauseOfSamsara(feeling) & CauseOfSamsara(reflection))>] -> 'vivāda-mūla-saṃsāra-hetutvāt'
ORDER[CASE=abl,SEM=<OrderReasonForSeparateBases(base_order)>] -> 'krama-kāraṇāt'
OTHER[CASE=abl,NUM=pl,SEM=<other_mental_factors>] -> 'caittebhyaḥ'
PAIR[CASE=nom,NUM=dual,GEN=f,SEM=<\P.(P(feeling) & P(reflection))>] -> 'vedanā-saṃjñe'
SEPARATE[OBJCASE=abl,SEM=<\o.\P.\x.(P(x) & SeparatelyFrom(x,o))>] -> 'pṛthak'
BASES[CASE=nom,NUM=dual,GEN=m,SEM=<\x.SeparateBasePlacement(x)>] -> 'skandhau'
PLACED[CASE=nom,NUM=dual,GEN=m,SEM=<\P.\x.P(x)>] -> 'niveśitau'
"""

dependency_text = r"""'niveśitau' -> 'vivāda-mūla-saṃsāra-hetutvāt' | 'krama-kāraṇāt' | 'vedanā-saṃjñe' | 'skandhau'
'skandhau' -> 'pṛthak'
'pṛthak' -> 'caittebhyaḥ'
"""

sem_text = r"""# derived by the section 4 parse
((CauseOfDisputeRoots(feeling) & CauseOfDisputeRoots(reflection) & CauseOfSamsara(feeling) & CauseOfSamsara(reflection)) & OrderReasonForSeparateBases(base_order) & ((SeparateBasePlacement(feeling) & SeparatelyFrom(feeling,other_mental_factors)) & (SeparateBasePlacement(reflection) & SeparatelyFrom(reflection,other_mental_factors))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((CauseOfDisputeRoots(feeling) & CauseOfDisputeRoots(reflection) & CauseOfSamsara(feeling) & CauseOfSamsara(reflection)) & OrderReasonForSeparateBases(base_order) & ((SeparateBasePlacement(feeling) & SeparatelyFrom(feeling,other_mental_factors)) & (SeparateBasePlacement(reflection) & SeparatelyFrom(reflection,other_mental_factors))))
# THEORY prose
# EXPECT prose T
MentalFactor(feeling) & MentalFactor(reflection)
PrincipalCause(feeling,sensual_fixation) & PrincipalCause(reflection,view_fixation)
DisputeRoot(sensual_fixation) & DisputeRoot(view_fixation)
MediatedBy(feeling,savor) & MediatedBy(reflection,inversion)
GreedyForSavor(wanderer) & InvertedReflection(wanderer) & Wanders(wanderer)
# THEORY rejected_separate_bases_are_not_mental_factors
# EXPECT rejected_separate_bases_are_not_mental_factors F
all x.(SeparateBasePlacement(x) -> -MentalFactor(x))
# THEORY rejected_principal_means_sole
# EXPECT rejected_principal_means_sole F
all x.all y.(PrincipalCause(x,y) -> SoleCause(x,y))
# THEORY rejected_feeling_and_reflection_are_the_roots
# EXPECT rejected_feeling_and_reflection_are_the_roots F
DisputeRoot(feeling) & DisputeRoot(reflection)
"""

valuation_text = r"""feeling => feeling_factor
reflection => reflection_factor
other_mental_factors => remaining_factor_class
base_order => announced_order_reason
sensual_fixation => sensual_root
view_fixation => view_root
savor => savor_mediation
inversion => inversion_mediation
wanderer => wandering_person
CauseOfDisputeRoots => {feeling_factor,reflection_factor}
CauseOfSamsara => {feeling_factor,reflection_factor}
OrderReasonForSeparateBases => {announced_order_reason}
SeparateBasePlacement => {feeling_factor,reflection_factor}
SeparatelyFrom => {(feeling_factor,remaining_factor_class),(reflection_factor,remaining_factor_class)}
MentalFactor => {feeling_factor,reflection_factor,another_factor}
PrincipalCause => {(feeling_factor,sensual_root),(reflection_factor,view_root)}
DisputeRoot => {sensual_root,view_root}
MediatedBy => {(feeling_factor,savor_mediation),(reflection_factor,inversion_mediation)}
SoleCause => {}
GreedyForSavor => {wandering_person}
InvertedReflection => {wandering_person}
Wanders => {wandering_person}
"""


In [44]:
studies[21] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[21]["tree"].pformat())
print("Derived SEM:", studies[21]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_separate_bases_are_not_mental_factors=F rejected_principal_means_sole=F rejected_feeling_and_reflection_are_the_roots=F
(S[SEM=<(CauseOfDisputeRoots(feeling) & CauseOfDisputeRoots(reflection) & CauseOfSamsara(feeling) & CauseOfSamsara(reflection) & OrderReasonForSeparateBases(base_order) & SeparateBasePlacement(feeling) & SeparatelyFrom(feeling,other_mental_factors) & SeparateBasePlacement(reflection) & SeparatelyFrom(reflection,other_mental_factors))>]
  (REASONS[SEM=<\P.(CauseOfDisputeRoots(feeling) & CauseOfDisputeRoots(reflection) & CauseOfSamsara(feeling) & CauseOfSamsara(reflection) & OrderReasonForSeparateBases(base_order) & P)>]
    (CAUSAL[CASE='abl', SEM=<(CauseOfDisputeRoots(feeling) & CauseOfDisputeRoots(reflection) & CauseOfSamsara(feeling) & CauseOfSamsara(reflection))>]
      vivāda-mūla-saṃsāra-hetutvāt)
    (ORDER[CASE='abl', SEM=<OrderReasonForSeparateBases(base_order)>]
      krama-kāraṇāt))
  (PLACEMENT[SEM=<(SeparateBasePlac

# VAK_1.22 — The Unconditioned, and the Order of the Bases

Source: `VAK_1.22.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
skandheṣv asaṃskṛtaṃ noktam arthāyogāt kramaḥ punaḥ /
yathaudārikasaṃkleśabhājanādyarthadhātutaḥ // 1.22 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| skandheṣu | locative plural | among the bases |
| asaṃskṛtam | nominative neuter | the unconditioned |
| na | negative particle | not |
| uktam | nominative neuter singular passive participle | stated; agrees with the unconditioned |
| arthāyogāt | ablative | the defining meaning does not apply |
| kramaḥ | nominative | the order |
| punaḥ | indeclinable | a second question |
| yathā-audārika-saṃkleśa-bhājana-ādi-artha-dhātutaḥ | compressed adverbial compound with ablatival ending | according to grossness, affliction, vessel and the other roles, and Realm or level |

The second line names four grounds. The commentary unfolds them. They are not one label.

**Grammar tokens.** Verse-order IAST with sandhi opened and compound
boundaries marked. The final compressed compound stays one leaf; its
fourfold interpretation follows the Bhāṣya. No covert token is supplied.

```text
# tokens: skandheṣu asaṃskṛtam na uktam artha-ayogāt kramaḥ punaḥ yathā-audārika-saṃkleśa-bhājana-ādi-artha-dhātutaḥ
```

## 5. Translation

### Literal Translation

> The unconditioned is not stated among the bases, because the meaning does
> not apply. The order, again, is according to grossness, affliction, the
> meaning beginning with a vessel, and Realm or level.

### Bhāṣya-informed study translation

> The unconditioned is not stated among the Bases, since the meaning of
> collection does not apply. The order, however, is according to grossness,
> affliction, the meanings beginning with a vessel, and the Realms and levels.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| asaṃskṛta | unconditioned | not a base |
| artha-ayoga | the meaning does not apply | collection does not fit |
| rāśi | collection | the meaning of base |
| audārika | gross | first ground of order |
| saṃkleśa | affliction | second ground; direction runs back |
| bhājana | vessel | form |
| bhojana | food | feeling; the recap says vessel |
| vyañjana | seasoning | reflection |
| kartṛ | maker | formations |
| bhoktṛ | eater | the cognition |
| vijñāna-sthiti | station of the cognition | the levels |
| kṣetra-bīja | field and seed | the sequence displays this |
| dhātutaḥ | according to the levels | not the eighteen |
| citta | consciousness | afflicted by the afflictions |
| bhavāgra | summit of existence | formations predominate |

In [45]:
fcfg_text = r"""% start S
# tokens: skandheṣu asaṃskṛtam na uktam artha-ayogāt kramaḥ punaḥ yathā-audārika-saṃkleśa-bhājana-ādi-artha-dhātutaḥ
S[SEM=<(?exclusion & ?ordering)>] -> EXCLUSION[SEM=?exclusion] ORDERING[SEM=?ordering]
EXCLUSION[SEM=<all x.(?unconditioned(x) -> (?negative(?stated(?bases))(x) & ?reason(x)))>] -> BASES[CASE=loc,SEM=?bases] UNCONDITIONED[CASE=nom,NUM=?n,GEN=?g,SEM=?unconditioned] NEG[SEM=?negative] STATED[CASE=nom,NUM=?n,GEN=?g,SEM=?stated] REASON[CASE=abl,SEM=?reason]
ORDERING[SEM=<?grounds(?order)>] -> ORDER[CASE=nom,NUM=sg,SEM=?order] RESUME GROUNDS[SEM=?grounds]
BASES[CASE=loc,NUM=pl,SEM=<\x.BaseMember(x)>] -> 'skandheṣu'
UNCONDITIONED[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Unconditioned(x)>] -> 'asaṃskṛtam'
NEG[SEM=<\P.\x.-P(x)>] -> 'na'
STATED[CASE=nom,NUM=sg,GEN=n,SEM=<\P.P>] -> 'uktam'
REASON[CASE=abl,SEM=<\x.CollectionMeaningInapplicable(x)>] -> 'artha-ayogāt'
ORDER[CASE=nom,NUM=sg,GEN=m,SEM=<base_order>] -> 'kramaḥ'
RESUME -> 'punaḥ'
GROUNDS[SEM=<\o.(OrderingGround(o,grossness) & OrderingGround(o,affliction) & OrderingGround(o,meal_roles) & OrderingGround(o,levels))>] -> 'yathā-audārika-saṃkleśa-bhājana-ādi-artha-dhātutaḥ'
"""

dependency_text = r"""'uktam' -> 'skandheṣu' | 'asaṃskṛtam' | 'na' | 'artha-ayogāt'
'kramaḥ' -> 'punaḥ' | 'yathā-audārika-saṃkleśa-bhājana-ādi-artha-dhātutaḥ'
"""

sem_text = r"""# derived by the section 4 parse
((all x.(Unconditioned(x) -> (-BaseMember(x) & CollectionMeaningInapplicable(x)))) & (OrderingGround(base_order,grossness) & OrderingGround(base_order,affliction) & OrderingGround(base_order,meal_roles) & OrderingGround(base_order,levels)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
((all x.(Unconditioned(x) -> (-BaseMember(x) & CollectionMeaningInapplicable(x)))) & (OrderingGround(base_order,grossness) & OrderingGround(base_order,affliction) & OrderingGround(base_order,meal_roles) & OrderingGround(base_order,levels)))
# THEORY prose
# EXPECT prose T
all x.(Unconditioned(x) -> (DharmaEssenceMember(x) & DharmaPrincipleMember(x)))
Precedes(form,feeling) & Precedes(feeling,reflection) & Precedes(reflection,formations) & Precedes(formations,cognition)
MealRole(form,vessel) & MealRole(feeling,food) & MealRole(reflection,seasoning) & MealRole(formations,maker) & MealRole(cognition,eater)
CharacterizedBy(desire_realm,form) & CharacterizedBy(dhyanas,feeling) & CharacterizedBy(first_three_formless,reflection) & CharacterizedBy(summit,formations)
# THEORY rejected_excluded_from_the_system
# EXPECT rejected_excluded_from_the_system F
all x.(Unconditioned(x) -> -DharmaEssenceMember(x))
# THEORY rejected_sixth_base
# EXPECT rejected_sixth_base F
exists x.(Unconditioned(x) & BaseMember(x))
# THEORY rejected_meal_recap_replaces_food
# EXPECT rejected_meal_recap_replaces_food F
all x.(x = feeling -> MealRole(x,vessel))
"""

valuation_text = r"""base_order => five_base_sequence
grossness => grossness_account
affliction => affliction_account
meal_roles => meal_account
levels => realm_level_account
form => form_base
feeling => feeling_base
reflection => reflection_base
formations => formations_base
cognition => cognition_base
vessel => vessel_role
food => food_role
seasoning => seasoning_role
maker => maker_role
eater => eater_role
desire_realm => desire_level
dhyanas => dhyana_levels
first_three_formless => lower_formless_levels
summit => summit_level
Unconditioned => {space_case,discrimination_cessation,nonarising_cessation}
CollectionMeaningInapplicable => {space_case,discrimination_cessation,nonarising_cessation}
BaseMember => {form_case,feeling_case,reflection_case,formation_case,cognition_case}
DharmaEssenceMember => {space_case,discrimination_cessation,nonarising_cessation}
DharmaPrincipleMember => {space_case,discrimination_cessation,nonarising_cessation}
OrderingGround => {(five_base_sequence,grossness_account),(five_base_sequence,affliction_account),(five_base_sequence,meal_account),(five_base_sequence,realm_level_account)}
BaseKind => {form_base,feeling_base,reflection_base,formations_base,cognition_base}
Precedes => {(form_base,feeling_base),(feeling_base,reflection_base),(reflection_base,formations_base),(formations_base,cognition_base)}
MealRole => {(form_base,vessel_role),(feeling_base,food_role),(reflection_base,seasoning_role),(formations_base,maker_role),(cognition_base,eater_role)}
CharacterizedBy => {(desire_level,form_base),(dhyana_levels,feeling_base),(lower_formless_levels,reflection_base),(summit_level,formations_base)}
"""


In [46]:
studies[22] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[22]["tree"].pformat())
print("Derived SEM:", studies[22]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_excluded_from_the_system=F rejected_sixth_base=F rejected_meal_recap_replaces_food=F
(S[SEM=<(all x.(Unconditioned(x) -> (-BaseMember(x) & CollectionMeaningInapplicable(x))) & OrderingGround(base_order,grossness) & OrderingGround(base_order,affliction) & OrderingGround(base_order,meal_roles) & OrderingGround(base_order,levels))>]
  (EXCLUSION[SEM=<all x.(Unconditioned(x) -> (-BaseMember(x) & CollectionMeaningInapplicable(x)))>]
    (BASES[CASE='loc', NUM='pl', SEM=<\x.BaseMember(x)>] skandheṣu)
    (UNCONDITIONED[CASE='nom', GEN='n', NUM='sg', SEM=<\x.Unconditioned(x)>]
      asaṃskṛtam)
    (NEG[SEM=<\P x.-P(x)>] na)
    (STATED[CASE='nom', GEN='n', NUM='sg', SEM=<\P.P>] uktam)
    (REASON[CASE='abl', SEM=<\x.CollectionMeaningInapplicable(x)>]
      artha-ayogāt))
  (ORDERING[SEM=<(OrderingGround(base_order,grossness) & OrderingGround(base_order,affliction) & OrderingGround(base_order,meal_roles) & OrderingGround(base_order,levels))>]
    (ORDE

# VAK_1.23 — The Order of the Six

Source: `VAK_1.23.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
prāk pañca vārttamānārthyāt bhautikārthyāc catuṣṭayam /
dūrāśutaravṛttyānyat yathāsthānaṃ kramo 'thavā // 1.23 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| prāk | indeclinable | first; carried forward to the second clause |
| pañca | numeral | the five, beginning with the eye |
| vārttamāna-arthyāt | ablative abstract | because their Condition is present |
| bhautika-arthyāt | ablative abstract | because their Condition is Element-dependent Form |
| catuṣṭayam | nominative neuter singular collective | the four |
| dūra-āśutara-vṛttyā | instrumental feminine singular compound | by farther or quicker operation |
| anyat | nominative neuter singular | the remainder |
| yathā-sthānam | adverbial compound | according to site |
| kramaḥ | nominative masculine singular | the order |
| athavā | particle | an alternative |

The long instrumental is kept. The prose repeats it.

**Grammar tokens.** Verse-order IAST with sandhi opened and compound
boundaries marked. *Prāk* is not repeated as a token in the second clause;
the Bhāṣya's *prāg iti vartate* ("first" carries forward) is resolved in
the second reason's semantics.

```text
# tokens: prāk pañca vārttamāna-arthyāt bhautika-arthyāt catuṣṭayam dūra-āśutara-vṛttyā anyat yathā-sthānam kramaḥ athavā
```

## 5. Translation

### Literal Translation

> The five come first, because their Conditions are present. The four, because
> their Conditions are Element-dependent Form. The rest, by farther or quicker
> operation. Or the order is according to location.

### Bhāṣya-informed study translation

> The five Faculties come first because their Conditions are present. Of
> these, four precede the Body because they take Element-dependent Form as
> Condition. The remaining order follows distance or quicker operation—or,
> alternatively, bodily location.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| pañca | the five | eye, ear, nose, tongue, body |
| vārttamāna-viṣaya | present Condition | why the five are first |
| manas | Mind | unrestricted range; no place |
| bhautika | Element-dependent Form | Condition of the four |
| bhūta | Element | possible Condition of Body |
| dūra-vṛtti | farther operation | eye before ear |
| āśutara-vṛtti | quicker operation | nose before tongue |
| adhiṣṭhāna | site | the alternative; not the Faculty |
| viṣaya | Condition | the range; not Object, not a finished thing |
| vijñāna | Cognition | follows the order of the six; not Principle |

In [47]:
fcfg_text = r"""% start S
# tokens: prāk pañca vārttamāna-arthyāt bhautika-arthyāt catuṣṭayam dūra-āśutara-vṛttyā anyat yathā-sthānam kramaḥ athavā
S[SEM=<(?first & ?second & ?third & ?alternative)>] -> FIRST[SEM=?first] SECOND[SEM=?second] THIRD[SEM=?third] ALT[SEM=?alternative]
FIRST[SEM=<?reason(?group)>] -> PRIOR GROUP[K=five,SEM=?group] REASON[CASE=abl,K=present,SEM=?reason]
SECOND[SEM=<?reason(?group)>] -> REASON[CASE=abl,K=bhautika,SEM=?reason] GROUP[K=four,SEM=?group]
THIRD[SEM=<?ground(?group)>] -> GROUND[CASE=ins,SEM=?ground] GROUP[K=rest,SEM=?group]
ALT[SEM=<?site(?order)>] -> SITE[SEM=?site] ORDER[CASE=nom,NUM=sg,SEM=?order] ALTERNATIVE
PRIOR -> 'prāk'
GROUP[K=five,SEM=<five_faculties>] -> 'pañca'
REASON[CASE=abl,K=present,SEM=<\g.PrecedesBecause(g,mind,present_conditions)>] -> 'vārttamāna-arthyāt'
REASON[CASE=abl,K=bhautika,SEM=<\g.PrecedesBecause(g,body,element_dependent_conditions)>] -> 'bhautika-arthyāt'
GROUP[K=four,SEM=<first_four>] -> 'catuṣṭayam'
GROUND[CASE=ins,SEM=<\g.(OrderedBy(g,farther_operation) & OrderedBy(g,quicker_operation))>] -> 'dūra-āśutara-vṛttyā'
GROUP[K=rest,SEM=<remaining_order>] -> 'anyat'
SITE[SEM=<\o.AlternativelyOrderedBy(o,site)>] -> 'yathā-sthānam'
ORDER[CASE=nom,NUM=sg,GEN=m,SEM=<six_order>] -> 'kramaḥ'
ALTERNATIVE -> 'athavā'
"""

dependency_text = r"""'pañca' -> 'prāk' | 'vārttamāna-arthyāt'
'catuṣṭayam' -> 'bhautika-arthyāt'
'anyat' -> 'dūra-āśutara-vṛttyā'
'kramaḥ' -> 'yathā-sthānam' | 'athavā'
"""

sem_text = r"""# derived by the section 4 parse
(PrecedesBecause(five_faculties,mind,present_conditions) & PrecedesBecause(first_four,body,element_dependent_conditions) & (OrderedBy(remaining_order,farther_operation) & OrderedBy(remaining_order,quicker_operation)) & AlternativelyOrderedBy(six_order,site))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(PrecedesBecause(five_faculties,mind,present_conditions) & PrecedesBecause(first_four,body,element_dependent_conditions) & (OrderedBy(remaining_order,farther_operation) & OrderedBy(remaining_order,quicker_operation)) & AlternativelyOrderedBy(six_order,site))
# THEORY prose
# EXPECT prose T
Precedes(eye,ear) & Precedes(ear,nose) & Precedes(nose,tongue) & Precedes(tongue,body) & Precedes(body,mind)
all x.(InGroup(x,five_faculties) -> PresentConditionOnly(x))
-PresentConditionOnly(mind) & TakesNonPresent(mind)
all x.(InGroup(x,first_four) -> InGroup(x,five_faculties))
all x.(InGroup(x,first_four) -> (TakesElementDependent(x) & -TakesGreatElement(x)))
TakesGreatElement(body) & TakesElementDependent(body)
DistantCondition(eye) & DistantCondition(ear) & -DistantCondition(nose) & -DistantCondition(tongue)
FartherOperation(eye,ear) & QuickerOperation(nose,tongue)
SiteAbove(eye,ear) & SiteAbove(ear,nose) & SiteAbove(nose,tongue)
Unlocalized(mind) & DependsOnFaculties(mind) & -Unconditioned(mind)
# THEORY rejected_mind_restricted_to_present
# EXPECT rejected_mind_restricted_to_present F
exists x.(x = mind & PresentConditionOnly(x))
# THEORY rejected_body_only_element_dependent
# EXPECT rejected_body_only_element_dependent F
all x.(x = body -> -TakesGreatElement(x))
# THEORY rejected_nose_operates_at_distance
# EXPECT rejected_nose_operates_at_distance F
exists x.(x = nose & DistantCondition(x))
# THEORY rejected_unlocalized_means_unconditioned
# EXPECT rejected_unlocalized_means_unconditioned F
all x.(Unlocalized(x) -> Unconditioned(x))
"""

valuation_text = r"""five_faculties => five_group
first_four => four_group
remaining_order => remainder_comparisons
six_order => sixfold_sequence
present_conditions => present_ground
element_dependent_conditions => element_dependent_ground
farther_operation => farther_ground
quicker_operation => quicker_ground
site => site_ground
eye => eye_faculty
ear => ear_faculty
nose => nose_faculty
tongue => tongue_faculty
body => body_faculty
mind => mind_faculty
PrecedesBecause => {(five_group,mind_faculty,present_ground),(four_group,body_faculty,element_dependent_ground)}
OrderedBy => {(remainder_comparisons,farther_ground),(remainder_comparisons,quicker_ground)}
AlternativelyOrderedBy => {(sixfold_sequence,site_ground)}
Precedes => {(eye_faculty,ear_faculty),(ear_faculty,nose_faculty),(nose_faculty,tongue_faculty),(tongue_faculty,body_faculty),(body_faculty,mind_faculty)}
InGroup => {(eye_faculty,five_group),(ear_faculty,five_group),(nose_faculty,five_group),(tongue_faculty,five_group),(body_faculty,five_group),(eye_faculty,four_group),(ear_faculty,four_group),(nose_faculty,four_group),(tongue_faculty,four_group)}
PresentConditionOnly => {eye_faculty,ear_faculty,nose_faculty,tongue_faculty,body_faculty}
TakesNonPresent => {mind_faculty}
TakesElementDependent => {eye_faculty,ear_faculty,nose_faculty,tongue_faculty,body_faculty}
TakesGreatElement => {body_faculty}
DistantCondition => {eye_faculty,ear_faculty}
FartherOperation => {(eye_faculty,ear_faculty)}
QuickerOperation => {(nose_faculty,tongue_faculty)}
SiteAbove => {(eye_faculty,ear_faculty),(ear_faculty,nose_faculty),(nose_faculty,tongue_faculty)}
Unlocalized => {mind_faculty}
DependsOnFaculties => {mind_faculty}
Unconditioned => {}
"""


In [48]:
studies[23] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[23]["tree"].pformat())
print("Derived SEM:", studies[23]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_mind_restricted_to_present=F rejected_body_only_element_dependent=F rejected_nose_operates_at_distance=F rejected_unlocalized_means_unconditioned=F
(S[SEM=<(PrecedesBecause(five_faculties,mind,present_conditions) & PrecedesBecause(first_four,body,element_dependent_conditions) & OrderedBy(remaining_order,farther_operation) & OrderedBy(remaining_order,quicker_operation) & AlternativelyOrderedBy(six_order,site))>]
  (FIRST[SEM=<PrecedesBecause(five_faculties,mind,present_conditions)>]
    (PRIOR[] prāk)
    (GROUP[K='five', SEM=<five_faculties>] pañca)
    (REASON[CASE='abl', K='present', SEM=<\g.PrecedesBecause(g,mind,present_conditions)>]
      vārttamāna-arthyāt))
  (SECOND[SEM=<PrecedesBecause(first_four,body,element_dependent_conditions)>]
    (REASON[CASE='abl', K='bhautika', SEM=<\g.PrecedesBecause(g,body,element_dependent_conditions)>]
      bhautika-arthyāt)
    (GROUP[K='four', SEM=<first_four>] catuṣṭayam))
  (THIRD[SEM=<(OrderedBy(remai

# VAK_1.24 — Form and Dharma

Source: `VAK_1.24.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
viśeṣaṇārthaṃ prādhānyād bahudharmāgrasaṃgrahāt /
ekam āyatanaṃ rūpam ekaṃ dharmākhyam ucyate // 1.24 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| viśeṣaṇa-artham | viśeṣaṇa + artham; accusative neuter singular, adverbial | for the sake of distinction; applies to both names |
| prādhānyāt | ablative singular | because of primacy; grounds the name Form |
| bahu-dharma-agra-saṃgrahāt | bahu-dharma-agra-saṃgrahāt; ablative singular | because many dharmas, and the highest, are gathered; grounds the name Dharma |
| ekam | nominative neuter singular | one; first occurrence |
| āyatanam | nominative neuter singular | Essence; understood again with the second *ekam* |
| rūpam | nominative neuter singular, predicate | called Form |
| ekam | nominative neuter singular | one; second occurrence |
| dharma-ākhyam | dharma + ākhyā, bahuvrīhi; nominative neuter singular | having the designation Dharma |
| ucyate | third person singular present passive | is called; one verb, both names |

The sandhi `prādhānyād` voices `t` before `b`. The compound gathers two things, not one: many dharmas, and the highest dharma. The verse does not call the many highest.

**Grammar tokens.** Verse-order IAST with sandhi opened and compound
boundaries marked. *Āyatanam* is not repeated as a token; the second
*ekam* inherits it through agreement, and the parse supplies the
restrictor in the rule rather than as a covert leaf.

```text
# tokens: viśeṣaṇa-artham prādhānyāt bahu-dharma-agra-saṃgrahāt ekam āyatanam rūpam ekam dharma-ākhyam ucyate
```

## 5. Translation

### Literal Translation

> For the sake of distinction, because of primacy, and because many dharmas
> and the highest are gathered, one Essence is called Form, and one is
> called Dharma.

### Bhāṣya-informed study translation

> For distinction, one Essence is called Form because of primacy, and one is
> called Dharma because it gathers many dharmas and the highest.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| āyatana | Essence | one called Form; one called Dharma |
| rūpa | Form | the name, and the Form Base that holds the ten |
| dharma | Dharma | the name of the gathering; not a second word for Essence |
| viśeṣaṇārtham | for distinction | both names |
| prādhānya | primacy | the name Form |
| bahu-dharma-saṃgraha | gathering of many dharmas | beginning with Feeling |
| agra | the highest | nirvāṇa; not a name for the many |
| rūpa-skandha | Form Base | includes the ten; is not the one Essence called Form |
| viṣaya-viṣayin | Condition and what has the Condition | how the ten are arranged |
| sapratigha | resistant | shared by the ten; joined with showability for primacy |
| sanidarśana | able to be shown | primacy; only the Form Essence |
| vedanā | Feeling | first of the many gathered |
| saṃjñā | Reflection | gathered under the Dharma name; not manas |
| saṃskāra | Formation | gathered under the Dharma name |
| nirvāṇa | nirvāṇa | the highest dharma |

In [49]:
fcfg_text = r"""% start S
# tokens: viśeṣaṇa-artham prādhānyāt bahu-dharma-agra-saṃgrahāt ekam āyatanam rūpam ekam dharma-ākhyam ucyate
S[SEM=<(?one1(?essence)(?called(?formname))(\x.(?purpose(x) & ?primacy(x))) & ?one2(?essence)(?called(?dharmaname))(\x.(?purpose(x) & ?gathering(x))))>] -> PURPOSE[CASE=acc,SEM=?purpose] PRIMACY[CASE=abl,SEM=?primacy] GATHERING[CASE=abl,SEM=?gathering] ONE[CASE=nom,NUM=?n,GEN=?g,SEM=?one1] ESSENCE[CASE=nom,NUM=?n,GEN=?g,SEM=?essence] FORMNAME[CASE=nom,NUM=?n,GEN=?g,SEM=?formname] ONE[CASE=nom,NUM=?n,GEN=?g,SEM=?one2] DHARMANAME[CASE=nom,NUM=?n,GEN=?g,SEM=?dharmaname] CALLED[NUM=?n,SEM=?called]
PURPOSE[CASE=acc,SEM=<\x.NamedForDistinction(x)>] -> 'viśeṣaṇa-artham'
PRIMACY[CASE=abl,SEM=<\x.NamedForPrimacy(x)>] -> 'prādhānyāt'
GATHERING[CASE=abl,SEM=<\x.(GathersMany(x) & GathersHighest(x))>] -> 'bahu-dharma-agra-saṃgrahāt'
ONE[CASE=nom,NUM=sg,GEN=n,SEM=<\P Q R.exists x.(P(x) & Q(x) & R(x) & all y.((P(y) & Q(y)) -> (y = x)))>] -> 'ekam'
ESSENCE[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Essence(x)>] -> 'āyatanam'
FORMNAME[CASE=nom,NUM=sg,GEN=n,SEM=<form_name>] -> 'rūpam'
DHARMANAME[CASE=nom,NUM=sg,GEN=n,SEM=<dharma_name>] -> 'dharma-ākhyam'
CALLED[NUM=sg,SEM=<\n x.CalledBy(x,n)>] -> 'ucyate'
"""

dependency_text = r"""'ucyate' -> 'āyatanam' | 'rūpam' | 'dharma-ākhyam' | 'viśeṣaṇa-artham' | 'prādhānyāt' | 'bahu-dharma-agra-saṃgrahāt'
'āyatanam' -> 'ekam'
'dharma-ākhyam' -> 'ekam'
"""

sem_text = r"""# derived by the section 4 parse
(exists x.(Essence(x) & CalledBy(x,form_name) & NamedForDistinction(x) & NamedForPrimacy(x) & all y.((Essence(y) & CalledBy(y,form_name)) -> (y = x))) & exists x.(Essence(x) & CalledBy(x,dharma_name) & NamedForDistinction(x) & GathersMany(x) & GathersHighest(x) & all y.((Essence(y) & CalledBy(y,dharma_name)) -> (y = x))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(exists x.(Essence(x) & CalledBy(x,form_name) & NamedForDistinction(x) & NamedForPrimacy(x) & all y.((Essence(y) & CalledBy(y,form_name)) -> (y = x))) & exists x.(Essence(x) & CalledBy(x,dharma_name) & NamedForDistinction(x) & GathersMany(x) & GathersHighest(x) & all y.((Essence(y) & CalledBy(y,dharma_name)) -> (y = x))))
# THEORY prose
# EXPECT prose T
all x.(InFormBase(x) -> (Essence(x) & Resistant(x)))
all x.((InFormBase(x) & -(x = form_essence)) -> HasSpecificName(x))
exists x.(x = form_essence & InFormBase(x) & -HasSpecificName(x))
all x.(Showable(x) -> (x = form_essence))
Showable(form_essence) & WorldlyForm(form_essence) & Resistant(form_essence)
CorrespondsTo(eye,form_essence) & CorrespondsTo(ear,sound) & CorrespondsTo(nose,odor) & CorrespondsTo(tongue,taste) & CorrespondsTo(body,tangible)
Base(form_base) & -Essence(form_base) & -(form_base = form_essence)
all x.(Essence(x) -> DharmaByNature(x))
Gathers(dharma_essence,feeling) & Gathers(dharma_essence,reflection) & Gathers(dharma_essence,formations) & Gathers(dharma_essence,nirvana)
Base(feeling) & Base(reflection) & Base(formations) & -Base(nirvana)
Highest(nirvana) & -Highest(feeling) & -Highest(reflection) & -Highest(formations)
all x.((Essence(x) & Gathers(x,nirvana)) -> (x = dharma_essence))
AttributedToOthers(three_eye_account) & -GivenInVerse(three_eye_account)
# THEORY rejected_form_base_is_form_essence
# EXPECT rejected_form_base_is_form_essence F
exists x.(x = form_base & x = form_essence)
# THEORY rejected_every_essence_called_dharma
# EXPECT rejected_every_essence_called_dharma F
all x.(Essence(x) -> CalledBy(x,dharma_name))
# THEORY rejected_primacy_names_dharma
# EXPECT rejected_primacy_names_dharma F
exists x.(x = dharma_essence & NamedForPrimacy(x))
# THEORY rejected_many_are_highest
# EXPECT rejected_many_are_highest F
all x.(Gathers(dharma_essence,x) -> Highest(x))
# THEORY rejected_only_faculties_already_named
# EXPECT rejected_only_faculties_already_named F
all x.((InFormBase(x) & HasSpecificName(x)) -> SenseFaculty(x))
# THEORY rejected_resistance_alone_distinguishes
# EXPECT rejected_resistance_alone_distinguishes F
all x.(Resistant(x) -> (x = form_essence))
"""

valuation_text = r"""form_name => form_designation
dharma_name => dharma_designation
form_essence => form_ess
dharma_essence => dharma_ess
form_base => form_skandha
eye => eye_ess
ear => ear_ess
nose => nose_ess
tongue => tongue_ess
body => body_ess
sound => sound_ess
odor => odor_ess
taste => taste_ess
tangible => tangible_ess
feeling => feeling_base
reflection => reflection_base
formations => formations_base
nirvana => nirvana_dharma
three_eye_account => others_three_eyes
Essence => {eye_ess,ear_ess,nose_ess,tongue_ess,body_ess,form_ess,sound_ess,odor_ess,taste_ess,tangible_ess,mind_ess,dharma_ess}
CalledBy => {(form_ess,form_designation),(dharma_ess,dharma_designation)}
NamedForDistinction => {form_ess,dharma_ess}
NamedForPrimacy => {form_ess}
GathersMany => {dharma_ess}
GathersHighest => {dharma_ess}
InFormBase => {eye_ess,ear_ess,nose_ess,tongue_ess,body_ess,form_ess,sound_ess,odor_ess,taste_ess,tangible_ess}
HasSpecificName => {eye_ess,ear_ess,nose_ess,tongue_ess,body_ess,sound_ess,odor_ess,taste_ess,tangible_ess}
SenseFaculty => {eye_ess,ear_ess,nose_ess,tongue_ess,body_ess}
Resistant => {eye_ess,ear_ess,nose_ess,tongue_ess,body_ess,form_ess,sound_ess,odor_ess,taste_ess,tangible_ess}
Showable => {form_ess}
WorldlyForm => {form_ess}
CorrespondsTo => {(eye_ess,form_ess),(ear_ess,sound_ess),(nose_ess,odor_ess),(tongue_ess,taste_ess),(body_ess,tangible_ess)}
Base => {form_skandha,feeling_base,reflection_base,formations_base}
DharmaByNature => {eye_ess,ear_ess,nose_ess,tongue_ess,body_ess,form_ess,sound_ess,odor_ess,taste_ess,tangible_ess,mind_ess,dharma_ess}
Gathers => {(dharma_ess,feeling_base),(dharma_ess,reflection_base),(dharma_ess,formations_base),(dharma_ess,nirvana_dharma)}
Highest => {nirvana_dharma}
AttributedToOthers => {others_three_eyes}
GivenInVerse => {}
"""


In [50]:
studies[24] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[24]["tree"].pformat())
print("Derived SEM:", studies[24]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_form_base_is_form_essence=F rejected_every_essence_called_dharma=F rejected_primacy_names_dharma=F rejected_many_are_highest=F rejected_only_faculties_already_named=F rejected_resistance_alone_distinguishes=F
(S[SEM=<(exists x.(Essence(x) & CalledBy(x,form_name) & NamedForDistinction(x) & NamedForPrimacy(x) & all y.((Essence(y) & CalledBy(y,form_name)) -> (y = x))) & exists x.(Essence(x) & CalledBy(x,dharma_name) & NamedForDistinction(x) & GathersMany(x) & GathersHighest(x) & all y.((Essence(y) & CalledBy(y,dharma_name)) -> (y = x))))>]
  (PURPOSE[CASE='acc', SEM=<\x.NamedForDistinction(x)>]
    viśeṣaṇa-artham)
  (PRIMACY[CASE='abl', SEM=<\x.NamedForPrimacy(x)>] prādhānyāt)
  (GATHERING[CASE='abl', SEM=<\x.(GathersMany(x) & GathersHighest(x))>]
    bahu-dharma-agra-saṃgrahāt)
  (ONE[CASE='nom', GEN='n', NUM='sg', SEM=<\P Q R.exists x.(P(x) & Q(x) & R(x) & all y.((P(y) & Q(y)) -> (y = x)))>]
    ekam)
  (ESSENCE[CASE='nom', GEN='n', NUM='sg', SE

# VAK_1.25 — Speech or Name

Source: `VAK_1.25.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
dharmaskandhasahasrāṇi yāny aśītiṃ jagau muniḥ /
tāni vāṅnāma vety eṣāṃ rūpasaṃskārasaṃgrahaḥ // 1.25 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| dharma-skandha-sahasrāṇi | dharma-skandha-sahasrāṇi; nominative/accusative neuter plural | Dharma-collections, in thousands; antecedent |
| yāni | relative; accusative neuter plural | which; object of *jagau* |
| aśītim | numeral; accusative singular | eighty; with the thousands, eighty thousand |
| jagau | √gai, perfect, third person singular | proclaimed |
| muniḥ | nominative masculine singular | the Sage; agent |
| tāni | correlative; nominative neuter plural | those; resumes *yāni* |
| vāk | nominative feminine singular | speech; first account |
| nāma | nominative neuter singular | name; second account |
| vā | particle | or; keeps the two accounts as alternatives |
| iti | particle | closes the report of the two accounts |
| eṣām | genitive plural | of these; the collections |
| rūpa-saṃskāra-saṃgrahaḥ | rūpa-saṃskāra-saṃgrahaḥ; nominative masculine singular | inclusion in Form and Formations, respectively |

`Skandha` in this compound is the collection of teaching. It is not a sixth Base.

**Grammar tokens.** Verse-order IAST with sandhi opened (`yāny` →
*yāni*, `vāṅnāma` → *vāk nāma*, `vety` → *vā iti*) and compound
boundaries marked. No covert token is supplied.

```text
# tokens: dharma-skandha-sahasrāṇi yāni aśītim jagau muniḥ tāni vāk nāma vā iti eṣām rūpa-saṃskāra-saṃgrahaḥ
```

## 5. Translation

### Literal Translation

> The eighty thousand Dharma-collections which the Sage proclaimed — speech,
> or name — their inclusion is in Form and Formations.

### Bhāṣya-informed study translation

> The Sage proclaimed eighty thousand Dharma-collections: they are included in
> the Form Base if Buddha-word is speech by nature, or in the Formations Base
> if it is name by nature.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| dharmaskandha | Dharma-collection | a teaching-collection; not a sixth Base |
| aśīti sahasrāṇi | eighty thousand | the number proclaimed |
| muni | the Sage | the one who proclaimed them |
| vāk | speech | one account of Buddha-word |
| nāma | name | the other account |
| buddhavacana | Buddha-word | speech by nature, or name by nature |
| rūpa-skandha | Form Base | inclusion on the speech account |
| saṃskāra-skandha | Formations Base | inclusion on the name account |
| saṃgraha | inclusion | in these very classifications |
| vyatireka | standing apart | denied |

In [51]:
fcfg_text = r"""% start S
# tokens: dharma-skandha-sahasrāṇi yāni aśītim jagau muniḥ tāni vāk nāma vā iti eṣām rūpa-saṃskāra-saṃgrahaḥ
S[SEM=<all c.((?coll(c) & ?rel(c)) -> ?pred(c))>] -> COLL[NUM=?n,GEN=?g,SEM=?coll] RELCL[NUM=?n,GEN=?g,SEM=?rel] PRED[NUM=?n,GEN=?g,SEM=?pred]
RELCL[NUM=?n,GEN=?g,SEM=<\c.(?num(c) & ?verb(?agent)(c))>] -> REL[CASE=acc,NUM=?n,GEN=?g] NUMBER[SEM=?num] VERB[NUM=?m,SEM=?verb] AGENT[CASE=nom,NUM=?m,SEM=?agent]
PRED[NUM=?n,GEN=?g,SEM=<\c.(?or(?first)(?second) & ?incl(?first)(?second)(c))>] -> CORREL[CASE=nom,NUM=?n,GEN=?g] ACCOUNT[K=speech,SEM=?first] ACCOUNT[K=name,SEM=?second] OR[SEM=?or] QUOTE GEN[CASE=gen,NUM=?n] INCL[CASE=nom,SEM=?incl]
COLL[NUM=pl,GEN=n,SEM=<\c.DharmaCollection(c)>] -> 'dharma-skandha-sahasrāṇi'
REL[CASE=acc,NUM=pl,GEN=n] -> 'yāni'
NUMBER[SEM=<\c.InTotal(c,eighty_thousand)>] -> 'aśītim'
VERB[NUM=sg,SEM=<\a c.Proclaimed(a,c)>] -> 'jagau'
AGENT[CASE=nom,NUM=sg,GEN=m,SEM=<sage>] -> 'muniḥ'
CORREL[CASE=nom,NUM=pl,GEN=n] -> 'tāni'
ACCOUNT[K=speech,SEM=<speech_nature>] -> 'vāk'
ACCOUNT[K=name,SEM=<name_nature>] -> 'nāma'
OR[SEM=<\a b.(ReportedAccount(a) & ReportedAccount(b) & Alternative(a,b))>] -> 'vā'
QUOTE -> 'iti'
GEN[CASE=gen,NUM=pl] -> 'eṣām'
INCL[CASE=nom,SEM=<\a b c.(IncludedUnder(c,a,form_base) & IncludedUnder(c,b,formations_base))>] -> 'rūpa-saṃskāra-saṃgrahaḥ'
"""

dependency_text = r"""'jagau' -> 'yāni' | 'aśītim' | 'muniḥ'
'dharma-skandha-sahasrāṇi' -> 'jagau'
'tāni' -> 'dharma-skandha-sahasrāṇi'
'rūpa-saṃskāra-saṃgrahaḥ' -> 'tāni' | 'eṣām' | 'iti'
'iti' -> 'vā'
'vā' -> 'vāk' | 'nāma'
"""

sem_text = r"""# derived by the section 4 parse
all c.((DharmaCollection(c) & InTotal(c,eighty_thousand) & Proclaimed(sage,c)) -> (ReportedAccount(speech_nature) & ReportedAccount(name_nature) & Alternative(speech_nature,name_nature) & IncludedUnder(c,speech_nature,form_base) & IncludedUnder(c,name_nature,formations_base)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
all c.((DharmaCollection(c) & InTotal(c,eighty_thousand) & Proclaimed(sage,c)) -> (ReportedAccount(speech_nature) & ReportedAccount(name_nature) & Alternative(speech_nature,name_nature) & IncludedUnder(c,speech_nature,form_base) & IncludedUnder(c,name_nature,formations_base)))
# THEORY prose
# EXPECT prose T
all x.(ScripturalDesignation(x) -> (IncludedInEstablished(x) & -StandsApart(x)))
all c.(DharmaCollection(c) -> (-Base(c) & ScripturalDesignation(c)))
all c.(DharmaCollection(c) -> (-IncludedUnder(c,speech_nature,formations_base) & -IncludedUnder(c,name_nature,form_base)))
all h.(HoldsSpeech(h) -> all c.(DharmaCollection(c) -> IncludedFor(h,c,form_base)))
all h.(HoldsName(h) -> all c.(DharmaCollection(c) -> IncludedFor(h,c,formations_base)))
-(form_base = formations_base) & Base(form_base) & Base(formations_base)
all a.-Selected(a)
# THEORY rejected_collection_is_further_base
# EXPECT rejected_collection_is_further_base F
exists c.(DharmaCollection(c) & Base(c))
# THEORY rejected_scriptural_name_stands_apart
# EXPECT rejected_scriptural_name_stands_apart F
exists x.(ScripturalDesignation(x) & StandsApart(x))
# THEORY rejected_cross_assignment
# EXPECT rejected_cross_assignment F
all c.(DharmaCollection(c) -> IncludedUnder(c,speech_nature,formations_base))
# THEORY rejected_one_account_selected
# EXPECT rejected_one_account_selected F
exists a.(ReportedAccount(a) & Selected(a))
# THEORY rejected_each_holder_includes_in_both
# EXPECT rejected_each_holder_includes_in_both F
exists h c.(IncludedFor(h,c,form_base) & IncludedFor(h,c,formations_base))
"""

valuation_text = r"""sage => muni
eighty_thousand => eighty_thousand_total
speech_nature => speech_account
name_nature => name_account
form_base => rupa_skandha
formations_base => samskara_skandha
DharmaCollection => {collection_a,collection_b}
InTotal => {(collection_a,eighty_thousand_total),(collection_b,eighty_thousand_total)}
Proclaimed => {(muni,collection_a),(muni,collection_b)}
ReportedAccount => {speech_account,name_account}
Alternative => {(speech_account,name_account)}
IncludedUnder => {(collection_a,speech_account,rupa_skandha),(collection_a,name_account,samskara_skandha),(collection_b,speech_account,rupa_skandha),(collection_b,name_account,samskara_skandha)}
ScripturalDesignation => {collection_a,collection_b,other_sutra_designation}
IncludedInEstablished => {collection_a,collection_b,other_sutra_designation}
StandsApart => {}
Base => {rupa_skandha,samskara_skandha}
HoldsSpeech => {speech_holder}
HoldsName => {name_holder}
IncludedFor => {(speech_holder,collection_a,rupa_skandha),(speech_holder,collection_b,rupa_skandha),(name_holder,collection_a,samskara_skandha),(name_holder,collection_b,samskara_skandha)}
Selected => {}
"""


In [52]:
studies[25] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[25]["tree"].pformat())
print("Derived SEM:", studies[25]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_collection_is_further_base=F rejected_scriptural_name_stands_apart=F rejected_cross_assignment=F rejected_one_account_selected=F rejected_each_holder_includes_in_both=F
(S[SEM=<all c.((DharmaCollection(c) & InTotal(c,eighty_thousand) & Proclaimed(sage,c)) -> (ReportedAccount(speech_nature) & ReportedAccount(name_nature) & Alternative(speech_nature,name_nature) & IncludedUnder(c,speech_nature,form_base) & IncludedUnder(c,name_nature,formations_base)))>]
  (COLL[GEN='n', NUM='pl', SEM=<\c.DharmaCollection(c)>]
    dharma-skandha-sahasrāṇi)
  (RELCL[GEN='n', NUM='pl', SEM=<\c.(InTotal(c,eighty_thousand) & Proclaimed(sage,c))>]
    (REL[CASE='acc', GEN='n', NUM='pl'] yāni)
    (NUMBER[SEM=<\c.InTotal(c,eighty_thousand)>] aśītim)
    (VERB[NUM='sg', SEM=<\a c.Proclaimed(a,c)>] jagau)
    (AGENT[CASE='nom', GEN='m', NUM='sg', SEM=<sage>] muniḥ))
  (PRED[GEN='n', NUM='pl', SEM=<\c.(ReportedAccount(speech_nature) & ReportedAccount(name_nature) & Alterna

# VAK_1.26 — The Measure of a Dharma-Collection

Source: `VAK_1.26.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
śāstrapramāṇa ity eke skandhādīnāṃ kathaikaśaḥ /
caritapratipakṣas tu dharmaskandho 'nuvarṇitaḥ // 1.26 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| śāstra-pramāṇaḥ | śāstra-pramāṇa, bahuvrīhi; nominative masculine singular | having a treatise as its measure; said of the Dharma-collection |
| iti | quotative particle | closes the reported account |
| eke | nominative masculine plural | some; holders of the first account |
| skandha-ādīnām | skandha-ādi; genitive plural | of the Bases and the rest; the topics |
| kathā | nominative feminine singular | exposition; the second measure |
| ekaśaḥ | distributive adverb | each separately |
| carita-pratipakṣaḥ | carita-pratipakṣa; nominative masculine singular | counteragent to a disposition; predicate |
| tu | particle | but; contrasts, does not refute |
| dharma-skandhaḥ | dharma-skandha; nominative masculine singular | a Dharma-collection; subject; not a sixth Base |
| anuvarṇitaḥ | anu-√varṇ, past passive participle; nominative masculine singular | has been described; agrees with the subject |

The research verse has `śāstrapramāṇā`. The running verse and the commentary read `śāstrapramāṇa`. That reading is kept. The long vowel is recorded, not adopted.

**Grammar tokens.** Verse-order IAST with sandhi opened (`kathaikaśaḥ` →
*kathā ekaśaḥ*, `'nuvarṇitaḥ` → *anuvarṇitaḥ*) and compound boundaries
marked. The second account's holders ("others") and the measured
Dharma-collection of the first two accounts are supplied by the Bhāṣya
inside the rules, not as covert leaves.

```text
# tokens: śāstra-pramāṇaḥ iti eke skandha-ādīnām kathā ekaśaḥ carita-pratipakṣaḥ tu dharma-skandhaḥ anuvarṇitaḥ
```

## 5. Translation

### Literal Translation

> Some say the measure is a treatise; an exposition of each of the Bases and
> the rest. But a Dharma-collection is described as a counteragent to a
> disposition.

### Bhāṣya-informed study translation

> Some say a treatise is the measure of a Dharma-collection; others, an
> exposition of each of the Bases and the rest. But a Dharma-collection is
> described as a counteragent to a disposition.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| dharmaskandha | Dharma-collection | a teaching-unit; not a sixth Base |
| pramāṇa | measure | extent; not a means of knowledge |
| śāstra | treatise | the work named Dharmaskandha |
| ṣaṭsahasrāṇi | six thousand | unit unstated |
| kathā | exposition | the second measure |
| ekaśaḥ | each separately | not one collective topic |
| skandha | Base | in the topic list; the five |
| āyatana | Essence | in the topic list; the twelve |
| dhātu | Principle | in the topic list; the eighteen |
| carita | disposition | attachment, hatred, delusion, pride, and the rest |
| pratipakṣa | counteragent | the third measure |
| aśītisahasra | eighty thousand | dispositions, and collections |

In [53]:
fcfg_text = r"""% start S
# tokens: śāstra-pramāṇaḥ iti eke skandha-ādīnām kathā ekaśaḥ carita-pratipakṣaḥ tu dharma-skandhaḥ anuvarṇitaḥ
S[SEM=<(?first & ?second & ?third)>] -> FIRST[SEM=?first] SECOND[SUPPLIED=prose,SEM=?second] THIRD[SEM=?third]
FIRST[SEM=<?holder(?measure)>] -> MEASURE[CASE=nom,NUM=sg,GEN=m,SEM=?measure] QUOTE HOLDER[CASE=nom,NUM=pl,SEM=?holder]
SECOND[SUPPLIED=prose,SEM=<(Holds(other_teachers,?exp) & MeasureOf(?exp,dharma_collection) & ?topics(?exp) & ?each(?exp))>] -> TOPICS[CASE=gen,NUM=pl,SEM=?topics] EXPOSITION[CASE=nom,NUM=sg,GEN=f,SEM=?exp] EACH[SEM=?each]
THIRD[SEM=<(?described(?coll)(?pred) & ?contrast(?pred))>] -> PRED[CASE=nom,NUM=?n,GEN=?g,SEM=?pred] TU[SEM=?contrast] COLL[CASE=nom,NUM=?n,GEN=?g,SEM=?coll] DESCRIBED[CASE=nom,NUM=?n,GEN=?g,SEM=?described]
MEASURE[CASE=nom,NUM=sg,GEN=m,SEM=<treatise_measure>] -> 'śāstra-pramāṇaḥ'
QUOTE -> 'iti'
HOLDER[CASE=nom,NUM=pl,GEN=m,SEM=<\m.(Holds(some_teachers,m) & MeasureOf(m,dharma_collection))>] -> 'eke'
TOPICS[CASE=gen,NUM=pl,SEM=<\m.ConcernsTopics(m,bases_and_rest)>] -> 'skandha-ādīnām'
EXPOSITION[CASE=nom,NUM=sg,GEN=f,SEM=<exposition_measure>] -> 'kathā'
EACH[SEM=<\m.Distributive(m)>] -> 'ekaśaḥ'
PRED[CASE=nom,NUM=sg,GEN=m,SEM=<counteragent_measure>] -> 'carita-pratipakṣaḥ'
TU[SEM=<\m.(ContrastsWith(m,treatise_measure) & ContrastsWith(m,exposition_measure))>] -> 'tu'
COLL[CASE=nom,NUM=sg,GEN=m,SEM=<dharma_collection>] -> 'dharma-skandhaḥ'
DESCRIBED[CASE=nom,NUM=sg,GEN=m,SEM=<\c m.(DescribedAs(c,m) & MeasureOf(m,c))>] -> 'anuvarṇitaḥ'
"""

dependency_text = r"""'eke' -> 'śāstra-pramāṇaḥ' | 'iti'
'kathā' -> 'skandha-ādīnām' | 'ekaśaḥ'
'anuvarṇitaḥ' -> 'dharma-skandhaḥ' | 'carita-pratipakṣaḥ' | 'tu'
"""

sem_text = r"""# derived by the section 4 parse
(Holds(some_teachers,treatise_measure) & MeasureOf(treatise_measure,dharma_collection) & Holds(other_teachers,exposition_measure) & MeasureOf(exposition_measure,dharma_collection) & ConcernsTopics(exposition_measure,bases_and_rest) & Distributive(exposition_measure) & DescribedAs(dharma_collection,counteragent_measure) & MeasureOf(counteragent_measure,dharma_collection) & ContrastsWith(counteragent_measure,treatise_measure) & ContrastsWith(counteragent_measure,exposition_measure))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(Holds(some_teachers,treatise_measure) & MeasureOf(treatise_measure,dharma_collection) & Holds(other_teachers,exposition_measure) & MeasureOf(exposition_measure,dharma_collection) & ConcernsTopics(exposition_measure,bases_and_rest) & Distributive(exposition_measure) & DescribedAs(dharma_collection,counteragent_measure) & MeasureOf(counteragent_measure,dharma_collection) & ContrastsWith(counteragent_measure,treatise_measure) & ContrastsWith(counteragent_measure,exposition_measure))
# THEORY prose
# EXPECT prose T
exists t.(t = dharmaskandha_treatise & NamedDharmaskandha(t) & Abhidharma(t) & HasCount(t,six_thousand))
exists n.(n = six_thousand & -UnitStated(n))
Topic(bases_topic) & Topic(essences_topic) & Topic(principles_topic)
all t.(Topic(t) -> exists x.(ExpositionOf(x,t) & DharmaCollection(x)))
all x t u.((ExpositionOf(x,t) & ExpositionOf(x,u)) -> (t = u))
all d.(Disposition(d) -> exists c.(Counteragent(c,d) & DharmaCollection(c) & Taught(blessed_one,c)))
all c d f.((Counteragent(c,d) & Counteragent(c,f)) -> (d = f))
all d.(Disposition(d) -> exists k.(DifferentiatedBy(d,k) & DispositionKind(k)))
DispositionKind(attachment) & DispositionKind(hatred) & DispositionKind(delusion) & DispositionKind(pride)
all m.-Refuted(m)
all c.(DharmaCollection(c) -> -Base(c))
# THEORY rejected_tu_refutes
# EXPECT rejected_tu_refutes F
exists m.(MeasureOf(m,dharma_collection) & Refuted(m))
# THEORY rejected_collection_is_base
# EXPECT rejected_collection_is_base F
exists c.(DharmaCollection(c) & Base(c))
# THEORY rejected_one_exposition_of_all_topics
# EXPECT rejected_one_exposition_of_all_topics F
exists x.all t.(Topic(t) -> ExpositionOf(x,t))
# THEORY rejected_unit_stated
# EXPECT rejected_unit_stated F
exists n.(n = six_thousand & UnitStated(n))
# THEORY rejected_pramana_as_means_of_knowledge
# EXPECT rejected_pramana_as_means_of_knowledge F
exists m.(MeasureOf(m,dharma_collection) & MeansOfKnowledge(m))
# THEORY rejected_one_counteragent_for_many
# EXPECT rejected_one_counteragent_for_many F
exists c d f.(Counteragent(c,d) & Counteragent(c,f) & -(d = f))
"""

valuation_text = r"""treatise_measure => treatise_account
exposition_measure => exposition_account
counteragent_measure => counteragent_account
dharma_collection => dharma_collection_kind
some_teachers => eke_holders
other_teachers => apare_holders
bases_and_rest => topic_list
dharmaskandha_treatise => dharmaskandha_sastra
six_thousand => six_thousand_count
bases_topic => skandha_topic
essences_topic => ayatana_topic
principles_topic => dhatu_topic
blessed_one => bhagavat
attachment => raga
hatred => dvesa
delusion => moha
pride => mana
Holds => {(eke_holders,treatise_account),(apare_holders,exposition_account)}
MeasureOf => {(treatise_account,dharma_collection_kind),(exposition_account,dharma_collection_kind),(counteragent_account,dharma_collection_kind)}
ConcernsTopics => {(exposition_account,topic_list)}
Distributive => {exposition_account}
ContrastsWith => {(counteragent_account,treatise_account),(counteragent_account,exposition_account)}
DescribedAs => {(dharma_collection_kind,counteragent_account)}
NamedDharmaskandha => {dharmaskandha_sastra}
Abhidharma => {dharmaskandha_sastra}
HasCount => {(dharmaskandha_sastra,six_thousand_count)}
UnitStated => {}
Topic => {skandha_topic,ayatana_topic,dhatu_topic}
ExpositionOf => {(skandha_exposition,skandha_topic),(ayatana_exposition,ayatana_topic),(dhatu_exposition,dhatu_topic)}
DharmaCollection => {skandha_exposition,ayatana_exposition,dhatu_exposition,counter_a,counter_b}
Disposition => {disposition_a,disposition_b}
Counteragent => {(counter_a,disposition_a),(counter_b,disposition_b)}
Taught => {(bhagavat,counter_a),(bhagavat,counter_b)}
DifferentiatedBy => {(disposition_a,raga),(disposition_b,dvesa)}
DispositionKind => {raga,dvesa,moha,mana}
Refuted => {}
Base => {rupa_skandha,vedana_skandha}
MeansOfKnowledge => {}
"""


In [54]:
studies[26] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[26]["tree"].pformat())
print("Derived SEM:", studies[26]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_tu_refutes=F rejected_collection_is_base=F rejected_one_exposition_of_all_topics=F rejected_unit_stated=F rejected_pramana_as_means_of_knowledge=F rejected_one_counteragent_for_many=F
(S[SEM=<(Holds(some_teachers,treatise_measure) & MeasureOf(treatise_measure,dharma_collection) & Holds(other_teachers,exposition_measure) & MeasureOf(exposition_measure,dharma_collection) & ConcernsTopics(exposition_measure,bases_and_rest) & Distributive(exposition_measure) & DescribedAs(dharma_collection,counteragent_measure) & MeasureOf(counteragent_measure,dharma_collection) & ContrastsWith(counteragent_measure,treatise_measure) & ContrastsWith(counteragent_measure,exposition_measure))>]
  (FIRST[SEM=<(Holds(some_teachers,treatise_measure) & MeasureOf(treatise_measure,dharma_collection))>]
    (MEASURE[CASE='nom', GEN='m', NUM='sg', SEM=<treatise_measure>]
      śāstra-pramāṇaḥ)
    (QUOTE[] iti)
    (HOLDER[CASE='nom', GEN='m', NUM='pl', SEM=<\m.(Holds(some_tea

# VAK_1.27 — Determine, Then Assign

Source: `VAK_1.27.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
tathānye 'pi yathāyogaṃ skandhāyatanadhātavaḥ /
pratipādyā yathokteṣu saṃpradhārya svalakṣaṇam // 1.27 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| tathā | indeclinable | likewise; the eighty thousand were already placed in the five Bases |
| anye | nominative masculine plural | others; designations in other sūtras |
| api | particle | also |
| yathā-yogam | avyayībhāva; adverbial | as appropriate to each |
| skandha-āyatana-dhātavaḥ | dvandva; nominative masculine plural | Bases, Essences, and Principles; subject |
| pratipādyāḥ | prati-√pad, causative gerundive; nominative masculine plural | are to be assigned; agrees with the subject |
| yathā-ukteṣu | locative plural | among those already stated |
| saṃpradhārya | sam-pra-√dhṛ, causative absolutive | having determined; precedes the assignment |
| sva-lakṣaṇam | sva-lakṣaṇa; accusative neuter singular | own-characteristic; object of the absolutive |

The commentary doubles the characteristic: each one's own. It is to be determined as set out in this treatise.

**Grammar tokens.** Verse-order IAST with sandhi opened (`tathānye` →
*tathā anye*, `'pi` → *api*, `yathokteṣu` → *yathā-ukteṣu*) and compound
boundaries marked. No covert token is supplied.

```text
# tokens: tathā anye api yathā-yogam skandha-āyatana-dhātavaḥ pratipādyāḥ yathā-ukteṣu saṃpradhārya sva-lakṣaṇam
```

## 5. Translation

### Literal Translation

> Likewise, other Bases, Essences, and Principles also are to be assigned,
> as appropriate, among those already stated, having determined the own-
> characteristic.

### Bhāṣya-informed study translation

> Likewise, other Bases, Essences, and Principles are to be assigned as
> appropriate among those already stated, after determining each one's own
> characteristic.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| tathā | likewise | the eighty thousand already placed in the five Bases |
| yathāyogam | as appropriate | not one rule for every name |
| yathokteṣu | among those already stated | five Bases, twelve Essences, eighteen Principles |
| saṃpradhārya | having determined | before assignment |
| svalakṣaṇa | own-characteristic | each one; as set out in this treatise |
| saparivāra | with attendants | a wider scope than the principal nature |
| śīlaskandha | discipline-collection | included in the Form Base |
| kṛtsnāyatana | totality | eight are non-greed |
| abhibhvāyatana | mastery | as the totalities |
| vimuktyāyatana | liberation | wisdom by nature |
| alobha | non-greed | principal nature of the eight |
| prajñā | wisdom | principal nature of the five liberations |
| asaṃjñisattva | non-percipient beings | ten Essences; Odor and Taste absent |
| bahudhātuka | discourse of many Principles | sixty-two; inclusion not listed |

In [55]:
fcfg_text = r"""% start S
# tokens: tathā anye api yathā-yogam skandha-āyatana-dhātavaḥ pratipādyāḥ yathā-ukteṣu saṃpradhārya sva-lakṣaṇam
S[SEM=<(?likewise & all x.((?others(x) & ?names(x)) -> exists k p.(?own(x)(k) & ?determined(k)(x) & ?among(p) & ?assign(x)(p) & ?manner(k)(p))))>] -> LIKEWISE[SEM=?likewise] OTHERS[CASE=nom,NUM=?n,GEN=?g,SEM=?others] ALSO MANNER[SEM=?manner] NAMES[CASE=nom,NUM=?n,GEN=?g,SEM=?names] GERUNDIVE[CASE=nom,NUM=?n,GEN=?g,SEM=?assign] AMONG[CASE=loc,NUM=pl,SEM=?among] ABS[SEM=?determined] OWN[CASE=acc,SEM=?own]
LIKEWISE[SEM=<PlacedIn(eighty_thousand,five_bases)>] -> 'tathā'
OTHERS[CASE=nom,NUM=pl,GEN=m,SEM=<\x.InOtherSutra(x)>] -> 'anye'
ALSO -> 'api'
MANNER[SEM=<\k p.FitsAsAppropriate(k,p)>] -> 'yathā-yogam'
NAMES[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(BaseName(x) | EssenceName(x) | PrincipleName(x))>] -> 'skandha-āyatana-dhātavaḥ'
GERUNDIVE[CASE=nom,NUM=pl,GEN=m,SEM=<\x p.ToBeAssigned(x,p)>] -> 'pratipādyāḥ'
AMONG[CASE=loc,NUM=pl,SEM=<\p.Established(p)>] -> 'yathā-ukteṣu'
ABS[SEM=<\k x.DeterminedBeforeAssignment(x,k)>] -> 'saṃpradhārya'
OWN[CASE=acc,NUM=sg,GEN=n,SEM=<\x k.OwnCharacteristic(x,k)>] -> 'sva-lakṣaṇam'
"""

dependency_text = r"""'pratipādyāḥ' -> 'skandha-āyatana-dhātavaḥ' | 'tathā' | 'yathā-yogam' | 'yathā-ukteṣu' | 'saṃpradhārya'
'skandha-āyatana-dhātavaḥ' -> 'anye'
'anye' -> 'api'
'saṃpradhārya' -> 'sva-lakṣaṇam'
"""

sem_text = r"""# derived by the section 4 parse
(PlacedIn(eighty_thousand,five_bases) & all x.((InOtherSutra(x) & (BaseName(x) | EssenceName(x) | PrincipleName(x))) -> exists k p.(OwnCharacteristic(x,k) & DeterminedBeforeAssignment(x,k) & Established(p) & ToBeAssigned(x,p) & FitsAsAppropriate(k,p))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(PlacedIn(eighty_thousand,five_bases) & all x.((InOtherSutra(x) & (BaseName(x) | EssenceName(x) | PrincipleName(x))) -> exists k p.(OwnCharacteristic(x,k) & DeterminedBeforeAssignment(x,k) & Established(p) & ToBeAssigned(x,p) & FitsAsAppropriate(k,p))))
# THEORY prose
# EXPECT prose T
ToBeAssigned(discipline_collection,form_base) & -ToBeAssigned(discipline_collection,formations_base)
ToBeAssigned(concentration_collection,formations_base) & ToBeAssigned(wisdom_collection,formations_base) & ToBeAssigned(liberation_collection,formations_base) & ToBeAssigned(knowledge_vision_collection,formations_base)
OwnCharacteristic(eight_totalities,non_greed) & ToBeAssigned(eight_totalities,dharma_essence)
OwnCharacteristic(five_liberations,wisdom) & ToBeAssigned(five_liberations,dharma_essence)
all x y.(AttendedOf(x,y) -> all p.(ToBeAssigned(y,p) -> ToBeAssigned(x,p)))
all x y.(AttendedOf(x,y) -> exists p.(ToBeAssigned(x,p) & -ToBeAssigned(y,p)))
all x p.(ToBeAssigned(x,p) -> Established(p))
-ToBeAssigned(nonpercipient_beings,odor_essence) & -ToBeAssigned(nonpercipient_beings,taste_essence) & ToBeAssigned(nonpercipient_beings,mind_essence)
ToBeAssigned(neither_perception_beings,mind_essence) & ToBeAssigned(neither_perception_beings,dharma_essence)
-Enumerated(sixty_two_principles) & KnownAsAppropriate(sixty_two_principles)
# THEORY rejected_assignment_without_determination
# EXPECT rejected_assignment_without_determination F
exists x p.(ToBeAssigned(x,p) & -exists k.DeterminedBeforeAssignment(x,k))
# THEORY rejected_new_place
# EXPECT rejected_new_place F
exists x p.(ToBeAssigned(x,p) & -Established(p))
# THEORY rejected_attendants_same_scope
# EXPECT rejected_attendants_same_scope F
all x y.(AttendedOf(x,y) -> all p.(ToBeAssigned(x,p) -> ToBeAssigned(y,p)))
# THEORY rejected_discipline_in_formations
# EXPECT rejected_discipline_in_formations F
exists x.(x = discipline_collection & ToBeAssigned(x,formations_base))
# THEORY rejected_mind_essence_means_cognition_occurring
# EXPECT rejected_mind_essence_means_cognition_occurring F
all x.(ToBeAssigned(x,mind_essence) -> CognitionOccurring(x))
# THEORY rejected_sixty_two_enumerated
# EXPECT rejected_sixty_two_enumerated F
exists x.(x = sixty_two_principles & Enumerated(x))
"""

valuation_text = r"""eighty_thousand => eighty_thousand_collections
five_bases => pancaskandha
discipline_collection => sila_coll
concentration_collection => samadhi_coll
wisdom_collection => prajna_coll
liberation_collection => vimukti_coll
knowledge_vision_collection => jnanadarsana_coll
eight_totalities => totalities8
five_liberations => liberations5
nonpercipient_beings => asamjnisattva
neither_perception_beings => naivasamjna_beings
sixty_two_principles => sixty_two
form_base => rupa_skandha
formations_base => samskara_skandha
dharma_essence => dharma_ayatana
mind_essence => mano_ayatana
odor_essence => gandha_ayatana
taste_essence => rasa_ayatana
non_greed => alobha
wisdom => prajna
PlacedIn => {(eighty_thousand_collections,pancaskandha)}
InOtherSutra => {sila_coll,samadhi_coll,prajna_coll,vimukti_coll,jnanadarsana_coll,totalities8,totalities8_att,masteries,masteries_att,space_cognition_totalities,formless4,liberations5,liberations5_att,asamjnisattva,naivasamjna_beings,sixty_two}
BaseName => {sila_coll,samadhi_coll,prajna_coll,vimukti_coll,jnanadarsana_coll}
EssenceName => {totalities8,totalities8_att,masteries,masteries_att,space_cognition_totalities,formless4,liberations5,liberations5_att,asamjnisattva,naivasamjna_beings}
PrincipleName => {sixty_two}
OwnCharacteristic => {(sila_coll,form_nature),(samadhi_coll,formation_nature),(prajna_coll,formation_nature),(vimukti_coll,formation_nature),(jnanadarsana_coll,formation_nature),(totalities8,alobha),(totalities8_att,five_bases_nature),(masteries,alobha),(masteries_att,five_bases_nature),(space_cognition_totalities,four_bases_nature),(formless4,four_bases_nature),(liberations5,prajna),(liberations5_att,liberation_complex),(asamjnisattva,ten_essence_nature),(naivasamjna_beings,mental_nature),(sixty_two,each_own_nature)}
DeterminedBeforeAssignment => {(sila_coll,form_nature),(samadhi_coll,formation_nature),(prajna_coll,formation_nature),(vimukti_coll,formation_nature),(jnanadarsana_coll,formation_nature),(totalities8,alobha),(totalities8_att,five_bases_nature),(masteries,alobha),(masteries_att,five_bases_nature),(space_cognition_totalities,four_bases_nature),(formless4,four_bases_nature),(liberations5,prajna),(liberations5_att,liberation_complex),(asamjnisattva,ten_essence_nature),(naivasamjna_beings,mental_nature),(sixty_two,each_own_nature)}
Established => {rupa_skandha,samskara_skandha,dharma_ayatana,mano_ayatana,sabda_ayatana,caksur_ayatana,srotra_ayatana,ghrana_ayatana,jihva_ayatana,kaya_ayatana,rupa_ayatana,sprastavya_ayatana,gandha_ayatana,rasa_ayatana,eighteen_principles}
ToBeAssigned => {(sila_coll,rupa_skandha),(samadhi_coll,samskara_skandha),(prajna_coll,samskara_skandha),(vimukti_coll,samskara_skandha),(jnanadarsana_coll,samskara_skandha),(totalities8,dharma_ayatana),(totalities8_att,mano_ayatana),(totalities8_att,dharma_ayatana),(masteries,dharma_ayatana),(masteries_att,mano_ayatana),(masteries_att,dharma_ayatana),(space_cognition_totalities,mano_ayatana),(space_cognition_totalities,dharma_ayatana),(formless4,mano_ayatana),(formless4,dharma_ayatana),(liberations5,dharma_ayatana),(liberations5_att,sabda_ayatana),(liberations5_att,mano_ayatana),(liberations5_att,dharma_ayatana),(asamjnisattva,caksur_ayatana),(asamjnisattva,srotra_ayatana),(asamjnisattva,ghrana_ayatana),(asamjnisattva,jihva_ayatana),(asamjnisattva,kaya_ayatana),(asamjnisattva,rupa_ayatana),(asamjnisattva,sabda_ayatana),(asamjnisattva,sprastavya_ayatana),(asamjnisattva,mano_ayatana),(asamjnisattva,dharma_ayatana),(naivasamjna_beings,mano_ayatana),(naivasamjna_beings,dharma_ayatana),(sixty_two,eighteen_principles)}
FitsAsAppropriate => {(form_nature,rupa_skandha),(formation_nature,samskara_skandha),(alobha,dharma_ayatana),(five_bases_nature,mano_ayatana),(five_bases_nature,dharma_ayatana),(four_bases_nature,mano_ayatana),(four_bases_nature,dharma_ayatana),(prajna,dharma_ayatana),(liberation_complex,sabda_ayatana),(liberation_complex,mano_ayatana),(liberation_complex,dharma_ayatana),(ten_essence_nature,caksur_ayatana),(mental_nature,mano_ayatana),(mental_nature,dharma_ayatana),(each_own_nature,eighteen_principles)}
AttendedOf => {(totalities8_att,totalities8),(masteries_att,masteries),(liberations5_att,liberations5)}
CognitionOccurring => {}
Enumerated => {}
KnownAsAppropriate => {sixty_two}
"""


In [56]:
studies[27] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[27]["tree"].pformat())
print("Derived SEM:", studies[27]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_assignment_without_determination=F rejected_new_place=F rejected_attendants_same_scope=F rejected_discipline_in_formations=F rejected_mind_essence_means_cognition_occurring=F rejected_sixty_two_enumerated=F
(S[SEM=<(PlacedIn(eighty_thousand,five_bases) & all x.((InOtherSutra(x) & (BaseName(x) | EssenceName(x) | PrincipleName(x))) -> exists k p.(OwnCharacteristic(x,k) & DeterminedBeforeAssignment(x,k) & Established(p) & ToBeAssigned(x,p) & FitsAsAppropriate(k,p))))>]
  (LIKEWISE[SEM=<PlacedIn(eighty_thousand,five_bases)>] tathā)
  (OTHERS[CASE='nom', GEN='m', NUM='pl', SEM=<\x.InOtherSutra(x)>]
    anye)
  (ALSO[] api)
  (MANNER[SEM=<\k p.FitsAsAppropriate(k,p)>] yathā-yogam)
  (NAMES[CASE='nom', GEN='m', NUM='pl', SEM=<\x.(BaseName(x) | EssenceName(x) | PrincipleName(x))>]
    skandha-āyatana-dhātavaḥ)
  (GERUNDIVE[CASE='nom', GEN='m', NUM='pl', SEM=<\x p.ToBeAssigned(x,p)>]
    pratipādyāḥ)
  (AMONG[CASE='loc', NUM='pl', SEM=<\p.Established(p)>

# VAK_1.28 — Space and Cognition within the Six Supports of Birth

Source: `VAK_1.28.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
chidram ākāśadhātvākhyam ālokatamasī kila /
vijñānadhātur vijñānaṃ sāsravaṃ janmaniśrayāḥ // 1.28 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| chidram | nominative neuter singular | opening, aperture; subject of the first clause |
| ākāśa-dhātu-ākhyam | bahuvrīhi; nominative neuter singular | named the Space Principle; agrees with *chidram* |
| āloka-tamasī | dvandva; nominative neuter dual | light and darkness; the reported nature of the opening |
| kila | reportive particle | it is said; marks a received account |
| vijñāna-dhātuḥ | nominative masculine singular | Cognition Principle; subject of the second clause |
| vijñānam | nominative neuter singular | Cognition; predicate |
| sa-āsravam | nominative neuter singular | Impure; agrees with *vijñānam* |
| janma-niśrayāḥ | nominative masculine plural | supports of birth; the Bhāṣya supplies the six Principles as subject |

`Dhātu` is Principle. *Vijñāna* is Cognition. The Bhāṣya identifies
the larger list as six Principles: earth, water, fire, wind, space, and
Cognition. The verse's plural `janmaniśrayāḥ` predicates birth-support of
the six together, not of the singular Cognition Principle alone.

**Grammar tokens.** Verse-order IAST with sandhi opened (`vijñānadhātur`
→ *vijñāna-dhātuḥ*, `vijñānaṃ` → *vijñānam*, `sāsravaṃ` → *sa-āsravam*)
and compound boundaries marked. The subject of the last clause, *ime ṣaḍ
dhātavaḥ*, is supplied by the Bhāṣya inside the rule (`SUPPLIED=prose`),
not as a covert leaf.

```text
# tokens: chidram ākāśa-dhātu-ākhyam āloka-tamasī kila vijñāna-dhātuḥ vijñānam sa-āsravam janma-niśrayāḥ
```

## 5. Translation

### Literal Translation

> An opening is named the Space Principle—light and darkness, it is said.
> The Cognition Principle is Impure Cognition; these [six] are
> supports of birth.

### Bhāṣya-informed study translation

> In the teaching of six Principles, an opening is called the Space
> Principle—light and darkness, it is said. The Cognition Principle is Impure
> Cognition; these six Principles are supports of birth.

## 7. Technical Vocabulary

| Sanskrit | Project rendering | Determination |
|---|---|---|
| dhātu | Principle | not Cognition |
| ṣaḍ-dhātu | six Principles | earth, water, fire, wind, space, and Cognition in the sūtra teaching |
| chidra | opening; aperture | conditioned referent named as the Space Principle here |
| ākāśa | unconditioned space; non-obstruction | distinct from the conditioned opening of this verse |
| ākāśadhātu | Space Principle | opening explained through light and darkness; assigned to Form |
| āloka | light | one reported determination of the opening |
| tamas | darkness | the other reported determination |
| kila | according to a received account | reportive; does not itself identify a school |
| vijñāna | Cognition | not Principle |
| vijñānadhātu | Cognition Principle | Impure Cognition in this six; one of the seven |
| sāsrava | Impure | the Cognition selected for this birth-support teaching |
| anāsrava | Pure | excluded from the Cognition Principle in this particular list |
| janmaniśraya | support of birth | shared function of the six Principles |
| pratisandhicitta | rebirth-linking Cognition | opening Cognition in the life-continuum described by the Bhāṣya |
| cyuticitta | death Cognition | terminal Cognition in that life-continuum |

In [57]:
fcfg_text = r"""% start S
# tokens: chidram ākāśa-dhātu-ākhyam āloka-tamasī kila vijñāna-dhātuḥ vijñānam sa-āsravam janma-niśrayāḥ
S[SEM=<(?space & ?cognition & ?support)>] -> SPACE[SEM=?space] COGNITION[SEM=?cognition] SUPPORT[SUPPLIED=prose,SEM=?support]
SPACE[SEM=<exists x.(?opening(x) & ?named(x) & ?report(?nature)(x))>] -> OPENING[CASE=nom,NUM=?n,GEN=?g,SEM=?opening] NAMED[CASE=nom,NUM=?n,GEN=?g,SEM=?named] NATURE[CASE=nom,NUM=du,GEN=n,SEM=?nature] KILA[SEM=?report]
COGNITION[SEM=<exists y.(?cog(y) & ?impure(y) & ?principle(y))>] -> PRINCIPLE[CASE=nom,NUM=sg,GEN=m,SEM=?principle] COG[CASE=nom,NUM=?n,GEN=?g,SEM=?cog] IMPURE[CASE=nom,NUM=?n,GEN=?g,SEM=?impure]
SUPPORT[SUPPLIED=prose,SEM=<all p.(InSixfold(p) -> ?supports(p))>] -> SUPPORTS[CASE=nom,NUM=pl,GEN=m,SEM=?supports]
OPENING[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Opening(x)>] -> 'chidram'
NAMED[CASE=nom,NUM=sg,GEN=n,SEM=<\x.Named(x,space_principle)>] -> 'ākāśa-dhātu-ākhyam'
NATURE[CASE=nom,NUM=du,GEN=n,SEM=<light_and_darkness>] -> 'āloka-tamasī'
KILA[SEM=<\n x.ReportedNature(x,n)>] -> 'kila'
PRINCIPLE[CASE=nom,NUM=sg,GEN=m,SEM=<\y.Named(y,cognition_principle)>] -> 'vijñāna-dhātuḥ'
COG[CASE=nom,NUM=sg,GEN=n,SEM=<\y.Cognition(y)>] -> 'vijñānam'
IMPURE[CASE=nom,NUM=sg,GEN=n,SEM=<\y.Impure(y)>] -> 'sa-āsravam'
SUPPORTS[CASE=nom,NUM=pl,GEN=m,SEM=<\p.SupportOfBirth(p)>] -> 'janma-niśrayāḥ'
"""

dependency_text = r"""'ākāśa-dhātu-ākhyam' -> 'chidram' | 'āloka-tamasī'
'āloka-tamasī' -> 'kila'
'vijñānam' -> 'vijñāna-dhātuḥ' | 'sa-āsravam'
'janma-niśrayāḥ' -> 'ākāśa-dhātu-ākhyam' | 'vijñānam'
"""

sem_text = r"""# derived by the section 4 parse
(exists x.(Opening(x) & Named(x,space_principle) & ReportedNature(x,light_and_darkness)) & exists y.(Cognition(y) & Impure(y) & Named(y,cognition_principle)) & all p.(InSixfold(p) -> SupportOfBirth(p)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(exists x.(Opening(x) & Named(x,space_principle) & ReportedNature(x,light_and_darkness)) & exists y.(Cognition(y) & Impure(y) & Named(y,cognition_principle)) & all p.(InSixfold(p) -> SupportOfBirth(p)))
# THEORY prose
# EXPECT prose T
InSixfold(earth) & InSixfold(water) & InSixfold(fire) & InSixfold(wind) & InSixfold(space_member) & InSixfold(cognition_member)
all x.(Named(x,space_principle) -> (Opening(x) & -Unconditioned(x) & IncludedIn(x,form_principle)))
Unconditioned(unconditioned_space) & -Named(unconditioned_space,space_principle)
all y.(Named(y,cognition_principle) -> (Impure(y) & IncludedIn(y,seven_cognition_principles)))
exists z.(Cognition(z) & Pure(z) & -Named(z,cognition_principle) & -Unconditioned(z))
all x.((InSixfold(x) & ElementPrinciple(x)) -> IncludedIn(x,tangible_principle))
ElementPrinciple(earth) & ElementPrinciple(water) & ElementPrinciple(fire) & ElementPrinciple(wind)
all x.(InSixfold(x) -> exists t.(Eighteenfold(t) & IncludedIn(x,t)))
SpansFrom(birth_span,rebirth_linking) & SpansTo(birth_span,death_cognition)
AttributedToOthers(second_agha_account) & -Resolved(cittastha)
# THEORY rejected_space_principle_is_unconditioned
# EXPECT rejected_space_principle_is_unconditioned F
exists x.(Named(x,space_principle) & Unconditioned(x))
# THEORY rejected_every_cognition_is_the_principle_here
# EXPECT rejected_every_cognition_is_the_principle_here F
all y.(Cognition(y) -> Named(y,cognition_principle))
# THEORY rejected_only_cognition_supports_birth
# EXPECT rejected_only_cognition_supports_birth F
all p.(SupportOfBirth(p) -> Named(p,cognition_principle))
# THEORY rejected_pure_means_unconditioned
# EXPECT rejected_pure_means_unconditioned F
all z.(Pure(z) -> Unconditioned(z))
# THEORY rejected_six_add_principles
# EXPECT rejected_six_add_principles F
exists x.(InSixfold(x) & -exists t.(Eighteenfold(t) & IncludedIn(x,t)))
# THEORY rejected_cittastha_resolved
# EXPECT rejected_cittastha_resolved F
exists w.(w = cittastha & Resolved(w))
"""

valuation_text = r"""space_principle => space_name
cognition_principle => cognition_name
light_and_darkness => aloka_tamas
earth => earth_m
water => water_m
fire => fire_m
wind => wind_m
space_member => opening_m
cognition_member => impure_cognition_m
unconditioned_space => akasa_asamskrta
form_principle => rupa_dhatu
tangible_principle => sprastavya_dhatu
seven_cognition_principles => seven_vijnana_dhatu
birth_span => janma_span
rebirth_linking => pratisandhi_citta
death_cognition => cyuti_citta
second_agha_account => apare_agha
cittastha => cittastha_reading
Opening => {opening_m}
Named => {(opening_m,space_name),(impure_cognition_m,cognition_name)}
ReportedNature => {(opening_m,aloka_tamas)}
Cognition => {impure_cognition_m,pure_cognition_m}
Impure => {impure_cognition_m}
Pure => {pure_cognition_m}
InSixfold => {earth_m,water_m,fire_m,wind_m,opening_m,impure_cognition_m}
SupportOfBirth => {earth_m,water_m,fire_m,wind_m,opening_m,impure_cognition_m}
Unconditioned => {akasa_asamskrta}
ElementPrinciple => {earth_m,water_m,fire_m,wind_m}
Eighteenfold => {rupa_dhatu,sprastavya_dhatu,seven_vijnana_dhatu}
IncludedIn => {(earth_m,sprastavya_dhatu),(water_m,sprastavya_dhatu),(fire_m,sprastavya_dhatu),(wind_m,sprastavya_dhatu),(opening_m,rupa_dhatu),(impure_cognition_m,seven_vijnana_dhatu)}
SpansFrom => {(janma_span,pratisandhi_citta)}
SpansTo => {(janma_span,cyuti_citta)}
AttributedToOthers => {apare_agha}
Resolved => {}
"""


In [58]:
studies[28] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[28]["tree"].pformat())
print("Derived SEM:", studies[28]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_space_principle_is_unconditioned=F rejected_every_cognition_is_the_principle_here=F rejected_only_cognition_supports_birth=F rejected_pure_means_unconditioned=F rejected_six_add_principles=F rejected_cittastha_resolved=F
(S[SEM=<(exists x.(Opening(x) & Named(x,space_principle) & ReportedNature(x,light_and_darkness)) & exists y.(Cognition(y) & Impure(y) & Named(y,cognition_principle)) & all p.(InSixfold(p) -> SupportOfBirth(p)))>]
  (SPACE[SEM=<exists x.(Opening(x) & Named(x,space_principle) & ReportedNature(x,light_and_darkness))>]
    (OPENING[CASE='nom', GEN='n', NUM='sg', SEM=<\x.Opening(x)>]
      chidram)
    (NAMED[CASE='nom', GEN='n', NUM='sg', SEM=<\x.Named(x,space_principle)>]
      ākāśa-dhātu-ākhyam)
    (NATURE[CASE='nom', GEN='n', NUM='du', SEM=<light_and_darkness>]
      āloka-tamasī)
    (KILA[SEM=<\n x.ReportedNature(x,n)>] kila))
  (COGNITION[SEM=<exists y.(Cognition(y) & Impure(y) & Named(y,cognition_principle))>]
    (PRINCIPL

# VAK_1.29 — Visible, Resistant, Indeterminate

Source: `VAK_1.29.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
sanidarśana eko 'tra rūpaṃ sapratighā daśa /
rūpiṇo 'vyākṛtā aṣṭau ta evārūpaśabdakāḥ // 1.29 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| sa-nidarśanaḥ | sa-nidarśana, bahuvrīhi; nominative masculine singular | visible; can be pointed out; agrees with understood *dhātuḥ* |
| ekaḥ | nominative masculine singular | one |
| atra | indeclinable | here, among the eighteen Principles |
| rūpam | nominative neuter singular | Visible Form; names the one |
| sa-pratighāḥ | sa-pratigha, bahuvrīhi; nominative masculine plural | resistant |
| daśa | numeral | ten |
| rūpiṇaḥ | rūpin; nominative masculine plural | Form-bearing; the ten in the Form Base |
| avyākṛtāḥ | a-vyākṛta; nominative masculine plural | indeterminate |
| aṣṭau | numeral | eight |
| te | nominative masculine plural | those; the same ten |
| eva | particle | the very same |
| a-rūpa-śabdakāḥ | bahuvrīhi; nominative masculine plural | without Visible Form and Sound; excluding those two |

The final compound does not say the eight are without Form. It excludes two of the ten: the Visible Form Principle and the Sound Principle.

**Grammar tokens.** Verse-order IAST with sandhi opened (`sanidarśana
eko 'tra` → *sanidarśanaḥ ekaḥ atra*, `rūpiṇo 'vyākṛtā` → *rūpiṇaḥ
avyākṛtāḥ*, `ta evārūpa-` → *te eva a-rūpa-*) and compound boundaries
marked. The understood *dhātavaḥ* is carried by masculine agreement, not
supplied as a leaf.

```text
# tokens: sa-nidarśanaḥ ekaḥ atra rūpam sa-pratighāḥ daśa rūpiṇaḥ avyākṛtāḥ aṣṭau te eva a-rūpa-śabdakāḥ
```

## 5. Translation

### Literal Translation

> One here is visible: Form. Ten are resistant. Eight Form-bearing ones are
> indeterminate: those same, excluding Form and Sound.

### Bhāṣya-informed study translation

> Among the eighteen Principles, one is visible: the Visible Form Principle.
> Ten included in the Form Base are resistant. Eight of those ten are
> indeterminate: those same ten excluding Visible Form and Sound.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| dhātu | Principle | the eighteen |
| sanidarśana | visible | can be pointed out |
| anidarśana | not visible | the other seventeen |
| rūpa-dhātu | Visible Form Principle | the one visible |
| sapratigha | resistant | the ten, by obstruction |
| pratigha | resistance | three kinds; the count uses one |
| āvaraṇa-pratighāta | obstruction | blocking another's arising in one's place |
| viṣaya-pratighāta | Condition-resistance | the Faculty in its Condition |
| ālambana-pratighāta | support-resistance | consciousness and the concomitants |
| rūpin | Form-bearing | the ten in the Form Base |
| avyākṛta | indeterminate | not declared wholesome or unwholesome |
| śabda-dhātu | Sound Principle | excluded from the eight |
| vijñāna | Cognition | not this count |
| citta | consciousness | the seven, in the comparison only |

In [59]:
fcfg_text = r"""% start S
# tokens: sa-nidarśanaḥ ekaḥ atra rūpam sa-pratighāḥ daśa rūpiṇaḥ avyākṛtāḥ aṣṭau te eva a-rūpa-śabdakāḥ
S[SEM=<(?visible & ?resistant & ?indeterminate)>] -> VISIBLE[SEM=?visible] RESISTANT[SEM=?resistant] INDETERMINATE[SEM=?indeterminate]
VISIBLE[SEM=<all x.(?here(x) -> (?vis(x) <-> (x = ?name)))>] -> VIS[CASE=nom,NUM=?n,GEN=?g,SEM=?vis] ONE[CASE=nom,NUM=?n,GEN=?g] HERE[SEM=?here] NAME[CASE=nom,NUM=sg,GEN=n,SEM=?name]
RESISTANT[SEM=<all x.(AmongEighteen(x) -> (?res(x) <-> ?formb(x)))>] -> RES[CASE=nom,NUM=pl,GEN=?g,SEM=?res] TEN FORMB[CASE=nom,NUM=pl,GEN=?g,SEM=?formb]
INDETERMINATE[SEM=<all x.(AmongEighteen(x) -> (?ind(x) <-> (?same(x) & ?excl(x))))>] -> IND[CASE=nom,NUM=pl,GEN=?g,SEM=?ind] EIGHT SAME[CASE=nom,NUM=pl,GEN=?g,SEM=?same] EVA EXCL[CASE=nom,NUM=pl,GEN=?g,SEM=?excl]
VIS[CASE=nom,NUM=sg,GEN=m,SEM=<\x.Visible(x)>] -> 'sa-nidarśanaḥ'
ONE[CASE=nom,NUM=sg,GEN=m] -> 'ekaḥ'
HERE[SEM=<\x.AmongEighteen(x)>] -> 'atra'
NAME[CASE=nom,NUM=sg,GEN=n,SEM=<visible_form>] -> 'rūpam'
RES[CASE=nom,NUM=pl,GEN=m,SEM=<\x.Resistant(x)>] -> 'sa-pratighāḥ'
TEN -> 'daśa'
FORMB[CASE=nom,NUM=pl,GEN=m,SEM=<\x.FormBearing(x)>] -> 'rūpiṇaḥ'
IND[CASE=nom,NUM=pl,GEN=m,SEM=<\x.OnlyIndeterminate(x)>] -> 'avyākṛtāḥ'
EIGHT -> 'aṣṭau'
SAME[CASE=nom,NUM=pl,GEN=m,SEM=<\x.Resistant(x)>] -> 'te'
EVA -> 'eva'
EXCL[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(-(x = visible_form) & -(x = sound))>] -> 'a-rūpa-śabdakāḥ'
"""

dependency_text = r"""'sa-nidarśanaḥ' -> 'ekaḥ' | 'atra' | 'rūpam'
'sa-pratighāḥ' -> 'daśa' | 'rūpiṇaḥ'
'avyākṛtāḥ' -> 'aṣṭau' | 'te'
'te' -> 'eva' | 'a-rūpa-śabdakāḥ'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(AmongEighteen(x) -> (Visible(x) <-> (x = visible_form))) & all x.(AmongEighteen(x) -> (Resistant(x) <-> FormBearing(x))) & all x.(AmongEighteen(x) -> (OnlyIndeterminate(x) <-> (Resistant(x) & -(x = visible_form) & -(x = sound)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(AmongEighteen(x) -> (Visible(x) <-> (x = visible_form))) & all x.(AmongEighteen(x) -> (Resistant(x) <-> FormBearing(x))) & all x.(AmongEighteen(x) -> (OnlyIndeterminate(x) <-> (Resistant(x) & -(x = visible_form) & -(x = sound)))))
# THEORY prose
# EXPECT prose T
-AmongEighteen(form_base) & -Visible(form_base)
all x.(AmongEighteen(x) -> (Resistant(x) <-> ObstructionResistant(x)))
all x.(Item(x) -> ((ConditionResistant(x) & -ObstructionResistant(x)) <-> (CittaPrinciple(x) | x = dharma_associated)))
all x.(Item(x) -> ((ObstructionResistant(x) & -ConditionResistant(x)) <-> ConditionMeaning(x)))
all x.(Item(x) -> ((ConditionResistant(x) & ObstructionResistant(x)) <-> SenseFaculty(x)))
all x.(Item(x) -> ((-ConditionResistant(x) & -ObstructionResistant(x)) <-> (x = dharma_nonassociated)))
all x.(SupportResistant(x) -> ConditionResistant(x))
all x.((ConditionResistant(x) & -SupportResistant(x)) <-> SenseFaculty(x))
PortionOf(dharma_associated,dharma_principle) & PortionOf(dharma_nonassociated,dharma_principle)
AttributedTo(kumaralata_definition,kumaralata) & AttributedToOthers(maturation_account) & OverextendsTo(maturation_account,pure_dharmas)
# THEORY rejected_form_base_is_visible
# EXPECT rejected_form_base_is_visible F
exists x.(x = form_base & Visible(x))
# THEORY rejected_count_by_condition_resistance
# EXPECT rejected_count_by_condition_resistance F
all x.(AmongEighteen(x) -> (Resistant(x) <-> ConditionResistant(x)))
# THEORY rejected_eight_are_formless
# EXPECT rejected_eight_are_formless F
all x.(OnlyIndeterminate(x) -> -FormBearing(x))
# THEORY rejected_sound_only_indeterminate
# EXPECT rejected_sound_only_indeterminate F
exists x.(x = sound & OnlyIndeterminate(x))
# THEORY rejected_mind_is_resistant
# EXPECT rejected_mind_is_resistant F
exists x.(x = mind_principle & Resistant(x))
# THEORY rejected_support_without_condition
# EXPECT rejected_support_without_condition F
exists x.(SupportResistant(x) & -ConditionResistant(x))
"""

valuation_text = r"""visible_form => rupa_d
sound => sabda_d
mind_principle => mano_d
dharma_principle => dharma_d
form_base => rupa_skandha
dharma_associated => dharma_samprayukta
dharma_nonassociated => dharma_asamprayukta
kumaralata => bhadanta_kumaralata
kumaralata_definition => kumaralata_def
maturation_account => vipaka_account
pure_dharmas => anasrava_dharmas
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
Visible => {rupa_d}
FormBearing => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
Resistant => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
OnlyIndeterminate => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,gandha_d,rasa_d,sprastavya_d}
Item => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_samprayukta,dharma_asamprayukta}
ObstructionResistant => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
ConditionResistant => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_samprayukta}
SupportResistant => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_samprayukta}
CittaPrinciple => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
ConditionMeaning => {rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
SenseFaculty => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
PortionOf => {(dharma_samprayukta,dharma_d),(dharma_asamprayukta,dharma_d)}
AttributedTo => {(kumaralata_def,bhadanta_kumaralata)}
AttributedToOthers => {vipaka_account}
OverextendsTo => {(vipaka_account,anasrava_dharmas)}
"""


In [60]:
studies[29] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[29]["tree"].pformat())
print("Derived SEM:", studies[29]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_form_base_is_visible=F rejected_count_by_condition_resistance=F rejected_eight_are_formless=F rejected_sound_only_indeterminate=F rejected_mind_is_resistant=F rejected_support_without_condition=F
(S[SEM=<(all x.(AmongEighteen(x) -> (Visible(x) <-> (x = visible_form))) & all x.(AmongEighteen(x) -> (Resistant(x) <-> FormBearing(x))) & all x.(AmongEighteen(x) -> (OnlyIndeterminate(x) <-> (Resistant(x) & -(x = visible_form) & -(x = sound)))))>]
  (VISIBLE[SEM=<all x.(AmongEighteen(x) -> (Visible(x) <-> (x = visible_form)))>]
    (VIS[CASE='nom', GEN='m', NUM='sg', SEM=<\x.Visible(x)>]
      sa-nidarśanaḥ)
    (ONE[CASE='nom', GEN='m', NUM='sg'] ekaḥ)
    (HERE[SEM=<\x.AmongEighteen(x)>] atra)
    (NAME[CASE='nom', GEN='n', NUM='sg', SEM=<visible_form>] rūpam))
  (RESISTANT[SEM=<all x.(AmongEighteen(x) -> (Resistant(x) <-> FormBearing(x)))>]
    (RES[CASE='nom', GEN='m', NUM='pl', SEM=<\x.Resistant(x)>]
      sa-pratighāḥ)
    (TEN[] daśa)
    (FORMB

# VAK_1.30 — Threefold, and Fourteen

Source: `VAK_1.30.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
tridhānye kāmadhātvāptāḥ sarve rūpe caturdaśa /
vinā gandharasaghrāṇajihvāvijñānadhātubhiḥ // 1.30 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| tridhā | indeclinable | in three ways: wholesome, unwholesome, indeterminate |
| anye | anya; nominative masculine plural | the others; the ten remaining after the eight of 1.29 |
| kāma-dhātu-āptāḥ | tatpuruṣa; āpta, past participle of *āp*; nominative masculine plural | connected with the Desire Realm |
| sarve | sarva; nominative masculine plural | all eighteen |
| rūpe | rūpa; locative singular, for *rūpa-dhātau* | in the Form Realm |
| caturdaśa | numeral | fourteen |
| vinā | indeclinable, governing the instrumental | excluding |
| gandha-rasa-ghrāṇa-jihvā-vijñāna-dhātubhiḥ | dvandva inside tatpuruṣa; instrumental plural | the Odor, Taste, nose-Cognition, and tongue-Cognition Principles |

The closing compound excludes four Principles, not six: *vijñāna* distributes over *ghrāṇa* and *jihvā*. Nose and tongue, the Faculties, are not in the exclusion.

**Grammar tokens.** Verse-order IAST with sandhi opened (`tridhānye` →
*tridhā anye*) and compound boundaries marked. The closing compound stays
one token; its internal distribution is stated in the instrumental's SEM.

```text
# tokens: tridhā anye kāma-dhātu-āptāḥ sarve rūpe caturdaśa vinā gandha-rasa-ghrāṇa-jihvā-vijñāna-dhātubhiḥ
```

## 5. Translation

### Literal Translation

> The others are threefold. All are connected with the Desire Realm. In the
> Form Realm, fourteen, excluding the Odor, Taste, nose-Cognition, and
> tongue-Cognition Principles.

### Bhāṣya-informed study translation

> The other ten are threefold. All eighteen are connected with the Desire
> Realm; fourteen occur in the Form Realm, excluding the Principles of Odor,
> Taste, nose-Cognition, and tongue-Cognition.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| anye | the others | the ten after the eight |
| tridhā | threefold | wholesome, unwholesome, indeterminate |
| kuśala | wholesome | not the Pure |
| akuśala | unwholesome | not the Impure |
| avyākṛta | indeterminate | not declared either |
| kāmadhātu | Desire Realm | a Principle of Principles; not one of the eighteen |
| rūpadhātu | Form Realm | a Principle of Principles; not the Visible Form Principle |
| āpta | connected | not separated from |
| gandha-dhātu | Odor Principle | excluded from the fourteen |
| rasa-dhātu | Taste Principle | excluded from the fourteen |
| ghrāṇa-vijñāna-dhātu | nose-Cognition Principle | one Principle, not two |
| jihvā-vijñāna-dhātu | tongue-Cognition Principle | likewise |
| vijñāna | Cognition | not Principle |
| dhātu | Principle | the eighteen, except the Realm names |

In [61]:
fcfg_text = r"""% start S
# tokens: tridhā anye kāma-dhātu-āptāḥ sarve rūpe caturdaśa vinā gandha-rasa-ghrāṇa-jihvā-vijñāna-dhātubhiḥ
S[SEM=<(?ethical & ?desire & ?form)>] -> ETHICAL[SEM=?ethical] DESIRE[SEM=?desire] FORMREALM[SEM=?form]
ETHICAL[SEM=<all x.(?others(x) -> ?tri(x))>] -> TRI[SEM=?tri] OTHERS[CASE=nom,NUM=pl,GEN=m,SEM=?others]
DESIRE[SEM=<all x.(?all(x) -> ?conn(x))>] -> CONN[CASE=nom,NUM=?n,GEN=?g,SEM=?conn] ALL[CASE=nom,NUM=?n,GEN=?g,SEM=?all]
FORMREALM[SEM=<all x.(AmongEighteen(x) -> (ConnectedWith(x,?realm) <-> -?excl(x)))>] -> IN[CASE=loc,SEM=?realm] FOURTEEN WITHOUT[GOV=?c] EXCL[CASE=?c,SEM=?excl]
TRI[SEM=<\x.Threefold(x)>] -> 'tridhā'
OTHERS[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(AmongEighteen(x) & -OnlyIndeterminate(x))>] -> 'anye'
CONN[CASE=nom,NUM=pl,GEN=m,SEM=<\x.ConnectedWith(x,desire_realm)>] -> 'kāma-dhātu-āptāḥ'
ALL[CASE=nom,NUM=pl,GEN=m,SEM=<\x.AmongEighteen(x)>] -> 'sarve'
IN[CASE=loc,SEM=<form_realm>] -> 'rūpe'
FOURTEEN -> 'caturdaśa'
WITHOUT[GOV=ins] -> 'vinā'
EXCL[CASE=ins,SEM=<\x.((x = odor) | (x = taste) | (x = nose_cognition) | (x = tongue_cognition))>] -> 'gandha-rasa-ghrāṇa-jihvā-vijñāna-dhātubhiḥ'
"""

dependency_text = r"""'tridhā' -> 'anye'
'kāma-dhātu-āptāḥ' -> 'sarve'
'caturdaśa' -> 'rūpe' | 'vinā'
'vinā' -> 'gandha-rasa-ghrāṇa-jihvā-vijñāna-dhātubhiḥ'
"""

sem_text = r"""# derived by the section 4 parse
(all x.((AmongEighteen(x) & -OnlyIndeterminate(x)) -> Threefold(x)) & all x.(AmongEighteen(x) -> ConnectedWith(x,desire_realm)) & all x.(AmongEighteen(x) -> (ConnectedWith(x,form_realm) <-> -((x = odor) | (x = taste) | (x = nose_cognition) | (x = tongue_cognition)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.((AmongEighteen(x) & -OnlyIndeterminate(x)) -> Threefold(x)) & all x.(AmongEighteen(x) -> ConnectedWith(x,desire_realm)) & all x.(AmongEighteen(x) -> (ConnectedWith(x,form_realm) <-> -((x = odor) | (x = taste) | (x = nose_cognition) | (x = tongue_cognition)))))
# THEORY prose
# EXPECT prose T
all x.(Threefold(x) <-> (CanBeWholesome(x) & CanBeUnwholesome(x) & CanBeIndeterminate(x)))
all x.(CittaPrinciple(x) -> ThreefoldBy(x,association))
ThreefoldBy(dharma_principle,nature) & ThreefoldBy(dharma_principle,association) & ThreefoldBy(dharma_principle,arising)
ThreefoldBy(visible_form,vijnapti) & ThreefoldBy(sound,vijnapti)
all x.(AmongEighteen(x) -> (Threefold(x) <-> exists g.ThreefoldBy(x,g)))
PortionOf(pratisamkhya_nirodha,dharma_principle) & CanBeWholesome(pratisamkhya_nirodha)
all x.(Principle(x) <-> (AmongEighteen(x) | RealmLevel(x)))
BaseLevel(form_base) & RealmLevel(desire_realm) & RealmLevel(form_realm)
all x.(RealmLevel(x) -> -AmongEighteen(x))
all r.(RealmLevel(r) -> exists x.(AmongEighteen(x) & ConnectedWith(x,r)))
all x.(BaseLevel(x) -> -Principle(x))
all x.(AmongEighteen(x) -> (GatheredIn(x,form_base) <-> FormBearing(x)))
MorselFood(odor) & MorselFood(taste) & MorselFood(tangible)
SupportOf(odor,nose_cognition) & SupportOf(taste,tongue_cognition)
all x.all y.((SupportOf(y,x) & -ConnectedWith(y,form_realm)) -> -ConnectedWith(x,form_realm))
HasUse(tangible) & HasUse(nose) & HasUse(tongue)
AttributedToOthers(dhyana_account) & Explained(varnayanti_account)
# THEORY rejected_realm_among_eighteen
# EXPECT rejected_realm_among_eighteen F
exists x.(RealmLevel(x) & AmongEighteen(x))
# THEORY rejected_realm_not_principle
# EXPECT rejected_realm_not_principle F
exists x.(RealmLevel(x) & -Principle(x))
# THEORY rejected_form_realm_is_visible_form
# EXPECT rejected_form_realm_is_visible_form F
exists x.(x = form_realm & x = visible_form)
# THEORY rejected_form_base_is_principle
# EXPECT rejected_form_base_is_principle F
exists x.(x = form_base & Principle(x))
# THEORY rejected_eight_are_threefold
# EXPECT rejected_eight_are_threefold F
exists x.(OnlyIndeterminate(x) & Threefold(x))
# THEORY rejected_faculties_excluded
# EXPECT rejected_faculties_excluded F
exists x.(((x = nose) | (x = tongue)) & -ConnectedWith(x,form_realm))
# THEORY rejected_tangible_excluded_as_food
# EXPECT rejected_tangible_excluded_as_food F
exists x.(x = tangible & -ConnectedWith(x,form_realm))
# THEORY rejected_cognition_without_support
# EXPECT rejected_cognition_without_support F
exists x.exists y.(SupportOf(y,x) & -ConnectedWith(y,form_realm) & ConnectedWith(x,form_realm))
"""

valuation_text = r"""desire_realm => kama_dhatu
form_realm => rupa_dhatu_realm
form_base => rupa_skandha
visible_form => rupa_d
sound => sabda_d
odor => gandha_d
taste => rasa_d
tangible => sprastavya_d
nose => ghrana_d
tongue => jihva_d
nose_cognition => ghranavijnana_d
tongue_cognition => jihvavijnana_d
dharma_principle => dharma_d
pratisamkhya_nirodha => pratisamkhya_nirodha_m
association => samprayoga_g
nature => svabhava_g
arising => samuttha_g
vijnapti => vijnapti_g
dhyana_account => dhyana_account_m
varnayanti_account => varnayanti_account_m
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
Principle => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,kama_dhatu,rupa_dhatu_realm}
BaseLevel => {rupa_skandha}
RealmLevel => {kama_dhatu,rupa_dhatu_realm}
FormBearing => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
GatheredIn => {(caksur_d,rupa_skandha),(srotra_d,rupa_skandha),(ghrana_d,rupa_skandha),(jihva_d,rupa_skandha),(kaya_d,rupa_skandha),(rupa_d,rupa_skandha),(sabda_d,rupa_skandha),(gandha_d,rupa_skandha),(rasa_d,rupa_skandha),(sprastavya_d,rupa_skandha)}
OnlyIndeterminate => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,gandha_d,rasa_d,sprastavya_d}
Threefold => {rupa_d,sabda_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
CanBeWholesome => {rupa_d,sabda_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,pratisamkhya_nirodha_m}
CanBeUnwholesome => {rupa_d,sabda_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
CanBeIndeterminate => {rupa_d,sabda_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
CittaPrinciple => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
ThreefoldBy => {(mano_d,samprayoga_g),(caksurvijnana_d,samprayoga_g),(srotravijnana_d,samprayoga_g),(ghranavijnana_d,samprayoga_g),(jihvavijnana_d,samprayoga_g),(kayavijnana_d,samprayoga_g),(manovijnana_d,samprayoga_g),(dharma_d,svabhava_g),(dharma_d,samprayoga_g),(dharma_d,samuttha_g),(rupa_d,vijnapti_g),(sabda_d,vijnapti_g)}
PortionOf => {(pratisamkhya_nirodha_m,dharma_d)}
ConnectedWith => {(caksur_d,kama_dhatu),(srotra_d,kama_dhatu),(ghrana_d,kama_dhatu),(jihva_d,kama_dhatu),(kaya_d,kama_dhatu),(rupa_d,kama_dhatu),(sabda_d,kama_dhatu),(gandha_d,kama_dhatu),(rasa_d,kama_dhatu),(sprastavya_d,kama_dhatu),(mano_d,kama_dhatu),(dharma_d,kama_dhatu),(caksurvijnana_d,kama_dhatu),(srotravijnana_d,kama_dhatu),(ghranavijnana_d,kama_dhatu),(jihvavijnana_d,kama_dhatu),(kayavijnana_d,kama_dhatu),(manovijnana_d,kama_dhatu),(caksur_d,rupa_dhatu_realm),(srotra_d,rupa_dhatu_realm),(ghrana_d,rupa_dhatu_realm),(jihva_d,rupa_dhatu_realm),(kaya_d,rupa_dhatu_realm),(rupa_d,rupa_dhatu_realm),(sabda_d,rupa_dhatu_realm),(sprastavya_d,rupa_dhatu_realm),(mano_d,rupa_dhatu_realm),(dharma_d,rupa_dhatu_realm),(caksurvijnana_d,rupa_dhatu_realm),(srotravijnana_d,rupa_dhatu_realm),(kayavijnana_d,rupa_dhatu_realm),(manovijnana_d,rupa_dhatu_realm)}
MorselFood => {gandha_d,rasa_d,sprastavya_d}
SupportOf => {(gandha_d,ghranavijnana_d),(rasa_d,jihvavijnana_d)}
HasUse => {sprastavya_d,ghrana_d,jihva_d}
AttributedToOthers => {dhyana_account_m}
Explained => {varnayanti_account_m}
"""


In [62]:
studies[30] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[30]["tree"].pformat())
print("Derived SEM:", studies[30]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_realm_among_eighteen=F rejected_realm_not_principle=F rejected_form_realm_is_visible_form=F rejected_form_base_is_principle=F rejected_eight_are_threefold=F rejected_faculties_excluded=F rejected_tangible_excluded_as_food=F rejected_cognition_without_support=F
(S[SEM=<(all x.((AmongEighteen(x) & -OnlyIndeterminate(x)) -> Threefold(x)) & all x.(AmongEighteen(x) -> ConnectedWith(x,desire_realm)) & all x.(AmongEighteen(x) -> (ConnectedWith(x,form_realm) <-> -((x = odor) | (x = taste) | (x = nose_cognition) | (x = tongue_cognition)))))>]
  (ETHICAL[SEM=<all x.((AmongEighteen(x) & -OnlyIndeterminate(x)) -> Threefold(x))>]
    (TRI[SEM=<\x.Threefold(x)>] tridhā)
    (OTHERS[CASE='nom', GEN='m', NUM='pl', SEM=<\x.(AmongEighteen(x) & -OnlyIndeterminate(x))>]
      anye))
  (DESIRE[SEM=<all x.(AmongEighteen(x) -> ConnectedWith(x,desire_realm))>]
    (CONN[CASE='nom', GEN='m', NUM='pl', SEM=<\x.ConnectedWith(x,desire_realm)>]
      kāma-dhātu-āptāḥ)
    (

# VAK_1.31 — Three in the Formless

Source: `VAK_1.31.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
ārūpyāptā manodharmamanovijñānadhātavaḥ /
sāsravānāsravā ete trayaḥ śeṣās tu sāsravāḥ // 1.31 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| ārūpya-āptāḥ | tatpuruṣa; āpta, past participle of *āp*; nominative masculine plural | connected with the Formless Realm |
| mano-dharma-mano-vijñāna-dhātavaḥ | dvandva inside tatpuruṣa; nominative masculine plural | the Mind, Dharma, and Mind-Cognition Principles |
| sa-āsrava-an-āsravāḥ | dvandva of two bahuvrīhis; nominative masculine plural | Impure and Pure |
| ete | etad; nominative masculine plural | these; the three just named |
| trayaḥ | tri; nominative masculine plural | three |
| śeṣāḥ | śeṣa; nominative masculine plural | the remaining fifteen |
| tu | particle | but; contrasts the fifteen with the three |
| sa-āsravāḥ | sa-āsrava, bahuvrīhi; nominative masculine plural | Impure |

The compound names three Principles, not three Essences. The Dharma Principle is not the Dharma Essence.

**Grammar tokens.** Verse-order IAST with sandhi opened (`ārūpyāptā` →
*ārūpya-āptāḥ*, `sāsravānāsravā` → *sa-āsrava-an-āsravāḥ*, `śeṣās tu` →
*śeṣāḥ tu*) and compound boundaries marked. Each compound stays one token;
its members are stated in the SEM.

```text
# tokens: ārūpya-āptāḥ mano-dharma-mano-vijñāna-dhātavaḥ sa-āsrava-an-āsravāḥ ete trayaḥ śeṣāḥ tu sa-āsravāḥ
```

## 5. Translation

### Literal Translation

> Connected with the Formless are the Mind Principle, the Dharma Principle,
> and the Mind-Cognition Principle. These three are Impure and Pure. The
> rest are Impure.

### Bhāṣya-informed study translation

> Connected with the Formless are the Mind Principle, Dharma Principle, and
> Mind-Cognition Principle. These three are Impure and Pure; the remaining
> fifteen are Impure.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| ārūpya | Formless Realm | a Principle of Principles; not one of the eighteen |
| āpta | connected | not separated from |
| mano-dhātu | Mind Principle | the seventh of the Cognition Principle |
| dharma-dhātu | Dharma Principle | not the Dharma Essence |
| mano-vijñāna-dhātu | Mind-Cognition Principle | one of the six |
| vijñāna | Cognition | not Principle |
| sāsrava | Impure | not outflow-bearing |
| anāsrava | Pure | Path or unconditioned, in this cut |
| śeṣāḥ | the remaining fifteen | exclusively Impure |

In [63]:
fcfg_text = r"""% start S
# tokens: ārūpya-āptāḥ mano-dharma-mano-vijñāna-dhātavaḥ sa-āsrava-an-āsravāḥ ete trayaḥ śeṣāḥ tu sa-āsravāḥ
S[SEM=<(?formless & ?both & ?rest)>] -> FORMLESS[SEM=?formless] BOTH[SEM=?both] REST[SEM=?rest]
FORMLESS[SEM=<all x.(AmongEighteen(x) -> (?conn(x) <-> ?three(x)))>] -> CONN[CASE=nom,NUM=?n,GEN=?g,SEM=?conn] THREE[CASE=nom,NUM=?n,GEN=?g,SEM=?three]
BOTH[SEM=<all x.(?these(x) -> ?pair(x))>] -> PAIR[CASE=nom,NUM=?n,GEN=?g,SEM=?pair] THESE[CASE=nom,NUM=?n,GEN=?g,SEM=?these] TRI[CASE=nom,NUM=?n,GEN=?g]
REST[SEM=<all x.(?rest(x) -> (?imp(x) & -HasPureInstance(x)))>] -> REMAIN[CASE=nom,NUM=?n,GEN=?g,SEM=?rest] TU IMP[CASE=nom,NUM=?n,GEN=?g,SEM=?imp]
CONN[CASE=nom,NUM=pl,GEN=m,SEM=<\x.ConnectedWith(x,formless_realm)>] -> 'ārūpya-āptāḥ'
THREE[CASE=nom,NUM=pl,GEN=m,SEM=<\x.((x = mind) | (x = dharma_principle) | (x = mind_cognition))>] -> 'mano-dharma-mano-vijñāna-dhātavaḥ'
PAIR[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(HasImpureInstance(x) & HasPureInstance(x))>] -> 'sa-āsrava-an-āsravāḥ'
THESE[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(AmongEighteen(x) & ConnectedWith(x,formless_realm))>] -> 'ete'
TRI[CASE=nom,NUM=pl,GEN=m] -> 'trayaḥ'
REMAIN[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(AmongEighteen(x) & -ConnectedWith(x,formless_realm))>] -> 'śeṣāḥ'
TU -> 'tu'
IMP[CASE=nom,NUM=pl,GEN=m,SEM=<\x.HasImpureInstance(x)>] -> 'sa-āsravāḥ'
"""

dependency_text = r"""'ārūpya-āptāḥ' -> 'mano-dharma-mano-vijñāna-dhātavaḥ'
'sa-āsrava-an-āsravāḥ' -> 'ete'
'ete' -> 'trayaḥ'
'sa-āsravāḥ' -> 'śeṣāḥ' | 'tu'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(AmongEighteen(x) -> (ConnectedWith(x,formless_realm) <-> ((x = mind) | (x = dharma_principle) | (x = mind_cognition)))) & all x.((AmongEighteen(x) & ConnectedWith(x,formless_realm)) -> (HasImpureInstance(x) & HasPureInstance(x))) & all x.((AmongEighteen(x) & -ConnectedWith(x,formless_realm)) -> (HasImpureInstance(x) & -HasPureInstance(x))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(AmongEighteen(x) -> (ConnectedWith(x,formless_realm) <-> ((x = mind) | (x = dharma_principle) | (x = mind_cognition)))) & all x.((AmongEighteen(x) & ConnectedWith(x,formless_realm)) -> (HasImpureInstance(x) & HasPureInstance(x))) & all x.((AmongEighteen(x) & -ConnectedWith(x,formless_realm)) -> (HasImpureInstance(x) & -HasPureInstance(x))))
# THEORY prose
# EXPECT prose T
all x.(AmongEighteen(x) -> (-ConnectedWith(x,formless_realm) <-> (FormNatured(x) | exists y.(FormNatured(y) & SupportAndCondition(y,x)))))
all z.(FreeOfFormPassion(z) <-> BornIn(z,formless_realm))
all i.(Pure(i) <-> (InPathTruth(i) | Unconditioned(i)))
all x.(HasPureInstance(x) <-> exists i.(InstanceOf(i,x) & Pure(i)))
all x.(HasImpureInstance(x) <-> exists i.(InstanceOf(i,x) & -Pure(i)))
all i.(Unconditioned(i) -> InstanceOf(i,dharma_principle))
all x.(Principle(x) <-> (AmongEighteen(x) | RealmLevel(x)))
EssenceLevel(dharma_essence) & RealmLevel(formless_realm)
all x.(RealmLevel(x) -> -AmongEighteen(x))
all r.(RealmLevel(r) -> exists x.(AmongEighteen(x) & ConnectedWith(x,r)))
all x.(EssenceLevel(x) -> -Principle(x))
# THEORY rejected_form_natured_in_formless
# EXPECT rejected_form_natured_in_formless F
exists x.(FormNatured(x) & ConnectedWith(x,formless_realm))
# THEORY rejected_five_cognitions_in_formless
# EXPECT rejected_five_cognitions_in_formless F
exists x.(exists y.(FormNatured(y) & SupportAndCondition(y,x)) & ConnectedWith(x,formless_realm))
# THEORY rejected_pure_means_formless
# EXPECT rejected_pure_means_formless F
all i.(Pure(i) <-> exists x.(InstanceOf(i,x) & ConnectedWith(x,formless_realm)))
# THEORY rejected_fifteen_have_pure
# EXPECT rejected_fifteen_have_pure F
exists x.(AmongEighteen(x) & -ConnectedWith(x,formless_realm) & HasPureInstance(x))
# THEORY rejected_three_only_pure
# EXPECT rejected_three_only_pure F
exists x.(ConnectedWith(x,formless_realm) & -HasImpureInstance(x))
# THEORY rejected_dharma_principle_is_essence
# EXPECT rejected_dharma_principle_is_essence F
exists x.(x = dharma_principle & EssenceLevel(x))
# THEORY rejected_formless_among_eighteen
# EXPECT rejected_formless_among_eighteen F
exists x.(x = formless_realm & AmongEighteen(x))
# THEORY rejected_formless_not_principle
# EXPECT rejected_formless_not_principle F
exists x.(x = formless_realm & -Principle(x))
"""

valuation_text = r"""mind => mano_d
dharma_principle => dharma_d
mind_cognition => manovijnana_d
dharma_essence => dharma_ayatana
formless_realm => arupya_dhatu
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,mano_d,dharma_d,manovijnana_d}
Principle => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,mano_d,dharma_d,manovijnana_d,arupya_dhatu}
EssenceLevel => {dharma_ayatana}
RealmLevel => {arupya_dhatu}
ConnectedWith => {(mano_d,arupya_dhatu),(dharma_d,arupya_dhatu),(manovijnana_d,arupya_dhatu)}
FormNatured => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
SupportAndCondition => {(rupa_d,caksurvijnana_d),(sabda_d,srotravijnana_d),(gandha_d,ghranavijnana_d),(rasa_d,jihvavijnana_d),(sprastavya_d,kayavijnana_d),(caksur_d,caksurvijnana_d),(srotra_d,srotravijnana_d),(ghrana_d,ghranavijnana_d),(jihva_d,jihvavijnana_d),(kaya_d,kayavijnana_d)}
InstanceOf => {(caksur_d_imp,caksur_d),(srotra_d_imp,srotra_d),(ghrana_d_imp,ghrana_d),(jihva_d_imp,jihva_d),(kaya_d_imp,kaya_d),(rupa_d_imp,rupa_d),(sabda_d_imp,sabda_d),(gandha_d_imp,gandha_d),(rasa_d_imp,rasa_d),(sprastavya_d_imp,sprastavya_d),(caksurvijnana_d_imp,caksurvijnana_d),(srotravijnana_d_imp,srotravijnana_d),(ghranavijnana_d_imp,ghranavijnana_d),(jihvavijnana_d_imp,jihvavijnana_d),(kayavijnana_d_imp,kayavijnana_d),(mano_d_imp,mano_d),(dharma_d_imp,dharma_d),(manovijnana_d_imp,manovijnana_d),(mano_d_path,mano_d),(dharma_d_path,dharma_d),(manovijnana_d_path,manovijnana_d),(dharma_d_unc,dharma_d)}
Pure => {mano_d_path,dharma_d_path,manovijnana_d_path,dharma_d_unc}
InPathTruth => {mano_d_path,dharma_d_path,manovijnana_d_path}
Unconditioned => {dharma_d_unc}
HasPureInstance => {mano_d,dharma_d,manovijnana_d}
HasImpureInstance => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,mano_d,dharma_d,manovijnana_d}
FreeOfFormPassion => {arupya_born}
BornIn => {(arupya_born,arupya_dhatu)}
"""


In [64]:
studies[31] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[31]["tree"].pformat())
print("Derived SEM:", studies[31]["sem"])


parses 1 SEM=ok
verse=T prose=T rejected_form_natured_in_formless=F rejected_five_cognitions_in_formless=F rejected_pure_means_formless=F rejected_fifteen_have_pure=F rejected_three_only_pure=F rejected_dharma_principle_is_essence=F rejected_formless_among_eighteen=F rejected_formless_not_principle=F
(S[SEM=<(all x.(AmongEighteen(x) -> (ConnectedWith(x,formless_realm) <-> ((x = mind) | (x = dharma_principle) | (x = mind_cognition)))) & all x.((AmongEighteen(x) & ConnectedWith(x,formless_realm)) -> (HasImpureInstance(x) & HasPureInstance(x))) & all x.((AmongEighteen(x) & -ConnectedWith(x,formless_realm)) -> (HasImpureInstance(x) & -HasPureInstance(x))))>]
  (FORMLESS[SEM=<all x.(AmongEighteen(x) -> (ConnectedWith(x,formless_realm) <-> ((x = mind) | (x = dharma_principle) | (x = mind_cognition))))>]
    (CONN[CASE='nom', GEN='m', NUM='pl', SEM=<\x.ConnectedWith(x,formless_realm)>]
      ārūpya-āptāḥ)
    (THREE[CASE='nom', GEN='m', NUM='pl', SEM=<\x.((x = mind) | (x = dharma_principle) |

# VAK_1.32 — With Both, With One, With Neither

Source: `VAK_1.32.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
savitarkavicārā hi pañca vijñānadhātavaḥ /
antyās trayas triprakārāḥ śeṣā ubhayavarjitāḥ // 1.32 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| sa-vitarka-vicārāḥ | bahuvrīhi; nominative masculine plural | with vitarka and with vicāra |
| hi | particle | restrictive (*niyamārtha*): always, not sometimes |
| pañca | numeral | five |
| vijñāna-dhātavaḥ | tatpuruṣa; nominative masculine plural | the Cognition Principles |
| antyāḥ | antya; nominative masculine plural | the last; Mind, Dharma, Mind-Cognition |
| trayaḥ | tri; nominative masculine plural | three |
| tri-prakārāḥ | bahuvrīhi; nominative masculine plural | of three kinds |
| śeṣāḥ | śeṣa; nominative masculine plural | the remaining ten |
| ubhaya-varjitāḥ | tatpuruṣa; varjita, past participle; nominative masculine plural | without both |

Vitarka and vicāra stay Sanskrit. The last three are the Mind Principle, the Dharma Principle, and the Mind-Cognition Principle. The Dharma Principle is not the Dharma Essence.

**Grammar tokens.** Verse-order IAST with sandhi opened (`antyās trayas`
→ *antyāḥ trayaḥ*, `śeṣā ubhaya-` → *śeṣāḥ ubhaya-*) and compound
boundaries marked.

```text
# tokens: sa-vitarka-vicārāḥ hi pañca vijñāna-dhātavaḥ antyāḥ trayaḥ tri-prakārāḥ śeṣāḥ ubhaya-varjitāḥ
```

## 5. Translation

### Literal Translation

> The five Cognition Principles are with vitarka and with vicāra. The last
> three are of three kinds. The rest are without both.

### Bhāṣya-informed study translation

> The five Cognition Principles are accompanied by both vitarka and vicāra.
> The last three—Mind Principle, Dharma Principle, and Mind-Cognition
> Principle—are of three kinds; the remaining ten are without both.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| vitarka | vitarka | not translated on this page |
| vicāra | vicāra | not translated on this page |
| pañca vijñāna-dhātavaḥ | five Cognition Principles | always with both |
| antyāḥ trayaḥ | the last three | Mind, Dharma, Mind-Cognition |
| dharma-dhātu | Dharma Principle | not the Dharma Essence |
| tri-prakāra | of three kinds | by level, not by name |
| dhyānāntara | intermediate concentration | without vitarka, with vicāra |
| bhavāgra | summit of existence | without both |
| ubhaya-varjita | without both | the ten whose nature is Form |
| asaṃprayukta | unassociated | the rest of the Dharma Principle |

In [65]:
fcfg_text = r"""% start S
# tokens: sa-vitarka-vicārāḥ hi pañca vijñāna-dhātavaḥ antyāḥ trayaḥ tri-prakārāḥ śeṣāḥ ubhaya-varjitāḥ
S[SEM=<(?five & ?last & ?rest)>] -> FIVECL[SEM=?five] LASTCL[SEM=?last] RESTCL[SEM=?rest]
FIVECL[SEM=<all x.(?subj(x) -> all k.(Kind(x,k) <-> ?pred(k)))>] -> PRED[CASE=nom,NUM=?n,GEN=?g,SEM=?pred] HI[RESTR=yes] FIVE SUBJ[CASE=nom,NUM=?n,GEN=?g,SEM=?subj]
LASTCL[SEM=<all x.(?subj(x) -> ?tri(x))>] -> LAST[CASE=nom,NUM=?n,GEN=?g,SEM=?subj] THREE[CASE=nom,NUM=?n,GEN=?g] TRI[CASE=nom,NUM=?n,GEN=?g,SEM=?tri]
RESTCL[SEM=<all x.(?subj(x) -> all k.(Kind(x,k) <-> ?pred(k)))>] -> REST[CASE=nom,NUM=?n,GEN=?g,SEM=?subj] PRED[CASE=nom,NUM=?n,GEN=?g,SEM=?pred]
PRED[CASE=nom,NUM=pl,GEN=m,SEM=<\k.(k = with_both)>] -> 'sa-vitarka-vicārāḥ'
HI[RESTR=yes] -> 'hi'
FIVE -> 'pañca'
SUBJ[CASE=nom,NUM=pl,GEN=m,SEM=<\x.FiveCognition(x)>] -> 'vijñāna-dhātavaḥ'
LAST[CASE=nom,NUM=pl,GEN=m,SEM=<\x.LastThree(x)>] -> 'antyāḥ'
THREE[CASE=nom,NUM=pl,GEN=m] -> 'trayaḥ'
TRI[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(Kind(x,with_both) & Kind(x,vicara_only) & Kind(x,without_both))>] -> 'tri-prakārāḥ'
REST[CASE=nom,NUM=pl,GEN=m,SEM=<\x.(AmongEighteen(x) & -FiveCognition(x) & -LastThree(x))>] -> 'śeṣāḥ'
PRED[CASE=nom,NUM=pl,GEN=m,SEM=<\k.(k = without_both)>] -> 'ubhaya-varjitāḥ'
"""

dependency_text = r"""'sa-vitarka-vicārāḥ' -> 'hi' | 'vijñāna-dhātavaḥ'
'vijñāna-dhātavaḥ' -> 'pañca'
'tri-prakārāḥ' -> 'trayaḥ'
'trayaḥ' -> 'antyāḥ'
'ubhaya-varjitāḥ' -> 'śeṣāḥ'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(FiveCognition(x) -> all k.(Kind(x,k) <-> (k = with_both))) & all x.(LastThree(x) -> (Kind(x,with_both) & Kind(x,vicara_only) & Kind(x,without_both))) & all x.((AmongEighteen(x) & -FiveCognition(x) & -LastThree(x)) -> all k.(Kind(x,k) <-> (k = without_both))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(FiveCognition(x) -> all k.(Kind(x,k) <-> (k = with_both))) & all x.(LastThree(x) -> (Kind(x,with_both) & Kind(x,vicara_only) & Kind(x,without_both))) & all x.((AmongEighteen(x) & -FiveCognition(x) & -LastThree(x)) -> all k.(Kind(x,k) <-> (k = without_both))))
# THEORY prose
# EXPECT prose T
all i.(AssocNonPair(i) -> (KindAt(i,kama,with_both) & KindAt(i,first_dhyana,with_both) & KindAt(i,intermediate,vicara_only) & KindAt(i,second_onward,without_both)))
all i.(Unassociated(i) -> all l.(Level(l) -> KindAt(i,l,without_both)))
exists x.(x = vicara & KindAt(x,intermediate,without_both))
all l.(BothLevel(l) -> (KindAt(vitarka,l,vicara_only) & KindAt(vicara,l,vitarka_only)))
-(exists l.KindAt(vitarka,l,with_both)) & -(exists l.KindAt(vicara,l,with_both))
PortionOf(vitarka,dharma_principle) & PortionOf(vicara,dharma_principle)
all x.(LastThree(x) -> all k.(Kind(x,k) <-> exists i.exists l.(PortionOf(i,x) & KindAt(i,l,k))))
all l.(BothLevel(l) -> (exists i.KindAt(i,l,with_both) & exists i.KindAt(i,l,vicara_only) & exists i.KindAt(i,l,vitarka_only) & exists i.KindAt(i,l,without_both)))
all x.(FormNatured(x) -> NotAssociated(x))
all x.(NotAssociated(x) -> all k.(Kind(x,k) -> (k = without_both)))
# THEORY rejected_five_sometimes_without
# EXPECT rejected_five_sometimes_without F
exists x.(FiveCognition(x) & Kind(x,without_both))
# THEORY rejected_dharma_exactly_three_kinds
# EXPECT rejected_dharma_exactly_three_kinds F
-Kind(dharma_principle,vitarka_only)
# THEORY rejected_one_kind_per_principle
# EXPECT rejected_one_kind_per_principle F
exists x.(LastThree(x) & exists k.all j.(Kind(x,j) <-> (j = k)))
# THEORY rejected_vitarka_with_second_vitarka
# EXPECT rejected_vitarka_with_second_vitarka F
exists l.KindAt(vitarka,l,with_both)
# THEORY rejected_intermediate_has_vitarka
# EXPECT rejected_intermediate_has_vitarka F
exists i.KindAt(i,intermediate,with_both)
# THEORY rejected_form_natured_with_pair
# EXPECT rejected_form_natured_with_pair F
exists x.(FormNatured(x) & (Kind(x,with_both) | Kind(x,vicara_only) | Kind(x,vitarka_only)))
"""

valuation_text = r"""with_both => both_k
vicara_only => vicara_only_k
vitarka_only => vitarka_only_k
without_both => neither_k
kama => kama_l
first_dhyana => dhyana1_l
intermediate => antara_l
second_onward => higher_l
vitarka => vitarka_i
vicara => vicara_i
dharma_principle => dharma_d
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,mano_d,dharma_d,manovijnana_d}
FiveCognition => {caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d}
LastThree => {mano_d,dharma_d,manovijnana_d}
FormNatured => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
NotAssociated => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
Kind => {(caksurvijnana_d,both_k),(srotravijnana_d,both_k),(ghranavijnana_d,both_k),(jihvavijnana_d,both_k),(kayavijnana_d,both_k),(mano_d,both_k),(mano_d,vicara_only_k),(mano_d,neither_k),(manovijnana_d,both_k),(manovijnana_d,vicara_only_k),(manovijnana_d,neither_k),(dharma_d,both_k),(dharma_d,vicara_only_k),(dharma_d,neither_k),(dharma_d,vitarka_only_k),(caksur_d,neither_k),(srotra_d,neither_k),(ghrana_d,neither_k),(jihva_d,neither_k),(kaya_d,neither_k),(rupa_d,neither_k),(sabda_d,neither_k),(gandha_d,neither_k),(rasa_d,neither_k),(sprastavya_d,neither_k)}
Level => {kama_l,dhyana1_l,antara_l,higher_l}
BothLevel => {kama_l,dhyana1_l}
KindAt => {(mano_i,kama_l,both_k),(mano_i,dhyana1_l,both_k),(mano_i,antara_l,vicara_only_k),(mano_i,higher_l,neither_k),(manovijnana_i,kama_l,both_k),(manovijnana_i,dhyana1_l,both_k),(manovijnana_i,antara_l,vicara_only_k),(manovijnana_i,higher_l,neither_k),(dharma_assoc_i,kama_l,both_k),(dharma_assoc_i,dhyana1_l,both_k),(dharma_assoc_i,antara_l,vicara_only_k),(dharma_assoc_i,higher_l,neither_k),(dharma_unassoc_i,kama_l,neither_k),(dharma_unassoc_i,dhyana1_l,neither_k),(dharma_unassoc_i,antara_l,neither_k),(dharma_unassoc_i,higher_l,neither_k),(vitarka_i,kama_l,vicara_only_k),(vitarka_i,dhyana1_l,vicara_only_k),(vicara_i,kama_l,vitarka_only_k),(vicara_i,dhyana1_l,vitarka_only_k),(vicara_i,antara_l,neither_k)}
PortionOf => {(mano_i,mano_d),(manovijnana_i,manovijnana_d),(dharma_assoc_i,dharma_d),(dharma_unassoc_i,dharma_d),(vitarka_i,dharma_d),(vicara_i,dharma_d)}
AssocNonPair => {mano_i,manovijnana_i,dharma_assoc_i}
Unassociated => {dharma_unassoc_i}
"""


In [67]:
studies[32] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[32]["tree"].pformat())
print("Derived SEM:", studies[32]["sem"])


ValueError: theory prose: Refusing to evaluate formula: over a domain of 32 element(s) its nested quantifiers would explore mo

# VAK_1.33 — Without the Other Two

Source: `VAK_1.33.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
nirūpaṇānusmaraṇavikalpenāvikalpakāḥ /
tau prajñā mānasī vyagrā smṛtiḥ sarvaiva mānasī // 1.33 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| nirūpaṇa-anusmaraṇa-vikalpena | dvandva inside tatpuruṣa; instrumental singular | in respect of examining and recollective vikalpa |
| avikalpakāḥ | a-vikalpaka, bahuvrīhi; nominative masculine plural | called without vikalpa |
| tau | tad; nominative masculine dual | those two: the examining and the recollective |
| prajñā | nominative feminine singular | prajñā |
| mānasī | mānasa; nominative feminine singular | mental: associated with Mind-Cognition |
| vyagrā | vy-agra; nominative feminine singular | unconcentrated |
| smṛtiḥ | nominative feminine singular | smṛti |
| sarvā | sarva; nominative feminine singular | all |
| eva | particle | just; all of it without restriction |

The understood subject is the five Cognition Principles of 1.32. The verse says nirūpaṇa. The commentary says abhinirūpaṇa. Both stay.

**Grammar tokens.** Verse-order IAST with sandhi opened
(`-vikalpenāvikalpakāḥ` → *-vikalpena avikalpakāḥ*, `sarvaiva` → *sarvā
eva*) and compound boundaries marked. *Mānasī* occurs twice, once in each
identification.

```text
# tokens: nirūpaṇa-anusmaraṇa-vikalpena avikalpakāḥ tau prajñā mānasī vyagrā smṛtiḥ sarvā eva mānasī
```

## 5. Translation

### Literal Translation

> They are without vikalpa in respect of examining and recollective vikalpa.
> Those two are unconcentrated mental prajñā, and all mental smṛti.

### Bhāṣya-informed study translation

> The five sensory Cognition Principles lack examining and recollective
> discrimination. Those two are, respectively, unconcentrated mental
> discriminative comprehension and all mental mindfulness.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| avikalpaka | without vikalpa | in respect of the other two |
| svabhāva-vikalpa | intrinsic vikalpa | vitarka |
| abhinirūpaṇa-vikalpa | examining vikalpa | unconcentrated mental prajñā |
| anusmaraṇa-vikalpa | recollective vikalpa | all mental smṛti |
| mānasī | mental | associated with Mind-Cognition |
| vyagrā | unconcentrated | asamāhitā |
| prajñā | prajñā | not science on this page |
| smṛti | smṛti | all of it that is mental |

In [ ]:
fcfg_text = r"""% start S
# tokens: nirūpaṇa-anusmaraṇa-vikalpena avikalpakāḥ tau prajñā mānasī vyagrā smṛtiḥ sarvā eva mānasī
S[SEM=<(?neg & ?ident)>] -> NEG[SEM=?neg] IDENT[SEM=?ident]
NEG[SEM=<all x.(FiveCognition(x) -> (CalledAvikalpaka(x) & -exists p.(Possesses(x,p) & ?resp(p))))>] -> RESPECT[CASE=ins,SEM=?resp] AVIK[CASE=nom,NUM=pl,GEN=m]
IDENT[SEM=<(?a & ?b)>] -> TAU[NUM=dual] FIRST[SEM=?a] SECOND[SEM=?b]
FIRST[SEM=<all p.(Vikalpa(p,examining) <-> (?n(p) & ?m(p) & ?v(p)))>] -> HEADN[CASE=nom,NUM=sg,GEN=?g,SEM=?n] MENT[CASE=nom,NUM=sg,GEN=?g,SEM=?m] VY[CASE=nom,NUM=sg,GEN=?g,SEM=?v]
SECOND[SEM=<all p.(Vikalpa(p,recollective) <-> (?n(p) & ?m(p)))>] -> HEADN[CASE=nom,NUM=sg,GEN=?g,SEM=?n] ALLQ[CASE=nom,NUM=sg,GEN=?g] EVA MENT[CASE=nom,NUM=sg,GEN=?g,SEM=?m]
RESPECT[CASE=ins,SEM=<\p.(Vikalpa(p,examining) | Vikalpa(p,recollective))>] -> 'nirūpaṇa-anusmaraṇa-vikalpena'
AVIK[CASE=nom,NUM=pl,GEN=m] -> 'avikalpakāḥ'
TAU[NUM=dual] -> 'tau'
HEADN[CASE=nom,NUM=sg,GEN=f,SEM=<\p.Prajna(p)>] -> 'prajñā'
MENT[CASE=nom,NUM=sg,GEN=f,SEM=<\p.Mental(p)>] -> 'mānasī'
VY[CASE=nom,NUM=sg,GEN=f,SEM=<\p.Unconcentrated(p)>] -> 'vyagrā'
HEADN[CASE=nom,NUM=sg,GEN=f,SEM=<\p.Smrti(p)>] -> 'smṛtiḥ'
ALLQ[CASE=nom,NUM=sg,GEN=f] -> 'sarvā'
EVA -> 'eva'
"""

dependency_text = r"""'avikalpakāḥ' -> 'nirūpaṇa-anusmaraṇa-vikalpena'
'tau' -> 'prajñā' | 'smṛtiḥ'
'prajñā' -> 'mānasī' | 'vyagrā'
'smṛtiḥ' -> 'sarvā' | 'mānasī'
'sarvā' -> 'eva'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(FiveCognition(x) -> (CalledAvikalpaka(x) & -exists p.(Possesses(x,p) & (Vikalpa(p,examining) | Vikalpa(p,recollective))))) & all p.(Vikalpa(p,examining) <-> (Prajna(p) & Mental(p) & Unconcentrated(p))) & all p.(Vikalpa(p,recollective) <-> (Smrti(p) & Mental(p))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(FiveCognition(x) -> (CalledAvikalpaka(x) & -exists p.(Possesses(x,p) & (Vikalpa(p,examining) | Vikalpa(p,recollective))))) & all p.(Vikalpa(p,examining) <-> (Prajna(p) & Mental(p) & Unconcentrated(p))) & all p.(Vikalpa(p,recollective) <-> (Smrti(p) & Mental(p))))
# THEORY prose
# EXPECT prose T
all p.all k.(Vikalpa(p,k) -> ((k = intrinsic) | (k = examining) | (k = recollective)))
all p.(Vikalpa(p,intrinsic) <-> Vitarka(p))
all x.(FiveCognition(x) -> exists p.(Possesses(x,p) & Vikalpa(p,intrinsic)))
all p.(Mental(p) <-> AssociatedWith(p,mind_cognition))
all p.(Vikalpa(p,recollective) -> (Concentrated(p) | Unconcentrated(p)))
exists p.(Vikalpa(p,recollective) & Concentrated(p)) & exists p.(Vikalpa(p,recollective) & Unconcentrated(p))
exists x.(FiveCognition(x) & exists p.(Possesses(x,p) & Prajna(p) & -Vikalpa(p,examining)))
exists h.(h = horse & OneFooted(h) & CalledFootless(h))
exists v.(v = vitarka & DeferredTo(v,caitasika_section))
# THEORY rejected_five_without_any_vikalpa
# EXPECT rejected_five_without_any_vikalpa F
exists x.(FiveCognition(x) & -exists p.(Possesses(x,p) & Vikalpa(p,intrinsic)))
# THEORY rejected_fourth_vikalpa
# EXPECT rejected_fourth_vikalpa F
exists p.exists k.(Vikalpa(p,k) & -(k = intrinsic) & -(k = examining) & -(k = recollective))
# THEORY rejected_concentrated_prajna_examines
# EXPECT rejected_concentrated_prajna_examines F
exists p.(Prajna(p) & Concentrated(p) & Vikalpa(p,examining))
# THEORY rejected_recollection_only_unconcentrated
# EXPECT rejected_recollection_only_unconcentrated F
all p.(Vikalpa(p,recollective) -> Unconcentrated(p))
# THEORY rejected_sense_prajna_examines
# EXPECT rejected_sense_prajna_examines F
exists p.(Prajna(p) & -Mental(p) & Vikalpa(p,examining))
# THEORY rejected_five_examine
# EXPECT rejected_five_examine F
exists x.exists p.(FiveCognition(x) & Possesses(x,p) & Vikalpa(p,examining))
"""

valuation_text = r"""intrinsic => intrinsic_k
examining => examining_k
recollective => recollective_k
mind_cognition => manovijnana_d
vitarka => vitarka_i
caitasika_section => caitasika_ch2
horse => asva
FiveCognition => {caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d}
Possesses => {(caksurvijnana_d,vitarka_i),(caksurvijnana_d,prajna_sense),(caksurvijnana_d,smrti_sense),(srotravijnana_d,vitarka_i),(srotravijnana_d,prajna_sense),(srotravijnana_d,smrti_sense),(ghranavijnana_d,vitarka_i),(ghranavijnana_d,prajna_sense),(ghranavijnana_d,smrti_sense),(jihvavijnana_d,vitarka_i),(jihvavijnana_d,prajna_sense),(jihvavijnana_d,smrti_sense),(kayavijnana_d,vitarka_i),(kayavijnana_d,prajna_sense),(kayavijnana_d,smrti_sense)}
Vikalpa => {(vitarka_i,intrinsic_k),(prajna_vyagra,examining_k),(smrti_vyagra,recollective_k),(smrti_samahita,recollective_k)}
Vitarka => {vitarka_i}
Prajna => {prajna_sense,prajna_vyagra,prajna_samahita}
Smrti => {smrti_sense,smrti_vyagra,smrti_samahita}
Mental => {prajna_vyagra,prajna_samahita,smrti_vyagra,smrti_samahita}
AssociatedWith => {(prajna_vyagra,manovijnana_d),(prajna_samahita,manovijnana_d),(smrti_vyagra,manovijnana_d),(smrti_samahita,manovijnana_d)}
Unconcentrated => {prajna_vyagra,smrti_vyagra}
Concentrated => {prajna_samahita,smrti_samahita}
CalledAvikalpaka => {caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d}
OneFooted => {asva}
CalledFootless => {asva}
DeferredTo => {(vitarka_i,caitasika_ch2)}
"""


In [ ]:
studies[33] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[33]["tree"].pformat())
print("Derived SEM:", studies[33]["sem"])


# VAK_1.34 — Taking a Support, and Appropriated

Source: `VAK_1.34.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
sapta sālambanāś cittadhātavo 'rdhaṃ ca dharmataḥ /
navānupāttās te cāṣṭau śabdaś cānye nava dvidhā // 1.34 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| sapta | numeral, nominative | seven: the count of the consciousness Principles |
| sālambanāḥ | sa-ālambana, bahuvrīhi; nominative masculine plural | having a Support: they grasp a Condition |
| citta-dhātavaḥ | tatpuruṣa; nominative masculine plural | the consciousness Principles: the six Cognition Principles and the Mind Principle |
| ardham | neuter accusative/nominative singular | a half |
| ca | particle | and |
| dharmataḥ | dharma + -tas; ablatival adverb | with respect to the Dharma Principle |
| nava | numeral | nine |
| anupāttāḥ | an-upātta; nominative plural | Unappropriated: never taken up as a basis |
| te | tad; nominative plural | those: resumes the first count |
| aṣṭau | numeral | eight: the seven and the Dharma Principle as eighth |
| śabdaḥ | nominative masculine singular | Sound |
| anye | anya; nominative plural | the others: the rest of the eighteen |
| dvidhā | adverb of manner | twofold: Appropriated and Unappropriated |

The half is the associated portion. The eight of the second count are the
seven plus the whole Dharma Principle. The scope changes: the first count
runs over portions, the second over Principles.

**Grammar tokens.** Verse-order IAST with sandhi opened (`sālambanāś` →
*sālambanāḥ*, `cittadhātavo 'rdhaṃ` → *citta-dhātavaḥ ardham*,
`navānupāttās` → *nava anupāttāḥ*, `cāṣṭau` → *ca aṣṭau*, `śabdaś cānye`
→ *śabdaḥ ca anye*) and the compound boundary marked. *Ca* occurs three
times; *nava* twice.

```text
# tokens: sapta sālambanāḥ citta-dhātavaḥ ardham ca dharmataḥ nava anupāttāḥ te ca aṣṭau śabdaḥ ca anye nava dvidhā
```

## 5. Translation

### Literal Translation

> Seven consciousness Principles have a Support, and a half with respect to
> the Dharma Principle. Nine are Unappropriated: those eight, and Sound. The
> other nine are twofold.

### Bhāṣya-informed study translation

> Seven consciousness Principles and the mental-factor portion of the Dharma
> Principle take a Support. Nine are Unappropriated: those eight and Sound.
> The other nine are twofold.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| sālambana | taking a Support | glossed *viṣaya-grahaṇa*, grasping a Condition |
| anālambana | not taking a Support | the ten of Form, and the unassociated half |
| viṣaya | Condition | what is grasped (1.29) |
| citta-dhātu | consciousness Principle | the six Cognition Principles and the Mind Principle |
| ardham dharmataḥ | a half of the Dharma Principle | the associated mental factors |
| anupātta | Unappropriated | nine, in every instance |
| upātta | Appropriated | a present Faculty, or a present meaning not separate from it |
| adhiṣṭhāna | basis | what citta and the mental factors take up |
| śabda | Sound | among the nine always Unappropriated |
| sacetana | animate | the world's name for Appropriated |

In [ ]:
fcfg_text = r"""% start S
# tokens: sapta sālambanāḥ citta-dhātavaḥ ardham ca dharmataḥ nava anupāttāḥ te ca aṣṭau śabdaḥ ca anye nava dvidhā
S[SEM=<(?a & ?b)>] -> SUPPORTCUT[SEM=?a] APPROPCUT[SEM=?b]
SUPPORTCUT[SEM=<(all x.(?c(x) -> ?s(x)) & exists h.(PortionOf(h,?d) & ?s(h)))>] -> NUMR[CNT=?k] SAL[CASE=nom,NUM=pl,SEM=?s] CITTA[CASE=nom,NUM=pl,CNT=?k,SEM=?c] HALF CA ABLT[SEM=?d]
APPROPCUT[SEM=<(all x.(?n(x) -> all i.(InstanceOf(i,x) -> -Appropriated(i))) & all x.((AmongEighteen(x) & -?n(x)) -> ?tw(x)))>] -> NUMR[CNT=?k] ANUP[CASE=nom,NUM=pl] LIST[CNT=?k,SEM=?n] ANYE NUMR[CNT=?k] DVIDHA[SEM=?tw]
LIST[CNT=9,SEM=<\x.(?e(x) | ?w(x))>] -> TE CA EIGHT[CNT=8,SEM=?e] SABDA[CNT=1,SEM=?w] CA
NUMR[CNT=7] -> 'sapta'
NUMR[CNT=9] -> 'nava'
SAL[CASE=nom,NUM=pl,SEM=<\x.TakesSupport(x)>] -> 'sālambanāḥ'
CITTA[CASE=nom,NUM=pl,CNT=7,SEM=<\x.CittaPrinciple(x)>] -> 'citta-dhātavaḥ'
HALF -> 'ardham'
CA -> 'ca'
ABLT[SEM=<dharma_principle>] -> 'dharmataḥ'
ANUP[CASE=nom,NUM=pl] -> 'anupāttāḥ'
TE -> 'te'
EIGHT[CNT=8,SEM=<\x.(CittaPrinciple(x) | (x = dharma_principle))>] -> 'aṣṭau'
SABDA[CNT=1,SEM=<\x.(x = sound)>] -> 'śabdaḥ'
ANYE -> 'anye'
DVIDHA[SEM=<\x.(exists i.(InstanceOf(i,x) & Appropriated(i)) & exists i.(InstanceOf(i,x) & -Appropriated(i)))>] -> 'dvidhā'
"""

dependency_text = r"""'sālambanāḥ' -> 'citta-dhātavaḥ' | 'ardham'
'citta-dhātavaḥ' -> 'sapta'
'ardham' -> 'dharmataḥ' | 'ca'
'anupāttāḥ' -> 'nava' | 'te' | 'śabdaḥ'
'te' -> 'aṣṭau' | 'ca'
'śabdaḥ' -> 'ca'
'dvidhā' -> 'anye'
'anye' -> 'nava'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(CittaPrinciple(x) -> TakesSupport(x)) & exists h.(PortionOf(h,dharma_principle) & TakesSupport(h)) & all x.((CittaPrinciple(x) | (x = dharma_principle) | (x = sound)) -> all i.(InstanceOf(i,x) -> -Appropriated(i))) & all x.((AmongEighteen(x) & -(CittaPrinciple(x) | (x = dharma_principle) | (x = sound))) -> (exists i.(InstanceOf(i,x) & Appropriated(i)) & exists i.(InstanceOf(i,x) & -Appropriated(i)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(CittaPrinciple(x) -> TakesSupport(x)) & exists h.(PortionOf(h,dharma_principle) & TakesSupport(h)) & all x.((CittaPrinciple(x) | (x = dharma_principle) | (x = sound)) -> all i.(InstanceOf(i,x) -> -Appropriated(i))) & all x.((AmongEighteen(x) & -(CittaPrinciple(x) | (x = dharma_principle) | (x = sound))) -> (exists i.(InstanceOf(i,x) & Appropriated(i)) & exists i.(InstanceOf(i,x) & -Appropriated(i)))))
# THEORY prose
# EXPECT prose T
all x.(Item(x) -> (TakesSupport(x) <-> (CittaPrinciple(x) | (x = dharma_associated))))
all x.(TakesSupport(x) <-> GraspsCondition(x))
all x.((AmongEighteen(x) & FormBearing(x)) -> -TakesSupport(x))
exists x.(x = dharma_nonassociated & PortionOf(x,dharma_principle) & -TakesSupport(x))
exists x.(x = dharma_principle & AmongEighteen(x) & -Item(x))
all x.(FourMeaning(x) <-> (ConditionMeaning(x) & -(x = sound)))
all i.all x.((InstanceOf(i,x) & SenseFaculty(x)) -> (Appropriated(i) <-> Present(i)))
all i.all x.((InstanceOf(i,x) & FourMeaning(x)) -> (Appropriated(i) <-> (Present(i) & NotSeparateFromFaculty(i))))
all i.(Appropriated(i) <-> TakenAsBasis(i))
all i.(TakenAsBasis(i) -> MutualBenefitInjury(i))
all i.(Appropriated(i) <-> CalledAnimate(i))
exists i.(InstanceOf(i,sound) & Present(i) & -Appropriated(i))
# THEORY rejected_unassociated_half_takes_support
# EXPECT rejected_unassociated_half_takes_support F
exists x.(x = dharma_nonassociated & TakesSupport(x))
# THEORY rejected_form_takes_support
# EXPECT rejected_form_takes_support F
exists x.(AmongEighteen(x) & FormBearing(x) & TakesSupport(x))
# THEORY rejected_the_two_cuts_coincide
# EXPECT rejected_the_two_cuts_coincide F
all x.(AmongEighteen(x) -> (TakesSupport(x) <-> exists i.(InstanceOf(i,x) & Appropriated(i))))
# THEORY rejected_past_faculty_appropriated
# EXPECT rejected_past_faculty_appropriated F
exists i.exists x.(InstanceOf(i,x) & SenseFaculty(x) & -Present(i) & Appropriated(i))
# THEORY rejected_presence_suffices
# EXPECT rejected_presence_suffices F
all i.all x.((InstanceOf(i,x) & FourMeaning(x) & Present(i)) -> Appropriated(i))
# THEORY rejected_sound_twofold
# EXPECT rejected_sound_twofold F
exists i.(InstanceOf(i,sound) & Appropriated(i))
"""

valuation_text = r"""sound => sabda_d
dharma_principle => dharma_d
dharma_associated => dharma_samprayukta
dharma_nonassociated => dharma_asamprayukta
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
Item => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_samprayukta,dharma_asamprayukta}
CittaPrinciple => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
TakesSupport => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_samprayukta}
GraspsCondition => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_samprayukta}
FormBearing => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
SenseFaculty => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
ConditionMeaning => {rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
FourMeaning => {rupa_d,gandha_d,rasa_d,sprastavya_d}
PortionOf => {(dharma_samprayukta,dharma_d),(dharma_asamprayukta,dharma_d)}
InstanceOf => {(caksur_present,caksur_d),(caksur_past,caksur_d),(srotra_present,srotra_d),(srotra_past,srotra_d),(ghrana_present,ghrana_d),(ghrana_past,ghrana_d),(jihva_present,jihva_d),(jihva_past,jihva_d),(kaya_present,kaya_d),(kaya_past,kaya_d),(rupa_bound,rupa_d),(rupa_hair,rupa_d),(rupa_earth,rupa_d),(gandha_bound,gandha_d),(gandha_hair,gandha_d),(gandha_earth,gandha_d),(rasa_bound,rasa_d),(rasa_hair,rasa_d),(rasa_earth,rasa_d),(sprastavya_bound,sprastavya_d),(sprastavya_hair,sprastavya_d),(sprastavya_earth,sprastavya_d),(sabda_present,sabda_d),(mano_now,mano_d),(caksurvijnana_now,caksurvijnana_d),(srotravijnana_now,srotravijnana_d),(ghranavijnana_now,ghranavijnana_d),(jihvavijnana_now,jihvavijnana_d),(kayavijnana_now,kayavijnana_d),(manovijnana_now,manovijnana_d),(dharma_now,dharma_d)}
Present => {caksur_present,srotra_present,ghrana_present,jihva_present,kaya_present,rupa_bound,rupa_hair,rupa_earth,gandha_bound,gandha_hair,gandha_earth,rasa_bound,rasa_hair,rasa_earth,sprastavya_bound,sprastavya_hair,sprastavya_earth,sabda_present,mano_now,caksurvijnana_now,srotravijnana_now,ghranavijnana_now,jihvavijnana_now,kayavijnana_now,manovijnana_now,dharma_now}
NotSeparateFromFaculty => {rupa_bound,gandha_bound,rasa_bound,sprastavya_bound}
Appropriated => {caksur_present,srotra_present,ghrana_present,jihva_present,kaya_present,rupa_bound,gandha_bound,rasa_bound,sprastavya_bound}
TakenAsBasis => {caksur_present,srotra_present,ghrana_present,jihva_present,kaya_present,rupa_bound,gandha_bound,rasa_bound,sprastavya_bound}
MutualBenefitInjury => {caksur_present,srotra_present,ghrana_present,jihva_present,kaya_present,rupa_bound,gandha_bound,rasa_bound,sprastavya_bound}
CalledAnimate => {caksur_present,srotra_present,ghrana_present,jihva_present,kaya_present,rupa_bound,gandha_bound,rasa_bound,sprastavya_bound}
"""


In [ ]:
studies[34] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[34]["tree"].pformat())
print("Derived SEM:", studies[34]["sem"])


# VAK_1.35 — Element, Element-dependent, Accumulated

Source: `VAK_1.35.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
spraṣṭavyaṃ dvividhaṃ śeṣā rūpiṇo nava bhautikāḥ /
dharmadhātvekadeśaś ca saṃcitā daśa rūpiṇaḥ // 1.35 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| spraṣṭavyam | spraṣṭavya; nominative neuter singular | the Tangible |
| dvividham | dvi-vidha; nominative neuter singular | twofold: Element and Element-dependent |
| śeṣāḥ | śeṣa; nominative masculine plural | the remaining: Form Principles other than the Tangible |
| rūpiṇaḥ | rūpin; nominative masculine plural | whose nature is Form |
| nava | numeral | nine |
| bhautikāḥ | bhautika, from bhūta; nominative masculine plural | Element-dependent |
| dharma-dhātu-ekadeśaḥ | tatpuruṣa; nominative masculine singular | one portion of the Dharma Principle |
| ca | particle | and: adds the portion to the nine |
| saṃcitāḥ | saṃ-√ci, past participle; nominative masculine plural | Accumulated: clusters of atoms |
| daśa | numeral | ten |
| rūpiṇaḥ | rūpin; nominative masculine plural | whose nature is Form |

Element is *bhūta*. Principle is *dhātu*. The portion of the Dharma
Principle is avijñapti. The two occurrences of *rūpiṇaḥ* name the same
class of ten; *śeṣāḥ* subtracts the Tangible.

**Grammar tokens.** Verse-order IAST with sandhi opened (`spraṣṭavyaṃ` →
*spraṣṭavyam*, `śeṣā rūpiṇo` → *śeṣāḥ rūpiṇaḥ*, `dharmadhātvekadeśaś` →
*dharma-dhātu-ekadeśaḥ*, `saṃcitā` → *saṃcitāḥ*) and compound boundaries
marked.

```text
# tokens: spraṣṭavyam dvividham śeṣāḥ rūpiṇaḥ nava bhautikāḥ dharma-dhātu-ekadeśaḥ ca saṃcitāḥ daśa rūpiṇaḥ
```

## 5. Translation

### Literal Translation

> The Tangible is twofold. The other nine whose nature is Form are Element-
> dependent. And a portion of the Dharma Principle. The ten whose nature is
> Form are Accumulated.

### Bhāṣya-informed study translation

> The Tangible is twofold. The other nine Form Principles, and one portion
> of the Dharma Principle, are Element-dependent. The ten Form Principles
> are Accumulated.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| bhūta, mahābhūta | Element, Great Element | four, in the Tangible |
| bhautika | Element-dependent | *upādāya-rūpa*: Form depending on the Great Elements |
| spraṣṭavya | Tangible | the only twofold |
| avijñapti | avijñapti | the Element-dependent portion of the Dharma Principle |
| saṃcita | Accumulated | clusters of atoms |
| daśa rūpiṇaḥ | ten whose nature is Form | five Faculties and five Condition-Principles |
| viṣaya | Condition | the four Conditions among the nine |
| āyatana | Essence | the sūtra’s word |
| saṃjñā | reflection | follows consciousness |

In [ ]:
fcfg_text = r"""% start S
# tokens: spraṣṭavyam dvividham śeṣāḥ rūpiṇaḥ nava bhautikāḥ dharma-dhātu-ekadeśaḥ ca saṃcitāḥ daśa rūpiṇaḥ
S[SEM=<(?a & ?b & ?c)>] -> TWOFOLD[SEM=?a] DEPENDENT[SEM=?b] ACCUM[SEM=?c]
TWOFOLD[SEM=<?d(?t)>] -> SPR[GEN=n,NUM=sg,SEM=?t] DVI[GEN=n,NUM=sg,SEM=?d]
DEPENDENT[SEM=<(all x.(?r(x) -> all i.(InstanceOf(i,x) -> ?b(i))) & exists h.(PortionOf(h,?p) & all i.(InstanceOf(i,h) -> ?b(i))))>] -> REST[CNT=9,SEM=?r] BHAUT[NUM=pl,SEM=?b] EKADESA[SEM=?p] CA
REST[CNT=9,SEM=<\x.(?f(x) & -(x = tangible))>] -> SESA[NUM=pl] RUPIN[NUM=pl,CNT=10,SEM=?f] NUMR[CNT=9]
ACCUM[SEM=<all x.(?r(x) -> ?s(x))>] -> SAMCITA[NUM=pl,SEM=?s] NUMR[CNT=?k] RUPIN[NUM=pl,CNT=?k,SEM=?r]
SPR[GEN=n,NUM=sg,SEM=<tangible>] -> 'spraṣṭavyam'
DVI[GEN=n,NUM=sg,SEM=<\t.(exists i.(InstanceOf(i,t) & GreatElement(i)) & exists i.(InstanceOf(i,t) & ElementDependent(i)))>] -> 'dvividham'
SESA[NUM=pl] -> 'śeṣāḥ'
RUPIN[NUM=pl,CNT=10,SEM=<\x.FormBearing(x)>] -> 'rūpiṇaḥ'
NUMR[CNT=9] -> 'nava'
NUMR[CNT=10] -> 'daśa'
BHAUT[NUM=pl,SEM=<\i.ElementDependent(i)>] -> 'bhautikāḥ'
EKADESA[SEM=<dharma_principle>] -> 'dharma-dhātu-ekadeśaḥ'
CA -> 'ca'
SAMCITA[NUM=pl,SEM=<\x.Accumulated(x)>] -> 'saṃcitāḥ'
"""

dependency_text = r"""'dvividham' -> 'spraṣṭavyam'
'bhautikāḥ' -> 'rūpiṇaḥ' | 'dharma-dhātu-ekadeśaḥ'
'rūpiṇaḥ' -> 'śeṣāḥ' | 'nava' | 'daśa'
'dharma-dhātu-ekadeśaḥ' -> 'ca'
'saṃcitāḥ' -> 'rūpiṇaḥ'
"""

sem_text = r"""# derived by the section 4 parse
(exists i.(InstanceOf(i,tangible) & GreatElement(i)) & exists i.(InstanceOf(i,tangible) & ElementDependent(i)) & all x.((FormBearing(x) & -(x = tangible)) -> all i.(InstanceOf(i,x) -> ElementDependent(i))) & exists h.(PortionOf(h,dharma_principle) & all i.(InstanceOf(i,h) -> ElementDependent(i))) & all x.(FormBearing(x) -> Accumulated(x)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(exists i.(InstanceOf(i,tangible) & GreatElement(i)) & exists i.(InstanceOf(i,tangible) & ElementDependent(i)) & all x.((FormBearing(x) & -(x = tangible)) -> all i.(InstanceOf(i,x) -> ElementDependent(i))) & exists h.(PortionOf(h,dharma_principle) & all i.(InstanceOf(i,h) -> ElementDependent(i))) & all x.(FormBearing(x) -> Accumulated(x)))
# THEORY prose
# EXPECT prose T
all i.(GreatElement(i) <-> ((i = earth) | (i = water) | (i = fire) | (i = wind)))
all i.((InstanceOf(i,tangible) & ElementDependent(i)) <-> ((i = smooth) | (i = rough) | (i = heavy) | (i = light) | (i = cold) | (i = hunger) | (i = thirst)))
all i.-(GreatElement(i) & ElementDependent(i))
all x.(AmongEighteen(x) -> (exists i.(InstanceOf(i,x) & GreatElement(i)) <-> (x = tangible)))
exists h.(h = avijnapti_portion & PortionOf(h,dharma_principle) & all i.(InstanceOf(i,h) -> ElementDependent(i)))
all x.(CittaPrinciple(x) -> all i.(InstanceOf(i,x) -> (-GreatElement(i) & -ElementDependent(i))))
exists h.(h = dharma_rest & PortionOf(h,dharma_principle) & all i.(InstanceOf(i,h) -> (-GreatElement(i) & -ElementDependent(i))))
all i.(GreatElement(i) -> all f.(GraspedBy(i,f) -> (f = body)))
all i.(InstanceOf(i,visible_form) -> -GraspedBy(i,body))
exists m.(m = eye_fleshy_mass & Contains(m,earth) & -SenseFaculty(m))
exists p.(p = person & SixPrincipleShowing(p))
exists f.(f = vedana & MentalFactor(f) & FollowsCitta(f) & -Citta(f))
all x.(AmongEighteen(x) -> (Accumulated(x) <-> AtomCluster(x)))
exists t.(t = buddhadeva_thesis & AttributedTo(t,buddhadeva) & Refuted(t))
# THEORY rejected_ten_are_only_elements
# EXPECT rejected_ten_are_only_elements F
all x.(FormBearing(x) -> all i.(InstanceOf(i,x) -> GreatElement(i)))
# THEORY rejected_elements_outside_the_tangible
# EXPECT rejected_elements_outside_the_tangible F
exists i.exists x.(GreatElement(i) & InstanceOf(i,x) & -(x = tangible))
# THEORY rejected_eye_grasps_hardness
# EXPECT rejected_eye_grasps_hardness F
exists i.(GreatElement(i) & GraspedBy(i,eye))
# THEORY rejected_fleshy_mass_is_the_faculty
# EXPECT rejected_fleshy_mass_is_the_faculty F
exists m.(m = eye_fleshy_mass & SenseFaculty(m))
# THEORY rejected_mental_factors_are_citta
# EXPECT rejected_mental_factors_are_citta F
all f.(MentalFactor(f) -> Citta(f))
# THEORY rejected_avijnapti_accumulated
# EXPECT rejected_avijnapti_accumulated F
exists h.(h = avijnapti_portion & Accumulated(h))
# THEORY rejected_consciousness_element_dependent
# EXPECT rejected_consciousness_element_dependent F
exists i.exists x.(CittaPrinciple(x) & InstanceOf(i,x) & ElementDependent(i))
"""

valuation_text = r"""tangible => sprastavya_d
visible_form => rupa_d
eye => caksur_d
body => kaya_d
dharma_principle => dharma_d
avijnapti_portion => dharma_avijnapti
dharma_rest => dharma_rest
earth => earth_i
water => water_i
fire => fire_i
wind => wind_i
smooth => smooth_i
rough => rough_i
heavy => heavy_i
light => light_i
cold => cold_i
hunger => hunger_i
thirst => thirst_i
vedana => vedana_i
eye_fleshy_mass => caksur_mamsapinda
person => purusa
buddhadeva => bhadanta_buddhadeva
buddhadeva_thesis => bhutamatra_thesis
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
FormBearing => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
SenseFaculty => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
CittaPrinciple => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
PortionOf => {(dharma_avijnapti,dharma_d),(dharma_rest,dharma_d)}
InstanceOf => {(caksur_i,caksur_d),(srotra_i,srotra_d),(ghrana_i,ghrana_d),(jihva_i,jihva_d),(kaya_i,kaya_d),(rupa_i,rupa_d),(sabda_i,sabda_d),(gandha_i,gandha_d),(rasa_i,rasa_d),(earth_i,sprastavya_d),(water_i,sprastavya_d),(fire_i,sprastavya_d),(wind_i,sprastavya_d),(smooth_i,sprastavya_d),(rough_i,sprastavya_d),(heavy_i,sprastavya_d),(light_i,sprastavya_d),(cold_i,sprastavya_d),(hunger_i,sprastavya_d),(thirst_i,sprastavya_d),(mano_i,mano_d),(caksurvijnana_i,caksurvijnana_d),(srotravijnana_i,srotravijnana_d),(ghranavijnana_i,ghranavijnana_d),(jihvavijnana_i,jihvavijnana_d),(kayavijnana_i,kayavijnana_d),(manovijnana_i,manovijnana_d),(avijnapti_i,dharma_avijnapti),(vedana_i,dharma_rest),(samjna_i,dharma_rest)}
GreatElement => {earth_i,water_i,fire_i,wind_i}
ElementDependent => {avijnapti_i,caksur_i,srotra_i,ghrana_i,jihva_i,kaya_i,rupa_i,sabda_i,gandha_i,rasa_i,smooth_i,rough_i,heavy_i,light_i,cold_i,hunger_i,thirst_i}
Accumulated => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
AtomCluster => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
GraspedBy => {(earth_i,kaya_d),(water_i,kaya_d),(fire_i,kaya_d),(wind_i,kaya_d),(rupa_i,caksur_d)}
Contains => {(caksur_mamsapinda,earth_i)}
MentalFactor => {vedana_i,samjna_i}
FollowsCitta => {vedana_i,samjna_i}
Citta => {mano_i,caksurvijnana_i,srotravijnana_i,ghranavijnana_i,jihvavijnana_i,kayavijnana_i,manovijnana_i}
SixPrincipleShowing => {purusa}
AttributedTo => {(bhutamatra_thesis,bhadanta_buddhadeva)}
Refuted => {bhutamatra_thesis}
"""


In [ ]:
studies[35] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[35]["tree"].pformat())
print("Derived SEM:", studies[35]["sem"])


# VAK_1.36 — The Outer Four

Source: `VAK_1.36.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
chinatti chidyate caiva bāhyaṃ dhātucatuṣṭayam /
dahyate tulayaty evaṃ vivādo dagdhṛtulyayoḥ // 1.36 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| chinatti | √chid, present active 3rd singular | cuts |
| chidyate | √chid, present passive 3rd singular | is cut |
| ca eva | particles | and also: both voices of the one root |
| bāhyam | bāhya; nominative neuter singular | external |
| dhātu-catuṣṭayam | tatpuruṣa; nominative neuter singular | the tetrad of Principles: Visible Form, Odor, Taste, Tangible |
| dahyate | √dah, present passive 3rd singular | is burned |
| tulayati | √tul, causative-stem present active 3rd singular | weighs |
| evam | adverb | likewise: the same tetrad |
| vivādaḥ | vi-√vad; nominative masculine singular | a dispute |
| dagdhṛ-tulyayoḥ | dvandva; genitive dual | about the burner and the weighed |

*Dagdhṛ* is the agent noun of √dah; *tulya* is the gerundive of √tul. The
dispute therefore turns on the converse voices of the two settled verbs:
the tetrad is burned, but what burns is disputed; the tetrad weighs, but
what is weighed is disputed. The fire of the second position is the Fire
Principle, an Element Principle: an instance in the Tangible, not one of
the eighteen.

**Grammar tokens.** Verse-order IAST with sandhi opened (`caiva` → *ca
eva*, `bāhyaṃ` → *bāhyam*, `tulayaty evaṃ` → *tulayati evam*, `vivādo` →
*vivādaḥ*) and compound boundaries marked.

```text
# tokens: chinatti chidyate ca eva bāhyam dhātu-catuṣṭayam dahyate tulayati evam vivādaḥ dagdhṛ-tulyayoḥ
```

## 5. Translation

### Literal Translation

> The external tetrad of Principles cuts and is also cut. Likewise it is
> burned and it weighs. There is a dispute about the burner and the weighed.

### Bhāṣya-informed study translation

> The four external Principles—Visible Form, Odor, Taste, and Tangible—cut
> and are cut; they are burned and weigh. Whether this same fourfold group is
> also the burner and the weighed is disputed.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| bāhya-dhātu-catuṣṭaya | external tetrad of Principles | Visible Form, Odor, Taste, Tangible |
| cheda | cutting | divided arising of a stream of clusters arising in connection |
| indriya | Faculty | not cut, does not cut |
| maṇi-prabhā | jewel-radiance | the clarity |
| śabda | Sound | cut off, so not burned or weighed |
| dagdhṛ | burner | agent noun of √dah: the disputed agent |
| tulya | the weighed | gerundive of √tul: the disputed patient |
| tejo-dhātu | Fire Principle | an Element Principle: the second position's burner, an instance in the Tangible |
| gurutva | heaviness | the second position's weighed; Element-dependent Tangible |

In [ ]:
fcfg_text = r"""% start S
# tokens: chinatti chidyate ca eva bāhyam dhātu-catuṣṭayam dahyate tulayati evam vivādaḥ dagdhṛ-tulyayoḥ
S[SEM=<(?cut & ?bw & ?d)>] -> CUTCL[SEM=?cut] BWCL[VA=?a,VB=?b,SEM=?bw] DISP[VA=?b,VB=?a,SEM=?d]
CUTCL[SEM=<all x.(?t(x) -> (?p(x) & ?q(x)))>] -> V[ROOT=chid,VOICE=act,NUM=sg,SEM=?p] V[ROOT=chid,VOICE=pass,NUM=sg,SEM=?q] CA EVA BAHYA[GEN=n,NUM=sg] TETRAD[GEN=n,NUM=sg,CNT=4,SEM=?t]
BWCL[VA=?a,VB=?b,SEM=<all x.(OuterFour(x) -> (?p(x) & ?q(x)))>] -> V[ROOT=dah,VOICE=?a,NUM=sg,SEM=?p] V[ROOT=tul,VOICE=?b,NUM=sg,SEM=?q] EVAM
DISP[VA=?x,VB=?y,SEM=<(Dispute(?d) & About(?d,burner) & About(?d,weighed))>] -> VIVADA[SEM=?d] DVANDVA[CASE=gen,NUM=dual,V1=?x,V2=?y]
V[ROOT=chid,VOICE=act,NUM=sg,SEM=<\x.Cuts(x)>] -> 'chinatti'
V[ROOT=chid,VOICE=pass,NUM=sg,SEM=<\x.IsCut(x)>] -> 'chidyate'
V[ROOT=dah,VOICE=pass,NUM=sg,SEM=<\x.IsBurned(x)>] -> 'dahyate'
V[ROOT=tul,VOICE=act,NUM=sg,SEM=<\x.Weighs(x)>] -> 'tulayati'
CA -> 'ca'
EVA -> 'eva'
EVAM -> 'evam'
BAHYA[GEN=n,NUM=sg] -> 'bāhyam'
TETRAD[GEN=n,NUM=sg,CNT=4,SEM=<\x.OuterFour(x)>] -> 'dhātu-catuṣṭayam'
VIVADA[SEM=<dispute>] -> 'vivādaḥ'
DVANDVA[CASE=gen,NUM=dual,V1=act,V2=pass] -> 'dagdhṛ-tulyayoḥ'
"""

dependency_text = r"""'chinatti' -> 'dhātu-catuṣṭayam' | 'chidyate'
'chidyate' -> 'ca' | 'eva'
'dhātu-catuṣṭayam' -> 'bāhyam'
'dahyate' -> 'evam' | 'tulayati'
'vivādaḥ' -> 'dagdhṛ-tulyayoḥ'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(OuterFour(x) -> (Cuts(x) & IsCut(x))) & all x.(OuterFour(x) -> (IsBurned(x) & Weighs(x))) & Dispute(dispute) & About(dispute,burner) & About(dispute,weighed))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(OuterFour(x) -> (Cuts(x) & IsCut(x))) & all x.(OuterFour(x) -> (IsBurned(x) & Weighs(x))) & Dispute(dispute) & About(dispute,burner) & About(dispute,weighed))
# THEORY prose
# EXPECT prose T
all x.(OuterFour(x) <-> ((x = visible_form) | (x = odor) | (x = taste) | (x = tangible)))
all x.(AmongEighteen(x) -> ((Cuts(x) | IsCut(x) | IsBurned(x) | Weighs(x)) -> OuterFour(x)))
all x.(IsCut(x) -> ClusterStream(x))
all x.(SenseFaculty(x) -> (Clear(x) & -Cuts(x) & -IsCut(x) & -IsBurned(x) & -Weighs(x)))
exists l.(l = severed_limb & -HasFaculty(l))
exists s.(s = sound & DiscontinuousStream(s) & -IsBurned(s) & -Weighs(s))
all x.(BurnerIn(position_a,x) <-> OuterFour(x))
all x.(WeighedIn(position_a,x) <-> OuterFour(x))
all x.(BurnerIn(position_b,x) <-> (x = fire_element))
all x.(WeighedIn(position_b,x) <-> (x = heaviness))
exists f.(f = fire_element & GreatElement(f) & InstanceOf(f,tangible) & -AmongEighteen(f))
exists h.(h = heaviness & ElementDependent(h) & InstanceOf(h,tangible))
exists d.(d = dispute & -Decided(d))
# THEORY rejected_faculty_is_cut
# EXPECT rejected_faculty_is_cut F
exists x.(SenseFaculty(x) & IsCut(x))
# THEORY rejected_faculty_burns_or_weighs
# EXPECT rejected_faculty_burns_or_weighs F
exists x.(SenseFaculty(x) & (IsBurned(x) | Weighs(x)))
# THEORY rejected_sound_is_burned
# EXPECT rejected_sound_is_burned F
exists s.(s = sound & IsBurned(s))
# THEORY rejected_fire_is_a_fifth_principle
# EXPECT rejected_fire_is_a_fifth_principle F
exists f.(f = fire_element & AmongEighteen(f))
# THEORY rejected_positions_agree
# EXPECT rejected_positions_agree F
all x.(BurnerIn(position_a,x) <-> BurnerIn(position_b,x))
# THEORY rejected_dispute_decided
# EXPECT rejected_dispute_decided F
exists d.(d = dispute & Decided(d))
"""

valuation_text = r"""visible_form => rupa_d
odor => gandha_d
taste => rasa_d
tangible => sprastavya_d
sound => sabda_d
burner => burner_q
weighed => weighed_q
dispute => vivada
position_a => kecid_a
position_b => kecid_b
fire_element => tejas_i
heaviness => gurutva_i
severed_limb => chinna_anga
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
SenseFaculty => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
OuterFour => {rupa_d,gandha_d,rasa_d,sprastavya_d}
Cuts => {rupa_d,gandha_d,rasa_d,sprastavya_d}
IsCut => {rupa_d,gandha_d,rasa_d,sprastavya_d}
IsBurned => {rupa_d,gandha_d,rasa_d,sprastavya_d}
Weighs => {rupa_d,gandha_d,rasa_d,sprastavya_d}
ClusterStream => {rupa_d,gandha_d,rasa_d,sprastavya_d}
Clear => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
DiscontinuousStream => {sabda_d}
Dispute => {vivada}
About => {(vivada,burner_q),(vivada,weighed_q)}
Decided => {}
BurnerIn => {(kecid_a,rupa_d),(kecid_a,gandha_d),(kecid_a,rasa_d),(kecid_a,sprastavya_d),(kecid_b,tejas_i)}
WeighedIn => {(kecid_a,rupa_d),(kecid_a,gandha_d),(kecid_a,rasa_d),(kecid_a,sprastavya_d),(kecid_b,gurutva_i)}
InstanceOf => {(tejas_i,sprastavya_d),(gurutva_i,sprastavya_d)}
GreatElement => {tejas_i}
ElementDependent => {gurutva_i}
HasFaculty => {}
"""


In [ ]:
studies[36] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[36]["tree"].pformat())
print("Derived SEM:", studies[36]["sem"])


# VAK_1.37 — Maturation, Reinforcement, and Causal Continuation

Source: `VAK_1.37.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
vipākajaupacayikāḥ pañcādhyātmaṃ vipākajaḥ /
na śabdo 'pratighā aṣṭau naiḥṣyandika-vipākajāḥ // 1.37 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| vipākaja-aupacayikāḥ | dvandva of two adjectives; nominative masculine plural | maturation-born and upacaya-reinforced |
| pañca | numeral | five |
| adhyātmam | adverbial accusative | internal: the five Faculty Principles |
| vipākajaḥ | vipāka-ja; nominative masculine singular | maturation-born |
| na | negative particle | not |
| śabdaḥ | nominative masculine singular | Sound |
| apratighāḥ | a-pratigha; nominative masculine plural | non-resistant |
| aṣṭau | numeral | eight: the seven consciousness Principles and the Dharma Principle |
| naiḥṣyandika-vipākajāḥ | dvandva of two adjectives; nominative masculine plural | continuity-born and maturation-born |

The line break joins *vipākajaḥ* with *na śabdaḥ*: Sound is not
maturation-born. *Vipākaja* is *vipāka-hetu-ja* with the middle member
dropped, as in *go-ratha*.

`Naiḥṣyandika` should not be confused with `sāsrava`, “with outflows.” Here
it names production through a corresponding causal flow, specified in the
Bhāṣya through `sabhāgahetu` and `sarvatragahetu`.

**Grammar tokens.** Verse-order IAST with sandhi opened
(`vipākajaupacayikāḥ` → *vipākaja-aupacayikāḥ*, `pañcādhyātmaṃ` → *pañca
adhyātmam*, `śabdo 'pratighā` → *śabdaḥ apratighāḥ*) and compound
boundaries marked. *Vipāka-ja* occurs three times: in the first dvandva,
alone, and in the last dvandva.

```text
# tokens: vipākaja-aupacayikāḥ pañca adhyātmam vipākajaḥ na śabdaḥ apratighāḥ aṣṭau naiḥṣyandika-vipākajāḥ
```

## 5. Translation

### Literal Translation

> The five internal [Faculty Principles] are maturation-born and upacaya-
> reinforced. Sound is not maturation-born. The eight non-resistant
> Principles include continuity-born and maturation-born instances.

### Bhāṣya-informed study translation

> The five internal Faculty Principles are maturation-born and reinforced
> through increase. Sound is not maturation-born. The eight non-resistant
> Principles have instances born of causal continuity and of maturation.

## 7. Technical Vocabulary

| Sanskrit | Project rendering | Determination |
|---|---|---|
| vipākaja | maturation-born | generated from karma that has reached the time of fruition |
| vipākahetu | maturation Ground | causal basis expanded from the abbreviated compound |
| vipākasantāna | maturation-continuity | karmically generated continuum |
| aupacayika | upacaya-reinforced | increased through nourishment and other living conditions |
| upacayasantāna | reinforcement-continuity | continuity protecting and strengthening maturation like a surrounding wall |
| anupaghāta | non-injury / preservation | absence of damage, distinguished from positive growth |
| naiḥṣyandika | continuity-born | generated through homogeneous or universally operative causes |
| sabhāgahetu | homogeneous Ground | produces corresponding factors in a causal series |
| sarvatragahetu | universally operative Ground | continuity-ground especially relevant to afflicted dharmas |
| īhā | effort / exertion | immediate initiating condition excluding sound from maturation-status |
| paramparā | mediated causal succession | explains how karma can indirectly condition voice without making sound itself vipākaja |
| apratigha | non-resistant | not resistant, not causally unconditioned |

In [ ]:
fcfg_text = r"""% start S
# tokens: vipākaja-aupacayikāḥ pañca adhyātmam vipākajaḥ na śabdaḥ apratighāḥ aṣṭau naiḥṣyandika-vipākajāḥ
S[SEM=<(?a & ?b & ?c)>] -> FIVECL[SEM=?a] SOUNDCL[SEM=?b] EIGHTCL[SEM=?c]
FIVECL[SEM=<all x.(?k(x) -> ?p(x))>] -> PRED[NUM=pl,SEM=?p] NUMR[CNT=?n] ADHY[CNT=?n,SEM=?k]
SOUNDCL[SEM=<-?p(?s)>] -> PRED[NUM=?num,SEM=?p] NA SABDA[NUM=?num,SEM=?s]
EIGHTCL[SEM=<all x.(?k(x) -> ?p(x))>] -> APRAT[NUM=pl,CNT=?n,SEM=?k] NUMR[CNT=?n] PRED[NUM=pl,SEM=?p]
PRED[NUM=pl,SEM=<\x.(exists i.(InstanceOf(i,x) & Born(i,vipaka)) & exists i.(InstanceOf(i,x) & Born(i,upacaya)))>] -> 'vipākaja-aupacayikāḥ'
PRED[NUM=sg,SEM=<\x.exists i.(InstanceOf(i,x) & Born(i,vipaka))>] -> 'vipākajaḥ'
PRED[NUM=pl,SEM=<\x.(exists i.(InstanceOf(i,x) & Born(i,naisyandika)) & exists i.(InstanceOf(i,x) & Born(i,vipaka)))>] -> 'naiḥṣyandika-vipākajāḥ'
NUMR[CNT=5] -> 'pañca'
NUMR[CNT=8] -> 'aṣṭau'
ADHY[CNT=5,SEM=<\x.InternalFive(x)>] -> 'adhyātmam'
NA -> 'na'
SABDA[NUM=sg,SEM=<sound>] -> 'śabdaḥ'
APRAT[NUM=pl,CNT=8,SEM=<\x.NonResistant(x)>] -> 'apratighāḥ'
"""

dependency_text = r"""'vipākaja-aupacayikāḥ' -> 'adhyātmam'
'adhyātmam' -> 'pañca'
'vipākajaḥ' -> 'na' | 'śabdaḥ'
'naiḥṣyandika-vipākajāḥ' -> 'apratighāḥ'
'apratighāḥ' -> 'aṣṭau'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(InternalFive(x) -> (exists i.(InstanceOf(i,x) & Born(i,vipaka)) & exists i.(InstanceOf(i,x) & Born(i,upacaya)))) & -exists i.(InstanceOf(i,sound) & Born(i,vipaka)) & all x.(NonResistant(x) -> (exists i.(InstanceOf(i,x) & Born(i,naisyandika)) & exists i.(InstanceOf(i,x) & Born(i,vipaka)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(InternalFive(x) -> (exists i.(InstanceOf(i,x) & Born(i,vipaka)) & exists i.(InstanceOf(i,x) & Born(i,upacaya)))) & -exists i.(InstanceOf(i,sound) & Born(i,vipaka)) & all x.(NonResistant(x) -> (exists i.(InstanceOf(i,x) & Born(i,naisyandika)) & exists i.(InstanceOf(i,x) & Born(i,vipaka)))))
# THEORY prose
# EXPECT prose T
all i.all m.(Born(i,m) -> ((m = vipaka) | (m = upacaya) | (m = naisyandika)))
all x.(InternalFive(x) <-> SenseFaculty(x))
all x.(NonResistant(x) <-> (CittaPrinciple(x) | (x = dharma_principle)))
all x.(InternalFive(x) -> -exists i.(InstanceOf(i,x) & Born(i,naisyandika)))
all i.(Born(i,vipaka) <-> GeneratedBy(i,vipaka_ground))
all i.(Born(i,naisyandika) <-> (GeneratedBy(i,sabhaga_ground) | GeneratedBy(i,sarvatraga_ground)))
exists i.(InstanceOf(i,sound) & Born(i,upacaya)) & exists i.(InstanceOf(i,sound) & Born(i,naisyandika))
all i.(InstanceOf(i,sound) -> FromEffort(i))
all i.(FromEffort(i) -> -Born(i,vipaka))
all i.all x.((InstanceOf(i,x) & Born(i,upacaya)) -> Accumulated(x))
all x.(NonResistant(x) -> -Accumulated(x))
exists b.(b = brahmacarya & PreventsInjury(b) & -Increases(b))
exists u.(u = upacaya_continuum & Protects(u,vipaka_continuum) & -(u = vipaka_continuum))
all a.(VoiceAccount(a) -> Mediates(a,karma,sound))
exists i.(InstanceOf(i,sound) & KarmaDescended(i) & -Born(i,vipaka))
exists i.(InstanceOf(i,dharma_principle) & Unconditioned(i) & -exists m.Born(i,m))
# THEORY rejected_sound_maturation_born
# EXPECT rejected_sound_maturation_born F
exists i.(InstanceOf(i,sound) & Born(i,vipaka))
# THEORY rejected_sound_without_any_mode
# EXPECT rejected_sound_without_any_mode F
all i.(InstanceOf(i,sound) -> -exists m.Born(i,m))
# THEORY rejected_each_instance_both_ways
# EXPECT rejected_each_instance_both_ways F
all x.(NonResistant(x) -> all i.(InstanceOf(i,x) -> (Born(i,naisyandika) & Born(i,vipaka))))
# THEORY rejected_faculty_continuity_class
# EXPECT rejected_faculty_continuity_class F
exists i.exists x.(InternalFive(x) & InstanceOf(i,x) & Born(i,naisyandika))
# THEORY rejected_eight_reinforced
# EXPECT rejected_eight_reinforced F
exists i.exists x.(NonResistant(x) & InstanceOf(i,x) & Born(i,upacaya))
# THEORY rejected_celibacy_increases
# EXPECT rejected_celibacy_increases F
exists b.(b = brahmacarya & Increases(b))
# THEORY rejected_unconditioned_produced
# EXPECT rejected_unconditioned_produced F
all i.(InstanceOf(i,dharma_principle) -> exists m.Born(i,m))
# THEORY rejected_remote_ancestry_decides
# EXPECT rejected_remote_ancestry_decides F
all i.(KarmaDescended(i) -> Born(i,vipaka))
"""

valuation_text = r"""vipaka => vipaka_m
upacaya => upacaya_m
naisyandika => naisyandika_m
sound => sabda_d
dharma_principle => dharma_d
vipaka_ground => vipaka_h
sabhaga_ground => sabhaga_h
sarvatraga_ground => sarvatraga_h
brahmacarya => brahmacarya_p
upacaya_continuum => upacaya_santana
vipaka_continuum => vipaka_santana
third_account => parampara_3
fifth_account => parampara_5
karma => karma_k
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_d}
SenseFaculty => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
InternalFive => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
CittaPrinciple => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
NonResistant => {mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d,dharma_d}
Accumulated => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
InstanceOf => {(dharma_asamskrta,dharma_d),(caksur_vip,caksur_d),(caksur_upa,caksur_d),(srotra_vip,srotra_d),(srotra_upa,srotra_d),(ghrana_vip,ghrana_d),(ghrana_upa,ghrana_d),(jihva_vip,jihva_d),(jihva_upa,jihva_d),(kaya_vip,kaya_d),(kaya_upa,kaya_d),(sabda_upa,sabda_d),(sabda_nai,sabda_d),(sabda_brahma,sabda_d),(mano_nai,mano_d),(mano_vip,mano_d),(caksurvijnana_nai,caksurvijnana_d),(caksurvijnana_vip,caksurvijnana_d),(srotravijnana_nai,srotravijnana_d),(srotravijnana_vip,srotravijnana_d),(ghranavijnana_nai,ghranavijnana_d),(ghranavijnana_vip,ghranavijnana_d),(jihvavijnana_nai,jihvavijnana_d),(jihvavijnana_vip,jihvavijnana_d),(kayavijnana_nai,kayavijnana_d),(kayavijnana_vip,kayavijnana_d),(manovijnana_nai,manovijnana_d),(manovijnana_vip,manovijnana_d),(dharma_nai,dharma_d),(dharma_vip,dharma_d)}
Born => {(caksur_vip,vipaka_m),(caksur_upa,upacaya_m),(srotra_vip,vipaka_m),(srotra_upa,upacaya_m),(ghrana_vip,vipaka_m),(ghrana_upa,upacaya_m),(jihva_vip,vipaka_m),(jihva_upa,upacaya_m),(kaya_vip,vipaka_m),(kaya_upa,upacaya_m),(sabda_upa,upacaya_m),(sabda_nai,naisyandika_m),(sabda_brahma,naisyandika_m),(mano_nai,naisyandika_m),(mano_vip,vipaka_m),(caksurvijnana_nai,naisyandika_m),(caksurvijnana_vip,vipaka_m),(srotravijnana_nai,naisyandika_m),(srotravijnana_vip,vipaka_m),(ghranavijnana_nai,naisyandika_m),(ghranavijnana_vip,vipaka_m),(jihvavijnana_nai,naisyandika_m),(jihvavijnana_vip,vipaka_m),(kayavijnana_nai,naisyandika_m),(kayavijnana_vip,vipaka_m),(manovijnana_nai,naisyandika_m),(manovijnana_vip,vipaka_m),(dharma_nai,naisyandika_m),(dharma_vip,vipaka_m)}
GeneratedBy => {(caksur_vip,vipaka_h),(srotra_vip,vipaka_h),(ghrana_vip,vipaka_h),(jihva_vip,vipaka_h),(kaya_vip,vipaka_h),(sabda_nai,sabhaga_h),(sabda_brahma,sabhaga_h),(mano_nai,sabhaga_h),(mano_vip,vipaka_h),(caksurvijnana_nai,sabhaga_h),(caksurvijnana_vip,vipaka_h),(srotravijnana_nai,sabhaga_h),(srotravijnana_vip,vipaka_h),(ghranavijnana_nai,sabhaga_h),(ghranavijnana_vip,vipaka_h),(jihvavijnana_nai,sabhaga_h),(jihvavijnana_vip,vipaka_h),(kayavijnana_nai,sabhaga_h),(kayavijnana_vip,vipaka_h),(manovijnana_nai,sabhaga_h),(manovijnana_vip,vipaka_h),(dharma_nai,sabhaga_h),(dharma_vip,vipaka_h),(dharma_nai,sarvatraga_h)}
FromEffort => {sabda_upa,sabda_nai,sabda_brahma}
KarmaDescended => {sabda_brahma}
Unconditioned => {dharma_asamskrta}
PreventsInjury => {brahmacarya_p}
Increases => {}
Protects => {(upacaya_santana,vipaka_santana)}
VoiceAccount => {parampara_3,parampara_5}
Mediates => {(parampara_3,karma_k,sabda_d),(parampara_5,karma_k,sabda_d)}
MemberCount => {(parampara_3,three_n),(parampara_5,five_n)}
"""


In [ ]:
studies[37] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[37]["tree"].pformat())
print("Derived SEM:", studies[37]["sem"])


# VAK_1.38 — Three Ways, One Dravya, Three Moments

Source: `VAK_1.38.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
tridhānye dravyavān ekaḥ kṣaṇikāḥ paścimās trayaḥ /
cakṣurvijñānadhātvoḥ syāt pṛthag lābhaḥ sahāpi ca // 1.38 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| tridhā | adverb of manner | in three ways: maturation-born, upacaya-reinforced, continuity-born |
| anye | anya; nominative masculine plural | the others: the four left by 1.37 |
| dravyavān | dravya-vat; nominative masculine singular | bearing *dravya* |
| ekaḥ | eka; nominative masculine singular | one: the Dharma Principle |
| kṣaṇikāḥ | kṣaṇa + -ika; nominative masculine plural | momentary |
| paścimāḥ | paścima; nominative masculine plural | last: in the order of recitation |
| trayaḥ | tri; nominative masculine plural | three: Mind, Dharma, Mind-Cognition |
| cakṣur-vijñāna-dhātvoḥ | dvandva; genitive dual | of the Eye Principle and the Eye-Cognition Principle |
| syāt | √as, optative 3rd singular | there may be |
| pṛthak | adverb | separately |
| lābhaḥ | lābha; nominative masculine singular | acquisition |
| saha | adverb | together |
| api ca | particles | also; and: gathers the further working for Visible Form |

The others are Visible Form, Taste, Odor, and Tangible: eighteen less the
five Faculties, Sound, and the eight of 1.37. The one is the Dharma
Principle. The last three are the Mind Principle, the Dharma Principle,
and the Mind-Cognition Principle.

**Grammar tokens.** Verse-order IAST with sandhi opened (`tridhānye` →
*tridhā anye*, `paścimās trayaḥ` → *paścimāḥ trayaḥ*, `pṛthag` → *pṛthak*,
`sahāpi` → *saha api*) and compound boundaries marked. The dvandva
*cakṣur-vijñāna-dhātvoḥ* is read *cakṣur-dhātu* + *cakṣur-vijñāna-dhātu*,
as the Bhāṣya does.

```text
# tokens: tridhā anye dravyavān ekaḥ kṣaṇikāḥ paścimāḥ trayaḥ cakṣur-vijñāna-dhātvoḥ syāt pṛthak lābhaḥ saha api ca
```

## 5. Translation

### Literal Translation

> The others are of three kinds. One is dravya-bearing. The last three are
> momentary. Of the eye-Principle and the Cognition Principle there may be
> separate acquisition, and also together.

### Bhāṣya-informed study translation

> The four remaining Principles are threefold. One is *dravya*-bearing. The
> last three are momentary. The Eye Principle and the Eye-Cognition Principle
> may be acquired separately or together.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| anye | the others | Visible Form, Taste, Odor, Tangible |
| vipākaja | maturation-born | one of the three |
| aupacayika | upacaya-reinforced | one of the three; not the atomic accumulation of 1.35 |
| naiḥṣyandika | homogeneous continuation | not an outflow |
| dravyavān | dravya-bearing | the Dharma Principle alone |
| sāra | core | the commentary's reason; not a substance |
| kṣaṇika | momentary | one moment in the first Pure complex |
| anāsrava | Pure | the first complex |
| cakṣur-dhātu | Eye Principle | the Faculty as Principle |
| cakṣur-vijñāna-dhātu | Eye-Cognition Principle | acquisition may split from the eye |
| lābha | acquisition | separate or together |
| samanvāgama | possession | four cases |

In [ ]:
fcfg_text = r"""% start S
# tokens: tridhā anye dravyavān ekaḥ kṣaṇikāḥ paścimāḥ trayaḥ cakṣur-vijñāna-dhātvoḥ syāt pṛthak lābhaḥ saha api ca
S[SEM=<(?a & ?b & ?c & ?d)>] -> THREEWAYS[SEM=?a] ONE[SEM=?b] LAST[SEM=?c] ACQ[SEM=?d]
THREEWAYS[SEM=<all x.(?k(x) -> ?p(x))>] -> TRIDHA[SEM=?p] ANYE[NUM=pl,CNT=4,SEM=?k]
ONE[SEM=<exists x.(?p(x) & all y.(?p(y) -> (y = x)))>] -> DRAVYAVAT[NUM=sg,SEM=?p] EKA[NUM=sg,CNT=1]
LAST[SEM=<all x.(?k(x) -> ?p(x))>] -> KSANIKA[NUM=pl,SEM=?p] PASCIMA[NUM=pl,CNT=?n,SEM=?k] TRI[NUM=pl,CNT=?n]
ACQ[SEM=<(?pair(?sep) & ?pair(?tog))>] -> PAIR[CASE=gen,NUM=dual,SEM=?pair] SYAT PRTHAK[SEM=?sep] LABHA[NUM=sg] SAHA[SEM=?tog] API CA
TRIDHA[SEM=<\x.(exists i.(InstanceOf(i,x) & Born(i,vipaka)) & exists i.(InstanceOf(i,x) & Born(i,upacaya)) & exists i.(InstanceOf(i,x) & Born(i,naisyandika)))>] -> 'tridhā'
ANYE[NUM=pl,CNT=4,SEM=<\x.Remaining(x)>] -> 'anye'
DRAVYAVAT[NUM=sg,SEM=<\x.DravyaBearing(x)>] -> 'dravyavān'
EKA[NUM=sg,CNT=1] -> 'ekaḥ'
KSANIKA[NUM=pl,SEM=<\x.Momentary(x)>] -> 'kṣaṇikāḥ'
PASCIMA[NUM=pl,CNT=3,SEM=<\x.LastThree(x)>] -> 'paścimāḥ'
TRI[NUM=pl,CNT=3] -> 'trayaḥ'
PAIR[CASE=gen,NUM=dual,SEM=<\R.R(eye_principle,eye_cognition)>] -> 'cakṣur-vijñāna-dhātvoḥ'
SYAT -> 'syāt'
PRTHAK[SEM=<\a b.(exists c.(AcqCase(c) & Acquires(c,a) & -Acquires(c,b)) & exists c.(AcqCase(c) & Acquires(c,b) & -Acquires(c,a)))>] -> 'pṛthak'
LABHA[NUM=sg] -> 'lābhaḥ'
SAHA[SEM=<\a b.exists c.(AcqCase(c) & Acquires(c,a) & Acquires(c,b))>] -> 'saha'
API -> 'api'
CA -> 'ca'
"""

dependency_text = r"""'tridhā' -> 'anye'
'dravyavān' -> 'ekaḥ'
'kṣaṇikāḥ' -> 'trayaḥ'
'trayaḥ' -> 'paścimāḥ'
'syāt' -> 'lābhaḥ'
'lābhaḥ' -> 'cakṣur-vijñāna-dhātvoḥ' | 'pṛthak' | 'saha'
'saha' -> 'api'
'api' -> 'ca'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(Remaining(x) -> (exists i.(InstanceOf(i,x) & Born(i,vipaka)) & exists i.(InstanceOf(i,x) & Born(i,upacaya)) & exists i.(InstanceOf(i,x) & Born(i,naisyandika)))) & exists x.(DravyaBearing(x) & all y.(DravyaBearing(y) -> (y = x))) & all x.(LastThree(x) -> Momentary(x)) & exists c.(AcqCase(c) & Acquires(c,eye_principle) & -Acquires(c,eye_cognition)) & exists c.(AcqCase(c) & Acquires(c,eye_cognition) & -Acquires(c,eye_principle)) & exists c.(AcqCase(c) & Acquires(c,eye_principle) & Acquires(c,eye_cognition)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(Remaining(x) -> (exists i.(InstanceOf(i,x) & Born(i,vipaka)) & exists i.(InstanceOf(i,x) & Born(i,upacaya)) & exists i.(InstanceOf(i,x) & Born(i,naisyandika)))) & exists x.(DravyaBearing(x) & all y.(DravyaBearing(y) -> (y = x))) & all x.(LastThree(x) -> Momentary(x)) & exists c.(AcqCase(c) & Acquires(c,eye_principle) & -Acquires(c,eye_cognition)) & exists c.(AcqCase(c) & Acquires(c,eye_cognition) & -Acquires(c,eye_principle)) & exists c.(AcqCase(c) & Acquires(c,eye_principle) & Acquires(c,eye_cognition)))
# THEORY prose
# EXPECT prose T
all x.(Remaining(x) <-> ((x = visible_form) | (x = taste) | (x = odor) | (x = tangible)))
all x.(AmongEighteen(x) -> (DravyaBearing(x) <-> exists i.(InstanceOf(i,x) & Unconditioned(i))))
all i.(Unconditioned(i) -> Core(i))
all x.(LastThree(x) <-> ((x = mind) | (x = dharma_principle) | (x = mind_cognition)))
all x.(AmongEighteen(x) -> (Momentary(x) <-> exists i.(InstanceOf(i,x) & InMoment(i,first_pure) & -Born(i,naisyandika))))
all i.(InMoment(i,first_pure) -> Pure(i))
all i.((InMoment(i,first_pure) & Citta(i)) <-> (InMoment(i,first_pure) & (InstanceOf(i,mind) | InstanceOf(i,mind_cognition))))
all i.((InMoment(i,first_pure) & -Citta(i)) -> InstanceOf(i,dharma_principle))
all x.(LastThree(x) -> all i.((InstanceOf(i,x) & -Unconditioned(i) & -InMoment(i,first_pure)) -> Born(i,naisyandika)))
exists c.(c = kama_gradual & Acquires(c,eye_principle) & -Acquires(c,eye_cognition))
exists c.(c = formless_to_second & Acquires(c,eye_principle) & -Acquires(c,eye_cognition))
exists c.(c = second_seeing & Acquires(c,eye_cognition) & -Acquires(c,eye_principle))
exists c.(c = second_to_below & Acquires(c,eye_cognition) & -Acquires(c,eye_principle))
exists c.(c = formless_to_desire & Acquires(c,eye_principle) & Acquires(c,eye_cognition))
exists c.(c = formless_to_brahma & Acquires(c,eye_principle) & Acquires(c,eye_cognition))
exists p.(p = second_not_seeing & Possesses(p,eye_principle) & -Possesses(p,eye_cognition))
exists p.(p = desire_without_eye & Possesses(p,eye_cognition) & -Possesses(p,eye_principle))
exists p.(p = desire_with_eye & Possesses(p,eye_principle) & Possesses(p,eye_cognition))
exists p.(p = formless_being & -Possesses(p,eye_principle) & -Possesses(p,eye_cognition))
SameWorkingFor(eye_principle,visible_form) & SameWorkingFor(eye_cognition,visible_form)
# THEORY rejected_dravya_in_every_principle
# EXPECT rejected_dravya_in_every_principle F
all x.(AmongEighteen(x) -> DravyaBearing(x))
# THEORY rejected_second_dravya_bearer
# EXPECT rejected_second_dravya_bearer F
exists x.(DravyaBearing(x) & -(x = dharma_principle))
# THEORY rejected_all_conditioned_momentary
# EXPECT rejected_all_conditioned_momentary F
all x.(AmongEighteen(x) -> Momentary(x))
# THEORY rejected_momentary_outside_first_pure
# EXPECT rejected_momentary_outside_first_pure F
exists i.exists x.(LastThree(x) & InstanceOf(i,x) & -Unconditioned(i) & -InMoment(i,first_pure) & -Born(i,naisyandika))
# THEORY rejected_one_acquisition
# EXPECT rejected_one_acquisition F
all c.(AcqCase(c) -> (Acquires(c,eye_principle) <-> Acquires(c,eye_cognition)))
# THEORY rejected_possession_follows_the_eye
# EXPECT rejected_possession_follows_the_eye F
all p.(Being(p) -> (Possesses(p,eye_cognition) -> Possesses(p,eye_principle)))
# THEORY rejected_remaining_twofold_only
# EXPECT rejected_remaining_twofold_only F
all x.(Remaining(x) -> -exists i.(InstanceOf(i,x) & Born(i,upacaya)))
"""

valuation_text = r"""vipaka => vipaka_m
upacaya => upacaya_m
naisyandika => naisyandika_m
visible_form => rupa_d
taste => rasa_d
odor => gandha_d
tangible => sprastavya_d
mind => mano_d
dharma_principle => dharma_d
mind_cognition => manovijnana_d
eye_principle => caksur_d
eye_cognition => caksurvijnana_d
first_pure => duhkhe_dharmajnanaksanti
kama_gradual => kama_kramasah
formless_to_second => arupya_to_dhyana2
second_seeing => dhyana2_sammukhi
second_to_below => dhyana2_to_adhah
formless_to_desire => arupya_to_kama
formless_to_brahma => arupya_to_brahmaloka
second_not_seeing => p_dhyana2_asammukhi
desire_without_eye => p_kama_alabdha
desire_with_eye => p_kama_labdha
formless_being => p_arupya
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
Remaining => {rupa_d,rasa_d,gandha_d,sprastavya_d}
DravyaBearing => {dharma_d}
LastThree => {mano_d,dharma_d,manovijnana_d}
Momentary => {mano_d,dharma_d,manovijnana_d}
InstanceOf => {(rupa_vip,rupa_d),(rupa_upa,rupa_d),(rupa_nai,rupa_d),(rasa_vip,rasa_d),(rasa_upa,rasa_d),(rasa_nai,rasa_d),(gandha_vip,gandha_d),(gandha_upa,gandha_d),(gandha_nai,gandha_d),(sprastavya_vip,sprastavya_d),(sprastavya_upa,sprastavya_d),(sprastavya_nai,sprastavya_d),(mano_ksanti,mano_d),(mano_later,mano_d),(manovijnana_ksanti,manovijnana_d),(manovijnana_later,manovijnana_d),(dharma_ksanti,dharma_d),(dharma_later,dharma_d),(dharma_asamskrta,dharma_d)}
Born => {(rupa_vip,vipaka_m),(rupa_upa,upacaya_m),(rupa_nai,naisyandika_m),(rasa_vip,vipaka_m),(rasa_upa,upacaya_m),(rasa_nai,naisyandika_m),(gandha_vip,vipaka_m),(gandha_upa,upacaya_m),(gandha_nai,naisyandika_m),(sprastavya_vip,vipaka_m),(sprastavya_upa,upacaya_m),(sprastavya_nai,naisyandika_m),(mano_later,naisyandika_m),(manovijnana_later,naisyandika_m),(dharma_later,naisyandika_m)}
Unconditioned => {dharma_asamskrta}
Core => {dharma_asamskrta}
InMoment => {(mano_ksanti,duhkhe_dharmajnanaksanti),(manovijnana_ksanti,duhkhe_dharmajnanaksanti),(dharma_ksanti,duhkhe_dharmajnanaksanti)}
Citta => {mano_ksanti,manovijnana_ksanti,mano_later,manovijnana_later}
Pure => {mano_ksanti,manovijnana_ksanti,dharma_ksanti}
AcqCase => {kama_kramasah,arupya_to_dhyana2,dhyana2_sammukhi,dhyana2_to_adhah,arupya_to_kama,arupya_to_brahmaloka}
Acquires => {(kama_kramasah,caksur_d),(arupya_to_dhyana2,caksur_d),(dhyana2_sammukhi,caksurvijnana_d),(dhyana2_to_adhah,caksurvijnana_d),(arupya_to_kama,caksur_d),(arupya_to_kama,caksurvijnana_d),(arupya_to_brahmaloka,caksur_d),(arupya_to_brahmaloka,caksurvijnana_d)}
Being => {p_dhyana2_asammukhi,p_kama_alabdha,p_kama_labdha,p_arupya}
Possesses => {(p_dhyana2_asammukhi,caksur_d),(p_kama_alabdha,caksurvijnana_d),(p_kama_labdha,caksur_d),(p_kama_labdha,caksurvijnana_d)}
SameWorkingFor => {(caksur_d,rupa_d),(caksurvijnana_d,rupa_d)}
"""


In [ ]:
studies[38] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[38]["tree"].pformat())
print("Derived SEM:", studies[38]["sem"])


# VAK_1.39 — Inner Twelve, and the Work

Source: `VAK_1.39.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
dvādaśādhyātmikā hitvā rūpādīn dharmasaṃjñakaḥ /
sabhāgas tatsabhāgāś ca śeṣā yo na svakarmakṛt // 1.39 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| dvādaśa | numeral | twelve |
| ādhyātmikāḥ | adhi-ātma + -ika; nominative masculine plural | inner: near citta as support |
| hitvā | √hā, absolutive | having set aside |
| rūpa-ādīn | bahuvrīhi; accusative masculine plural | those beginning with Visible Form: the six Condition Principles |
| dharma-saṃjñakaḥ | bahuvrīhi; nominative masculine singular | the one called Dharma: the Dharma Principle |
| sabhāgaḥ | sa-bhāga; nominative masculine singular | *sabhāga*: sharing in the work |
| tat-sabhāgāḥ | tatpuruṣa; nominative masculine plural | *tat-sabhāga*: of that kind, without the work |
| ca | particle | and: the rest are also *sabhāga* |
| śeṣāḥ | śeṣa; nominative masculine plural | the rest: the other seventeen |
| yaḥ | yad; nominative masculine singular | whichever |
| na | negative particle | not |
| sva-karma-kṛt | upapada compound; nominative masculine singular | doing its own work |

The one called Dharma is the Dharma Principle. Inner is not a self. It is
proximity to citta, and citta is designated self because it is the
support of the I-maker.

**Grammar tokens.** Verse-order IAST with sandhi opened (`dvādaśādhyātmikā`
→ *dvādaśa ādhyātmikāḥ*, `sabhāgas` → *sabhāgaḥ*, `tatsabhāgāś` →
*tat-sabhāgāḥ*, `śeṣā yo` → *śeṣāḥ yaḥ*) and compound boundaries marked.

```text
# tokens: dvādaśa ādhyātmikāḥ hitvā rūpa-ādīn dharma-saṃjñakaḥ sabhāgaḥ tat-sabhāgāḥ ca śeṣāḥ yaḥ na sva-karma-kṛt
```

## 5. Translation

### Literal Translation

> Twelve are inner, having set aside those beginning with visible. The one
> called dharma is sabhāga. And the rest are tat-sabhāga, whichever does not
> do its own work.

### Bhāṣya-informed study translation

> Twelve Principles are internal, excluding the six beginning with Visible
> Form. The Dharma Principle is *sabhāga*; the others are *sabhāga* when
> performing their own function, and *tat-sabhāga* when they do not.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| ādhyātmika | inner | near citta as support; not a self |
| bāhya | outer | the six Condition Principles, with the Dharma Principle |
| āśraya | Basis | the six with the Cognitions make twelve; distinct from *ālambana* (Support) |
| citta | citta | designated self as support of the I-maker |
| dharma-saṃjñaka | the one called dharma | the Dharma Principle; not an essence |
| sabhāga | sharing in the work | Faculty, Condition, and Cognition taking one another up; or contact as common result |
| tat-sabhāga | of that kind | the same kind, without the work |
| sva-karma-kṛt | does its own work | the sabhāga side |
| asādhāraṇa | uncommon | the eye, settled by one continuum |
| sādhāraṇa | common | Visible Form, settled by many |

In [ ]:
fcfg_text = r"""% start S
# tokens: dvādaśa ādhyātmikāḥ hitvā rūpa-ādīn dharma-saṃjñakaḥ sabhāgaḥ tat-sabhāgāḥ ca śeṣāḥ yaḥ na sva-karma-kṛt
S[SEM=<(?a & ?b)>] -> INNER[SEM=?a] WORK[SEM=?b]
INNER[SEM=<all x.(AmongEighteen(x) -> (?in(x) <-> -?ex(x)))>] -> NUMR[CNT=?r] ADHY[NUM=pl,SEM=?in] HITVA EXCL[CASE=acc,NUM=pl,REST=?r,SEM=?ex]
WORK[SEM=<(all i.(InstanceOf(i,?d) -> Sabhaga(i)) & all x.((AmongEighteen(x) & -(x = ?d)) -> all i.(InstanceOf(i,x) -> ((-?w(i) -> TatSabhaga(i)) & (?w(i) -> Sabhaga(i))))))>] -> DHARMA[NUM=sg,SEM=?d] SAB[NUM=sg] TAT[NUM=pl] CA SESA[NUM=pl] REL[NUM=sg,POL=neg,SEM=?w]
REL[NUM=sg,POL=neg,SEM=?w] -> YAH[NUM=sg] NA SVAKARMA[NUM=sg,SEM=?w]
NUMR[CNT=12] -> 'dvādaśa'
ADHY[NUM=pl,SEM=<\x.Inner(x)>] -> 'ādhyātmikāḥ'
HITVA -> 'hitvā'
EXCL[CASE=acc,NUM=pl,REST=12,SEM=<\x.OuterPrinciple(x)>] -> 'rūpa-ādīn'
DHARMA[NUM=sg,SEM=<dharma_principle>] -> 'dharma-saṃjñakaḥ'
SAB[NUM=sg] -> 'sabhāgaḥ'
TAT[NUM=pl] -> 'tat-sabhāgāḥ'
CA -> 'ca'
SESA[NUM=pl] -> 'śeṣāḥ'
YAH[NUM=sg] -> 'yaḥ'
NA -> 'na'
SVAKARMA[NUM=sg,SEM=<\i.DoesOwnWork(i)>] -> 'sva-karma-kṛt'
"""

dependency_text = r"""'ādhyātmikāḥ' -> 'dvādaśa' | 'hitvā'
'hitvā' -> 'rūpa-ādīn'
'sabhāgaḥ' -> 'dharma-saṃjñakaḥ'
'tat-sabhāgāḥ' -> 'śeṣāḥ' | 'ca'
'śeṣāḥ' -> 'sva-karma-kṛt'
'sva-karma-kṛt' -> 'yaḥ' | 'na'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(AmongEighteen(x) -> (Inner(x) <-> -OuterPrinciple(x))) & all i.(InstanceOf(i,dharma_principle) -> Sabhaga(i)) & all x.((AmongEighteen(x) & -(x = dharma_principle)) -> all i.(InstanceOf(i,x) -> ((-DoesOwnWork(i) -> TatSabhaga(i)) & (DoesOwnWork(i) -> Sabhaga(i))))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(AmongEighteen(x) -> (Inner(x) <-> -OuterPrinciple(x))) & all i.(InstanceOf(i,dharma_principle) -> Sabhaga(i)) & all x.((AmongEighteen(x) & -(x = dharma_principle)) -> all i.(InstanceOf(i,x) -> ((-DoesOwnWork(i) -> TatSabhaga(i)) & (DoesOwnWork(i) -> Sabhaga(i))))))
# THEORY prose
# EXPECT prose T
all x.(Inner(x) <-> (Basis(x) | CognitionPrinciple(x)))
all x.(Inner(x) <-> NearCittaAsBasis(x))
-exists s.Self(s)
exists c.(c = citta & DesignatedSelf(c) & SupportOfIMaker(c))
all x.(CognitionPrinciple(x) -> Inner(x))
exists m.(m = mind & MarkAcrossTimes(m))
all i.(InstanceOf(i,dharma_principle) -> DoesOwnWork(i))
all i.-(Sabhaga(i) & TatSabhaga(i))
all i.((InstanceOf(i,eye) & TatSabhaga(i)) -> (Ceased(i) | Ceasing(i) | AboutToCease(i) | NotToArise(i)))
exists v.(v = kasmira & KindCount(v,four_n)) & exists v.(v = western & Splits(v,not_to_arise))
all i.((InstanceOf(i,mind) & TatSabhaga(i)) -> NotToArise(i))
all i.all x.((InstanceOf(i,x) & -ConditionMeaning(x)) -> all p.all q.((Person(p) & Person(q)) -> (SabhagaFor(i,p) <-> SabhagaFor(i,q))))
exists i.(i = moon & InstanceOf(i,visible_form) & exists p.exists q.(SabhagaFor(i,p) & SabhagaFor(i,q) & -(p = q)) & exists r.(Person(r) & -SabhagaFor(i,r)))
all x.(ConditionMeaning(x) -> Common(x))
all x.(SenseFaculty(x) -> -Common(x))
exists x.(x = odor & ContactGrasped(x) & Common(x))
# THEORY rejected_inner_is_a_self
# EXPECT rejected_inner_is_a_self F
exists s.Self(s)
# THEORY rejected_cognitions_outer
# EXPECT rejected_cognitions_outer F
exists x.(CognitionPrinciple(x) & -Inner(x))
# THEORY rejected_dharma_tat_sabhaga
# EXPECT rejected_dharma_tat_sabhaga F
exists i.(InstanceOf(i,dharma_principle) & TatSabhaga(i))
# THEORY rejected_rest_always_sabhaga
# EXPECT rejected_rest_always_sabhaga F
all x.(AmongEighteen(x) -> all i.(InstanceOf(i,x) -> Sabhaga(i)))
# THEORY rejected_mind_fails_by_ceasing
# EXPECT rejected_mind_fails_by_ceasing F
exists i.(InstanceOf(i,mind) & TatSabhaga(i) & -NotToArise(i))
# THEORY rejected_eye_relative_to_seer
# EXPECT rejected_eye_relative_to_seer F
exists i.(InstanceOf(i,eye) & exists p.exists q.(SabhagaFor(i,p) & Person(q) & -SabhagaFor(i,q)))
# THEORY rejected_contact_grasped_uncommon
# EXPECT rejected_contact_grasped_uncommon F
exists x.(ContactGrasped(x) & -Common(x))
"""

valuation_text = r"""dharma_principle => dharma_d
eye => caksur_d
mind => mano_d
visible_form => rupa_d
odor => gandha_d
moon => rupa_moon
citta => citta_c
self_designation => atma_prajnapti
kasmira => kasmira_v
western => pascatya_v
four_n => four_n
five_n => five_n
not_to_arise => anutpatti_k
AmongEighteen => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,mano_d,dharma_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
SenseFaculty => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d}
Basis => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,mano_d}
CognitionPrinciple => {caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
ConditionMeaning => {rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
OuterPrinciple => {rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d,dharma_d}
Inner => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
NearCittaAsBasis => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,mano_d,caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d,manovijnana_d}
Self => {}
DesignatedSelf => {citta_c}
SupportOfIMaker => {citta_c}
MarkAcrossTimes => {mano_d}
InstanceOf => {(caksur_seen,caksur_d),(caksur_ceased,caksur_d),(caksur_ceasing,caksur_d),(caksur_about,caksur_d),(caksur_anutp,caksur_d),(srotra_seen,srotra_d),(srotra_ceased,srotra_d),(srotra_ceasing,srotra_d),(srotra_about,srotra_d),(srotra_anutp,srotra_d),(ghrana_seen,ghrana_d),(ghrana_ceased,ghrana_d),(ghrana_ceasing,ghrana_d),(ghrana_about,ghrana_d),(ghrana_anutp,ghrana_d),(jihva_seen,jihva_d),(jihva_ceased,jihva_d),(jihva_ceasing,jihva_d),(jihva_about,jihva_d),(jihva_anutp,jihva_d),(kaya_seen,kaya_d),(kaya_ceased,kaya_d),(kaya_ceasing,kaya_d),(kaya_about,kaya_d),(kaya_anutp,kaya_d),(mano_work,mano_d),(mano_anutp,mano_d),(caksurvijnana_work,caksurvijnana_d),(caksurvijnana_anutp,caksurvijnana_d),(srotravijnana_work,srotravijnana_d),(srotravijnana_anutp,srotravijnana_d),(ghranavijnana_work,ghranavijnana_d),(ghranavijnana_anutp,ghranavijnana_d),(jihvavijnana_work,jihvavijnana_d),(jihvavijnana_anutp,jihvavijnana_d),(kayavijnana_work,kayavijnana_d),(kayavijnana_anutp,kayavijnana_d),(manovijnana_work,manovijnana_d),(manovijnana_anutp,manovijnana_d),(rupa_moon,rupa_d),(rupa_unseen,rupa_d),(sabda_work,sabda_d),(sabda_unseen,sabda_d),(gandha_work,gandha_d),(gandha_unseen,gandha_d),(rasa_work,rasa_d),(rasa_unseen,rasa_d),(sprastavya_work,sprastavya_d),(sprastavya_unseen,sprastavya_d),(dharma_i,dharma_d),(dharma_asamskrta,dharma_d)}
DoesOwnWork => {caksur_seen,srotra_seen,ghrana_seen,jihva_seen,kaya_seen,mano_work,caksurvijnana_work,srotravijnana_work,ghranavijnana_work,jihvavijnana_work,kayavijnana_work,manovijnana_work,rupa_moon,sabda_work,gandha_work,rasa_work,sprastavya_work,dharma_i,dharma_asamskrta}
Sabhaga => {caksur_seen,srotra_seen,ghrana_seen,jihva_seen,kaya_seen,mano_work,caksurvijnana_work,srotravijnana_work,ghranavijnana_work,jihvavijnana_work,kayavijnana_work,manovijnana_work,rupa_moon,sabda_work,gandha_work,rasa_work,sprastavya_work,dharma_i,dharma_asamskrta}
TatSabhaga => {caksur_ceased,caksur_ceasing,caksur_about,caksur_anutp,srotra_ceased,srotra_ceasing,srotra_about,srotra_anutp,ghrana_ceased,ghrana_ceasing,ghrana_about,ghrana_anutp,jihva_ceased,jihva_ceasing,jihva_about,jihva_anutp,kaya_ceased,kaya_ceasing,kaya_about,kaya_anutp,mano_anutp,caksurvijnana_anutp,srotravijnana_anutp,ghranavijnana_anutp,jihvavijnana_anutp,kayavijnana_anutp,manovijnana_anutp,rupa_unseen,sabda_unseen,gandha_unseen,rasa_unseen,sprastavya_unseen}
NotToArise => {caksur_anutp,srotra_anutp,ghrana_anutp,jihva_anutp,kaya_anutp,mano_anutp,caksurvijnana_anutp,srotravijnana_anutp,ghranavijnana_anutp,jihvavijnana_anutp,kayavijnana_anutp,manovijnana_anutp}
Ceased => {caksur_ceased,srotra_ceased,ghrana_ceased,jihva_ceased,kaya_ceased}
Ceasing => {caksur_ceasing,srotra_ceasing,ghrana_ceasing,jihva_ceasing,kaya_ceasing}
AboutToCease => {caksur_about,srotra_about,ghrana_about,jihva_about,kaya_about}
Person => {p1,p2,p3}
SabhagaFor => {(caksur_seen,p1),(caksur_seen,p2),(caksur_seen,p3),(srotra_seen,p1),(srotra_seen,p2),(srotra_seen,p3),(ghrana_seen,p1),(ghrana_seen,p2),(ghrana_seen,p3),(jihva_seen,p1),(jihva_seen,p2),(jihva_seen,p3),(kaya_seen,p1),(kaya_seen,p2),(kaya_seen,p3),(mano_work,p1),(mano_work,p2),(mano_work,p3),(caksurvijnana_work,p1),(caksurvijnana_work,p2),(caksurvijnana_work,p3),(srotravijnana_work,p1),(srotravijnana_work,p2),(srotravijnana_work,p3),(ghranavijnana_work,p1),(ghranavijnana_work,p2),(ghranavijnana_work,p3),(jihvavijnana_work,p1),(jihvavijnana_work,p2),(jihvavijnana_work,p3),(kayavijnana_work,p1),(kayavijnana_work,p2),(kayavijnana_work,p3),(manovijnana_work,p1),(manovijnana_work,p2),(manovijnana_work,p3),(rupa_moon,p1),(rupa_moon,p2),(sabda_work,p1),(gandha_work,p1),(rasa_work,p1),(sprastavya_work,p1),(dharma_i,p1),(dharma_i,p2),(dharma_i,p3),(dharma_asamskrta,p1),(dharma_asamskrta,p2),(dharma_asamskrta,p3)}
Common => {rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
ContactGrasped => {gandha_d,rasa_d,sprastavya_d}
KindCount => {(kasmira_v,four_n),(pascatya_v,five_n)}
Splits => {(pascatya_v,anutpatti_k)}
"""


In [ ]:
studies[39] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[39]["tree"].pformat())
print("Derived SEM:", studies[39]["sem"])


# VAK_1.40 — Abandoned by Seeing, by Cultivation, or Not

Source: `VAK_1.40.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
daśa bhāvanayā heyāḥ pañca cāntyās trayas tridhā /
na dṛṣṭiheyam akliṣṭaṃ na rūpaṃ nāpy aṣaṣṭhajam // 1.40 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| daśa | numeral | ten: the Form Principles (*rūpiṇaḥ*) |
| bhāvanayā | bhāvanā; instrumental singular | by cultivation |
| heyāḥ | √hā, gerundive; nominative masculine plural | to be abandoned: said of Principles |
| pañca | numeral | five: the sensory Cognition Principles |
| ca | particle | and: *pañca* shares the predicate |
| antyāḥ | antya; nominative masculine plural | the last, in the order of recitation |
| trayaḥ | tri; nominative masculine plural | three: Mind, Dharma, Mind-Cognition |
| tridhā | adverb | of three kinds: seeing, cultivation, not abandoned |
| na | negative particle | not: once for each excluded class |
| dṛṣṭi-heyam | tatpuruṣa gerundive; nominative neuter singular | abandoned by seeing: said of any dharma |
| akliṣṭam | a-kliṣṭa; nominative neuter singular | the unafflicted |
| rūpam | rūpa; nominative neuter singular | Form |
| api | particle | also |
| a-ṣaṣṭha-jam | upapada compound; nominative neuter singular | not born from the sixth, the Mind Essence |

The sixth is the Mind Essence (*mana-āyatana*). The last three are the
Mind, Dharma, and Mind-Cognition Principles.

**Grammar tokens.** Verse-order IAST with sandhi opened (`cāntyās` → *ca
antyāḥ*, `trayas` → *trayaḥ*, `akliṣṭaṃ` → *akliṣṭam*, `nāpy` → *na api*)
and compound boundaries marked.

```text
# tokens: daśa bhāvanayā heyāḥ pañca ca antyāḥ trayaḥ tridhā na dṛṣṭi-heyam akliṣṭam na rūpam na api a-ṣaṣṭha-jam
```

## 5. Translation

### Literal Translation

> Ten are abandoned by cultivation, and five. The last three are threefold.
> Nothing unafflicted is abandoned by seeing, nor Form, nor what is not born
> from the sixth.

### Bhāṣya-informed study translation

> The ten Form Principles and the five sensory Cognition Principles are
> abandoned by cultivation. The last three—the Mind, Dharma, and
> Mind-Cognition Principles—are of three kinds. Nothing unafflicted is
> abandoned by seeing, nothing of Form, and nothing not born from the sixth
> Essence.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| darśana-heya | abandoned by seeing | the verse says dṛṣṭi-heya |
| bhāvanā-heya | abandoned by cultivation | the ten Form Principles and the five sensory Cognitions |
| aheya | not abandoned | the Pure |
| antyāḥ trayaḥ | the last three | Mind, Dharma, Mind-Cognition Principles |
| anuśaya | latent affliction | eighty-eight, with what arises with them |
| pṛthagjanatva | ordinary-person status | unafflicted and indeterminate |
| aṣaṣṭhaja | not born from the sixth | the sixth is the Mind Essence |
| mana-āyatana | Mind Essence | the sixth; Essence renders āyatana |
| sāsrava | Impure | the remainder, abandoned by cultivation |
| anāsrava | Pure | not abandoned |

In [ ]:
fcfg_text = r"""% start S
# tokens: daśa bhāvanayā heyāḥ pañca ca antyāḥ trayaḥ tridhā na dṛṣṭi-heyam akliṣṭam na rūpam na api a-ṣaṣṭha-jam
S[SEM=<(?a & ?b & ?c)>] -> CUL[CNT=15,SEM=?a] TRI[CNT=3,SEM=?b] EXCL[SEM=?c]
CUL[CNT=15,SEM=<all x.((?p(x) | ?q(x)) -> all i.(InstanceOf(i,x) -> ?m(i)))>] -> NUMR[CNT=10,SEM=?p] VP[GEN=m,NUM=pl,SEM=?m] NUMR[CNT=5,SEM=?q] CA
VP[GEN=?g,NUM=?n,SEM=?m] -> INS[SEM=?m] HEYA[GEN=?g,NUM=?n]
TRI[CNT=?k,SEM=<all x.(?l(x) -> (exists i.(InstanceOf(i,x) & DarsanaHeya(i)) & exists i.(InstanceOf(i,x) & BhavanaHeya(i)) & exists i.(InstanceOf(i,x) & Aheya(i))))>] -> ANTYA[NUM=pl,SEM=?l] NUMR[CNT=?k] TRIDHA[CNT=?k]
EXCL[SEM=<(all i.(?a(i) -> -?h(i)) & all i.(?b(i) -> -?h(i)) & all i.(?c(i) -> -?h(i)))>] -> NA DHEYA[GEN=?g,NUM=?n,SEM=?h] NP[GEN=?g,NUM=?n,SEM=?a] NA NP[GEN=?g,NUM=?n,SEM=?b] NA API NP[GEN=?g,NUM=?n,SEM=?c]
NUMR[CNT=10,SEM=<\x.Rupin(x)>] -> 'daśa'
NUMR[CNT=5,SEM=<\x.SensoryCognition(x)>] -> 'pañca'
NUMR[CNT=3] -> 'trayaḥ'
INS[SEM=<\i.BhavanaHeya(i)>] -> 'bhāvanayā'
HEYA[GEN=m,NUM=pl] -> 'heyāḥ'
CA -> 'ca'
ANTYA[NUM=pl,SEM=<\x.LastThree(x)>] -> 'antyāḥ'
TRIDHA[CNT=3] -> 'tridhā'
NA -> 'na'
API -> 'api'
DHEYA[GEN=n,NUM=sg,SEM=<\i.DarsanaHeya(i)>] -> 'dṛṣṭi-heyam'
NP[GEN=n,NUM=sg,SEM=<\i.Unafflicted(i)>] -> 'akliṣṭam'
NP[GEN=n,NUM=sg,SEM=<\i.Form(i)>] -> 'rūpam'
NP[GEN=n,NUM=sg,SEM=<\i.-BornFromSixth(i)>] -> 'a-ṣaṣṭha-jam'
"""

dependency_text = r"""'heyāḥ' -> 'bhāvanayā' | 'daśa' | 'pañca'
'pañca' -> 'ca'
'tridhā' -> 'trayaḥ'
'trayaḥ' -> 'antyāḥ'
'dṛṣṭi-heyam' -> 'na' | 'akliṣṭam' | 'rūpam' | 'a-ṣaṣṭha-jam'
'a-ṣaṣṭha-jam' -> 'api'
"""

sem_text = r"""# derived by the section 4 parse
(all x.((Rupin(x) | SensoryCognition(x)) -> all i.(InstanceOf(i,x) -> BhavanaHeya(i))) & all x.(LastThree(x) -> (exists i.(InstanceOf(i,x) & DarsanaHeya(i)) & exists i.(InstanceOf(i,x) & BhavanaHeya(i)) & exists i.(InstanceOf(i,x) & Aheya(i)))) & all i.(Unafflicted(i) -> -DarsanaHeya(i)) & all i.(Form(i) -> -DarsanaHeya(i)) & all i.(-BornFromSixth(i) -> -DarsanaHeya(i)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.((Rupin(x) | SensoryCognition(x)) -> all i.(InstanceOf(i,x) -> BhavanaHeya(i))) & all x.(LastThree(x) -> (exists i.(InstanceOf(i,x) & DarsanaHeya(i)) & exists i.(InstanceOf(i,x) & BhavanaHeya(i)) & exists i.(InstanceOf(i,x) & Aheya(i)))) & all i.(Unafflicted(i) -> -DarsanaHeya(i)) & all i.(Form(i) -> -DarsanaHeya(i)) & all i.(-BornFromSixth(i) -> -DarsanaHeya(i)))
# THEORY prose
# EXPECT prose T
all x.(LastThree(x) <-> (x = mind | x = dharma_principle | x = mind_cognition))
all i.(DarsanaHeya(i) <-> (SeeingAnusaya(i) | exists a.(SeeingAnusaya(a) & (ArisesWith(i,a) | AcquisitionOf(i,a) | AttendantOf(i,a)))))
all i.(BhavanaHeya(i) <-> (Impure(i) & -DarsanaHeya(i)))
all i.(Aheya(i) <-> Pure(i))
all i.(exists x.InstanceOf(i,x) -> (Pure(i) <-> -Impure(i)))
all i.(exists x.InstanceOf(i,x) -> (Unafflicted(i) <-> -Afflicted(i)))
all i.(BornOfFiveFaculties(i) -> -BornFromSixth(i))
exists p.(p = prthagjanatva & Unafflicted(p) & Indeterminate(p) & OpposesPath(p) & BhavanaHeya(p))
exists p.(p = prthagjanatva & PossessedBy(p,root_cutter) & PossessedBy(p,free_of_attachment))
exists k.(k = bad_destination_action & Form(k) & OpposesPath(k) & BhavanaHeya(k))
exists g.(g = greedy_eye_cognition & Afflicted(g) & BornOfFiveFaculties(g) & BhavanaHeya(g))
exists s.(s = sixth & s = mind_essence)
# THEORY rejected_opposition_suffices
# EXPECT rejected_opposition_suffices F
all i.(OpposesPath(i) -> DarsanaHeya(i))
# THEORY rejected_ordinary_status_by_seeing
# EXPECT rejected_ordinary_status_by_seeing F
exists p.(p = prthagjanatva & DarsanaHeya(p))
# THEORY rejected_form_by_seeing
# EXPECT rejected_form_by_seeing F
exists i.(Form(i) & DarsanaHeya(i))
# THEORY rejected_affliction_suffices
# EXPECT rejected_affliction_suffices F
all i.(Afflicted(i) -> DarsanaHeya(i))
# THEORY rejected_fifteen_have_pure
# EXPECT rejected_fifteen_have_pure F
exists x.((Rupin(x) | SensoryCognition(x)) & exists i.(InstanceOf(i,x) & Aheya(i)))
# THEORY rejected_pure_abandoned
# EXPECT rejected_pure_abandoned F
exists i.(Pure(i) & -Aheya(i))
# THEORY rejected_last_three_cultivation_only
# EXPECT rejected_last_three_cultivation_only F
exists x.(LastThree(x) & all i.(InstanceOf(i,x) -> BhavanaHeya(i)))
"""

valuation_text = r"""dharma_principle => dharma_d
mind => mano_d
mind_cognition => manovijnana_d
prthagjanatva => prthagjanatva_i
bad_destination_action => apayika_karma_i
greedy_eye_cognition => caksurvijnana_raga_i
root_cutter => samucchinna_p
free_of_attachment => vitaraga_p
sixth => mana_ayatana
mind_essence => mana_ayatana
Rupin => {caksur_d,srotra_d,ghrana_d,jihva_d,kaya_d,rupa_d,sabda_d,gandha_d,rasa_d,sprastavya_d}
SensoryCognition => {caksurvijnana_d,srotravijnana_d,ghranavijnana_d,jihvavijnana_d,kayavijnana_d}
LastThree => {mano_d,dharma_d,manovijnana_d}
InstanceOf => {(caksur_i,caksur_d),(srotra_i,srotra_d),(ghrana_i,ghrana_d),(jihva_i,jihva_d),(kaya_i,kaya_d),(rupa_i,rupa_d),(sabda_i,sabda_d),(gandha_i,gandha_d),(rasa_i,rasa_d),(sprastavya_i,sprastavya_d),(apayika_karma_i,rupa_d),(caksurvijnana_i,caksurvijnana_d),(srotravijnana_i,srotravijnana_d),(ghranavijnana_i,ghranavijnana_d),(jihvavijnana_i,jihvavijnana_d),(kayavijnana_i,kayavijnana_d),(caksurvijnana_raga_i,caksurvijnana_d),(anusaya_i,dharma_d),(anusaya_prapti_i,dharma_d),(anusaya_jati_i,dharma_d),(prthagjanatva_i,dharma_d),(avijnapti_i,dharma_d),(dharma_cult_i,dharma_d),(marga_i,dharma_d),(nirvana_i,dharma_d),(mano_view_i,mano_d),(mano_cult_i,mano_d),(mano_pure_i,mano_d),(manovijnana_view_i,manovijnana_d),(manovijnana_cult_i,manovijnana_d),(manovijnana_pure_i,manovijnana_d)}
DarsanaHeya => {anusaya_i,anusaya_prapti_i,anusaya_jati_i,mano_view_i,manovijnana_view_i}
BhavanaHeya => {caksur_i,srotra_i,ghrana_i,jihva_i,kaya_i,rupa_i,sabda_i,gandha_i,rasa_i,sprastavya_i,apayika_karma_i,caksurvijnana_i,srotravijnana_i,ghranavijnana_i,jihvavijnana_i,kayavijnana_i,caksurvijnana_raga_i,prthagjanatva_i,avijnapti_i,dharma_cult_i,mano_cult_i,manovijnana_cult_i}
Aheya => {marga_i,nirvana_i,mano_pure_i,manovijnana_pure_i}
Afflicted => {caksurvijnana_raga_i,anusaya_i,anusaya_prapti_i,anusaya_jati_i,dharma_cult_i,mano_view_i,mano_cult_i,manovijnana_view_i,manovijnana_cult_i}
Unafflicted => {caksur_i,srotra_i,ghrana_i,jihva_i,kaya_i,rupa_i,sabda_i,gandha_i,rasa_i,sprastavya_i,apayika_karma_i,caksurvijnana_i,srotravijnana_i,ghranavijnana_i,jihvavijnana_i,kayavijnana_i,prthagjanatva_i,avijnapti_i,marga_i,nirvana_i,mano_pure_i,manovijnana_pure_i}
Impure => {caksur_i,srotra_i,ghrana_i,jihva_i,kaya_i,rupa_i,sabda_i,gandha_i,rasa_i,sprastavya_i,apayika_karma_i,caksurvijnana_i,srotravijnana_i,ghranavijnana_i,jihvavijnana_i,kayavijnana_i,caksurvijnana_raga_i,anusaya_i,anusaya_prapti_i,anusaya_jati_i,prthagjanatva_i,avijnapti_i,dharma_cult_i,mano_view_i,mano_cult_i,manovijnana_view_i,manovijnana_cult_i}
Pure => {marga_i,nirvana_i,mano_pure_i,manovijnana_pure_i}
Form => {caksur_i,srotra_i,ghrana_i,jihva_i,kaya_i,rupa_i,sabda_i,gandha_i,rasa_i,sprastavya_i,apayika_karma_i,avijnapti_i}
BornFromSixth => {anusaya_i,anusaya_prapti_i,anusaya_jati_i,dharma_cult_i,marga_i,mano_view_i,mano_cult_i,mano_pure_i,manovijnana_view_i,manovijnana_cult_i,manovijnana_pure_i}
BornOfFiveFaculties => {caksurvijnana_i,srotravijnana_i,ghranavijnana_i,jihvavijnana_i,kayavijnana_i,caksurvijnana_raga_i}
OpposesPath => {apayika_karma_i,prthagjanatva_i}
Indeterminate => {prthagjanatva_i}
SeeingAnusaya => {anusaya_i}
ArisesWith => {(mano_view_i,anusaya_i),(manovijnana_view_i,anusaya_i)}
AcquisitionOf => {(anusaya_prapti_i,anusaya_i)}
AttendantOf => {(anusaya_jati_i,anusaya_i)}
PossessedBy => {(prthagjanatva_i,samucchinna_p),(prthagjanatva_i,vitaraga_p)}
"""


In [ ]:
studies[40] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[40]["tree"].pformat())
print("Derived SEM:", studies[40]["sem"])


# VAK_1.41 — Two Portions Are View

Source: `VAK_1.41.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
cakṣuś ca dharmadhātoś ca pradeśau dṛṣṭir aṣṭadhā /
pañcavijñānasahajā dhīr na dṛṣṭir atīraṇāt // 1.41 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| cakṣuḥ | cakṣus; nominative neuter singular | the eye: the whole Eye Principle |
| ca | particle | and |
| dharma-dhātoḥ | tatpuruṣa; genitive masculine singular | of the Dharma Principle: partitive |
| ca | particle | and |
| pradeśau | pradeśa; nominative masculine dual | two portions: of the eighteen |
| dṛṣṭiḥ | dṛṣṭi; nominative feminine singular | view |
| aṣṭadhā | adverb | eightfold: said of the Dharma Principle portion |
| pañca-vijñāna-sahajā | upapada compound; nominative feminine singular | born with the five sensory Cognitions |
| *dhīḥ* | dhī; nominative feminine singular | *prajñā* (Science of Principles), discriminative comprehension |
| na | negative particle | not |
| dṛṣṭiḥ | dṛṣṭi; nominative feminine singular | view |
| a-tīraṇāt | a-tīraṇa; ablative singular | because of not ascertaining: the ablative of ground |

*Cakṣuḥ* is nominative, not genitive. The eye is one of the two portions
of the eighteen as a whole Principle; only the Dharma Principle is divided.

**Grammar tokens.** Verse-order IAST with sandhi opened (`cakṣuś` →
*cakṣuḥ*, `dharmadhātoś` → *dharma-dhātoḥ*, `dṛṣṭir` → *dṛṣṭiḥ*, `dhīr` →
*dhīḥ*) and compound boundaries marked.

```text
# tokens: cakṣuḥ ca dharma-dhātoḥ ca pradeśau dṛṣṭiḥ aṣṭadhā pañca-vijñāna-sahajā dhīḥ na dṛṣṭiḥ a-tīraṇāt
```

## 5. Translation

### Literal Translation

> The eye and a portion of the Dharma Principle, two portions, are view,
> eightfold. Discriminative comprehension born with the five Cognitions is
> not view, because it does not ascertain.

### Bhāṣya-informed study translation

> The Eye Principle and an eightfold portion of the Dharma Principle, two
> portions of the eighteen, are view. The discriminative comprehension
> arising with the five sensory Cognitions is not view, because it does not
> ascertain.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| pradeśa | portion | of the eighteen: the whole eye, and part of the Dharma Principle |
| dṛṣṭi | view | eightfold in the Dharma Principle; disclosure in the eye |
| satkāya-dṛṣṭi | view of a real body | first of the five; explained later |
| laukikī samyag-dṛṣṭi | worldly right view | wholesome Impure discriminative comprehension with Mind-Cognition |
| śaikṣī | the trainee's view | Pure |
| aśaikṣī | the adept's view | Pure |
| dhī | discriminative comprehension | *prajñā* |
| atīraṇa | non-ascertainment | why the fivefold discriminative comprehension is not view |
| santīrikā | ascertaining | view proceeds by attentive examination |
| rūpālocana | disclosure of Visible Form | why the eye is view |

In [ ]:
fcfg_text = r"""% start S
# tokens: cakṣuḥ ca dharma-dhātoḥ ca pradeśau dṛṣṭiḥ aṣṭadhā pañca-vijñāna-sahajā dhīḥ na dṛṣṭiḥ a-tīraṇāt
S[SEM=<(?a & ?b)>] -> VIEW[SEM=?a] NOTV[SEM=?b]
VIEW[SEM=<(?q1(\i.View(i)) & ?q2(\i.View(i)) & all i.((InstanceOf(i,?r2) & View(i)) -> ?k(i)) & all x.all i.((InstanceOf(i,x) & View(i)) -> ((x = ?r1) | (x = ?r2))))>] -> NP[CASE=nom,REF=?r1,SEM=?q1] CA NP[CASE=gen,REF=?r2,SEM=?q2] CA PRADESA[NUM=du] DRSTI[GEN=f,NUM=sg] ADV[CNT=8,SEM=?k]
NOTV[SEM=<all i.((?s(i) & ?p(i)) -> (?w(i) & -View(i)))>] -> SAHAJA[GEN=?g,NUM=?n,SEM=?s] DHI[GEN=?g,NUM=?n,SEM=?p] NA DRSTI[GEN=?g,NUM=?n] ABL[SEM=?w]
NP[CASE=nom,REF=<eye>,SEM=<\P.all i.(InstanceOf(i,eye) -> P(i))>] -> 'cakṣuḥ'
NP[CASE=gen,REF=<dharma_principle>,SEM=<\P.(exists i.(InstanceOf(i,dharma_principle) & P(i)) & exists i.(InstanceOf(i,dharma_principle) & -P(i)))>] -> 'dharma-dhātoḥ'
CA -> 'ca'
PRADESA[NUM=du] -> 'pradeśau'
DRSTI[GEN=f,NUM=sg] -> 'dṛṣṭiḥ'
ADV[CNT=8,SEM=<\i.exists k.(EightKind(k) & ViewKind(i,k))>] -> 'aṣṭadhā'
SAHAJA[GEN=f,NUM=sg,SEM=<\i.BornWithFive(i)>] -> 'pañca-vijñāna-sahajā'
DHI[GEN=f,NUM=sg,SEM=<\i.Prajna(i)>] -> 'dhīḥ'
NA -> 'na'
ABL[SEM=<\i.-Ascertains(i)>] -> 'a-tīraṇāt'
"""

dependency_text = r"""'dṛṣṭiḥ' -> 'pradeśau' | 'aṣṭadhā' | 'dhīḥ' | 'na' | 'a-tīraṇāt'
'pradeśau' -> 'cakṣuḥ' | 'dharma-dhātoḥ'
'cakṣuḥ' -> 'ca'
'dharma-dhātoḥ' -> 'ca'
'dhīḥ' -> 'pañca-vijñāna-sahajā'
"""

sem_text = r"""# derived by the section 4 parse
(all i.(InstanceOf(i,eye) -> View(i)) & exists i.(InstanceOf(i,dharma_principle) & View(i)) & exists i.(InstanceOf(i,dharma_principle) & -View(i)) & all i.((InstanceOf(i,dharma_principle) & View(i)) -> exists k.(EightKind(k) & ViewKind(i,k))) & all x i.((InstanceOf(i,x) & View(i)) -> ((x = eye) | (x = dharma_principle))) & all i.((BornWithFive(i) & Prajna(i)) -> (-Ascertains(i) & -View(i))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all i.(InstanceOf(i,eye) -> View(i)) & exists i.(InstanceOf(i,dharma_principle) & View(i)) & exists i.(InstanceOf(i,dharma_principle) & -View(i)) & all i.((InstanceOf(i,dharma_principle) & View(i)) -> exists k.(EightKind(k) & ViewKind(i,k))) & all x i.((InstanceOf(i,x) & View(i)) -> ((x = eye) | (x = dharma_principle))) & all i.((BornWithFive(i) & Prajna(i)) -> (-Ascertains(i) & -View(i))))
# THEORY prose
# EXPECT prose T
all i.(InstanceOf(i,eye) -> (Discloses(i) & -Ascertains(i)))
all i.((InstanceOf(i,dharma_principle) & View(i)) -> Prajna(i))
all i.(Prajna(i) -> (View(i) <-> Ascertains(i)))
all i.((Prajna(i) & View(i)) -> AssociatedWith(i,mind_cognition))
exists w.(w = worldly_right_view & View(w) & Wholesome(w) & Impure(w))
exists t.(t = trainee_view & View(t) & Pure(t))
exists a.(a = adept_view & View(a) & Pure(a))
all i.((ViewKind(i,satkaya) | ViewKind(i,antagraha) | ViewKind(i,mithya) | ViewKind(i,drstiparamarsa) | ViewKind(i,silavrataparamarsa)) -> Afflicted(i))
exists o.(o = other_prajna & Prajna(o) & InstanceOf(o,dharma_principle) & -Ascertains(o) & -View(o))
exists v.(v = feeling & InstanceOf(v,dharma_principle) & -View(v))
# THEORY rejected_whole_dharma_principle_is_view
# EXPECT rejected_whole_dharma_principle_is_view F
all i.(InstanceOf(i,dharma_principle) -> View(i))
# THEORY rejected_only_part_of_eye
# EXPECT rejected_only_part_of_eye F
exists i.(InstanceOf(i,eye) & -View(i))
# THEORY rejected_eye_ascertains
# EXPECT rejected_eye_ascertains F
exists i.(InstanceOf(i,eye) & Ascertains(i))
# THEORY rejected_view_always_ascertains
# EXPECT rejected_view_always_ascertains F
all i.(View(i) -> Ascertains(i))
# THEORY rejected_five_born_prajna_is_view
# EXPECT rejected_five_born_prajna_is_view F
exists i.(Prajna(i) & BornWithFive(i) & View(i))
# THEORY rejected_mind_cognition_is_view
# EXPECT rejected_mind_cognition_is_view F
exists i.(InstanceOf(i,mind_cognition) & View(i))
# THEORY rejected_worldly_right_view_pure
# EXPECT rejected_worldly_right_view_pure F
exists w.(w = worldly_right_view & Pure(w))
"""

valuation_text = r"""eye => caksur_d
dharma_principle => dharma_d
mind_cognition => manovijnana_d
worldly_right_view => laukika_vi
trainee_view => saiksa_vi
adept_view => asaiksa_vi
other_prajna => other_prajna_i
feeling => vedana_i
satkaya => satkaya_k
antagraha => antagraha_k
mithya => mithya_k
drstiparamarsa => drstiparamarsa_k
silavrataparamarsa => silavrataparamarsa_k
InstanceOf => {(srotra_i,srotra_d),(ghrana_i,ghrana_d),(jihva_i,jihva_d),(kaya_i,kaya_d),(rupa_i,rupa_d),(sabda_i,sabda_d),(gandha_i,gandha_d),(rasa_i,rasa_d),(sprastavya_i,sprastavya_d),(mano_i,mano_d),(caksurvijnana_i,caksurvijnana_d),(srotravijnana_i,srotravijnana_d),(ghranavijnana_i,ghranavijnana_d),(jihvavijnana_i,jihvavijnana_d),(kayavijnana_i,kayavijnana_d),(manovijnana_i,manovijnana_d),(caksur_seen_i,caksur_d),(caksur_tat_i,caksur_d),(satkaya_vi,dharma_d),(antagraha_vi,dharma_d),(mithya_vi,dharma_d),(drstiparamarsa_vi,dharma_d),(silavrataparamarsa_vi,dharma_d),(laukika_vi,dharma_d),(saiksa_vi,dharma_d),(asaiksa_vi,dharma_d),(other_prajna_i,dharma_d),(five_prajna_i,dharma_d),(vedana_i,dharma_d)}
View => {caksur_seen_i,caksur_tat_i,satkaya_vi,antagraha_vi,mithya_vi,drstiparamarsa_vi,silavrataparamarsa_vi,laukika_vi,saiksa_vi,asaiksa_vi}
Discloses => {caksur_seen_i,caksur_tat_i}
Ascertains => {satkaya_vi,antagraha_vi,mithya_vi,drstiparamarsa_vi,silavrataparamarsa_vi,laukika_vi,saiksa_vi,asaiksa_vi}
Prajna => {satkaya_vi,antagraha_vi,mithya_vi,drstiparamarsa_vi,silavrataparamarsa_vi,laukika_vi,saiksa_vi,asaiksa_vi,other_prajna_i,five_prajna_i}
BornWithFive => {five_prajna_i}
EightKind => {satkaya_k,antagraha_k,mithya_k,drstiparamarsa_k,silavrataparamarsa_k,laukika_k,saiksa_k,asaiksa_k}
ViewKind => {(satkaya_vi,satkaya_k),(antagraha_vi,antagraha_k),(mithya_vi,mithya_k),(drstiparamarsa_vi,drstiparamarsa_k),(silavrataparamarsa_vi,silavrataparamarsa_k),(laukika_vi,laukika_k),(saiksa_vi,saiksa_k),(asaiksa_vi,asaiksa_k)}
AssociatedWith => {(satkaya_vi,manovijnana_d),(antagraha_vi,manovijnana_d),(mithya_vi,manovijnana_d),(drstiparamarsa_vi,manovijnana_d),(silavrataparamarsa_vi,manovijnana_d),(laukika_vi,manovijnana_d),(saiksa_vi,manovijnana_d),(asaiksa_vi,manovijnana_d)}
Wholesome => {laukika_vi}
Impure => {satkaya_vi,antagraha_vi,mithya_vi,drstiparamarsa_vi,silavrataparamarsa_vi,laukika_vi}
Pure => {saiksa_vi,asaiksa_vi}
Afflicted => {satkaya_vi,antagraha_vi,mithya_vi,drstiparamarsa_vi,silavrataparamarsa_vi}
"""


In [ ]:
studies[41] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[41]["tree"].pformat())
print("Derived SEM:", studies[41]["sem"])


# VAK_1.42 — The Eye Sees

Source: `VAK_1.42.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
cakṣuḥ paśyati rūpāṇi sabhāgaṃ na tadāśritam /
vijñānaṃ dṛśyate rūpaṃ na kilāntaritaṃ yataḥ // 1.42 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| cakṣuḥ | cakṣus; nominative neuter singular | the eye: agent of seeing |
| paśyati | √dṛś (paśya-), present active, third singular | sees |
| rūpāṇi | rūpa; accusative neuter plural | Visible Forms: what is seen |
| sabhāgam | sa-bhāga; nominative neuter singular | that does its own work: restricts *cakṣuḥ* |
| na | negative particle | not |
| tat-āśritam | tatpuruṣa, past participle of ā-√śri; nominative neuter singular | supported by it: *tat* is the eye |
| vijñānam | vijñāna; nominative neuter singular | Cognition: the excluded agent |
| dṛśyate | √dṛś, present passive, third singular | is seen |
| rūpam | rūpa; nominative neuter singular | a Visible Form: subject of the passive |
| na | negative particle | not |
| kila | particle | it is said: reported, not endorsed |
| antaritam | antar-√i, past participle; nominative neuter singular | screened: behind a wall |
| yataḥ | ablative adverb | since: the ground of the exclusion |

*Vijñānam* is nominative, not accusative. With *na tat-āśritam* it names the
rival agent the verse excludes. *Kila* marks the reason as the Vaibhāṣika
report; the Bhāṣya's own Sautrāntika voice does not accept it.

**Grammar tokens.** Verse-order IAST with sandhi opened (`sabhāgaṃ` →
*sabhāgam*, `tadāśritam` → *tat-āśritam*, `kilāntaritaṃ` → *kila
antaritam*) and compound boundaries marked.

```text
# tokens: cakṣuḥ paśyati rūpāṇi sabhāgam na tat-āśritam vijñānam dṛśyate rūpam na kila antaritam yataḥ
```

## 5. Translation

### Literal Translation

> The eye sees Visible Forms, the one that does its own work; not the
> Cognition supported by it. For, it is said, a screened Visible Form is not
> seen.

### Bhāṣya-informed study translation

> The eye sees Visible Forms when it does its own work; the Cognition
> supported by it does not see, since, it is said, a Visible Form screened by
> a wall is not seen.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| sabhāga | does its own work | the eye with Cognition |
| tad-āśrita | supported by it | Eye-Cognition; not what sees |
| antarita | screened | the wall; light blocked |
| sapratigha | resistant | the eye |
| apratigha | unresistant | Cognition |
| prāpta-viṣaya | Condition-reacher | the Body Faculty; not the eye |
| upacāra | attribution | the beds cry; the eye sees |
| nirvyāpāra | without activity | the Sautrāntika refusal |
| siddhānta | established position | the eye sees; mind cognizes |

In [ ]:
fcfg_text = r"""% start S
# tokens: cakṣuḥ paśyati rūpāṇi sabhāgam na tat-āśritam vijñānam dṛśyate rūpam na kila antaritam yataḥ
S[EVID=?ev,SEM=<(?a & ?g(?rel))>] -> ACT[ROOT=?rt,REL=?rel,SEM=?a] GROUND[ROOT=?rt,EVID=?ev,SEM=?g]
ACT[ROOT=?rt,REL=?rel,SEM=<(all a.((?s(a) & ?r(a)) -> exists v.(?o(v) & ?rel(a,v))) & all a.all v.(?rel(a,v) -> (?s(a) & ?r(a))) & all c.((?asr(?ref)(c) & ?n(c)) -> -exists v.?rel(c,v)))>] -> NP[CASE=nom,GEN=?g,NUM=?m,REF=?ref,SEM=?s] V[VOICE=act,ROOT=?rt,SEM=?rel] NP[CASE=acc,NUM=pl,SEM=?o] ADJ[GEN=?g,NUM=?m,SEM=?r] NA ASR[GEN=?g,NUM=?m,SEM=?asr] NP[CASE=nom,GEN=?g,NUM=?m,SEM=?n]
GROUND[ROOT=?rt,EVID=?ev,SEM=<\R.all v.((?o(v) & ?p(v)) -> -exists x.R(x,v))>] -> V[VOICE=pass,ROOT=?rt] NP[CASE=nom,NUM=sg,SEM=?o] NA KILA[EVID=?ev] PART[SEM=?p] YATAH
NP[CASE=nom,GEN=n,NUM=sg,REF=<eye>,SEM=<\a.InstanceOf(a,eye)>] -> 'cakṣuḥ'
V[VOICE=act,ROOT=drs,SEM=<\a v.Sees(a,v)>] -> 'paśyati'
NP[CASE=acc,NUM=pl,SEM=<\v.VisibleForm(v)>] -> 'rūpāṇi'
ADJ[GEN=n,NUM=sg,SEM=<\a.Sabhaga(a)>] -> 'sabhāgam'
NA -> 'na'
ASR[GEN=n,NUM=sg,SEM=<\r c.exists a.(InstanceOf(a,r) & SupportedBy(c,a))>] -> 'tat-āśritam'
NP[CASE=nom,GEN=n,NUM=sg,SEM=<\c.Cognition(c)>] -> 'vijñānam'
V[VOICE=pass,ROOT=drs] -> 'dṛśyate'
NP[CASE=nom,NUM=sg,SEM=<\v.VisibleForm(v)>] -> 'rūpam'
KILA[EVID=reported] -> 'kila'
PART[SEM=<\v.Screened(v)>] -> 'antaritam'
YATAH -> 'yataḥ'
"""

dependency_text = r"""'paśyati' -> 'cakṣuḥ' | 'rūpāṇi' | 'vijñānam' | 'dṛśyate'
'cakṣuḥ' -> 'sabhāgam'
'vijñānam' -> 'na' | 'tat-āśritam'
'dṛśyate' -> 'rūpam' | 'kila' | 'yataḥ'
'rūpam' -> 'antaritam'
'antaritam' -> 'na'
"""

sem_text = r"""# derived by the section 4 parse
(all a.((InstanceOf(a,eye) & Sabhaga(a)) -> exists v.(VisibleForm(v) & Sees(a,v))) & all a v.(Sees(a,v) -> (InstanceOf(a,eye) & Sabhaga(a))) & all c.((exists a.(InstanceOf(a,eye) & SupportedBy(c,a)) & Cognition(c)) -> -exists v.Sees(c,v)) & all v.((VisibleForm(v) & Screened(v)) -> -exists x.Sees(x,v)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all a.((InstanceOf(a,eye) & Sabhaga(a)) -> exists v.(VisibleForm(v) & Sees(a,v))) & all a v.(Sees(a,v) -> (InstanceOf(a,eye) & Sabhaga(a))) & all c.((exists a.(InstanceOf(a,eye) & SupportedBy(c,a)) & Cognition(c)) -> -exists v.Sees(c,v)) & all v.((VisibleForm(v) & Screened(v)) -> -exists x.Sees(x,v)))
# THEORY prose
# EXPECT prose T
all x.(Cognition(x) -> -Resistant(x))
all a.(InstanceOf(a,eye) -> Resistant(a))
all x.((exists v.Sees(x,v)) -> Resistant(x))
all v.(Screened(v) <-> (Covered(v) & LightBlocked(v)))
exists g.(g = glass_screened & Covered(g) & -LightBlocked(g) & exists a.Sees(a,g))
all a.all v.(Sees(a,v) -> exists c.(SupportedBy(c,a) & Experiences(c,v)))
all v.(Seen(v) <-> exists a.exists c.(Sees(a,v) & Experiences(c,v)))
exists a.(InstanceOf(a,eye) & -Sabhaga(a) & -exists v.Sees(a,v))
exists x.(x = body & ReachesByContact(x) & -exists y.(y = eye & ReachesByContact(y)))
all v.(Seen(v) -> -Cognized(v))
# THEORY rejected_every_eye_sees
# EXPECT rejected_every_eye_sees F
all a.(InstanceOf(a,eye) -> exists v.Sees(a,v))
# THEORY rejected_cognition_sees
# EXPECT rejected_cognition_sees F
exists c.exists v.(Cognition(c) & Sees(c,v))
# THEORY rejected_eye_reaches_by_contact
# EXPECT rejected_eye_reaches_by_contact F
exists x.(x = eye & ReachesByContact(x))
# THEORY rejected_any_cover_blocks
# EXPECT rejected_any_cover_blocks F
all v.(Covered(v) -> -exists a.Sees(a,v))
# THEORY rejected_eye_cognizes
# EXPECT rejected_eye_cognizes F
exists a.exists v.(InstanceOf(a,eye) & Experiences(a,v))
# THEORY sautrantika_no_agent
# EXPECT sautrantika_no_agent F
-exists x.exists v.Sees(x,v)
"""

valuation_text = r"""eye => caksur_d
body => kaya_d
glass_screened => glass_v
InstanceOf => {(eye_a,caksur_d),(eye_b,caksur_d),(cog_a,caksurvijnana_d),(moon_v,rupa_d),(wall_v,rupa_d),(glass_v,rupa_d)}
Sees => {(eye_a,moon_v),(eye_a,glass_v)}
VisibleForm => {moon_v,wall_v,glass_v}
Sabhaga => {eye_a,cog_a}
SupportedBy => {(cog_a,eye_a)}
Cognition => {cog_a}
Resistant => {eye_a,eye_b,wall_v}
Screened => {wall_v}
Covered => {wall_v,glass_v}
LightBlocked => {wall_v}
Experiences => {(cog_a,moon_v),(cog_a,glass_v)}
Seen => {moon_v,glass_v}
Cognized => {}
ReachesByContact => {kaya_d}
"""


In [ ]:
studies[42] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[42]["tree"].pformat())
print("Derived SEM:", studies[42]["sem"])


# VAK_1.43 — Both Eyes, and the Unreached

Source: `VAK_1.43.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
ubhābhyām api cakṣurbhyāṃ paśyati vyaktadarśanāt /
cakṣuḥśrotramano 'prāptaviṣayaṃ trayam anyathā // 1.43 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| ubhābhyām | ubha; instrumental dual | with both |
| api | particle | too: one eye is not excluded |
| cakṣurbhyām | cakṣus; instrumental dual | with the two eyes |
| paśyati | √dṛś (paśya-), present active, third singular | one sees |
| vyakta-darśanāt | karmadhāraya; ablative singular | because the seeing is clearer: ablative of ground |
| cakṣuḥ-śrotra-manaḥ | samāhāra dvandva; nominative neuter singular | Eye, Ear, and Mind, taken as one set of three |
| a-prāpta-viṣayam | bahuvrīhi; nominative neuter singular | whose Condition is not reached |
| trayam | traya; nominative neuter singular | the triad: the other three of the six |
| anyathā | adverb | otherwise: the converse of the preceding predicate |

The collective dvandva is neuter singular, and so are its predicate and
*trayam*. The other three are Nose, Tongue, and Body. They reach their
Condition.

**Grammar tokens.** Verse-order IAST with sandhi opened (`cakṣurbhyāṃ` →
*cakṣurbhyām*, `cakṣuḥśrotramano 'prāptaviṣayaṃ` → *cakṣuḥ-śrotra-manaḥ
a-prāpta-viṣayam*) and compound boundaries marked.

```text
# tokens: ubhābhyām api cakṣurbhyām paśyati vyakta-darśanāt cakṣuḥ-śrotra-manaḥ a-prāpta-viṣayam trayam anyathā
```

## 5. Translation

### Literal Translation

> One sees with both eyes too, because the seeing is clearer. Eye, Ear, and
> Mind do not reach their Condition. The three are otherwise.

### Bhāṣya-informed study translation

> One sees with both eyes as well, because the seeing is clearer. Eye, Ear,
> and Mind apprehend a Condition they do not reach; the other three—Nose,
> Tongue, and Body—apprehend a Condition they reach.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| ubhābhyām api | with both too | no exclusive rule |
| vyakta-darśana | clearer seeing | both eyes open |
| aprāpta-viṣaya | whose Condition is not reached | Eye, Ear, Mind |
| prāpta-viṣaya | apprehends a Condition it reaches | Nose, Tongue, Body |
| prāpti | reaching | arising without an interval |
| nirantara | without an interval | nothing between |
| paramāṇu | atom | partless; does not touch |
| vāyu-dhātu | Wind Principle | holds the heap, or scatters it |
| spṛṣṭa-saṃjñā | the designation touching | used where there is no interval |

In [ ]:
fcfg_text = r"""% start S
# tokens: ubhābhyām api cakṣurbhyām paśyati vyakta-darśanāt cakṣuḥ-śrotra-manaḥ a-prāpta-viṣayam trayam anyathā
S[SEM=<(?a & ?b)>] -> BOTH[SEM=?a] REACH[SEM=?b]
BOTH[SEM=<(exists s.(?v(s) & ?i(s)) & exists s.(?v(s) & -?i(s)) & all s.((?v(s) & ?i(s)) -> ?g(s)))>] -> UBHA[CASE=ins,NUM=du] API INST[CASE=ins,NUM=du,SEM=?i] V[SEM=?v] ABL[SEM=?g]
REACH[SEM=<(all x.(?m(x) -> ?u(x)) & all x.((Basis(x) & -?m(x)) -> ?n(?u,x)))>] -> DV[CNT=?k,GEN=?g,NUM=?q,SEM=?m] PRED[GEN=?g,NUM=?q,SEM=?u] TRAYA[CNT=?k,GEN=?g,NUM=?q] ANY[SEM=?n]
UBHA[CASE=ins,NUM=du] -> 'ubhābhyām'
API -> 'api'
INST[CASE=ins,NUM=du,SEM=<\s.BothEyes(s)>] -> 'cakṣurbhyām'
V[SEM=<\s.Seeing(s)>] -> 'paśyati'
ABL[SEM=<\s.Clearer(s)>] -> 'vyakta-darśanāt'
DV[CNT=3,GEN=n,NUM=sg,SEM=<\x.((x = eye) | (x = ear) | (x = mind))>] -> 'cakṣuḥ-śrotra-manaḥ'
PRED[GEN=n,NUM=sg,SEM=<\x.TakesUnreached(x)>] -> 'a-prāpta-viṣayam'
TRAYA[CNT=3,GEN=n,NUM=sg] -> 'trayam'
ANY[SEM=<\P x.-P(x)>] -> 'anyathā'
"""

dependency_text = r"""'paśyati' -> 'cakṣurbhyām' | 'vyakta-darśanāt'
'cakṣurbhyām' -> 'ubhābhyām' | 'api'
'a-prāpta-viṣayam' -> 'cakṣuḥ-śrotra-manaḥ' | 'trayam'
'trayam' -> 'anyathā'
"""

sem_text = r"""# derived by the section 4 parse
(exists s.(Seeing(s) & BothEyes(s)) & exists s.(Seeing(s) & -BothEyes(s)) & all s.((Seeing(s) & BothEyes(s)) -> Clearer(s)) & all x.(((x = eye) | (x = ear) | (x = mind)) -> TakesUnreached(x)) & all x.((Basis(x) & -((x = eye) | (x = ear) | (x = mind))) -> -TakesUnreached(x)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(exists s.(Seeing(s) & BothEyes(s)) & exists s.(Seeing(s) & -BothEyes(s)) & all s.((Seeing(s) & BothEyes(s)) -> Clearer(s)) & all x.(((x = eye) | (x = ear) | (x = mind)) -> TakesUnreached(x)) & all x.((Basis(x) & -((x = eye) | (x = ear) | (x = mind))) -> -TakesUnreached(x)))
# THEORY prose
# EXPECT prose T
all x.(Basis(x) -> (TakesUnreached(x) <-> exists v.(Takes(x,v) & -Nirantara(x,v))))
exists s.(HalfClosed(s) & DoubleMoon(s) & -BothEyes(s))
all c.(Cognition(c) -> -Located(c))
exists v.(v = distant_moon & Takes(eye,v) & -Nirantara(eye,v))
exists v.(v = collyrium & Nirantara(eye,v) & -Takes(eye,v))
exists v.(v = hidden_form & -Nirantara(eye,v) & -Takes(eye,v))
exists v.(v = distant_sound & Takes(ear,v) & -Nirantara(ear,v))
exists v.(v = other_realm_form & Takes(eye,v) & -Nirantara(eye,v))
exists m.(m = mind & -Rupin(m) & -exists v.Nirantara(m,v))
exists v.(v = coarising_odor & Nirantara(nose,v) & -Takes(nose,v))
all v.(Takes(nose,v) -> Breath(v))
all a.all b.((Atom(a) & Atom(b)) -> -Touch(a,b))
exists a.exists b.(Atom(a) & Atom(b) & Nirantara(a,b))
all a.all b.(DesignatedTouching(a,b) <-> Nirantara(a,b))
# THEORY reported_ear_both
# EXPECT reported_ear_both T
exists v.exists w.(Takes(ear,v) & Nirantara(ear,v) & Takes(ear,w) & -Nirantara(ear,w))
# THEORY rejected_one_eye_only
# EXPECT rejected_one_eye_only F
all s.(Seeing(s) -> -BothEyes(s))
# THEORY rejected_eye_reaches
# EXPECT rejected_eye_reaches F
all v.(Takes(eye,v) -> Nirantara(eye,v))
# THEORY rejected_every_unreached_seen
# EXPECT rejected_every_unreached_seen F
all v.((VisibleForm(v) & -Nirantara(eye,v)) -> Takes(eye,v))
# THEORY rejected_mind_reaches
# EXPECT rejected_mind_reaches F
exists v.Nirantara(mind,v)
# THEORY rejected_nose_unreached
# EXPECT rejected_nose_unreached F
exists v.(Takes(nose,v) & -Nirantara(nose,v))
# THEORY rejected_atoms_touch
# EXPECT rejected_atoms_touch F
exists a.exists b.(Atom(a) & Atom(b) & Touch(a,b))
# THEORY rejected_reaching_is_touching
# EXPECT rejected_reaching_is_touching F
all x.all v.(Nirantara(x,v) -> Touch(x,v))
"""

valuation_text = r"""eye => eye_f
ear => ear_f
mind => mind_f
nose => nose_f
distant_moon => moon_v
collyrium => anjana_v
hidden_form => hidden_v
distant_sound => far_sound_v
other_realm_form => divya_v
coarising_odor => sahabhu_odor_v
Basis => {eye_f,ear_f,mind_f,nose_f,tongue_f,body_f}
Seeing => {s_both,s_one,s_half}
BothEyes => {s_both}
Clearer => {s_both}
HalfClosed => {s_half}
DoubleMoon => {s_half}
Cognition => {cog_c}
Located => {}
Rupin => {eye_f,ear_f,nose_f,tongue_f,body_f}
VisibleForm => {moon_v,anjana_v,hidden_v,divya_v}
TakesUnreached => {eye_f,ear_f,mind_f}
Takes => {(eye_f,moon_v),(eye_f,divya_v),(ear_f,far_sound_v),(ear_f,inner_sound_v),(mind_f,dharma_v),(nose_f,odor_v),(tongue_f,taste_v),(body_f,tangible_v)}
Nirantara => {(eye_f,anjana_v),(ear_f,inner_sound_v),(nose_f,odor_v),(nose_f,sahabhu_odor_v),(tongue_f,taste_v),(body_f,tangible_v),(atom_a,atom_b)}
Breath => {odor_v}
Atom => {atom_a,atom_b}
Touch => {}
DesignatedTouching => {(eye_f,anjana_v),(ear_f,inner_sound_v),(nose_f,odor_v),(nose_f,sahabhu_odor_v),(tongue_f,taste_v),(body_f,tangible_v),(atom_a,atom_b)}
"""


In [ ]:
studies[43] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[43]["tree"].pformat())
print("Derived SEM:", studies[43]["sem"])


# VAK_1.44 — Equal Grasp, and the Past Basis

Source: `VAK_1.44.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
tribhir ghrāṇādibhis tulyaviṣayagrahaṇaṃ matam /
caramasyāśrayo 'tītaḥ pañcānāṃ sahajaś ca taiḥ // 1.44 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| tribhiḥ | tri; instrumental plural | by three: the count that closes the list |
| ghrāṇa-ādibhiḥ | bahuvrīhi; instrumental plural | by those beginning with the Nose: Nose, Tongue, Body |
| tulya-viṣaya-grahaṇam | tatpuruṣa; nominative neuter singular | apprehension of a Condition equal in extent |
| matam | √man, past passive participle; nominative neuter singular | is held: the doctrine, not a new proof |
| caramasya | carama; genitive singular | of the last: Mind-Cognition |
| āśrayaḥ | āśraya; nominative masculine singular | the Basis |
| atītaḥ | ati-√i, past participle; nominative masculine singular | past: ceased immediately before |
| pañcānām | pañcan; genitive plural | of the five: the sensory Cognitions |
| sahajaḥ | saha-ja; nominative masculine singular | born together: *āśrayaḥ* is understood |
| ca | particle | and: carries *atītaḥ* over to the five |
| taiḥ | tad; instrumental plural | with them: the five Cognitions |

The instrumental is the agent of the passive *matam*: by the three,
apprehension of an equal Condition is held to take place. *Sahajaḥ*
agrees with the elided *āśrayaḥ*. The Bhāṣya reads the *ca* as bringing in
the past Basis as well (*atītaś ceti ca-śabdaḥ*).

**Grammar tokens.** Verse-order IAST with sandhi opened (`tribhir
ghrāṇādibhis` → *tribhiḥ ghrāṇa-ādibhiḥ*, `caramasyāśrayo 'tītaḥ` →
*caramasya āśrayaḥ atītaḥ*, `sahajaś` → *sahajaḥ*) and compound
boundaries marked.

```text
# tokens: tribhiḥ ghrāṇa-ādibhiḥ tulya-viṣaya-grahaṇam matam caramasya āśrayaḥ atītaḥ pañcānām sahajaḥ ca taiḥ
```

## 5. Translation

### Literal Translation

> By the three beginning with the Nose, apprehension of an equal Condition
> is held. The Basis of the last is past. Of the five, a co-born one too,
> with them.

### Bhāṣya-informed study translation

> The three Faculties beginning with the Nose, it is held, apprehend a
> Condition equal in extent to themselves. The Basis of the last Cognition
> is past; the five sensory Cognitions have a Basis born together with
> them, and the past one as well.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| tulya-viṣaya-grahaṇa | apprehension of a Condition equal in extent | Nose, Tongue, Body |
| aniyama | no rule | eye and ear |
| amūrta | unformed | Mind; no size |
| sabhāga | does its own work | not all the body-atoms |
| tatsabhāga | corresponds and does not | the remainder |
| saṃcita | accumulated | one atom does not produce Cognition |
| carama | the last | Mind-Cognition |
| āśraya | Basis | distinct from *ālambana* (Support) |
| sahaja āśraya | co-born Basis | the eye, and so on |
| atīta āśraya | past Basis | Mind that ceased immediately before |
| samanantara-pratyaya | immediately preceding Condition | not the same as Basis |

In [ ]:
fcfg_text = r"""% start S
# tokens: tribhiḥ ghrāṇa-ādibhiḥ tulya-viṣaya-grahaṇam matam caramasya āśrayaḥ atītaḥ pañcānām sahajaḥ ca taiḥ
S[SEM=<(?a & ?b)>] -> EQ[SEM=?a] SUP[SEM=?b]
EQ[SEM=<all x.(?n(x) -> ?p(x))>] -> NUMR[CASE=ins,CNT=?k,NUM=pl] NI[CASE=ins,CNT=?k,NUM=pl,SEM=?n] PRED[GEN=?g,NUM=?s,SEM=?p] MATA[EVID=held,GEN=?g,NUM=?s]
SUP[SEM=<(all c.(?l(c) -> exists b.(?r(b,c) & ?t(b))) & all c.all b.((?l(c) & ?r(b,c)) -> ?t(b)) & all c.(?f(c) -> (exists b.(?r(b,c) & ?h(b,c)) & exists b.(?r(b,c) & ?k(?t,b)))))>] -> LASTG[CASE=gen,NUM=sg,SEM=?l] ASR[CASE=nom,GEN=?g,NUM=?s,SEM=?r] ATI[CASE=nom,GEN=?g,NUM=?s,SEM=?t] FIVEG[CASE=gen,NUM=?p,SEM=?f] SAHA[CASE=nom,GEN=?g,NUM=?s,SEM=?h] CA[SEM=?k] TAI[CASE=ins,NUM=?p]
NUMR[CASE=ins,CNT=3,NUM=pl] -> 'tribhiḥ'
NI[CASE=ins,CNT=3,NUM=pl,SEM=<\x.((x = nose) | (x = tongue) | (x = body))>] -> 'ghrāṇa-ādibhiḥ'
PRED[GEN=n,NUM=sg,SEM=<\x.EqualGrasp(x)>] -> 'tulya-viṣaya-grahaṇam'
MATA[EVID=held,GEN=n,NUM=sg] -> 'matam'
LASTG[CASE=gen,NUM=sg,SEM=<\x.MindCog(x)>] -> 'caramasya'
ASR[CASE=nom,GEN=m,NUM=sg,SEM=<\y z.BasisOf(y,z)>] -> 'āśrayaḥ'
ATI[CASE=nom,GEN=m,NUM=sg,SEM=<\y.Past(y)>] -> 'atītaḥ'
FIVEG[CASE=gen,CNT=5,NUM=pl,SEM=<\x.FiveCog(x)>] -> 'pañcānām'
SAHA[CASE=nom,GEN=m,NUM=sg,SEM=<\y z.Coborn(y,z)>] -> 'sahajaḥ'
CA[SEM=<\P y.P(y)>] -> 'ca'
TAI[CASE=ins,NUM=pl] -> 'taiḥ'
"""

dependency_text = r"""'matam' -> 'tulya-viṣaya-grahaṇam' | 'ghrāṇa-ādibhiḥ'
'ghrāṇa-ādibhiḥ' -> 'tribhiḥ'
'āśrayaḥ' -> 'caramasya' | 'atītaḥ'
'sahajaḥ' -> 'pañcānām' | 'taiḥ' | 'ca'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(((x = nose) | (x = tongue) | (x = body)) -> EqualGrasp(x)) & all c.(MindCog(c) -> exists b.(BasisOf(b,c) & Past(b))) & all c b.((MindCog(c) & BasisOf(b,c)) -> Past(b)) & all c.(FiveCog(c) -> (exists b.(BasisOf(b,c) & Coborn(b,c)) & exists b.(BasisOf(b,c) & Past(b)))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(((x = nose) | (x = tongue) | (x = body)) -> EqualGrasp(x)) & all c.(MindCog(c) -> exists b.(BasisOf(b,c) & Past(b))) & all c b.((MindCog(c) & BasisOf(b,c)) -> Past(b)) & all c.(FiveCog(c) -> (exists b.(BasisOf(b,c) & Coborn(b,c)) & exists b.(BasisOf(b,c) & Past(b)))))
# THEORY prose
# EXPECT prose T
all x.(Faculty(x) -> (EqualGrasp(x) <-> all g.(GraspBy(x,g) -> EqualAtoms(g))))
exists g.(GraspBy(eye,g) & FewerAtoms(g))
exists g.(GraspBy(eye,g) & EqualAtoms(g))
exists g.(GraspBy(eye,g) & MoreAtoms(g))
exists g.(GraspBy(ear,g) & FewerAtoms(g))
exists g.(GraspBy(ear,g) & MoreAtoms(g))
exists m.(m = mind & -Rupin(m) & all g.(GraspBy(m,g) -> -Measured(g)))
all x.all g.((Rupin(x) & GraspBy(x,g)) -> Accumulated(g))
all g.(GraspBy(body,g) -> -AllAtoms(g))
all c.all v.((FiveCog(c) & CondOf(v,c)) -> Present(v))
exists v.(CondOf(v,mc) & Past(v))
exists v.(CondOf(v,mc) & Present(v))
exists v.(CondOf(v,mc) & Future(v))
exists x.(x = eye & BasisOf(x,ec) & Coborn(x,ec) & -Samanantara(x,ec))
exists x.(x = pastmental & Samanantara(x,ec) & -BasisOf(x,ec))
exists x.(x = pastmind & BasisOf(x,ec) & Samanantara(x,ec))
exists x.(-BasisOf(x,ec) & -Samanantara(x,ec))
all x.(BasisOf(x,mc) -> Samanantara(x,mc))
exists x.(Samanantara(x,mc) & -BasisOf(x,mc))
# THEORY rejected_eye_equal
# EXPECT rejected_eye_equal F
all g.(GraspBy(eye,g) -> EqualAtoms(g))
# THEORY rejected_mind_measured
# EXPECT rejected_mind_measured F
exists g.(GraspBy(mind,g) & Measured(g))
# THEORY rejected_one_atom
# EXPECT rejected_one_atom F
exists x.exists g.(Rupin(x) & GraspBy(x,g) & -Accumulated(g))
# THEORY rejected_mind_coborn
# EXPECT rejected_mind_coborn F
exists c.exists b.(MindCog(c) & BasisOf(b,c) & Coborn(b,c))
# THEORY rejected_five_only_coborn
# EXPECT rejected_five_only_coborn F
all c.(FiveCog(c) -> all b.(BasisOf(b,c) -> Coborn(b,c)))
# THEORY rejected_five_past_condition
# EXPECT rejected_five_past_condition F
exists c.exists v.(FiveCog(c) & CondOf(v,c) & Past(v))
# THEORY rejected_basis_is_samanantara
# EXPECT rejected_basis_is_samanantara F
all x.all c.(BasisOf(x,c) <-> Samanantara(x,c))
"""

valuation_text = r"""nose => nose_f
tongue => tongue_f
body => body_f
eye => eye_f
ear => ear_f
mind => mind_f
pastmind => pastmind_b
pastmental => pastmental_d
mc => mano_c
ec => eye_c
Faculty => {eye_f,ear_f,nose_f,tongue_f,body_f,mind_f}
Rupin => {eye_f,ear_f,nose_f,tongue_f,body_f}
EqualGrasp => {nose_f,tongue_f,body_f}
GraspBy => {(eye_f,g_hair),(eye_f,g_grape),(eye_f,g_mountain),(ear_f,g_mosquito),(ear_f,g_thunder),(nose_f,g_nose),(tongue_f,g_tongue),(body_f,g_body),(mind_f,g_mind)}
EqualAtoms => {g_grape,g_nose,g_tongue,g_body}
FewerAtoms => {g_hair,g_mosquito}
MoreAtoms => {g_mountain,g_thunder}
Measured => {g_hair,g_grape,g_mountain,g_mosquito,g_thunder,g_nose,g_tongue,g_body}
Accumulated => {g_hair,g_grape,g_mountain,g_mosquito,g_thunder,g_nose,g_tongue,g_body}
AllAtoms => {}
MindCog => {mano_c}
FiveCog => {eye_c,body_c}
BasisOf => {(eye_f,eye_c),(pastmind_b,eye_c),(body_f,body_c),(pastmind_b,body_c),(pastmind_b,mano_c)}
Coborn => {(eye_f,eye_c),(body_f,body_c)}
Samanantara => {(pastmind_b,eye_c),(pastmental_d,eye_c),(pastmind_b,body_c),(pastmental_d,body_c),(pastmind_b,mano_c),(pastmental_d,mano_c)}
Past => {pastmind_b,pastmental_d,v_past}
Present => {v_present,v_touch}
Future => {v_future}
CondOf => {(v_present,eye_c),(v_touch,body_c),(v_past,mano_c),(v_present,mano_c),(v_future,mano_c)}
"""


In [ ]:
studies[44] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[44]["tree"].pformat())
print("Derived SEM:", studies[44]["sem"])


# VAK_1.45 — Named by the Basis

Source: `VAK_1.45.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
tadvikāravikāritvād āśrayāś cakṣurādayaḥ /
ato 'sādhāraṇatvād dhi vijñānaṃ tair nirucyate // 1.45 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| tad-vikāra-vikāritvāt | tatpuruṣa with abstract *-tva*; ablative singular | because of being altered by their alteration: ground of the Basis |
| āśrayāḥ | āśraya; nominative masculine plural | Bases |
| cakṣuḥ-ādayaḥ | bahuvrīhi; nominative masculine plural | the Eye and the rest |
| ataḥ | adverb from *idam*, ablative force | from this: from their being Bases |
| a-sādhāraṇatvāt | abstract noun; ablative singular | because of not being in common: ground of the name |
| hi | particle | for: the reason is stated |
| vijñānam | vijñāna; nominative neuter singular | Cognition |
| taiḥ | tad; instrumental plural | by them: the Eye and the rest |
| nirucyate | nir-√vac, present passive, third singular | is designated |

Two ablatives give two grounds. The first makes the Eye and the rest
Bases. The second, with *ataḥ*, takes that result and adds the ground of
the name. *Taiḥ* is plural because it points back to *cakṣuḥ-ādayaḥ*.

**Grammar tokens.** Verse-order IAST with sandhi opened
(`tadvikāravikāritvād āśrayāś cakṣurādayaḥ` → *tad-vikāra-vikāritvāt
āśrayāḥ cakṣuḥ-ādayaḥ*, `ato 'sādhāraṇatvād dhi` → *ataḥ a-sādhāraṇatvāt
hi*, `vijñānaṃ tair` → *vijñānam taiḥ*) and compound boundaries marked.

```text
# tokens: tad-vikāra-vikāritvāt āśrayāḥ cakṣuḥ-ādayaḥ ataḥ a-sādhāraṇatvāt hi vijñānam taiḥ nirucyate
```

## 5. Translation

### Literal Translation

> Because of being altered by their alteration, the Eye and the rest are
> Bases. Therefore, because they are not in common, Cognition is
> designated by them.

### Bhāṣya-informed study translation

> The Eye and the other Faculties are Bases, because Cognition is altered
> by their alteration. Therefore, because they are not shared, Cognition is
> designated by them.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| vikāra | alteration | benefit, impairment, acuity, dullness |
| āśraya | Basis | the Eye and the rest |
| ālambana | Support | the Visible Form and the rest; shared |
| asādhāraṇa | not in common | the eye is the Basis of its own Cognition only |
| nirucyate | is designated | the name follows the Basis |
| bherī-śabda | drum-sound | the Basis names the product |
| yava-aṅkura | barley-sprout | the same |

In [ ]:
fcfg_text = r"""% start S
# tokens: tad-vikāra-vikāritvāt āśrayāḥ cakṣuḥ-ādayaḥ ataḥ a-sādhāraṇatvāt hi vijñānam taiḥ nirucyate
S[SEM=<(?a & ?b)>] -> BASES[NUM=?q,SEM=?a] NAME[NUM=?q,SEM=?b]
BASES[NUM=?q,SEM=<all x.(?n(x) -> (?g(x) & ?b(x)))>] -> ABL[SEM=?g] ASR[CASE=nom,GEN=?k,NUM=?q,SEM=?b] NI[CASE=nom,GEN=?k,NUM=?q,SEM=?n]
NAME[NUM=?q,SEM=<all c.(?v(c) -> exists x.(?t(x) & ?a(x,c) & ?u(x) & ?r(c,x)))>] -> ATAH[SEM=?a] ABL2[SEM=?u] HI VIJ[CASE=nom,NUM=?s,SEM=?v] TAI[CASE=ins,NUM=?q,SEM=?t] V[VOICE=pass,NUM=?s,SEM=?r]
ABL[SEM=<\x.all p.(BasisOf(x,p) -> AltersWith(p,x))>] -> 'tad-vikāra-vikāritvāt'
ASR[CASE=nom,GEN=m,NUM=pl,SEM=<\x.exists k.BasisOf(x,k)>] -> 'āśrayāḥ'
NI[CASE=nom,GEN=m,NUM=pl,SEM=<\x.Faculty(x)>] -> 'cakṣuḥ-ādayaḥ'
ATAH[SEM=<\x y.BasisOf(x,y)>] -> 'ataḥ'
ABL2[SEM=<\x.all p.all q.((BasisOf(x,p) & BasisOf(x,q)) -> (p = q))>] -> 'a-sādhāraṇatvāt'
HI -> 'hi'
VIJ[CASE=nom,NUM=sg,SEM=<\c.Cognition(c)>] -> 'vijñānam'
TAI[CASE=ins,NUM=pl,SEM=<\x.Faculty(x)>] -> 'taiḥ'
V[VOICE=pass,NUM=sg,SEM=<\c x.NamedBy(c,x)>] -> 'nirucyate'
"""

dependency_text = r"""'āśrayāḥ' -> 'cakṣuḥ-ādayaḥ' | 'tad-vikāra-vikāritvāt'
'nirucyate' -> 'vijñānam' | 'taiḥ' | 'a-sādhāraṇatvāt' | 'ataḥ' | 'hi'
"""

sem_text = r"""# derived by the section 4 parse
(all x.(Faculty(x) -> (all p.(BasisOf(x,p) -> AltersWith(p,x)) & exists k.BasisOf(x,k))) & all c.(Cognition(c) -> exists x.(Faculty(x) & BasisOf(x,c) & all p q.((BasisOf(x,p) & BasisOf(x,q)) -> (p = q)) & NamedBy(c,x))))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.(Faculty(x) -> (all p.(BasisOf(x,p) -> AltersWith(p,x)) & exists k.BasisOf(x,k))) & all c.(Cognition(c) -> exists x.(Faculty(x) & BasisOf(x,c) & all p q.((BasisOf(x,p) & BasisOf(x,q)) -> (p = q)) & NamedBy(c,x))))
# THEORY prose
# EXPECT prose T
all c.(Cognition(c) -> (exists x.BasisOf(x,c) & exists v.SupportOf(v,c)))
exists v.(v = blue & SupportOf(v,ec1) & -AltersWith(ec1,v))
exists x.(x = eye1 & Impaired(x) & AltersWith(ec1,x))
exists v.(v = blue & SupportOf(v,ec1) & SupportOf(v,ec2) & SupportOf(v,mc1))
all c.all v.(SupportOf(v,c) -> -NamedBy(c,v))
exists s.(s = drum_sound & NamedBy(s,drum))
exists s.(s = barley_sprout & NamedBy(s,barley))
# THEORY rejected_form_is_basis
# EXPECT rejected_form_is_basis F
exists v.exists c.(VisibleForm(v) & BasisOf(v,c))
# THEORY rejected_follows_support
# EXPECT rejected_follows_support F
all c.all v.(SupportOf(v,c) -> AltersWith(c,v))
# THEORY rejected_basis_alone
# EXPECT rejected_basis_alone F
exists c.(Cognition(c) & -exists v.SupportOf(v,c))
# THEORY rejected_eye_shared
# EXPECT rejected_eye_shared F
exists x.exists c.exists d.(Faculty(x) & BasisOf(x,c) & BasisOf(x,d) & -(c = d))
# THEORY rejected_support_unshared
# EXPECT rejected_support_unshared F
all v.(VisibleForm(v) -> all p.all q.((SupportOf(v,p) & SupportOf(v,q)) -> (p = q)))
# THEORY rejected_named_by_support
# EXPECT rejected_named_by_support F
exists c.exists v.(SupportOf(v,c) & NamedBy(c,v))
"""

valuation_text = r"""blue => blue_v
eye1 => eye_a
ec1 => eyecog_a
ec2 => eyecog_b
mc1 => manocog_a
drum_sound => drum_sound_s
drum => drum_d
barley_sprout => sprout_s
barley => barley_d
Faculty => {eye_a,eye_b,body_a,mind_a}
VisibleForm => {blue_v}
Cognition => {eyecog_a,eyecog_b,bodycog_a,manocog_a}
BasisOf => {(eye_a,eyecog_a),(eye_b,eyecog_b),(body_a,bodycog_a),(mind_a,manocog_a)}
SupportOf => {(blue_v,eyecog_a),(blue_v,eyecog_b),(blue_v,manocog_a),(soft_v,bodycog_a)}
AltersWith => {(eyecog_a,eye_a),(eyecog_b,eye_b),(bodycog_a,body_a),(manocog_a,mind_a)}
Impaired => {eye_a}
NamedBy => {(eyecog_a,eye_a),(eyecog_b,eye_b),(bodycog_a,body_a),(manocog_a,mind_a),(drum_sound_s,drum_d),(sprout_s,barley_d)}
"""


In [ ]:
studies[45] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[45]["tree"].pformat())
print("Derived SEM:", studies[45]["sem"])


# VAK_1.46 — Not Lower, Not Higher

Source: `VAK_1.46.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
na kāyasyādharaṃ cakṣur ūrdhvaṃ rūpaṃ na cakṣuṣaḥ /
vijñānaṃ cāsya rūpaṃ tu kāyasyobhe ca sarvataḥ // 1.46 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| na | negative particle | not: the fixed rule is stated negatively |
| kāyasya | kāya; genitive singular | than the Body: standard of comparison |
| adharam | adhara; nominative neuter singular | lower: of a lower plane |
| cakṣuḥ | cakṣus; nominative neuter singular | the Eye |
| ūrdhvam | ūrdhva; nominative neuter singular | higher: of a higher plane |
| rūpam | rūpa; nominative neuter singular | Visible Form |
| cakṣuṣaḥ | cakṣus; genitive singular | than the Eye |
| vijñānam ca | vijñāna; nominative neuter singular, with particle | and Cognition: takes the same predicate |
| asya | idam; genitive singular | of this: of the Eye-Cognition just named |
| tu | particle | but: turns from the fixed rule to the open cases |
| ubhe ca | ubha; nominative neuter dual, with particle | and both: Visible Form and Cognition |
| sarvataḥ | adverb | in every way: higher, lower, or the same |

Higher and lower are planes (*bhūmi*), not height. The planes are the
Desire Realm and the four dhyānas. The genitive gives the standard of the
comparison.

**Grammar tokens.** Verse-order IAST with sandhi opened
(`kāyasyādharaṃ cakṣur ūrdhvaṃ` → *kāyasya adharam cakṣuḥ ūrdhvam*,
`vijñānaṃ cāsya` → *vijñānam ca asya*, `kāyasyobhe` → *kāyasya ubhe*).

```text
# tokens: na kāyasya adharam cakṣuḥ ūrdhvam rūpam na cakṣuṣaḥ vijñānam ca asya rūpam tu kāyasya ubhe ca sarvataḥ
```

## 5. Translation

### Literal Translation

> The Eye is not lower than the Body. A Visible Form is not higher than the
> Eye, and Cognition too. But relative to this, a Visible Form; and
> relative to the Body, both, in every way.

### Bhāṣya-informed study translation

> The Eye is never of a lower plane than the Body. A Visible Form is never
> of a higher plane than the Eye, nor is Eye-Cognition. Relative to that
> Cognition, however, a Visible Form is of a higher, lower, or the same
> plane; and relative to the Body, so are both.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| bhūmi | plane | the Desire Realm and the four dhyānas |
| adhara | lower | a lower plane, not a lower place |
| ūrdhva | higher | a higher plane |
| svabhūmika | of its own plane | the birth-plane, or the plane of the seer |
| kāma-dhātu | Desire Realm | the first of the five |
| dhyāna | dhyāna | four planes above the Desire Realm |
| sarvataḥ | in every way | higher, lower, or the same |
| niyama | fixed rule | the eye is not lower than the Body |

In [ ]:
fcfg_text = r"""% start S
# tokens: na kāyasya adharam cakṣuḥ ūrdhvam rūpam na cakṣuṣaḥ vijñānam ca asya rūpam tu kāyasya ubhe ca sarvataḥ
S[SEM=<(?a & (?b & ?c))>] -> CL1[SEM=?a] CL2[REF=?r,SEM=?b] OPEN[REF=?r,SEM=?c]
CL1[SEM=<all s.all x.all y.((?n(s,x) & ?g(s,y)) -> -?a(x,y))>] -> NEG GENT[SEM=?g] ADJ[GEN=?k,NUM=?q,SEM=?a] NOM[GEN=?k,NUM=?q,REF=eye,SEM=?n]
CL2[REF=?r,SEM=<(all s.all x.all y.((?n(s,x) & ?g(s,y)) -> -?a(x,y)) & all s.all x.all y.((?m(s,x) & ?g(s,y)) -> -?a(x,y)))>] -> ADJ[GEN=?k,NUM=?q,SEM=?a] NOM[GEN=?k,NUM=?q,REF=form,SEM=?n] NEG GENT[SEM=?g] NOM[GEN=?k,NUM=?q,REF=?r,SEM=?m] CA
OPEN[REF=?r,SEM=<(?o(?f,?h) & ?u(?o,?b))>] -> ASYA[REF=?r,SEM=?h] NOM[REF=form,SEM=?f] TU GENT[SEM=?b] UBHE[NUM=du,SEM=?u] CA SARV[SEM=?o]
NEG -> 'na'
CA -> 'ca'
TU -> 'tu'
GENT[SEM=<\s y.BodyIn(s,y)>] -> 'kāyasya'
GENT[SEM=<\s y.EyeIn(s,y)>] -> 'cakṣuṣaḥ'
ADJ[GEN=n,NUM=sg,SEM=<\x y.Below(x,y)>] -> 'adharam'
ADJ[GEN=n,NUM=sg,SEM=<\x y.Below(y,x)>] -> 'ūrdhvam'
NOM[GEN=n,NUM=sg,REF=eye,SEM=<\s x.EyeIn(s,x)>] -> 'cakṣuḥ'
NOM[GEN=n,NUM=sg,REF=form,SEM=<\s x.FormIn(s,x)>] -> 'rūpam'
NOM[GEN=n,NUM=sg,REF=cog,SEM=<\s x.CogIn(s,x)>] -> 'vijñānam'
ASYA[REF=cog,SEM=<\s y.CogIn(s,y)>] -> 'asya'
UBHE[NUM=du,SEM=<\O B.(O(\s x.FormIn(s,x),B) & O(\s x.CogIn(s,x),B))>] -> 'ubhe'
SARV[SEM=<\X Y.(exists s x y.(X(s,x) & Y(s,y) & Below(x,y)) & exists s x y.(X(s,x) & Y(s,y) & Below(y,x)) & exists s x y.(X(s,x) & Y(s,y) & Same(x,y)))>] -> 'sarvataḥ'
"""

dependency_text = r"""'adharam' -> 'cakṣuḥ' | 'kāyasya' | 'na'
'ūrdhvam' -> 'rūpam' | 'cakṣuṣaḥ' | 'na' | 'vijñānam'
'vijñānam' -> 'ca'
'sarvataḥ' -> 'rūpam' | 'asya' | 'ubhe' | 'tu'
'ubhe' -> 'kāyasya' | 'ca'
"""

sem_text = r"""# derived by the section 4 parse
(all s x y.((EyeIn(s,x) & BodyIn(s,y)) -> -Below(x,y)) & all s x y.((FormIn(s,x) & EyeIn(s,y)) -> -Below(y,x)) & all s x y.((CogIn(s,x) & EyeIn(s,y)) -> -Below(y,x)) & exists s x y.(FormIn(s,x) & CogIn(s,y) & Below(x,y)) & exists s x y.(FormIn(s,x) & CogIn(s,y) & Below(y,x)) & exists s x y.(FormIn(s,x) & CogIn(s,y) & Same(x,y)) & exists s x y.(FormIn(s,x) & BodyIn(s,y) & Below(x,y)) & exists s x y.(FormIn(s,x) & BodyIn(s,y) & Below(y,x)) & exists s x y.(FormIn(s,x) & BodyIn(s,y) & Same(x,y)) & exists s x y.(CogIn(s,x) & BodyIn(s,y) & Below(x,y)) & exists s x y.(CogIn(s,x) & BodyIn(s,y) & Below(y,x)) & exists s x y.(CogIn(s,x) & BodyIn(s,y) & Same(x,y)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all s x y.((EyeIn(s,x) & BodyIn(s,y)) -> -Below(x,y)) & all s x y.((FormIn(s,x) & EyeIn(s,y)) -> -Below(y,x)) & all s x y.((CogIn(s,x) & EyeIn(s,y)) -> -Below(y,x)) & exists s x y.(FormIn(s,x) & CogIn(s,y) & Below(x,y)) & exists s x y.(FormIn(s,x) & CogIn(s,y) & Below(y,x)) & exists s x y.(FormIn(s,x) & CogIn(s,y) & Same(x,y)) & exists s x y.(FormIn(s,x) & BodyIn(s,y) & Below(x,y)) & exists s x y.(FormIn(s,x) & BodyIn(s,y) & Below(y,x)) & exists s x y.(FormIn(s,x) & BodyIn(s,y) & Same(x,y)) & exists s x y.(CogIn(s,x) & BodyIn(s,y) & Below(x,y)) & exists s x y.(CogIn(s,x) & BodyIn(s,y) & Below(y,x)) & exists s x y.(CogIn(s,x) & BodyIn(s,y) & Same(x,y)))
# THEORY prose
# EXPECT prose T
all s.all c.(CogIn(s,c) -> (InPlane(c,kama) | InPlane(c,dh1)))
exists s.exists x.exists c.(EyeIn(s,x) & InPlane(x,dh2) & CogIn(s,c) & InPlane(c,dh1))
exists s.exists x.exists y.(EyeIn(s,x) & BodyIn(s,y) & Below(y,x))
exists s.exists x.(FormIn(s,x) & InPlane(x,dh4))
exists s.(BodyIn(s,kbody) & EyeIn(s,keye) & FormIn(s,kform) & CogIn(s,kcog) & InPlane(kcog,kama))
# THEORY rejected_one_plane
# EXPECT rejected_one_plane F
all s.all x.all y.((EyeIn(s,x) & CogIn(s,y)) -> Same(x,y))
# THEORY rejected_eye_below_body
# EXPECT rejected_eye_below_body F
exists s.exists x.exists y.(EyeIn(s,x) & BodyIn(s,y) & Below(x,y))
# THEORY rejected_higher_form_seen
# EXPECT rejected_higher_form_seen F
exists s.exists x.exists y.(FormIn(s,x) & EyeIn(s,y) & Below(y,x))
# THEORY rejected_cognition_above_eye
# EXPECT rejected_cognition_above_eye F
exists s.exists x.exists y.(CogIn(s,x) & EyeIn(s,y) & Below(y,x))
# THEORY rejected_cognition_in_second_dhyana
# EXPECT rejected_cognition_in_second_dhyana F
exists s.exists c.(CogIn(s,c) & InPlane(c,dh2))
# THEORY rejected_form_never_above_cognition
# EXPECT rejected_form_never_above_cognition F
all s.all x.all y.((FormIn(s,x) & CogIn(s,y)) -> -Below(y,x))
"""

valuation_text = r"""kama => kama_p
dh1 => dh1_p
dh2 => dh2_p
dh3 => dh3_p
dh4 => dh4_p
kbody => body0
keye => eye0
kform => form0
kcog => cog0
BodyIn => {(s1,body0),(s3,body0),(s4,body0),(s5,body0),(s6,body1),(s7,body1),(s8,body2),(s9,body0)}
EyeIn => {(s1,eye0),(s3,eye2),(s4,eye2),(s5,eye2),(s6,eye1),(s7,eye1),(s8,eye2),(s9,eye4)}
FormIn => {(s1,form0),(s3,form0),(s4,form1),(s5,form2),(s6,form0),(s7,form1),(s8,form2),(s9,form4)}
CogIn => {(s1,cog0),(s3,cog1),(s4,cog1),(s5,cog1),(s6,cog1),(s7,cog1),(s8,cog1),(s9,cog1)}
InPlane => {(body0,kama_p),(eye0,kama_p),(form0,kama_p),(cog0,kama_p),(eye2,dh2_p),(cog1,dh1_p),(form1,dh1_p),(form2,dh2_p),(body1,dh1_p),(eye1,dh1_p),(body2,dh2_p),(eye4,dh4_p),(form4,dh4_p)}
Below => {(body0,eye2),(body0,cog1),(body0,form1),(body0,form2),(body0,body1),(body0,eye1),(body0,body2),(body0,eye4),(body0,form4),(eye0,eye2),(eye0,cog1),(eye0,form1),(eye0,form2),(eye0,body1),(eye0,eye1),(eye0,body2),(eye0,eye4),(eye0,form4),(form0,eye2),(form0,cog1),(form0,form1),(form0,form2),(form0,body1),(form0,eye1),(form0,body2),(form0,eye4),(form0,form4),(cog0,eye2),(cog0,cog1),(cog0,form1),(cog0,form2),(cog0,body1),(cog0,eye1),(cog0,body2),(cog0,eye4),(cog0,form4),(eye2,eye4),(eye2,form4),(cog1,eye2),(cog1,form2),(cog1,body2),(cog1,eye4),(cog1,form4),(form1,eye2),(form1,form2),(form1,body2),(form1,eye4),(form1,form4),(form2,eye4),(form2,form4),(body1,eye2),(body1,form2),(body1,body2),(body1,eye4),(body1,form4),(eye1,eye2),(eye1,form2),(eye1,body2),(eye1,eye4),(eye1,form4),(body2,eye4),(body2,form4)}
Same => {(body0,body0),(body0,eye0),(body0,form0),(body0,cog0),(eye0,body0),(eye0,eye0),(eye0,form0),(eye0,cog0),(form0,body0),(form0,eye0),(form0,form0),(form0,cog0),(cog0,body0),(cog0,eye0),(cog0,form0),(cog0,cog0),(eye2,eye2),(eye2,form2),(eye2,body2),(cog1,cog1),(cog1,form1),(cog1,body1),(cog1,eye1),(form1,cog1),(form1,form1),(form1,body1),(form1,eye1),(form2,eye2),(form2,form2),(form2,body2),(body1,cog1),(body1,form1),(body1,body1),(body1,eye1),(eye1,cog1),(eye1,form1),(eye1,body1),(eye1,eye1),(body2,eye2),(body2,form2),(body2,body2),(eye4,eye4),(eye4,form4),(form4,eye4),(form4,form4)}
"""


In [ ]:
studies[46] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[46]["tree"].pformat())
print("Derived SEM:", studies[46]["sem"])


# VAK_1.47 — The Ear, the Three, and Mind

Source: `VAK_1.47.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
tathā śrotraṃ trayāṇāṃ tu sarvam eva svabhūmikam /
kāyavijñānam adharasvabhūmy aniyataṃ manaḥ // 1.47 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| tathā | adverb | likewise: the Eye's rule of 1.46, applied again |
| śrotram | śrotra; nominative neuter singular | the Ear |
| trayāṇām | tri; genitive plural | of the three: Nose, Tongue, Body |
| tu | particle | however: turns from the Ear to the three |
| sarvam | sarva; nominative neuter singular | everything: Faculty, Condition, and Cognition |
| eva | particle | only: no other plane |
| sva-bhūmikam | bahuvrīhi; nominative neuter singular | of its own plane: the plane of the body one is born in |
| kāya-vijñānam | tatpuruṣa; nominative neuter singular | Body-Cognition |
| adhara-sva-bhūmi | bahuvrīhi; nominative neuter singular | of a lower plane or its own |
| a-niyatam | past participle, negated; nominative neuter singular | not fixed |
| manaḥ | manas; nominative neuter singular | Mind |

The three are Nose, Tongue, and Body. The printed *adharasavabhūmi* is read
as *adhara-sva-bhūmi*. Every nominative is neuter singular, so agreement
does not separate the clauses; the particles and the order do.

**Grammar tokens.** Verse-order IAST with sandhi opened (`śrotraṃ` →
*śrotram*, `trayāṇāṃ` → *trayāṇām*, `adharasvabhūmy aniyataṃ` →
*adhara-sva-bhūmi a-niyatam*) and compound boundaries marked.

```text
# tokens: tathā śrotram trayāṇām tu sarvam eva sva-bhūmikam kāya-vijñānam adhara-sva-bhūmi a-niyatam manaḥ
```

## 5. Translation

### Literal Translation

> Likewise the Ear. Of the three, however, everything is of its own plane.
> Body-Cognition is of a lower plane or its own. Mind is not fixed.

### Bhāṣya-informed study translation

> The Ear follows the rule of the Eye. For the three beginning with the
> Nose, however, Faculty, Condition, and Cognition are all of their own
> plane, except that Body-Cognition is of a lower plane or its own. Mind is
> not fixed.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| tathā | likewise | the Ear follows the Eye |
| trayāṇām | of the three | Nose, Tongue, Body |
| svabhūmika | of its own plane | the general rule for the three |
| utsarga | general rule | everything of the three on its own plane |
| apavāda | exception | Body-Cognition |
| adhara-svabhūmi | lower or its own | not a higher plane |
| aniyata | not fixed | Mind |
| samāpatti | attainment | worked in its own chapter |

In [ ]:
fcfg_text = r"""% start S
# tokens: tathā śrotram trayāṇām tu sarvam eva sva-bhūmikam kāya-vijñānam adhara-sva-bhūmi a-niyatam manaḥ
S[SEM=<(?a & (?b & ?c))>] -> EAR[SEM=?a] THREE[SEM=?b] MIND[SEM=?c]
EAR[SEM=<?t(?h)>] -> TATHA[SEM=?t] SENSE[CASE=nom,SEM=?h]
THREE[SEM=<(all s x.((?g(s) & ?a(s,x) & -?k(s,x)) -> ?p(s,x)) & all s x.(?k(s,x) -> ?q(s,x)))>] -> GENP[CASE=gen,CNT=3,NUM=pl,SEM=?g] TU NOMN[GEN=?n,NUM=?m,SEM=?a] EVA PRED[GEN=?n,NUM=?m,SEM=?p] NOMN[GEN=?n,NUM=?m,SEM=?k] PRED[GEN=?n,NUM=?m,SEM=?q]
MIND[SEM=<?o(?f)>] -> ANI[GEN=?n,NUM=?m,SEM=?o] NOMN[GEN=?n,NUM=?m,SEM=?f]
TATHA[SEM=<\H.all s.(H(s) -> (all x y.((FacIn(s,x) & BodyIn(s,y)) -> -Below(x,y)) & all x y.((CondIn(s,x) & FacIn(s,y)) -> -Below(y,x)) & all x y.((CogIn(s,x) & FacIn(s,y)) -> -Below(y,x))))>] -> 'tathā'
SENSE[CASE=nom,SEM=<\s.Hearing(s)>] -> 'śrotram'
GENP[CASE=gen,CNT=3,NUM=pl,SEM=<\s.(Smelling(s) | Tasting(s) | Touching(s))>] -> 'trayāṇām'
TU -> 'tu'
EVA -> 'eva'
NOMN[GEN=n,NUM=sg,SEM=<\s x.(FacIn(s,x) | CondIn(s,x) | CogIn(s,x))>] -> 'sarvam'
PRED[GEN=n,NUM=sg,SEM=<\s x.exists b.(BodyIn(s,b) & Same(x,b))>] -> 'sva-bhūmikam'
NOMN[GEN=n,NUM=sg,SEM=<\s x.(Touching(s) & CogIn(s,x))>] -> 'kāya-vijñānam'
PRED[GEN=n,NUM=sg,SEM=<\s x.exists b.(BodyIn(s,b) & (Same(x,b) | Below(x,b)))>] -> 'adhara-sva-bhūmi'
ANI[GEN=n,NUM=sg,SEM=<\M.(exists s x y.(M(s,x) & BodyIn(s,y) & Same(x,y)) & exists s x y.(M(s,x) & BodyIn(s,y) & Below(y,x)) & exists s x y.(M(s,x) & BodyIn(s,y) & Below(x,y)))>] -> 'a-niyatam'
NOMN[GEN=n,NUM=sg,SEM=<\s x.(Thinking(s) & FacIn(s,x))>] -> 'manaḥ'
"""

dependency_text = r"""'tathā' -> 'śrotram'
'sva-bhūmikam' -> 'sarvam' | 'trayāṇām' | 'tu' | 'eva'
'adhara-sva-bhūmi' -> 'kāya-vijñānam'
'a-niyatam' -> 'manaḥ'
"""

sem_text = r"""# derived by the section 4 parse
(all s.(Hearing(s) -> (all x y.((FacIn(s,x) & BodyIn(s,y)) -> -Below(x,y)) & all x y.((CondIn(s,x) & FacIn(s,y)) -> -Below(y,x)) & all x y.((CogIn(s,x) & FacIn(s,y)) -> -Below(y,x)))) & all s x.(((Smelling(s) | Tasting(s) | Touching(s)) & (FacIn(s,x) | CondIn(s,x) | CogIn(s,x)) & -(Touching(s) & CogIn(s,x))) -> exists b.(BodyIn(s,b) & Same(x,b))) & all s x.((Touching(s) & CogIn(s,x)) -> exists b.(BodyIn(s,b) & (Same(x,b) | Below(x,b)))) & exists s x y.(Thinking(s) & FacIn(s,x) & BodyIn(s,y) & Same(x,y)) & exists s x y.(Thinking(s) & FacIn(s,x) & BodyIn(s,y) & Below(y,x)) & exists s x y.(Thinking(s) & FacIn(s,x) & BodyIn(s,y) & Below(x,y)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all s.(Hearing(s) -> (all x y.((FacIn(s,x) & BodyIn(s,y)) -> -Below(x,y)) & all x y.((CondIn(s,x) & FacIn(s,y)) -> -Below(y,x)) & all x y.((CogIn(s,x) & FacIn(s,y)) -> -Below(y,x)))) & all s x.(((Smelling(s) | Tasting(s) | Touching(s)) & (FacIn(s,x) | CondIn(s,x) | CogIn(s,x)) & -(Touching(s) & CogIn(s,x))) -> exists b.(BodyIn(s,b) & Same(x,b))) & all s x.((Touching(s) & CogIn(s,x)) -> exists b.(BodyIn(s,b) & (Same(x,b) | Below(x,b)))) & exists s x y.(Thinking(s) & FacIn(s,x) & BodyIn(s,y) & Same(x,y)) & exists s x y.(Thinking(s) & FacIn(s,x) & BodyIn(s,y) & Below(y,x)) & exists s x y.(Thinking(s) & FacIn(s,x) & BodyIn(s,y) & Below(x,y)))
# THEORY prose
# EXPECT prose T
all s.all x.((Touching(s) & (FacIn(s,x) | CondIn(s,x))) -> exists b.(BodyIn(s,b) & Same(x,b)))
exists s.exists c.exists b.(Touching(s) & CogIn(s,c) & BodyIn(s,b) & Below(c,b) & InPlane(b,dh2))
all s.all c.all b.((Touching(s) & CogIn(s,c) & BodyIn(s,b) & (InPlane(b,kama) | InPlane(b,dh1))) -> Same(c,b))
exists s.exists x.exists b.(Hearing(s) & FacIn(s,x) & BodyIn(s,b) & Below(b,x))
exists s.exists x.exists c.(Hearing(s) & FacIn(s,x) & InPlane(x,dh2) & CogIn(s,c) & InPlane(c,dh1))
# THEORY rejected_body_cognition_own_only
# EXPECT rejected_body_cognition_own_only F
all s.all c.all b.((Touching(s) & CogIn(s,c) & BodyIn(s,b)) -> Same(c,b))
# THEORY rejected_body_cognition_higher
# EXPECT rejected_body_cognition_higher F
exists s.exists c.exists b.(Touching(s) & CogIn(s,c) & BodyIn(s,b) & Below(b,c))
# THEORY rejected_tangible_lowered
# EXPECT rejected_tangible_lowered F
exists s.exists x.exists b.(Touching(s) & CondIn(s,x) & BodyIn(s,b) & -Same(x,b))
# THEORY rejected_odor_other_plane
# EXPECT rejected_odor_other_plane F
exists s.exists x.exists b.(Smelling(s) & (FacIn(s,x) | CondIn(s,x) | CogIn(s,x)) & BodyIn(s,b) & -Same(x,b))
# THEORY rejected_ear_below_body
# EXPECT rejected_ear_below_body F
exists s.exists x.exists b.(Hearing(s) & FacIn(s,x) & BodyIn(s,b) & Below(x,b))
# THEORY rejected_mind_fixed
# EXPECT rejected_mind_fixed F
all s.all x.all b.((Thinking(s) & FacIn(s,x) & BodyIn(s,b)) -> Same(x,b))
"""

valuation_text = r"""kama => kama_p
dh1 => dh1_p
dh2 => dh2_p
dh3 => dh3_p
dh4 => dh4_p
Hearing => {h1,h2,h3}
Smelling => {m1}
Tasting => {t1}
Touching => {k1,k2,k3}
Thinking => {n1,n2,n3}
BodyIn => {(h1,body0),(h2,body0),(h3,body0),(m1,body0),(t1,body0),(k1,body0),(k2,body1),(k3,body2),(n1,body0),(n2,body0),(n3,body2)}
FacIn => {(h1,ear0),(h2,ear1),(h3,ear2),(m1,nose0),(t1,tongue0),(k1,body0),(k2,body1),(k3,body2),(n1,mind0),(n2,mind1),(n3,mind1)}
CondIn => {(h1,sound0),(h2,sound1),(h3,sound0),(m1,odor0),(t1,taste0),(k1,tang0),(k2,tang1),(k3,tang2)}
CogIn => {(h1,cog0),(h2,cog1),(h3,cog1),(m1,cog0),(t1,cog0),(k1,cog0),(k2,cog1),(k3,cog1)}
InPlane => {(body0,kama_p),(ear0,kama_p),(sound0,kama_p),(cog0,kama_p),(ear1,dh1_p),(sound1,dh1_p),(cog1,dh1_p),(ear2,dh2_p),(nose0,kama_p),(odor0,kama_p),(tongue0,kama_p),(taste0,kama_p),(tang0,kama_p),(body1,dh1_p),(tang1,dh1_p),(body2,dh2_p),(tang2,dh2_p),(mind0,kama_p),(mind1,dh1_p)}
Below => {(body0,ear1),(body0,sound1),(body0,cog1),(body0,ear2),(body0,body1),(body0,tang1),(body0,body2),(body0,tang2),(body0,mind1),(ear0,ear1),(ear0,sound1),(ear0,cog1),(ear0,ear2),(ear0,body1),(ear0,tang1),(ear0,body2),(ear0,tang2),(ear0,mind1),(sound0,ear1),(sound0,sound1),(sound0,cog1),(sound0,ear2),(sound0,body1),(sound0,tang1),(sound0,body2),(sound0,tang2),(sound0,mind1),(cog0,ear1),(cog0,sound1),(cog0,cog1),(cog0,ear2),(cog0,body1),(cog0,tang1),(cog0,body2),(cog0,tang2),(cog0,mind1),(ear1,ear2),(ear1,body2),(ear1,tang2),(sound1,ear2),(sound1,body2),(sound1,tang2),(cog1,ear2),(cog1,body2),(cog1,tang2),(nose0,ear1),(nose0,sound1),(nose0,cog1),(nose0,ear2),(nose0,body1),(nose0,tang1),(nose0,body2),(nose0,tang2),(nose0,mind1),(odor0,ear1),(odor0,sound1),(odor0,cog1),(odor0,ear2),(odor0,body1),(odor0,tang1),(odor0,body2),(odor0,tang2),(odor0,mind1),(tongue0,ear1),(tongue0,sound1),(tongue0,cog1),(tongue0,ear2),(tongue0,body1),(tongue0,tang1),(tongue0,body2),(tongue0,tang2),(tongue0,mind1),(taste0,ear1),(taste0,sound1),(taste0,cog1),(taste0,ear2),(taste0,body1),(taste0,tang1),(taste0,body2),(taste0,tang2),(taste0,mind1),(tang0,ear1),(tang0,sound1),(tang0,cog1),(tang0,ear2),(tang0,body1),(tang0,tang1),(tang0,body2),(tang0,tang2),(tang0,mind1),(body1,ear2),(body1,body2),(body1,tang2),(tang1,ear2),(tang1,body2),(tang1,tang2),(mind0,ear1),(mind0,sound1),(mind0,cog1),(mind0,ear2),(mind0,body1),(mind0,tang1),(mind0,body2),(mind0,tang2),(mind0,mind1),(mind1,ear2),(mind1,body2),(mind1,tang2)}
Same => {(body0,body0),(body0,ear0),(body0,sound0),(body0,cog0),(body0,nose0),(body0,odor0),(body0,tongue0),(body0,taste0),(body0,tang0),(body0,mind0),(ear0,body0),(ear0,ear0),(ear0,sound0),(ear0,cog0),(ear0,nose0),(ear0,odor0),(ear0,tongue0),(ear0,taste0),(ear0,tang0),(ear0,mind0),(sound0,body0),(sound0,ear0),(sound0,sound0),(sound0,cog0),(sound0,nose0),(sound0,odor0),(sound0,tongue0),(sound0,taste0),(sound0,tang0),(sound0,mind0),(cog0,body0),(cog0,ear0),(cog0,sound0),(cog0,cog0),(cog0,nose0),(cog0,odor0),(cog0,tongue0),(cog0,taste0),(cog0,tang0),(cog0,mind0),(ear1,ear1),(ear1,sound1),(ear1,cog1),(ear1,body1),(ear1,tang1),(ear1,mind1),(sound1,ear1),(sound1,sound1),(sound1,cog1),(sound1,body1),(sound1,tang1),(sound1,mind1),(cog1,ear1),(cog1,sound1),(cog1,cog1),(cog1,body1),(cog1,tang1),(cog1,mind1),(ear2,ear2),(ear2,body2),(ear2,tang2),(nose0,body0),(nose0,ear0),(nose0,sound0),(nose0,cog0),(nose0,nose0),(nose0,odor0),(nose0,tongue0),(nose0,taste0),(nose0,tang0),(nose0,mind0),(odor0,body0),(odor0,ear0),(odor0,sound0),(odor0,cog0),(odor0,nose0),(odor0,odor0),(odor0,tongue0),(odor0,taste0),(odor0,tang0),(odor0,mind0),(tongue0,body0),(tongue0,ear0),(tongue0,sound0),(tongue0,cog0),(tongue0,nose0),(tongue0,odor0),(tongue0,tongue0),(tongue0,taste0),(tongue0,tang0),(tongue0,mind0),(taste0,body0),(taste0,ear0),(taste0,sound0),(taste0,cog0),(taste0,nose0),(taste0,odor0),(taste0,tongue0),(taste0,taste0),(taste0,tang0),(taste0,mind0),(tang0,body0),(tang0,ear0),(tang0,sound0),(tang0,cog0),(tang0,nose0),(tang0,odor0),(tang0,tongue0),(tang0,taste0),(tang0,tang0),(tang0,mind0),(body1,ear1),(body1,sound1),(body1,cog1),(body1,body1),(body1,tang1),(body1,mind1),(tang1,ear1),(tang1,sound1),(tang1,cog1),(tang1,body1),(tang1,tang1),(tang1,mind1),(body2,ear2),(body2,body2),(body2,tang2),(tang2,ear2),(tang2,body2),(tang2,tang2),(mind0,body0),(mind0,ear0),(mind0,sound0),(mind0,cog0),(mind0,nose0),(mind0,odor0),(mind0,tongue0),(mind0,taste0),(mind0,tang0),(mind0,mind0),(mind1,ear1),(mind1,sound1),(mind1,cog1),(mind1,body1),(mind1,tang1),(mind1,mind1)}
"""


In [ ]:
studies[47] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[47]["tree"].pformat())
print("Derived SEM:", studies[47]["sem"])


# VAK_1.48 — Two Cognitions, the Permanent, the Faculty

Source: `VAK_1.48.md` in the Organon Dhātu chapter.

## 2. Sanskrit (IAST)

```text
pañca bāhyā dvivijñeyā nityā dharmā asaṃskṛtāḥ /
dharmārdham indriyaṃ ye ca dvādaśādhyātmikāḥ smṛtāḥ // 1.48 //
```

## 3. Lexical Analysis

| Pada | Morphology | Force in this passage |
|---|---|---|
| pañca | pañcan; nominative | five |
| bāhyāḥ | bāhya; nominative masculine plural | external: the six outer of 1.39 less the Dharma Principle |
| dvi-vijñeyāḥ | dvigu with gerundive; nominative masculine plural | knowable by two Cognitions, and by no third |
| nityāḥ | nitya; nominative masculine plural | permanent: the predicate, placed first |
| dharmāḥ | dharma; nominative masculine plural | dharmas |
| a-saṃskṛtāḥ | past participle, negated; nominative masculine plural | unconditioned |
| dharma-ardham | tatpuruṣa; nominative neuter singular | a portion of the Dharma Principle |
| indriyam | indriya; nominative neuter singular | Faculty: predicate noun, keeps its own number |
| ye | yad; nominative masculine plural | those which |
| ca | particle | and: adds the twelve to the subject of *indriyam* |
| dvādaśa | dvādaśan; nominative | twelve |
| ādhyātmikāḥ | ādhyātmika; nominative masculine plural | internal: the twelve of 1.39 |
| smṛtāḥ | √smṛ, past passive participle; nominative masculine plural | taught: handed down as such |

The printed *divijñeyāḥ* is read as *dvi-vijñeyāḥ*. A portion (*ardha*) is
not a count of half. *Indriyam* stays neuter singular although one of its
subjects is plural: it names the class, not the members.

**Grammar tokens.** Verse-order IAST with sandhi opened (`bāhyā
dvivijñeyā nityā dharmā` → *bāhyāḥ dvi-vijñeyāḥ nityāḥ dharmāḥ*,
`dharmārdham` → *dharma-ardham*, `dvādaśādhyātmikāḥ` → *dvādaśa
ādhyātmikāḥ*) and compound boundaries marked.

```text
# tokens: pañca bāhyāḥ dvi-vijñeyāḥ nityāḥ dharmāḥ a-saṃskṛtāḥ dharma-ardham indriyam ye ca dvādaśa ādhyātmikāḥ smṛtāḥ
```

## 5. Translation

### Literal Translation

> The five external are knowable by two. Permanent are the unconditioned
> dharmas. A portion of the Dharma is Faculty, and the twelve taught as
> internal.

### Bhāṣya-informed study translation

> The five external Principles are each knowable by two Cognitions.
> Unconditioned dharmas are permanent. A portion of the Dharma Principle is
> Faculty, and so are the twelve Principles taught as internal.

## 7. Technical Vocabulary

| Sanskrit | Rendering | Note |
|---|---|---|
| bāhya | external | the five Conditions of the senses |
| dvi-vijñeya | knowable by two | own Cognition and Mind-Cognition |
| anubhūta | experienced | by the Cognition of its own sense |
| nitya | permanent | a portion only |
| asaṃskṛta | unconditioned | the permanent portion |
| dharma-ardha | a portion of the Dharma | not a count of half |
| adhyātmika | internal | the twelve |
| indriya | Faculty | the twenty-two of the sūtra |
| sālambana | takes a Support | why Mind is recited after life |

In [ ]:
fcfg_text = r"""% start S
# tokens: pañca bāhyāḥ dvi-vijñeyāḥ nityāḥ dharmāḥ a-saṃskṛtāḥ dharma-ardham indriyam ye ca dvādaśa ādhyātmikāḥ smṛtāḥ
S[SEM=<(?a & (?b & ?c))>] -> KNOWN[SEM=?a] PERM[SEM=?b] FAC[SEM=?c]
KNOWN[SEM=<all x.(?n(x) -> ?p(x))>] -> NUMR[CNT=?k] ADJ[CNT=?k,GEN=?g,NUM=?m,SEM=?n] KPRED[GEN=?g,NUM=?m,SEM=?p]
PERM[SEM=<all x.((?n(x) & ?a(x)) -> ?p(x))>] -> PPRED[GEN=?g,NUM=?m,SEM=?p] NOUN[GEN=?g,NUM=?m,SEM=?n] PART[GEN=?g,NUM=?m,SEM=?a]
FAC[SEM=<(?q(?f) & all x.(?r(x) -> ?f(x)))>] -> PORT[GEN=?g,NUM=?s,SEM=?q] FPRED[GEN=?g,NUM=?s,SEM=?f] REL[NUM=?m] CA NUMR[CNT=?k] ADJ[CNT=?k,GEN=m,NUM=?m,SEM=?r] SMR[EVID=taught,NUM=?m]
NUMR[CNT=5] -> 'pañca'
NUMR[CNT=12] -> 'dvādaśa'
ADJ[CNT=5,GEN=m,NUM=pl,SEM=<\x.(Outer(x) & -(x = dharma))>] -> 'bāhyāḥ'
ADJ[CNT=12,GEN=m,NUM=pl,SEM=<\x.Inner(x)>] -> 'ādhyātmikāḥ'
KPRED[GEN=m,NUM=pl,SEM=<\x.exists c.exists d.(Knows(c,x) & Knows(d,x) & -(c = d) & all k.(Knows(k,x) -> ((k = c) | (k = d))))>] -> 'dvi-vijñeyāḥ'
PPRED[GEN=m,NUM=pl,SEM=<\x.Permanent(x)>] -> 'nityāḥ'
NOUN[GEN=m,NUM=pl,SEM=<\x.InstanceOf(x,dharma)>] -> 'dharmāḥ'
PART[GEN=m,NUM=pl,SEM=<\x.Unconditioned(x)>] -> 'a-saṃskṛtāḥ'
PORT[GEN=n,NUM=sg,SEM=<\P.(exists x.(InstanceOf(x,dharma) & P(x)) & exists x.(InstanceOf(x,dharma) & -P(x)))>] -> 'dharma-ardham'
FPRED[GEN=n,NUM=sg,SEM=<\x.Faculty(x)>] -> 'indriyam'
REL[NUM=pl] -> 'ye'
CA -> 'ca'
SMR[EVID=taught,NUM=pl] -> 'smṛtāḥ'
"""

dependency_text = r"""'dvi-vijñeyāḥ' -> 'bāhyāḥ'
'bāhyāḥ' -> 'pañca'
'nityāḥ' -> 'dharmāḥ'
'dharmāḥ' -> 'a-saṃskṛtāḥ'
'indriyam' -> 'dharma-ardham' | 'ye' | 'ca'
'ye' -> 'smṛtāḥ'
'smṛtāḥ' -> 'ādhyātmikāḥ'
'ādhyātmikāḥ' -> 'dvādaśa'
"""

sem_text = r"""# derived by the section 4 parse
(all x.((Outer(x) & -(x = dharma)) -> exists c d.(Knows(c,x) & Knows(d,x) & -(c = d) & all k.(Knows(k,x) -> ((k = c) | (k = d))))) & all x.((InstanceOf(x,dharma) & Unconditioned(x)) -> Permanent(x)) & exists x.(InstanceOf(x,dharma) & Faculty(x)) & exists x.(InstanceOf(x,dharma) & -Faculty(x)) & all x.(Inner(x) -> Faculty(x)))
"""

fol_text = r"""# THEORY verse
# EXPECT verse T
(all x.((Outer(x) & -(x = dharma)) -> exists c d.(Knows(c,x) & Knows(d,x) & -(c = d) & all k.(Knows(k,x) -> ((k = c) | (k = d))))) & all x.((InstanceOf(x,dharma) & Unconditioned(x)) -> Permanent(x)) & exists x.(InstanceOf(x,dharma) & Faculty(x)) & exists x.(InstanceOf(x,dharma) & -Faculty(x)) & all x.(Inner(x) -> Faculty(x)))
# THEORY prose
# EXPECT prose T
all x.((Principle(x) & -(Outer(x) & -(x = dharma))) -> all k.(Knows(k,x) -> (k = mcog)))
all x.(Principle(x) -> Knows(mcog,x))
all x.(Permanent(x) -> (Unconditioned(x) & InstanceOf(x,dharma)))
all p.(Principle(p) -> -Permanent(p))
all x.((Outer(x) & -(x = dharma)) -> -Faculty(x))
exists f.(f = female & InstanceOf(f,body) & Faculty(f))
exists m.(m = mind & Faculty(m) & TakesSupport(m))
exists l.(l = life & Faculty(l) & InstanceOf(l,dharma) & -TakesSupport(l))
# THEORY rejected_dharma_known_by_two
# EXPECT rejected_dharma_known_by_two F
exists c.exists d.(Knows(c,dharma) & Knows(d,dharma) & -(c = d))
# THEORY rejected_principle_permanent
# EXPECT rejected_principle_permanent F
exists p.(p = dharma & Permanent(p))
# THEORY rejected_conditioned_permanent
# EXPECT rejected_conditioned_permanent F
exists x.(Permanent(x) & -Unconditioned(x))
# THEORY rejected_all_dharma_faculty
# EXPECT rejected_all_dharma_faculty F
all x.(InstanceOf(x,dharma) -> Faculty(x))
# THEORY rejected_no_dharma_faculty
# EXPECT rejected_no_dharma_faculty F
all x.(InstanceOf(x,dharma) -> -Faculty(x))
# THEORY rejected_external_faculty
# EXPECT rejected_external_faculty F
exists x.(Outer(x) & -(x = dharma) & Faculty(x))
"""

valuation_text = r"""dharma => dharma_d
body => body_d
mind => mind_d
mcog => mcog_d
female => female_f
life => life_f
Principle => {eye_d,ear_d,nose_d,tongue_d,body_d,mind_d,form_d,sound_d,odor_d,taste_d,tangible_d,dharma_d,ecog_d,scog_d,gcog_d,jcog_d,kcog_d,mcog_d}
Inner => {eye_d,ear_d,nose_d,tongue_d,body_d,mind_d,ecog_d,scog_d,gcog_d,jcog_d,kcog_d,mcog_d}
Outer => {form_d,sound_d,odor_d,taste_d,tangible_d,dharma_d}
Knows => {(ecog_d,form_d),(scog_d,sound_d),(gcog_d,odor_d),(jcog_d,taste_d),(kcog_d,tangible_d),(mcog_d,eye_d),(mcog_d,ear_d),(mcog_d,nose_d),(mcog_d,tongue_d),(mcog_d,body_d),(mcog_d,mind_d),(mcog_d,form_d),(mcog_d,sound_d),(mcog_d,odor_d),(mcog_d,taste_d),(mcog_d,tangible_d),(mcog_d,dharma_d),(mcog_d,ecog_d),(mcog_d,scog_d),(mcog_d,gcog_d),(mcog_d,jcog_d),(mcog_d,kcog_d),(mcog_d,mcog_d)}
InstanceOf => {(space_u,dharma_d),(pratisamkhya_u,dharma_d),(apratisamkhya_u,dharma_d),(life_f,dharma_d),(sukha_f,dharma_d),(sraddha_f,dharma_d),(samjna_x,dharma_d),(female_f,body_d),(male_f,body_d)}
Unconditioned => {space_u,pratisamkhya_u,apratisamkhya_u}
Permanent => {space_u,pratisamkhya_u,apratisamkhya_u}
Faculty => {eye_d,ear_d,nose_d,tongue_d,body_d,mind_d,ecog_d,scog_d,gcog_d,jcog_d,kcog_d,mcog_d,life_f,sukha_f,sraddha_f,female_f,male_f}
TakesSupport => {mind_d,ecog_d,scog_d,gcog_d,jcog_d,kcog_d,mcog_d}
"""


In [ ]:
studies[48] = run_study(
    fcfg_text, dependency_text, sem_text, fol_text, valuation_text
)
print(studies[48]["tree"].pformat())
print("Derived SEM:", studies[48]["sem"])


## Countermodel playground: 1.48

Copy the valuation, then give Sound a third Cognition that knows it. The derived verse becomes false. Unary predicate extensions use 1-tuples, e.g. `{('space_u',)}`; binary relations use pairs. Copy a relation's set before editing it so the original stays intact.

In [ ]:
original = studies[48]
changed = Valuation(list(original["valuation"].items()))
# Give Sound a third knower without changing the original valuation.
changed["Knows"] = set(changed["Knows"]) | {("ecog_d", "sound_d")}
before = satisfies(original["valuation"], original["sem"])
after = satisfies(changed, original["sem"])
print("Original verse:", before, "| With a third knower of Sound:", after)
assert before is True and after is False
